In [ ]:
import os
import shutil
import hashlib
from google.colab import drive

# ==========================================================
# 1) DRIVE MOUNT
# ==========================================================

MOUNT_POINT = "/content/drive"

if not os.path.ismount(MOUNT_POINT):
    drive.mount(MOUNT_POINT, force_remount=True)

print("Drive mount tamam mı?:", os.path.ismount(MOUNT_POINT))


# ==========================================================
# 2) PROJE YOLLARI
# ==========================================================

OLD_BASE = "/content/drive/MyDrive/tez_transformer"
NEW_BASE = "/content/drive/MyDrive/tez_transformer_v4_repro"

old_raw = os.path.join(
    OLD_BASE,
    "data",
    "raw",
    "raw_prices.csv"
)

new_raw_dir = os.path.join(
    NEW_BASE,
    "data",
    "raw"
)

new_raw = os.path.join(
    new_raw_dir,
    "raw_prices.csv"
)


# ==========================================================
# 3) YENİ PROJE KLASÖRLERİNİ OLUŞTUR
# ==========================================================

folders = [
    NEW_BASE,
    os.path.join(NEW_BASE, "config"),
    os.path.join(NEW_BASE, "scripts"),
    os.path.join(NEW_BASE, "data"),
    os.path.join(NEW_BASE, "data", "raw"),
    os.path.join(NEW_BASE, "data", "processed"),
    os.path.join(NEW_BASE, "data", "sequences"),
    os.path.join(NEW_BASE, "models"),
    os.path.join(NEW_BASE, "results"),
    os.path.join(NEW_BASE, "logs"),
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("\nYeni proje klasörleri oluşturuldu.")


# ==========================================================
# 4) SHA-256 FONKSİYONU
# ==========================================================

def sha256_file(path, chunk_size=1024 * 1024):
    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


# ==========================================================
# 5) FROZEN RAW VERİYİ KOPYALA
# ==========================================================

if not os.path.exists(old_raw):
    raise FileNotFoundError(
        f"Eski frozen raw_prices.csv bulunamadı:\n{old_raw}"
    )

if not os.path.exists(new_raw):
    shutil.copy2(old_raw, new_raw)
    print("\nFrozen raw_prices.csv yeni v4 projeye kopyalandı.")
else:
    print("\nYeni v4 projede raw_prices.csv zaten mevcut.")


# ==========================================================
# 6) HASH DOĞRULAMA
# ==========================================================

old_hash = sha256_file(old_raw)
new_hash = sha256_file(new_raw)

print("\nESKİ v3 RAW HASH:")
print(old_hash)

print("\nYENİ v4 RAW HASH:")
print(new_hash)

print("\nHash aynı mı?:", old_hash == new_hash)

if old_hash != new_hash:
    raise RuntimeError(
        "HASH EŞLEŞMİYOR. Frozen veri birebir aynı değil."
    )


# ==========================================================
# 7) HASH KAYDI OLUŞTUR
# ==========================================================

hash_record_path = os.path.join(
    NEW_BASE,
    "config",
    "frozen_raw_sha256_v4.txt"
)

with open(hash_record_path, "w", encoding="utf-8") as f:
    f.write("FROZEN RAW DATA SHA-256 — v4\n")
    f.write("=" * 60 + "\n")
    f.write(f"Source: {old_raw}\n")
    f.write(f"Copied to: {new_raw}\n")
    f.write(f"SHA-256: {new_hash}\n")

print("\nHash kaydı oluşturuldu:")
print(hash_record_path)


# ==========================================================
# 8) SON DURUM
# ==========================================================

print("\n" + "=" * 70)
print("v4 REPRO BAŞLANGICI TAMAMLANDI")
print("=" * 70)

print("Yeni proje:", NEW_BASE)
print("Frozen raw veri:", new_raw)
print("Hash doğrulandı:", old_hash == new_hash)

In [ ]:
# ==========================================================
# 00_setup_v4.py OLUŞTUR + ÇALIŞTIR
# ==========================================================

import os

NEW_BASE = "/content/drive/MyDrive/tez_transformer_v4_repro"

script_path = os.path.join(
    NEW_BASE,
    "scripts",
    "00_setup_v4.py"
)

script_code = r'''
# ==========================================================
# 00_setup_v4.py
# TEZ v4 — TEMİZ VE YENİDEN ÜRETİLEBİLİR PROJE KURULUMU
# ==========================================================

import os
import json
import hashlib
from datetime import datetime


# ==========================================================
# 1. ANA PROJE YOLU
# ==========================================================

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"


# ==========================================================
# 2. KLASÖR YAPISI
# ==========================================================

FOLDERS = [
    "config",
    "scripts",

    "data",
    "data/raw",
    "data/processed",

    "data/sequences",
    "data/sequences/baseline",
    "data/sequences/full",

    "models",

    "results",
    "results/baselines",
    "results/small_model_test",
    "results/grid_search",
    "results/multiseed",
    "results/final_test",
    "results/dm_tests",
    "results/shap",
    "results/robustness",

    "logs",
]


for rel_path in FOLDERS:
    full_path = os.path.join(BASE_DIR, rel_path)
    os.makedirs(full_path, exist_ok=True)


# ==========================================================
# 3. FROZEN RAW VERİ DOĞRULAMASI
# ==========================================================

RAW_PATH = os.path.join(
    BASE_DIR,
    "data",
    "raw",
    "raw_prices.csv"
)

EXPECTED_RAW_SHA256 = (
    "ab5f275d38dc98057b1cedcf58019adb26be7402c7ed5ae6ee3d6877b2444893"
)


def sha256_file(path, chunk_size=1024 * 1024):

    sha256 = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


if not os.path.exists(RAW_PATH):

    raise FileNotFoundError(
        f"Frozen raw veri bulunamadı:\n{RAW_PATH}"
    )


actual_raw_hash = sha256_file(RAW_PATH)


if actual_raw_hash != EXPECTED_RAW_SHA256:

    raise RuntimeError(
        "Frozen raw veri hash'i beklenen değerle eşleşmiyor.\n"
        f"Beklenen: {EXPECTED_RAW_SHA256}\n"
        f"Gerçek   : {actual_raw_hash}"
    )


# ==========================================================
# 4. RESMÎ v4 ŞEMASI
# ==========================================================

schema = {

    "project_version": "v4_repro",

    "project_title":
        "Çoklu Görevli Transformer ile Finansal Risk ve Getiri Tahmini",

    "created_at": datetime.now().isoformat(),

    "data": {

        "source":
            "Frozen raw_prices.csv inherited from audited v3 source",

        "internet_redownload":
            False,

        "raw_file":
            "data/raw/raw_prices.csv",

        "raw_sha256":
            actual_raw_hash,

        "assets": [
            "BIST100",
            "USDTRY",
            "EURTRY",
            "GOLD"
        ],

        "tickers": {
            "BIST100": "XU100.IS",
            "USDTRY": "USDTRY=X",
            "EURTRY": "EURTRY=X",
            "GOLD": "GC=F"
        },

        "original_period": {
            "start": "2010-01-01",
            "end": "2024-12-31"
        }
    },


    "features": {

        "baseline": [
            "LogRet",
            "Vol20"
        ],

        "full": [
            "LogRet",
            "Vol20",
            "MA5_Ratio",
            "MA20_Ratio",
            "RSI14",
            "MACD",
            "MACDSignal"
        ],

        "baseline_dim": 8,

        "full_dim": 28,

        "rsi14_zero_loss_rule":
            "If avg_loss == 0 and avg_gain > 0, RSI14 = 100"
    },


    "targets": {

        "definition": [
            "BIST100_NextRet",
            "USDTRY_NextRet",
            "EURTRY_NextRet",
            "GOLD_NextRet",
            "BIST100_NextVol",
            "USDTRY_NextVol",
            "EURTRY_NextVol",
            "GOLD_NextVol"
        ],

        "return_rule":
            "NextRet[t] = LogRet[t+1]",

        "volatility_rule":
            "NextVol[t] = Vol20[t+1]"
    },


    "split": {

        "type":
            "chronological_target_realization_aware",

        "ratios": {
            "train": 0.70,
            "validation": 0.15,
            "test": 0.15
        },

        "rule":
            "Input history may cross backward into the previous split, "
            "but target realization may never cross forward into the next split.",

        "random_split":
            False
    },


    "scaler": {

        "type":
            "StandardScaler",

        "fit_on":
            "train_only",

        "validation":
            "transform_only",

        "test":
            "transform_only"
    },


    "sequence": {

        "lookbacks": [
            10,
            20,
            30,
            60
        ],

        "overlap_aware":
            True,

        "principle":
            "Past input window may be carried; target may not be carried."
    },


    "models": [

        "FullSharingMTL",
        "PartialSharingMTL",
        "HierarchicalMTL",
        "NoSharing"
    ],


    "loss_strategies": [

        "FixedLambda_0.3",
        "FixedLambda_0.5",
        "FixedLambda_0.7",
        "UncertaintyWeighting",
        "PCGrad"
    ],


    "model_sizes": {

        "small": {
            "d_model": 32,
            "n_head": 4,
            "n_layers": 2,
            "d_ff": 128
        },

        "medium": {
            "d_model": 64,
            "n_head": 4,
            "n_layers": 2,
            "d_ff": 256
        },

        "large": {
            "d_model": 128,
            "n_head": 8,
            "n_layers": 4,
            "d_ff": 512
        }
    },


    "grid": {

        "architectures": 4,
        "loss_strategies": 5,
        "lookbacks": 4,
        "sizes": 3,
        "feature_sets": 2,

        "total_configs": 480,

        "official_grid_max_epochs": 50
    },


    "test_policy": {

        "used_for_model_selection":
            False,

        "first_model_evaluation_stage":
            "07_final_test_evaluation_v4.py",

        "post_test_model_changes_allowed":
            False
    },


    "scientific_principle":
        "Kararlar kilitli. Sonuçlar kilitli değil. Veri karar verir."
}


schema_path = os.path.join(
    BASE_DIR,
    "config",
    "schema_v4.json"
)


with open(
    schema_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        schema,
        f,
        ensure_ascii=False,
        indent=2
    )


# ==========================================================
# 5. README_v4.md
# ==========================================================

readme_path = os.path.join(
    BASE_DIR,
    "README_v4.md"
)


readme_text = f"""# TEZ TRANSFORMER v4 REPRO

## Amaç

Bu proje, dört finansal varlık için getiri ve volatilitenin
Transformer tabanlı çok görevli mimariler ile tahmin edilmesini amaçlar.

## Varlıklar

- BIST100
- USDTRY
- EURTRY
- GOLD

## Bilimsel İlke

**Kararlar kilitli. Sonuçlar kilitli değil. Veri karar verir.**

## Frozen Raw Data

Dosya:

`data/raw/raw_prices.csv`

SHA-256:

`{actual_raw_hash}`

Bu veri dosyası yeniden internetten indirilmemiştir.
Audit edilmiş v3 kaynağındaki frozen raw veri birebir korunmuştur.

## v4 Temel Düzeltmeleri

1. RSI14 için zero-loss handling açıkça tanımlanmıştır.
2. Split, target-realization-aware olarak uygulanacaktır.
3. Input history taşınabilir; target split sınırını geçemez.
4. StandardScaler yalnızca train setine fit edilir.
5. Test model seçiminde kullanılmaz.

## Resmî Pipeline

- 00_setup_v4.py
- 01_rebuild_from_frozen_raw_v4.py
- 02_preprocessing_v4.py
- 03_baseline_sanity_v4.py
- 04_small_model_test_v4.py
- 05_grid_search_v4.py
- 06_best_model_multiseed_v4.py
- 07_final_test_evaluation_v4.py
- 08_baseline_full_v4.py
- 09_diebold_mariano_v4.py

"""


with open(
    readme_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(readme_text)


# ==========================================================
# 6. SONUÇ
# ==========================================================

print("=" * 70)
print("00_setup_v4.py TAMAMLANDI")
print("=" * 70)

print("\nProje klasörü:")
print(BASE_DIR)

print("\nFrozen raw veri:")
print(RAW_PATH)

print("\nFrozen raw SHA-256:")
print(actual_raw_hash)

print("\nHash doğrulandı:")
print(actual_raw_hash == EXPECTED_RAW_SHA256)

print("\nSchema:")
print(schema_path)

print("\nREADME:")
print(readme_path)

print("\nKlasör sayısı:")
print(len(FOLDERS))

print("\n✅ v4 proje iskeleti hazır.")
'''


# ==========================================================
# SCRIPT'İ DRIVE'A KAYDET
# ==========================================================

with open(
    script_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(script_code)


print("00_setup_v4.py oluşturuldu:")
print(script_path)


# ==========================================================
# SCRIPT'İ ÇALIŞTIR
# ==========================================================

print("\nScript çalıştırılıyor...\n")

exec(
    compile(
        open(
            script_path,
            "r",
            encoding="utf-8"
        ).read(),
        script_path,
        "exec"
    )
)

In [ ]:
# ==========================================================
# v4 — FROZEN RAW VERİ YAPISINI OKU
# SADECE OKUR, HİÇBİR DOSYAYI DEĞİŞTİRMEZ
# ==========================================================

import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

RAW_PATH = os.path.join(
    BASE_DIR,
    "data",
    "raw",
    "raw_prices.csv"
)

# Veriyi oku
raw = pd.read_csv(
    RAW_PATH,
    index_col=0,
    parse_dates=True
)

print("=" * 70)
print("v4 FROZEN RAW VERİ KONTROLÜ")
print("=" * 70)

print("\nDosya:")
print(RAW_PATH)

print("\nShape:")
print(raw.shape)

print("\nKolonlar:")
print(raw.columns.tolist())

print("\nİlk tarih:")
print(raw.index.min())

print("\nSon tarih:")
print(raw.index.max())

print("\nIndex kronolojik artıyor mu?:")
print(raw.index.is_monotonic_increasing)

print("\nDuplicate tarih sayısı:")
print(raw.index.duplicated().sum())

print("\nNaN sayıları:")
print(raw.isna().sum())

print("\nİlk 5 satır:")
print(raw.head())

print("\nSon 5 satır:")
print(raw.tail())

print("\n" + "=" * 70)
print("KONTROL TAMAMLANDI")
print("=" * 70)
print("Bu hücre hiçbir dosyayı değiştirmedi.")

In [ ]:
# ==========================================================
# 01_rebuild_from_frozen_raw_v4.py OLUŞTUR + ÇALIŞTIR
# ==========================================================

import os

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

script_path = os.path.join(
    BASE_DIR,
    "scripts",
    "01_rebuild_from_frozen_raw_v4.py"
)

script_code = r'''
# ==========================================================
# 01_rebuild_from_frozen_raw_v4.py
#
# AMAÇ:
# - İnternete bağlanmadan frozen raw_prices.csv dosyasını kullanmak
# - Fiyatları leakage-free biçimde temizlemek
# - Baseline ve full feature setlerini üretmek
# - RSI14 zero-loss durumunu doğru ele almak
# - NextRet ve NextVol hedeflerini üretmek
# - Anchor date ile target realization date'i ayrı kaydetmek
# ==========================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd
from datetime import datetime


# ==========================================================
# 1. PROJE YOLLARI
# ==========================================================

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

RAW_PATH = os.path.join(
    BASE_DIR,
    "data",
    "raw",
    "raw_prices.csv"
)

PROCESSED_DIR = os.path.join(
    BASE_DIR,
    "data",
    "processed"
)

CONFIG_DIR = os.path.join(
    BASE_DIR,
    "config"
)

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(CONFIG_DIR, exist_ok=True)


EXPECTED_RAW_SHA256 = (
    "ab5f275d38dc98057b1cedcf58019adb26be7402c7ed5ae6ee3d6877b2444893"
)


ASSETS = [
    "BIST100",
    "USDTRY",
    "EURTRY",
    "GOLD"
]


BASELINE_FEATURE_NAMES = [
    "LogRet",
    "Vol20"
]


FULL_FEATURE_NAMES = [
    "LogRet",
    "Vol20",
    "MA5_Ratio",
    "MA20_Ratio",
    "RSI14",
    "MACD",
    "MACDSignal"
]


TARGET_ORDER = [
    "BIST100_NextRet",
    "USDTRY_NextRet",
    "EURTRY_NextRet",
    "GOLD_NextRet",
    "BIST100_NextVol",
    "USDTRY_NextVol",
    "EURTRY_NextVol",
    "GOLD_NextVol"
]


# ==========================================================
# 2. SHA-256 FONKSİYONU
# ==========================================================

def sha256_file(path, chunk_size=1024 * 1024):

    sha256 = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


# ==========================================================
# 3. FROZEN RAW VERİYİ OKU VE DOĞRULA
# ==========================================================

if not os.path.exists(RAW_PATH):

    raise FileNotFoundError(
        f"Frozen raw_prices.csv bulunamadı:\n{RAW_PATH}"
    )


raw_hash = sha256_file(RAW_PATH)


if raw_hash != EXPECTED_RAW_SHA256:

    raise RuntimeError(
        "Frozen raw data SHA-256 uyuşmuyor.\n"
        f"Beklenen: {EXPECTED_RAW_SHA256}\n"
        f"Gerçek   : {raw_hash}"
    )


raw = pd.read_csv(
    RAW_PATH,
    index_col=0,
    parse_dates=True
)


# ==========================================================
# 4. RAW YAPI KONTROLLERİ
# ==========================================================

if list(raw.columns) != ASSETS:

    raise ValueError(
        "Raw kolon sırası beklenen yapıyla uyuşmuyor.\n"
        f"Beklenen: {ASSETS}\n"
        f"Gerçek   : {raw.columns.tolist()}"
    )


if not raw.index.is_monotonic_increasing:

    raise ValueError(
        "Raw tarih index'i kronolojik artan değil."
    )


duplicate_count = int(
    raw.index.duplicated().sum()
)


if duplicate_count != 0:

    raise ValueError(
        f"Duplicate tarih bulundu: {duplicate_count}"
    )


print("=" * 80)
print("01 — FROZEN RAW VERİ DOĞRULANDI")
print("=" * 80)

print("\nRaw shape:")
print(raw.shape)

print("\nTarih aralığı:")
print(raw.index.min(), "→", raw.index.max())

print("\nRaw NaN sayıları:")
print(raw.isna().sum())

print("\nSHA-256:")
print(raw_hash)


# ==========================================================
# 5. FİYAT TEMİZLEME
#
# Union calendar korunur.
# Sadece geçmişte bilinen son değer ileri taşınır.
# bfill YOK.
# ==========================================================

prices_clean = raw.ffill()


remaining_nan = prices_clean.isna().sum()


print("\n" + "=" * 80)
print("FFILL SONRASI NaN KONTROLÜ")
print("=" * 80)

print(remaining_nan)


if int(remaining_nan.sum()) != 0:

    raise ValueError(
        "ffill sonrası NaN kaldı. "
        "Leading missing durumları ayrıca incelenmeli; "
        "otomatik bfill yapılmayacak."
    )


# ==========================================================
# 6. RSI14 FONKSİYONU — DÜZELTİLMİŞ
#
# Kural:
# avg_loss == 0 ve avg_gain > 0  → RSI = 100
# avg_gain == 0 ve avg_loss > 0  → RSI = 0
# avg_gain == 0 ve avg_loss == 0 → RSI = 50
# diğer durum                     → standart RSI formülü
# ==========================================================

def compute_rsi14(close: pd.Series):

    delta = close.diff()

    gain = delta.clip(lower=0)

    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(
        window=14,
        min_periods=14
    ).mean()

    avg_loss = loss.rolling(
        window=14,
        min_periods=14
    ).mean()

    rsi = pd.Series(
        np.nan,
        index=close.index,
        dtype=float
    )

    normal_mask = (
        (avg_gain > 0) &
        (avg_loss > 0)
    )

    rs = (
        avg_gain[normal_mask] /
        avg_loss[normal_mask]
    )

    rsi.loc[normal_mask] = (
        100.0 -
        (100.0 / (1.0 + rs))
    )

    only_gain_mask = (
        (avg_gain > 0) &
        (avg_loss == 0)
    )

    rsi.loc[only_gain_mask] = 100.0

    only_loss_mask = (
        (avg_gain == 0) &
        (avg_loss > 0)
    )

    rsi.loc[only_loss_mask] = 0.0

    flat_mask = (
        (avg_gain == 0) &
        (avg_loss == 0)
    )

    rsi.loc[flat_mask] = 50.0

    return rsi


# ==========================================================
# 7. FEATURE VE TARGET ÜRETİMİ
# ==========================================================

baseline_parts = []
full_parts = []

next_ret_parts = []
next_vol_parts = []

rsi_audit_records = []


for asset in ASSETS:

    close = prices_clean[asset].astype(float)


    # ------------------------------------------------------
    # Log Return
    # ------------------------------------------------------

    logret = np.log(
        close / close.shift(1)
    )


    # ------------------------------------------------------
    # 20 günlük annualized historical volatility
    # ------------------------------------------------------

    vol20 = (
        logret
        .rolling(
            window=20,
            min_periods=20
        )
        .std()
        * np.sqrt(252)
    )


    # ------------------------------------------------------
    # Moving Average Ratios
    # ------------------------------------------------------

    ma5 = close.rolling(
        window=5,
        min_periods=5
    ).mean()

    ma20 = close.rolling(
        window=20,
        min_periods=20
    ).mean()

    ma5_ratio = (
        close / ma5
    ) - 1.0

    ma20_ratio = (
        close / ma20
    ) - 1.0


    # ------------------------------------------------------
    # RSI14 — corrected zero-loss handling
    # ------------------------------------------------------

    delta = close.diff()

    gain = delta.clip(lower=0)

    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(
        window=14,
        min_periods=14
    ).mean()

    avg_loss = loss.rolling(
        window=14,
        min_periods=14
    ).mean()

    rsi14 = compute_rsi14(close)

    zero_loss_positive_gain_count = int(
        (
            (avg_loss == 0) &
            (avg_gain > 0)
        ).sum()
    )

    zero_gain_positive_loss_count = int(
        (
            (avg_gain == 0) &
            (avg_loss > 0)
        ).sum()
    )

    flat_count = int(
        (
            (avg_gain == 0) &
            (avg_loss == 0)
        ).sum()
    )

    rsi_audit_records.append(
        {
            "asset":
                asset,

            "zero_loss_positive_gain_count":
                zero_loss_positive_gain_count,

            "zero_gain_positive_loss_count":
                zero_gain_positive_loss_count,

            "flat_count":
                flat_count
        }
    )


    # ------------------------------------------------------
    # MACD
    # ------------------------------------------------------

    ema12 = close.ewm(
        span=12,
        adjust=False
    ).mean()

    ema26 = close.ewm(
        span=26,
        adjust=False
    ).mean()

    macd = ema12 - ema26

    macd_signal = macd.ewm(
        span=9,
        adjust=False
    ).mean()


    # ------------------------------------------------------
    # Baseline feature set
    # ------------------------------------------------------

    baseline_asset = pd.DataFrame(
        {
            f"{asset}_LogRet":
                logret,

            f"{asset}_Vol20":
                vol20,
        },
        index=prices_clean.index
    )

    baseline_parts.append(
        baseline_asset
    )


    # ------------------------------------------------------
    # Full feature set
    # ------------------------------------------------------

    full_asset = pd.DataFrame(
        {
            f"{asset}_LogRet":
                logret,

            f"{asset}_Vol20":
                vol20,

            f"{asset}_MA5_Ratio":
                ma5_ratio,

            f"{asset}_MA20_Ratio":
                ma20_ratio,

            f"{asset}_RSI14":
                rsi14,

            f"{asset}_MACD":
                macd,

            f"{asset}_MACDSignal":
                macd_signal,
        },
        index=prices_clean.index
    )

    full_parts.append(
        full_asset
    )


    # ------------------------------------------------------
    # Targets
    #
    # Anchor date = t
    # NextRet[t]  = LogRet[t+1]
    # NextVol[t]  = Vol20[t+1]
    # ------------------------------------------------------

    next_ret_parts.append(
        logret.shift(-1).rename(
            f"{asset}_NextRet"
        )
    )

    next_vol_parts.append(
        vol20.shift(-1).rename(
            f"{asset}_NextVol"
        )
    )


# ==========================================================
# 8. BİRLEŞTİR
# ==========================================================

features_baseline_raw = pd.concat(
    baseline_parts,
    axis=1
)

features_full_raw = pd.concat(
    full_parts,
    axis=1
)


# Target order bilinçli olarak:
# önce 4 return, sonra 4 volatility

targets_raw = pd.concat(
    next_ret_parts + next_vol_parts,
    axis=1
)


if list(targets_raw.columns) != TARGET_ORDER:

    raise RuntimeError(
        "Target sırası kilitli sırayla uyuşmuyor.\n"
        f"Beklenen: {TARGET_ORDER}\n"
        f"Gerçek   : {targets_raw.columns.tolist()}"
    )


# ==========================================================
# 9. TARGET REALIZATION DATE ÜRET
#
# Her anchor date t için target, bir sonraki union-calendar
# tarihinde gerçekleşir.
# ==========================================================

target_realization_dates = pd.Series(
    data=prices_clean.index.to_series().shift(-1).values,
    index=prices_clean.index,
    name="target_realization_date"
)


# ==========================================================
# 10. ORTAK GEÇERLİ INDEX
#
# Baseline ve full feature setleri aynı örneklemde
# karşılaştırılsın diye ortak index kullanılır.
# ==========================================================

combined_for_index = pd.concat(
    [
        features_baseline_raw,
        features_full_raw,
        targets_raw,
        target_realization_dates
    ],
    axis=1
)


valid_index = (
    combined_for_index
    .dropna()
    .index
)


features_baseline = (
    features_baseline_raw
    .loc[valid_index]
    .copy()
)

features_full = (
    features_full_raw
    .loc[valid_index]
    .copy()
)

targets_all = (
    targets_raw
    .loc[valid_index]
    .copy()
)

target_dates = (
    target_realization_dates
    .loc[valid_index]
    .to_frame()
)


# ==========================================================
# 11. TEMEL BÜTÜNLÜK KONTROLLERİ
# ==========================================================

if not (
    features_baseline.index.equals(
        features_full.index
    )
    and
    features_full.index.equals(
        targets_all.index
    )
    and
    targets_all.index.equals(
        target_dates.index
    )
):

    raise RuntimeError(
        "Feature/target/target-date index hizası bozuk."
    )


if features_baseline.shape[1] != 8:

    raise RuntimeError(
        f"Baseline feature dim 8 değil: "
        f"{features_baseline.shape[1]}"
    )


if features_full.shape[1] != 28:

    raise RuntimeError(
        f"Full feature dim 28 değil: "
        f"{features_full.shape[1]}"
    )


if targets_all.shape[1] != 8:

    raise RuntimeError(
        f"Target dim 8 değil: "
        f"{targets_all.shape[1]}"
    )


if not (
    pd.to_datetime(
        target_dates[
            "target_realization_date"
        ]
    )
    >
    target_dates.index
).all():

    raise RuntimeError(
        "Bazı target realization date değerleri "
        "anchor date'ten ileri değil."
    )


# ==========================================================
# 12. ÇIKTI DOSYALARI
# ==========================================================

prices_clean_path = os.path.join(
    PROCESSED_DIR,
    "prices_clean.csv"
)

baseline_path = os.path.join(
    PROCESSED_DIR,
    "features_baseline.csv"
)

full_path = os.path.join(
    PROCESSED_DIR,
    "features_full.csv"
)

targets_path = os.path.join(
    PROCESSED_DIR,
    "targets_all.csv"
)

target_dates_path = os.path.join(
    PROCESSED_DIR,
    "target_realization_dates.csv"
)

rsi_audit_path = os.path.join(
    PROCESSED_DIR,
    "rsi14_audit_v4.csv"
)


prices_clean.to_csv(
    prices_clean_path
)

features_baseline.to_csv(
    baseline_path
)

features_full.to_csv(
    full_path
)

targets_all.to_csv(
    targets_path
)

target_dates.to_csv(
    target_dates_path
)

pd.DataFrame(
    rsi_audit_records
).to_csv(
    rsi_audit_path,
    index=False
)


# ==========================================================
# 13. HASH KAYITLARI
# ==========================================================

derived_files = [
    prices_clean_path,
    baseline_path,
    full_path,
    targets_path,
    target_dates_path,
    rsi_audit_path
]


hash_records = []


for path in derived_files:

    hash_records.append(
        {
            "file":
                os.path.relpath(
                    path,
                    BASE_DIR
                ),

            "sha256":
                sha256_file(path)
        }
    )


hash_df = pd.DataFrame(
    hash_records
)


hash_output_path = os.path.join(
    CONFIG_DIR,
    "derived_data_sha256_v4.csv"
)


hash_df.to_csv(
    hash_output_path,
    index=False
)


# ==========================================================
# 14. META DOSYASI
# ==========================================================

meta = {

    "project_version":
        "v4_repro",

    "created_at":
        datetime.now().isoformat(),

    "raw_data": {

        "file":
            "data/raw/raw_prices.csv",

        "sha256":
            raw_hash,

        "shape":
            list(raw.shape),

        "date_start":
            str(raw.index.min().date()),

        "date_end":
            str(raw.index.max().date())
    },

    "clean_prices": {

        "shape":
            list(prices_clean.shape),

        "date_start":
            str(prices_clean.index.min().date()),

        "date_end":
            str(prices_clean.index.max().date()),

        "fill_method":
            "ffill_only",

        "bfill_used":
            False
    },

    "features": {

        "baseline_shape":
            list(features_baseline.shape),

        "full_shape":
            list(features_full.shape),

        "common_index":
            True,

        "rsi14_rule":
            {
                "avg_loss_0_avg_gain_positive":
                    100.0,

                "avg_gain_0_avg_loss_positive":
                    0.0,

                "avg_gain_0_avg_loss_0":
                    50.0
            }
    },

    "targets": {

        "shape":
            list(targets_all.shape),

        "order":
            TARGET_ORDER,

        "return_rule":
            "NextRet[t] = LogRet[t+1]",

        "volatility_rule":
            "NextVol[t] = Vol20[t+1]",

        "target_realization_dates_saved":
            True
    },

    "final_common_data": {

        "rows":
            len(valid_index),

        "anchor_date_start":
            str(valid_index.min().date()),

        "anchor_date_end":
            str(valid_index.max().date()),

        "first_target_realization_date":
            str(
                pd.to_datetime(
                    target_dates.iloc[0, 0]
                ).date()
            ),

        "last_target_realization_date":
            str(
                pd.to_datetime(
                    target_dates.iloc[-1, 0]
                ).date()
            )
    },

    "scientific_principle":
        "Kararlar kilitli. Sonuçlar kilitli değil. Veri karar verir."
}


meta_path = os.path.join(
    PROCESSED_DIR,
    "meta_v4.json"
)


with open(
    meta_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        meta,
        f,
        ensure_ascii=False,
        indent=2
    )


# ==========================================================
# 15. SONUÇLARI YAZDIR
# ==========================================================

print("\n" + "=" * 80)
print("01_rebuild_from_frozen_raw_v4.py TAMAMLANDI")
print("=" * 80)

print("\nRAW:")
print("shape =", raw.shape)
print(
    "date  =",
    raw.index.min().date(),
    "→",
    raw.index.max().date()
)

print("\nCLEAN PRICES:")
print("shape =", prices_clean.shape)
print(
    "NaN   =",
    int(prices_clean.isna().sum().sum())
)

print("\nFEATURES:")
print(
    "baseline =",
    features_baseline.shape
)

print(
    "full     =",
    features_full.shape
)

print("\nTARGETS:")
print(
    "targets  =",
    targets_all.shape
)

print("\nTARGET ORDER:")
for i, col in enumerate(
    targets_all.columns
):
    print(
        f"[{i}] {col}"
    )

print("\nFINAL COMMON INDEX:")
print(
    len(valid_index),
    "satır"
)

print(
    "Anchor:",
    valid_index.min().date(),
    "→",
    valid_index.max().date()
)

print(
    "Target realization:",
    pd.to_datetime(
        target_dates.iloc[0, 0]
    ).date(),
    "→",
    pd.to_datetime(
        target_dates.iloc[-1, 0]
    ).date()
)

print("\nRSI14 AUDIT:")
print(
    pd.DataFrame(
        rsi_audit_records
    ).to_string(
        index=False
    )
)

print("\nDOSYALAR:")
print(" -", prices_clean_path)
print(" -", baseline_path)
print(" -", full_path)
print(" -", targets_path)
print(" -", target_dates_path)
print(" -", rsi_audit_path)
print(" -", hash_output_path)
print(" -", meta_path)

print("\n✅ Frozen raw veri değiştirilmedi.")
print("✅ İnternet/yfinance kullanılmadı.")
print("✅ bfill kullanılmadı.")
print("✅ RSI14 zero-loss handling düzeltildi.")
print("✅ Target realization date ayrı kaydedildi.")
'''


# ==========================================================
# SCRIPT'İ KAYDET
# ==========================================================

with open(
    script_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(script_code)


print("01_rebuild_from_frozen_raw_v4.py oluşturuldu:")
print(script_path)


# ==========================================================
# SCRIPT'İ ÇALIŞTIR
# ==========================================================

print("\nScript çalıştırılıyor...\n")

exec(
    compile(
        open(
            script_path,
            "r",
            encoding="utf-8"
        ).read(),
        script_path,
        "exec"
    )
)

In [ ]:
# ==========================================================
# 02_preprocessing_v4.py — SYNTAX HATASINI DÜZELT + ÇALIŞTIR
# ==========================================================

import os
import re

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

script_path = os.path.join(
    BASE_DIR,
    "scripts",
    "02_preprocessing_v4.py"
)


# ==========================================================
# 1. SCRIPT'İ OKU
# ==========================================================

with open(
    script_path,
    "r",
    encoding="utf-8"
) as f:
    code = f.read()


# ==========================================================
# 2. fit_transform_scalers FONKSİYONUNU
#    TAMAMEN DOĞRU SÜRÜMLE DEĞİŞTİR
# ==========================================================

correct_function = r'''
def fit_transform_scalers(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test
):

    """
    StandardScaler yalnızca train setine fit edilir.

    Validation ve test:
        transform only.
    """

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()


    # ------------------------------------------------------
    # FIT: SADECE TRAIN
    # ------------------------------------------------------

    X_train_scaled = x_scaler.fit_transform(
        X_train
    )

    y_train_scaled = y_scaler.fit_transform(
        y_train
    )


    # ------------------------------------------------------
    # TRANSFORM: VALIDATION VE TEST
    # ------------------------------------------------------

    X_val_scaled = x_scaler.transform(
        X_val
    )

    y_val_scaled = y_scaler.transform(
        y_val
    )

    X_test_scaled = x_scaler.transform(
        X_test
    )

    y_test_scaled = y_scaler.transform(
        y_test
    )


    # ------------------------------------------------------
    # KRİTİK LEAKAGE KONTROLÜ
    #
    # Scaler'ın gördüğü örnek sayısı yalnızca
    # train satır sayısına eşit olmalı.
    # ------------------------------------------------------

    x_seen_raw = np.atleast_1d(
        x_scaler.n_samples_seen_
    )

    y_seen_raw = np.atleast_1d(
        y_scaler.n_samples_seen_
    )

    x_seen = int(
        x_seen_raw[0]
    )

    y_seen = int(
        y_seen_raw[0]
    )


    if x_seen != len(X_train):

        raise RuntimeError(
            "X scaler train dışında örnek görmüş olabilir."
        )


    if y_seen != len(y_train):

        raise RuntimeError(
            "Y scaler train dışında örnek görmüş olabilir."
        )


    return (
        X_train_scaled,
        y_train_scaled,

        X_val_scaled,
        y_val_scaled,

        X_test_scaled,
        y_test_scaled,

        x_scaler,
        y_scaler
    )


'''


pattern = (
    r"def fit_transform_scalers\("
    r".*?"
    r"(?=def make_train_sequences\()"
)


new_code, replacement_count = re.subn(
    pattern,
    correct_function,
    code,
    count=1,
    flags=re.DOTALL
)


if replacement_count != 1:
    raise RuntimeError(
        "fit_transform_scalers fonksiyonu beklenen şekilde bulunamadı. "
        f"Replacement count = {replacement_count}"
    )


# ==========================================================
# 3. DÜZELTİLMİŞ SCRIPT'İ KAYDET
# ==========================================================

with open(
    script_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(new_code)


print("=" * 80)
print("02_preprocessing_v4.py DÜZELTİLDİ")
print("=" * 80)

print("\nDüzeltilen fonksiyon:")
print("fit_transform_scalers")

print("\nDeğiştirilen fonksiyon sayısı:")
print(replacement_count)


# ==========================================================
# 4. ÖNCE SADECE SYNTAX KONTROLÜ
# ==========================================================

with open(
    script_path,
    "r",
    encoding="utf-8"
) as f:
    corrected_code = f.read()


compile(
    corrected_code,
    script_path,
    "exec"
)


print("\n✅ Syntax kontrolü geçti.")


# ==========================================================
# 5. SCRIPT'İ ÇALIŞTIR
# ==========================================================

print("\n" + "=" * 80)
print("02_preprocessing_v4.py ÇALIŞTIRILIYOR")
print("=" * 80)
print()


exec(
    compile(
        corrected_code,
        script_path,
        "exec"
    )
)

In [ ]:
# ==========================================================
# 03_baseline_sanity_v4.py OLUŞTUR + ÇALIŞTIR
# ==========================================================

import os

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

script_path = os.path.join(
    BASE_DIR,
    "scripts",
    "03_baseline_sanity_v4.py"
)

script_code = r'''
# ==========================================================
# 03_baseline_sanity_v4.py
#
# AMAÇ:
# - Validation setinde naive finansal baseline'ları hesaplamak
# - ReturnZero
# - ReturnPersistence
# - VolPersistence
# - ValidationScore için denominator'ları üretmek
# - Sequence yapısını sanity-check etmek
#
# ÖNEMLİ:
# - Test performansı HESAPLANMAZ.
# - Test seti model seçimi için KULLANILMAZ.
# ==========================================================

import os
import json
import pickle
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ==========================================================
# 1. YOLLAR
# ==========================================================

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

PROCESSED_DIR = os.path.join(
    BASE_DIR,
    "data",
    "processed"
)

SEQUENCE_DIR = os.path.join(
    BASE_DIR,
    "data",
    "sequences"
)

RESULTS_DIR = os.path.join(
    BASE_DIR,
    "results",
    "baselines"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)


FEATURES_BASELINE_PATH = os.path.join(
    PROCESSED_DIR,
    "features_baseline.csv"
)

TARGETS_PATH = os.path.join(
    PROCESSED_DIR,
    "targets_all.csv"
)

TARGET_DATES_PATH = os.path.join(
    PROCESSED_DIR,
    "target_realization_dates.csv"
)

SPLIT_META_PATH = os.path.join(
    PROCESSED_DIR,
    "split_meta_v4.json"
)


required_files = [
    FEATURES_BASELINE_PATH,
    TARGETS_PATH,
    TARGET_DATES_PATH,
    SPLIT_META_PATH
]


for path in required_files:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Gerekli dosya bulunamadı:\n{path}"
        )


# ==========================================================
# 2. KİLİTLİ SABİTLER
# ==========================================================

ASSETS = [
    "BIST100",
    "USDTRY",
    "EURTRY",
    "GOLD"
]


RETURN_TARGETS = [
    "BIST100_NextRet",
    "USDTRY_NextRet",
    "EURTRY_NextRet",
    "GOLD_NextRet"
]


VOL_TARGETS = [
    "BIST100_NextVol",
    "USDTRY_NextVol",
    "EURTRY_NextVol",
    "GOLD_NextVol"
]


LOOKBACKS = [
    10,
    20,
    30,
    60
]


TAU = 0.5


# ==========================================================
# 3. METRİK FONKSİYONLARI
# ==========================================================

def rmse(
    y_true,
    y_pred
):

    return float(
        np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        )
    )


def pinball_loss(
    y_true,
    y_pred,
    tau=0.5
):

    error = (
        y_true - y_pred
    )

    loss = np.maximum(
        tau * error,
        (tau - 1.0) * error
    )

    return float(
        np.mean(loss)
    )


def safe_r2(
    y_true,
    y_pred
):

    try:

        return float(
            r2_score(
                y_true,
                y_pred
            )
        )

    except Exception:

        return float("nan")


# ==========================================================
# 4. VERİLERİ OKU
# ==========================================================

features_baseline = pd.read_csv(
    FEATURES_BASELINE_PATH,
    index_col=0,
    parse_dates=True
)


targets_all = pd.read_csv(
    TARGETS_PATH,
    index_col=0,
    parse_dates=True
)


target_dates = pd.read_csv(
    TARGET_DATES_PATH,
    index_col=0,
    parse_dates=True
)


target_dates[
    "target_realization_date"
] = pd.to_datetime(
    target_dates[
        "target_realization_date"
    ]
)


with open(
    SPLIT_META_PATH,
    "r",
    encoding="utf-8"
) as f:

    split_meta = json.load(f)


print("=" * 80)
print("03 — VALIDATION BASELINE SANITY")
print("=" * 80)

print(
    "\nBaseline feature shape:",
    features_baseline.shape
)

print(
    "Target shape:",
    targets_all.shape
)


# ==========================================================
# 5. SPLIT SINIRLARINI META'DAN OKU
# ==========================================================

train_start = int(
    split_meta[
        "train"
    ][
        "start_idx"
    ]
)

train_end = int(
    split_meta[
        "train"
    ][
        "end_idx_exclusive"
    ]
)


val_start = int(
    split_meta[
        "validation"
    ][
        "start_idx"
    ]
)

val_end = int(
    split_meta[
        "validation"
    ][
        "end_idx_exclusive"
    ]
)


test_start = int(
    split_meta[
        "test"
    ][
        "start_idx"
    ]
)

test_end = int(
    split_meta[
        "test"
    ][
        "end_idx_exclusive"
    ]
)


if not (
    train_end == val_start
    and
    val_end == test_start
    and
    test_end == len(
        targets_all
    )
):

    raise RuntimeError(
        "Split index sınırlarında tutarsızlık var."
    )


# ==========================================================
# 6. SADECE VALIDATION VERİSİNİ AYIR
# ==========================================================

X_val = (
    features_baseline
    .iloc[
        val_start:val_end
    ]
    .copy()
)


y_val = (
    targets_all
    .iloc[
        val_start:val_end
    ]
    .copy()
)


dates_val = (
    target_dates
    .iloc[
        val_start:val_end
    ]
    .copy()
)


if not (
    X_val.index.equals(
        y_val.index
    )
    and
    y_val.index.equals(
        dates_val.index
    )
):

    raise RuntimeError(
        "Validation feature/target/date index hizası bozuk."
    )


print("\nVALIDATION")

print(
    "Örnek sayısı:",
    len(
        y_val
    )
)

print(
    "Anchor:",
    y_val.index.min().date(),
    "→",
    y_val.index.max().date()
)

print(
    "Target realization:",
    dates_val[
        "target_realization_date"
    ].min().date(),
    "→",
    dates_val[
        "target_realization_date"
    ].max().date()
)


# ==========================================================
# 7. NAIVE BASELINE TAHMİNLERİ
#
# ReturnZero:
#   Next return tahmini = 0
#
# ReturnPersistence:
#   Next return tahmini = mevcut LogRet[t]
#
# VolPersistence:
#   Next volatility tahmini = mevcut Vol20[t]
# ==========================================================

results = []


selection_denominators = {

    "project_version":
        "v4_repro",

    "created_at":
        datetime.now().isoformat(),

    "split":
        "validation_only",

    "validation_rows":
        int(
            len(
                y_val
            )
        ),

    "validation_anchor_start":
        str(
            y_val.index.min().date()
        ),

    "validation_anchor_end":
        str(
            y_val.index.max().date()
        ),

    "validation_target_realization_start":
        str(
            dates_val[
                "target_realization_date"
            ].min().date()
        ),

    "validation_target_realization_end":
        str(
            dates_val[
                "target_realization_date"
            ].max().date()
        ),

    "return_denominator":
        {},

    "volatility_denominator":
        {},

    "selection_score_rule":
        (
            "ValidationScore = "
            "0.5 * AvgReturnRatio + "
            "0.5 * AvgVolRatio"
        ),

    "return_ratio_rule":
        (
            "Model return MAE / "
            "ReturnZero validation MAE"
        ),

    "volatility_ratio_rule":
        (
            "Model volatility PinballLoss(tau=0.5) / "
            "VolPersistence validation PinballLoss(tau=0.5)"
        ),

    "test_used":
        False
}


for i, asset in enumerate(
    ASSETS
):

    # ------------------------------------------------------
    # GERÇEK TARGET'LAR
    # ------------------------------------------------------

    y_true_ret = (
        y_val[
            RETURN_TARGETS[i]
        ]
        .values
        .astype(float)
    )


    y_true_vol = (
        y_val[
            VOL_TARGETS[i]
        ]
        .values
        .astype(float)
    )


    # ------------------------------------------------------
    # CURRENT FEATURES
    # ------------------------------------------------------

    current_ret = (
        X_val[
            f"{asset}_LogRet"
        ]
        .values
        .astype(float)
    )


    current_vol = (
        X_val[
            f"{asset}_Vol20"
        ]
        .values
        .astype(float)
    )


    # ------------------------------------------------------
    # RETURN ZERO
    # ------------------------------------------------------

    pred_return_zero = np.zeros_like(
        y_true_ret
    )


    rz_mae = float(
        mean_absolute_error(
            y_true_ret,
            pred_return_zero
        )
    )


    rz_rmse = rmse(
        y_true_ret,
        pred_return_zero
    )


    rz_r2 = safe_r2(
        y_true_ret,
        pred_return_zero
    )


    results.append(
        {
            "asset":
                asset,

            "task":
                "return",

            "baseline":
                "ReturnZero",

            "MAE":
                rz_mae,

            "RMSE":
                rz_rmse,

            "R2":
                rz_r2,

            "PinballLoss_tau_0.5":
                np.nan
        }
    )


    # ------------------------------------------------------
    # RETURN PERSISTENCE
    # ------------------------------------------------------

    rp_mae = float(
        mean_absolute_error(
            y_true_ret,
            current_ret
        )
    )


    rp_rmse = rmse(
        y_true_ret,
        current_ret
    )


    rp_r2 = safe_r2(
        y_true_ret,
        current_ret
    )


    results.append(
        {
            "asset":
                asset,

            "task":
                "return",

            "baseline":
                "ReturnPersistence",

            "MAE":
                rp_mae,

            "RMSE":
                rp_rmse,

            "R2":
                rp_r2,

            "PinballLoss_tau_0.5":
                np.nan
        }
    )


    # ------------------------------------------------------
    # VOL PERSISTENCE
    # ------------------------------------------------------

    vp_mae = float(
        mean_absolute_error(
            y_true_vol,
            current_vol
        )
    )


    vp_rmse = rmse(
        y_true_vol,
        current_vol
    )


    vp_r2 = safe_r2(
        y_true_vol,
        current_vol
    )


    vp_pinball = pinball_loss(
        y_true_vol,
        current_vol,
        tau=TAU
    )


    results.append(
        {
            "asset":
                asset,

            "task":
                "volatility",

            "baseline":
                "VolPersistence",

            "MAE":
                vp_mae,

            "RMSE":
                vp_rmse,

            "R2":
                vp_r2,

            "PinballLoss_tau_0.5":
                vp_pinball
        }
    )


    # ------------------------------------------------------
    # SELECTION DENOMINATOR'LARI
    # ------------------------------------------------------

    selection_denominators[
        "return_denominator"
    ][asset] = {

        "baseline":
            "ReturnZero",

        "metric":
            "MAE",

        "value":
            rz_mae
    }


    selection_denominators[
        "volatility_denominator"
    ][asset] = {

        "baseline":
            "VolPersistence",

        "metric":
            "PinballLoss_tau_0.5",

        "tau":
            TAU,

        "value":
            vp_pinball
    }


# ==========================================================
# 8. SONUÇ DATAFRAME
# ==========================================================

results_df = pd.DataFrame(
    results
)


results_path = os.path.join(
    RESULTS_DIR,
    "validation_naive_baselines_v4.csv"
)


results_df.to_csv(
    results_path,
    index=False
)


print("\n" + "=" * 80)
print("VALIDATION NAIVE BASELINE SONUÇLARI")
print("=" * 80)

print(
    results_df.to_string(
        index=False
    )
)


# ==========================================================
# 9. DENOMINATOR JSON
# ==========================================================

denominator_path = os.path.join(
    PROCESSED_DIR,
    "selection_baseline_denominators_v4.json"
)


with open(
    denominator_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        selection_denominators,
        f,
        ensure_ascii=False,
        indent=2
    )


print("\nDenominator dosyası:")
print(
    denominator_path
)


# ==========================================================
# 10. SEQUENCE SANITY CHECKS
#
# Bu bölüm test performansı HESAPLAMAZ.
# Yalnızca sequence shape/date yapısını kontrol eder.
# ==========================================================

sequence_records = []


for feature_set in [
    "baseline",
    "full"
]:

    expected_dim = (
        8
        if feature_set == "baseline"
        else 28
    )


    for lookback in LOOKBACKS:

        lb_dir = os.path.join(
            SEQUENCE_DIR,
            feature_set,
            f"lb{lookback}"
        )


        required_sequence_files = [

            "X_train.npy",
            "y_train.npy",

            "X_val.npy",
            "y_val.npy",

            "X_test.npy",
            "y_test.npy",

            "anchor_dates_train.npy",
            "target_realization_dates_train.npy",

            "anchor_dates_val.npy",
            "target_realization_dates_val.npy",

            "anchor_dates_test.npy",
            "target_realization_dates_test.npy"
        ]


        for filename in required_sequence_files:

            path = os.path.join(
                lb_dir,
                filename
            )

            if not os.path.exists(path):

                raise FileNotFoundError(
                    f"Sequence dosyası eksik:\n{path}"
                )


        # Memory-map:
        # dosya yapısı okunur, test performansı hesaplanmaz.

        X_train_seq = np.load(
            os.path.join(
                lb_dir,
                "X_train.npy"
            ),
            mmap_mode="r"
        )


        y_train_seq = np.load(
            os.path.join(
                lb_dir,
                "y_train.npy"
            ),
            mmap_mode="r"
        )


        X_val_seq = np.load(
            os.path.join(
                lb_dir,
                "X_val.npy"
            ),
            mmap_mode="r"
        )


        y_val_seq = np.load(
            os.path.join(
                lb_dir,
                "y_val.npy"
            ),
            mmap_mode="r"
        )


        X_test_seq = np.load(
            os.path.join(
                lb_dir,
                "X_test.npy"
            ),
            mmap_mode="r"
        )


        y_test_seq = np.load(
            os.path.join(
                lb_dir,
                "y_test.npy"
            ),
            mmap_mode="r"
        )


        train_target_dates_seq = np.load(
            os.path.join(
                lb_dir,
                "target_realization_dates_train.npy"
            )
        )


        val_target_dates_seq = np.load(
            os.path.join(
                lb_dir,
                "target_realization_dates_val.npy"
            )
        )


        test_target_dates_seq = np.load(
            os.path.join(
                lb_dir,
                "target_realization_dates_test.npy"
            )
        )


        # --------------------------------------------------
        # SHAPE KONTROLÜ
        # --------------------------------------------------

        if X_train_seq.shape[1:] != (
            lookback,
            expected_dim
        ):

            raise RuntimeError(
                f"X_train shape yanlış: "
                f"{feature_set}, lb={lookback}, "
                f"{X_train_seq.shape}"
            )


        if X_val_seq.shape != (
            584,
            lookback,
            expected_dim
        ):

            raise RuntimeError(
                f"X_val shape yanlış: "
                f"{feature_set}, lb={lookback}, "
                f"{X_val_seq.shape}"
            )


        if X_test_seq.shape != (
            584,
            lookback,
            expected_dim
        ):

            raise RuntimeError(
                f"X_test shape yanlış: "
                f"{feature_set}, lb={lookback}, "
                f"{X_test_seq.shape}"
            )


        if y_val_seq.shape != (
            584,
            8
        ):

            raise RuntimeError(
                f"y_val shape yanlış: "
                f"{feature_set}, lb={lookback}, "
                f"{y_val_seq.shape}"
            )


        if y_test_seq.shape != (
            584,
            8
        ):

            raise RuntimeError(
                f"y_test shape yanlış: "
                f"{feature_set}, lb={lookback}, "
                f"{y_test_seq.shape}"
            )


        # --------------------------------------------------
        # TARGET TARİH AYRIŞMASI
        # --------------------------------------------------

        train_target_dt = pd.to_datetime(
            train_target_dates_seq
        )


        val_target_dt = pd.to_datetime(
            val_target_dates_seq
        )


        test_target_dt = pd.to_datetime(
            test_target_dates_seq
        )


        train_val_disjoint = bool(
            train_target_dt.max()
            <
            val_target_dt.min()
        )


        val_test_disjoint = bool(
            val_target_dt.max()
            <
            test_target_dt.min()
        )


        if not train_val_disjoint:

            raise RuntimeError(
                "Train/validation target dates ayrık değil."
            )


        if not val_test_disjoint:

            raise RuntimeError(
                "Validation/test target dates ayrık değil."
            )


        sequence_records.append(
            {
                "feature_set":
                    feature_set,

                "lookback":
                    lookback,

                "X_train_shape":
                    str(
                        tuple(
                            X_train_seq.shape
                        )
                    ),

                "y_train_shape":
                    str(
                        tuple(
                            y_train_seq.shape
                        )
                    ),

                "X_val_shape":
                    str(
                        tuple(
                            X_val_seq.shape
                        )
                    ),

                "y_val_shape":
                    str(
                        tuple(
                            y_val_seq.shape
                        )
                    ),

                "X_test_shape":
                    str(
                        tuple(
                            X_test_seq.shape
                        )
                    ),

                "y_test_shape":
                    str(
                        tuple(
                            y_test_seq.shape
                        )
                    ),

                "validation_window_loss":
                    int(
                        584 - len(
                            X_val_seq
                        )
                    ),

                "test_window_loss":
                    int(
                        584 - len(
                            X_test_seq
                        )
                    ),

                "train_val_target_dates_disjoint":
                    train_val_disjoint,

                "val_test_target_dates_disjoint":
                    val_test_disjoint,

                "test_metrics_computed":
                    False
            }
        )


sequence_sanity_df = pd.DataFrame(
    sequence_records
)


sequence_sanity_path = os.path.join(
    RESULTS_DIR,
    "sequence_sanity_checks_v4.csv"
)


sequence_sanity_df.to_csv(
    sequence_sanity_path,
    index=False
)


print("\n" + "=" * 80)
print("SEQUENCE SANITY CHECKS")
print("=" * 80)

print(
    sequence_sanity_df.to_string(
        index=False
    )
)


# ==========================================================
# 11. ÖZET JSON
# ==========================================================

summary = {

    "project_version":
        "v4_repro",

    "created_at":
        datetime.now().isoformat(),

    "script":
        "03_baseline_sanity_v4.py",

    "validation_only_metrics":
        True,

    "test_metrics_computed":
        False,

    "validation_rows":
        int(
            len(
                y_val
            )
        ),

    "baselines": [
        "ReturnZero",
        "ReturnPersistence",
        "VolPersistence"
    ],

    "selection_denominators": {

        "return":
            "ReturnZero MAE",

        "volatility":
            "VolPersistence PinballLoss tau=0.5"
    },

    "validation_target_period": {

        "start":
            str(
                dates_val[
                    "target_realization_date"
                ].min().date()
            ),

        "end":
            str(
                dates_val[
                    "target_realization_date"
                ].max().date()
            )
    },

    "sequence_configs_checked":
        int(
            len(
                sequence_sanity_df
            )
        ),

    "all_train_val_target_dates_disjoint":
        bool(
            sequence_sanity_df[
                "train_val_target_dates_disjoint"
            ].all()
        ),

    "all_val_test_target_dates_disjoint":
        bool(
            sequence_sanity_df[
                "val_test_target_dates_disjoint"
            ].all()
        ),

    "all_validation_window_loss_zero":
        bool(
            (
                sequence_sanity_df[
                    "validation_window_loss"
                ]
                == 0
            ).all()
        ),

    "all_test_window_loss_zero":
        bool(
            (
                sequence_sanity_df[
                    "test_window_loss"
                ]
                == 0
            ).all()
        )
}


summary_path = os.path.join(
    RESULTS_DIR,
    "baseline_sanity_summary_v4.json"
)


with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2
    )


# ==========================================================
# 12. KRİTİK SON KONTROLLER
# ==========================================================

if len(
    results_df
) != 12:

    raise RuntimeError(
        "Beklenen 12 baseline sonucu üretilemedi."
    )


if not (
    sequence_sanity_df[
        "validation_window_loss"
    ]
    == 0
).all():

    raise RuntimeError(
        "Validation pencere kaybı sıfır değil."
    )


if not (
    sequence_sanity_df[
        "test_window_loss"
    ]
    == 0
).all():

    raise RuntimeError(
        "Test pencere kaybı sıfır değil."
    )


if not sequence_sanity_df[
    "train_val_target_dates_disjoint"
].all():

    raise RuntimeError(
        "Bazı train/validation target tarihleri ayrık değil."
    )


if not sequence_sanity_df[
    "val_test_target_dates_disjoint"
].all():

    raise RuntimeError(
        "Bazı validation/test target tarihleri ayrık değil."
    )


# Denominator'lar pozitif olmalı.

for asset in ASSETS:

    return_value = (
        selection_denominators[
            "return_denominator"
        ][asset][
            "value"
        ]
    )


    vol_value = (
        selection_denominators[
            "volatility_denominator"
        ][asset][
            "value"
        ]
    )


    if return_value <= 0:

        raise RuntimeError(
            f"{asset} ReturnZero denominator pozitif değil."
        )


    if vol_value <= 0:

        raise RuntimeError(
            f"{asset} VolPersistence denominator pozitif değil."
        )


# ==========================================================
# 13. SON ÇIKTI
# ==========================================================

print("\n")
print("=" * 80)
print("03_baseline_sanity_v4.py BAŞARIYLA TAMAMLANDI")
print("=" * 80)

print("\nDosyalar:")

print(
    " -",
    results_path
)

print(
    " -",
    denominator_path
)

print(
    " -",
    sequence_sanity_path
)

print(
    " -",
    summary_path
)


print("\nKURAL KONTROLÜ:")

print(
    "✅ ReturnZero hesaplandı."
)

print(
    "✅ ReturnPersistence hesaplandı."
)

print(
    "✅ VolPersistence hesaplandı."
)

print(
    "✅ ValidationScore denominator'ları kaydedildi."
)

print(
    "✅ 8 sequence konfigürasyonu kontrol edildi."
)

print(
    "✅ Validation pencere kaybı = 0."
)

print(
    "✅ Test pencere kaybı = 0."
)

print(
    "✅ Train/validation target tarihleri ayrık."
)

print(
    "✅ Validation/test target tarihleri ayrık."
)

print(
    "✅ Test performansı hesaplanmadı."
)

print("=" * 80)
'''


# ==========================================================
# SCRIPT'İ KAYDET
# ==========================================================

with open(
    script_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        script_code
    )


print(
    "03_baseline_sanity_v4.py oluşturuldu:"
)

print(
    script_path
)


# ==========================================================
# ÖNCE SYNTAX KONTROLÜ
# ==========================================================

with open(
    script_path,
    "r",
    encoding="utf-8"
) as f:

    code_to_run = f.read()


compile(
    code_to_run,
    script_path,
    "exec"
)


print("\n✅ Syntax kontrolü geçti.")


# ==========================================================
# SCRIPT'İ ÇALIŞTIR
# ==========================================================

print(
    "\nScript çalıştırılıyor...\n"
)


exec(
    compile(
        code_to_run,
        script_path,
        "exec"
    )
)

In [ ]:
# ==========================================================
# 04_small_model_test_v4.py OLUŞTUR + SYNTAX KONTROLÜ + ÇALIŞTIR
# ==========================================================

import os

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

script_path = os.path.join(
    BASE_DIR,
    "scripts",
    "04_small_model_test_v4.py"
)

script_code = r'''
# ==========================================================
# 04_small_model_test_v4.py
#
# AMAÇ:
# - Küçük bir FullSharingMTL Transformer ile smoke test yapmak
# - Veri -> DataLoader -> Model -> Loss -> Backprop -> Validation
#   -> inverse scaling -> ValidationScore hattını doğrulamak
#
# ÖNEMLİ:
# - Grid search YOK.
# - Final model seçimi YOK.
# - Test seti YÜKLENMEZ ve test metriği HESAPLANMAZ.
# ==========================================================

import os
import json
import pickle
import random
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
except ImportError as e:
    raise ImportError(
        "PyTorch bulunamadı. Colab runtime'ında PyTorch kurulumu kontrol edilmeli."
    ) from e


# ==========================================================
# 1. YOLLAR
# ==========================================================

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

CONFIG_DIR = os.path.join(
    BASE_DIR,
    "config"
)

SEQUENCE_DIR = os.path.join(
    BASE_DIR,
    "data",
    "sequences"
)

PROCESSED_DIR = os.path.join(
    BASE_DIR,
    "data",
    "processed"
)

MODEL_DIR = os.path.join(
    BASE_DIR,
    "models"
)

RESULTS_DIR = os.path.join(
    BASE_DIR,
    "results",
    "small_model_test"
)

for path in [
    MODEL_DIR,
    RESULTS_DIR
]:
    os.makedirs(
        path,
        exist_ok=True
    )


# ==========================================================
# 2. SEED VE DEVICE
# ==========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("[DEVICE]", device)
print("[SEED]", SEED)


# ==========================================================
# 3. ŞEMA VE DENOMINATOR DOSYALARINI OKU
# ==========================================================

schema_path = os.path.join(
    CONFIG_DIR,
    "schema_v4.json"
)

denominator_path = os.path.join(
    PROCESSED_DIR,
    "selection_baseline_denominators_v4.json"
)

split_meta_path = os.path.join(
    PROCESSED_DIR,
    "split_meta_v4.json"
)

for path in [
    schema_path,
    denominator_path,
    split_meta_path
]:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Gerekli dosya bulunamadı:\n{path}"
        )


with open(
    schema_path,
    "r",
    encoding="utf-8"
) as f:

    schema = json.load(f)


with open(
    denominator_path,
    "r",
    encoding="utf-8"
) as f:

    denominators = json.load(f)


with open(
    split_meta_path,
    "r",
    encoding="utf-8"
) as f:

    split_meta = json.load(f)


ASSET_ORDER = schema[
    "data"
][
    "assets"
]


TARGET_NAMES = schema[
    "targets"
][
    "definition"
]


if ASSET_ORDER != [
    "BIST100",
    "USDTRY",
    "EURTRY",
    "GOLD"
]:

    raise RuntimeError(
        f"Asset sırası beklenenden farklı: {ASSET_ORDER}"
    )


if len(TARGET_NAMES) != 8:

    raise RuntimeError(
        f"Target sayısı 8 değil: {len(TARGET_NAMES)}"
    )


print(
    "\n[OK] Schema, split meta ve denominator dosyaları okundu."
)

print(
    "Asset order:",
    ASSET_ORDER
)

print(
    "Target order:",
    TARGET_NAMES
)


# ==========================================================
# 4. SMOKE TEST KONFİGÜRASYONU
# ==========================================================

FEATURE_SET = "baseline"
LOOKBACK = 10

D_MODEL = 32
N_HEAD = 4
N_LAYERS = 2
D_FF = 128
DROPOUT = 0.10

LOSS_STRATEGY = "FixedLambda_0.5"
LAMBDA_RETURN = 0.5
TAU = 0.5

BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0


print("\n" + "=" * 80)
print("SMOKE TEST KONFİGÜRASYONU")
print("=" * 80)

print(
    "Feature set :",
    FEATURE_SET
)

print(
    "Lookback    :",
    LOOKBACK
)

print(
    "Architecture:",
    "FullSharingMTL"
)

print(
    "Size        :",
    "small"
)

print(
    "Loss        :",
    LOSS_STRATEGY
)

print(
    "d_model     :",
    D_MODEL
)

print(
    "n_head      :",
    N_HEAD
)

print(
    "n_layers    :",
    N_LAYERS
)

print(
    "d_ff        :",
    D_FF
)

print(
    "epochs      :",
    EPOCHS
)

print(
    "batch_size  :",
    BATCH_SIZE
)


# ==========================================================
# 5. SADECE TRAIN VE VALIDATION DOSYALARINI YÜKLE
# ==========================================================

seq_dir = os.path.join(
    SEQUENCE_DIR,
    FEATURE_SET,
    f"lb{LOOKBACK}"
)

scaler_path = os.path.join(
    SEQUENCE_DIR,
    FEATURE_SET,
    "scalers.pkl"
)


required_files = [

    os.path.join(
        seq_dir,
        "X_train.npy"
    ),

    os.path.join(
        seq_dir,
        "y_train.npy"
    ),

    os.path.join(
        seq_dir,
        "y_train_raw.npy"
    ),

    os.path.join(
        seq_dir,
        "X_val.npy"
    ),

    os.path.join(
        seq_dir,
        "y_val.npy"
    ),

    os.path.join(
        seq_dir,
        "y_val_raw.npy"
    ),

    os.path.join(
        seq_dir,
        "anchor_dates_val.npy"
    ),

    os.path.join(
        seq_dir,
        "target_realization_dates_val.npy"
    ),

    scaler_path
]


for file_path in required_files:

    if not os.path.exists(file_path):

        raise FileNotFoundError(
            f"Gerekli dosya bulunamadı:\n{file_path}"
        )


X_train = np.load(
    os.path.join(
        seq_dir,
        "X_train.npy"
    )
)


y_train = np.load(
    os.path.join(
        seq_dir,
        "y_train.npy"
    )
)


y_train_raw = np.load(
    os.path.join(
        seq_dir,
        "y_train_raw.npy"
    )
)


X_val = np.load(
    os.path.join(
        seq_dir,
        "X_val.npy"
    )
)


y_val = np.load(
    os.path.join(
        seq_dir,
        "y_val.npy"
    )
)


y_val_raw = np.load(
    os.path.join(
        seq_dir,
        "y_val_raw.npy"
    )
)


anchor_dates_val = np.load(
    os.path.join(
        seq_dir,
        "anchor_dates_val.npy"
    )
)


target_dates_val = np.load(
    os.path.join(
        seq_dir,
        "target_realization_dates_val.npy"
    )
)


with open(
    scaler_path,
    "rb"
) as f:

    scaler_obj = pickle.load(f)


y_scaler = scaler_obj[
    "y_scaler"
]


print("\n" + "=" * 80)
print("VERİ SHAPE")
print("=" * 80)

print(
    "X_train   :",
    X_train.shape
)

print(
    "y_train   :",
    y_train.shape
)

print(
    "X_val     :",
    X_val.shape
)

print(
    "y_val     :",
    y_val.shape
)

print(
    "y_val_raw :",
    y_val_raw.shape
)


if (
    X_train.ndim != 3
    or
    X_val.ndim != 3
):

    raise ValueError(
        "X dizileri 3 boyutlu olmalı: "
        "(n, lookback, n_features)"
    )


if (
    y_train.ndim != 2
    or
    y_val.ndim != 2
):

    raise ValueError(
        "y dizileri 2 boyutlu olmalı: "
        "(n, n_targets)"
    )


if (
    X_train.shape[1] != LOOKBACK
    or
    X_val.shape[1] != LOOKBACK
):

    raise ValueError(
        "Lookback boyutu uyuşmuyor."
    )


if (
    y_train.shape[1] != 8
    or
    y_val.shape[1] != 8
):

    raise ValueError(
        "Target boyutu 8 olmalı."
    )


if len(X_val) != 584:

    raise ValueError(
        f"Validation örnek sayısı 584 değil: {len(X_val)}"
    )


if (
    len(anchor_dates_val) != len(X_val)
    or
    len(target_dates_val) != len(X_val)
):

    raise ValueError(
        "Validation tarih dizileri X_val ile hizalı değil."
    )


# ==========================================================
# 6. INVERSE-SCALING KONTROLÜ
# ==========================================================

y_val_inverse_check = (
    y_scaler.inverse_transform(
        y_val
    )
)


max_inverse_diff = float(
    np.max(
        np.abs(
            y_val_inverse_check
            -
            y_val_raw
        )
    )
)


if max_inverse_diff > 1e-5:

    raise RuntimeError(
        "y_val_raw ile "
        "y_scaler.inverse_transform(y_val) uyuşmuyor. "
        f"Maksimum fark: {max_inverse_diff}"
    )


val_target_dt = pd.to_datetime(
    target_dates_val
)


expected_val_target_start = pd.Timestamp(
    split_meta[
        "validation"
    ][
        "target_realization_start"
    ]
)


expected_val_target_end = pd.Timestamp(
    split_meta[
        "validation"
    ][
        "target_realization_end"
    ]
)


if (
    val_target_dt.min()
    !=
    expected_val_target_start
):

    raise RuntimeError(
        "Validation target başlangıç tarihi "
        "split meta ile uyuşmuyor."
    )


if (
    val_target_dt.max()
    !=
    expected_val_target_end
):

    raise RuntimeError(
        "Validation target bitiş tarihi "
        "split meta ile uyuşmuyor."
    )


N_FEATURES = X_train.shape[2]
N_TARGETS = y_train.shape[1]


print(
    "\n[OK] Veri şekilleri ve inverse-scaling kontrolü geçti."
)

print(
    "N_FEATURES:",
    N_FEATURES
)

print(
    "N_TARGETS :",
    N_TARGETS
)

print(
    "Max inverse diff:",
    max_inverse_diff
)

print(
    "Validation target realization:",
    val_target_dt.min().date(),
    "→",
    val_target_dt.max().date()
)


# ==========================================================
# 7. DATALOADER
# ==========================================================

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.float32
)


train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)


val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)


loader_generator = torch.Generator()

loader_generator.manual_seed(
    SEED
)


train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    drop_last=False,

    generator=loader_generator
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    drop_last=False
)


print(
    "[OK] DataLoader hazır."
)


# ==========================================================
# 8. MODEL: FullSharingMTL
# ==========================================================

class FullSharingMTL(nn.Module):

    """
    Tam paylaşım mimarisi:

    - ortak input projection
    - ortak positional embedding
    - ortak Transformer encoder
    - ortak latent representation
    - ayrı return head
    - ayrı volatility head

    Çıktı:
        ilk 4 = NextRet
        son 4 = NextVol
    """

    def __init__(

        self,

        n_features,

        lookback,

        d_model=32,

        n_head=4,

        n_layers=2,

        d_ff=128,

        dropout=0.10
    ):

        super().__init__()


        self.n_features = (
            n_features
        )

        self.lookback = (
            lookback
        )

        self.d_model = (
            d_model
        )


        self.input_projection = nn.Linear(
            n_features,
            d_model
        )


        self.positional_embedding = nn.Parameter(

            torch.zeros(
                1,
                lookback,
                d_model
            )
        )


        encoder_layer = nn.TransformerEncoderLayer(

            d_model=d_model,

            nhead=n_head,

            dim_feedforward=d_ff,

            dropout=dropout,

            activation="gelu",

            batch_first=True
        )


        self.encoder = nn.TransformerEncoder(

            encoder_layer,

            num_layers=n_layers
        )


        self.norm = nn.LayerNorm(
            d_model
        )


        self.return_head = nn.Sequential(

            nn.Linear(
                d_model,
                d_model
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                d_model,
                4
            )
        )


        self.vol_head = nn.Sequential(

            nn.Linear(
                d_model,
                d_model
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                d_model,
                4
            )
        )


    def forward(
        self,
        x
    ):

        h = self.input_projection(
            x
        )


        h = (
            h
            +
            self.positional_embedding[
                :,
                :h.size(1),
                :
            ]
        )


        h = self.encoder(
            h
        )


        h_last = self.norm(
            h[
                :,
                -1,
                :
            ]
        )


        ret_out = self.return_head(
            h_last
        )


        vol_out = self.vol_head(
            h_last
        )


        return torch.cat(

            [
                ret_out,
                vol_out
            ],

            dim=1
        )


model = FullSharingMTL(

    n_features=N_FEATURES,

    lookback=LOOKBACK,

    d_model=D_MODEL,

    n_head=N_HEAD,

    n_layers=N_LAYERS,

    d_ff=D_FF,

    dropout=DROPOUT

).to(device)


parameter_count = int(

    sum(

        p.numel()

        for p in model.parameters()

        if p.requires_grad
    )
)


print(
    "\n[MODEL]"
)

print(
    model
)

print(
    "\nTrainable parameter count:",
    parameter_count
)


# ==========================================================
# 9. LOSS VE OPTIMIZER
# ==========================================================

mse_loss = nn.MSELoss()


def pinball_loss_torch(

    y_pred,

    y_true,

    tau=0.5
):

    diff = (
        y_true
        -
        y_pred
    )


    loss = torch.maximum(

        tau * diff,

        (tau - 1.0) * diff
    )


    return loss.mean()


def multi_task_loss(

    y_pred,

    y_true
):

    pred_ret = (
        y_pred[
            :,
            :4
        ]
    )


    true_ret = (
        y_true[
            :,
            :4
        ]
    )


    pred_vol = (
        y_pred[
            :,
            4:
        ]
    )


    true_vol = (
        y_true[
            :,
            4:
        ]
    )


    return_loss = mse_loss(

        pred_ret,

        true_ret
    )


    vol_loss = pinball_loss_torch(

        pred_vol,

        true_vol,

        tau=TAU
    )


    total_loss = (

        LAMBDA_RETURN
        *
        return_loss

        +

        (
            1.0
            -
            LAMBDA_RETURN
        )
        *
        vol_loss
    )


    return (
        total_loss,
        return_loss,
        vol_loss
    )


optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY
)


# ==========================================================
# 10. TRAIN / VALIDATION FONKSİYONLARI
# ==========================================================

def train_one_epoch(

    model,

    loader,

    optimizer
):

    model.train()


    total_loss_sum = 0.0

    return_loss_sum = 0.0

    vol_loss_sum = 0.0

    n_obs = 0


    for (
        X_batch,
        y_batch
    ) in loader:


        X_batch = X_batch.to(
            device
        )


        y_batch = y_batch.to(
            device
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        y_pred = model(
            X_batch
        )


        (
            loss,
            return_loss,
            vol_loss

        ) = multi_task_loss(

            y_pred,

            y_batch
        )


        if not torch.isfinite(
            loss
        ):

            raise RuntimeError(
                "Training loss finite değil."
            )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=GRAD_CLIP
        )


        optimizer.step()


        batch_size = X_batch.size(
            0
        )


        total_loss_sum += (
            loss.item()
            *
            batch_size
        )


        return_loss_sum += (
            return_loss.item()
            *
            batch_size
        )


        vol_loss_sum += (
            vol_loss.item()
            *
            batch_size
        )


        n_obs += (
            batch_size
        )


    return {

        "loss":
            total_loss_sum
            /
            n_obs,

        "return_loss":
            return_loss_sum
            /
            n_obs,

        "vol_loss":
            vol_loss_sum
            /
            n_obs
    }


@torch.no_grad()
def evaluate_scaled_loss(

    model,

    loader
):

    model.eval()


    total_loss_sum = 0.0

    return_loss_sum = 0.0

    vol_loss_sum = 0.0

    n_obs = 0


    preds = []

    trues = []


    for (
        X_batch,
        y_batch
    ) in loader:


        X_batch = X_batch.to(
            device
        )


        y_batch = y_batch.to(
            device
        )


        y_pred = model(
            X_batch
        )


        (
            loss,
            return_loss,
            vol_loss

        ) = multi_task_loss(

            y_pred,

            y_batch
        )


        if not torch.isfinite(
            loss
        ):

            raise RuntimeError(
                "Validation loss finite değil."
            )


        batch_size = X_batch.size(
            0
        )


        total_loss_sum += (
            loss.item()
            *
            batch_size
        )


        return_loss_sum += (
            return_loss.item()
            *
            batch_size
        )


        vol_loss_sum += (
            vol_loss.item()
            *
            batch_size
        )


        n_obs += (
            batch_size
        )


        preds.append(

            y_pred
            .detach()
            .cpu()
            .numpy()
        )


        trues.append(

            y_batch
            .detach()
            .cpu()
            .numpy()
        )


    return {

        "loss":
            total_loss_sum
            /
            n_obs,

        "return_loss":
            return_loss_sum
            /
            n_obs,

        "vol_loss":
            vol_loss_sum
            /
            n_obs,

        "preds_scaled":
            np.vstack(
                preds
            ),

        "trues_scaled":
            np.vstack(
                trues
            )
    }


# ==========================================================
# 11. RAW METRİKLER
# ==========================================================

def mae_np(
    y_true,
    y_pred
):

    return float(

        np.mean(

            np.abs(
                y_true
                -
                y_pred
            )
        )
    )


def rmse_np(
    y_true,
    y_pred
):

    return float(

        np.sqrt(

            np.mean(

                (
                    y_true
                    -
                    y_pred
                )
                ** 2
            )
        )
    )


def r2_np(
    y_true,
    y_pred
):

    ss_res = np.sum(

        (
            y_true
            -
            y_pred
        )
        ** 2
    )


    ss_tot = np.sum(

        (
            y_true
            -
            np.mean(
                y_true
            )
        )
        ** 2
    )


    if ss_tot == 0:

        return float(
            "nan"
        )


    return float(

        1.0
        -
        ss_res
        /
        ss_tot
    )


def pinball_np(

    y_true,

    y_pred,

    tau=0.5
):

    diff = (
        y_true
        -
        y_pred
    )


    loss = np.maximum(

        tau
        *
        diff,

        (
            tau
            -
            1.0
        )
        *
        diff
    )


    return float(

        np.mean(
            loss
        )
    )


def compute_validation_raw_metrics(

    y_true_raw,

    y_pred_raw
):

    rows = []


    for (
        i,
        asset
    ) in enumerate(
        ASSET_ORDER
    ):


        true = y_true_raw[
            :,
            i
        ]


        pred = y_pred_raw[
            :,
            i
        ]


        rows.append(

            {

                "task":
                    "return",

                "asset":
                    asset,

                "MAE":
                    mae_np(
                        true,
                        pred
                    ),

                "RMSE":
                    rmse_np(
                        true,
                        pred
                    ),

                "R2":
                    r2_np(
                        true,
                        pred
                    ),

                "PinballLoss_tau_0.5":
                    np.nan
            }
        )


    for (
        i,
        asset
    ) in enumerate(
        ASSET_ORDER
    ):


        col = (
            4
            +
            i
        )


        true = y_true_raw[
            :,
            col
        ]


        pred = y_pred_raw[
            :,
            col
        ]


        rows.append(

            {

                "task":
                    "volatility",

                "asset":
                    asset,

                "MAE":
                    mae_np(
                        true,
                        pred
                    ),

                "RMSE":
                    rmse_np(
                        true,
                        pred
                    ),

                "R2":
                    r2_np(
                        true,
                        pred
                    ),

                "PinballLoss_tau_0.5":
                    pinball_np(

                        true,

                        pred,

                        tau=TAU
                    )
            }
        )


    return pd.DataFrame(
        rows
    )


# ==========================================================
# 12. VALIDATIONSCORE
# ==========================================================

def compute_validation_score(

    metrics_df,

    denominators
):

    return_ratios = []

    vol_ratios = []


    return_ratio_map = {}

    vol_ratio_map = {}


    for asset in ASSET_ORDER:


        model_return_mae = float(

            metrics_df.loc[

                (
                    metrics_df[
                        "task"
                    ]
                    ==
                    "return"
                )

                &

                (
                    metrics_df[
                        "asset"
                    ]
                    ==
                    asset
                ),

                "MAE"

            ].iloc[0]
        )


        denom_return_mae = float(

            denominators[

                "return_denominator"

            ][

                asset

            ][

                "value"

            ]
        )


        model_vol_pinball = float(

            metrics_df.loc[

                (
                    metrics_df[
                        "task"
                    ]
                    ==
                    "volatility"
                )

                &

                (
                    metrics_df[
                        "asset"
                    ]
                    ==
                    asset
                ),

                "PinballLoss_tau_0.5"

            ].iloc[0]
        )


        denom_vol_pinball = float(

            denominators[

                "volatility_denominator"

            ][

                asset

            ][

                "value"

            ]
        )


        if (
            denom_return_mae <= 0
            or
            denom_vol_pinball <= 0
        ):

            raise RuntimeError(

                f"{asset} denominator pozitif değil."
            )


        return_ratio = (

            model_return_mae

            /

            denom_return_mae
        )


        vol_ratio = (

            model_vol_pinball

            /

            denom_vol_pinball
        )


        return_ratios.append(
            return_ratio
        )


        vol_ratios.append(
            vol_ratio
        )


        return_ratio_map[
            asset
        ] = float(
            return_ratio
        )


        vol_ratio_map[
            asset
        ] = float(
            vol_ratio
        )


    avg_return_ratio = float(

        np.mean(
            return_ratios
        )
    )


    avg_vol_ratio = float(

        np.mean(
            vol_ratios
        )
    )


    validation_score = float(

        0.5
        *
        avg_return_ratio

        +

        0.5
        *
        avg_vol_ratio
    )


    return {

        "return_ratios":
            return_ratio_map,

        "vol_ratios":
            vol_ratio_map,

        "avg_return_ratio":
            avg_return_ratio,

        "avg_vol_ratio":
            avg_vol_ratio,

        "validation_score":
            validation_score,

        "lower_is_better":
            True
    }


# ==========================================================
# 13. EĞİTİM
# ==========================================================

print("\n" + "=" * 80)
print("EĞİTİM BAŞLIYOR — SMOKE TEST")
print("=" * 80)


history = []


for epoch in range(
    1,
    EPOCHS + 1
):


    train_metrics = train_one_epoch(

        model=model,

        loader=train_loader,

        optimizer=optimizer
    )


    val_scaled_metrics = evaluate_scaled_loss(

        model=model,

        loader=val_loader
    )


    row = {

        "epoch":
            epoch,

        "train_loss":
            train_metrics[
                "loss"
            ],

        "train_return_loss":
            train_metrics[
                "return_loss"
            ],

        "train_vol_loss":
            train_metrics[
                "vol_loss"
            ],

        "val_loss":
            val_scaled_metrics[
                "loss"
            ],

        "val_return_loss":
            val_scaled_metrics[
                "return_loss"
            ],

        "val_vol_loss":
            val_scaled_metrics[
                "vol_loss"
            ]
    }


    history.append(
        row
    )


    print(

        f"Epoch {epoch:02d} | "

        f"train_loss="
        f"{row['train_loss']:.6f} | "

        f"val_loss="
        f"{row['val_loss']:.6f} | "

        f"val_ret="
        f"{row['val_return_loss']:.6f} | "

        f"val_vol="
        f"{row['val_vol_loss']:.6f}"
    )


# ==========================================================
# 14. VALIDATION RAW METRİKLER VE SKOR
# ==========================================================

val_scaled_metrics = evaluate_scaled_loss(

    model,

    val_loader
)


y_val_pred_scaled = (
    val_scaled_metrics[
        "preds_scaled"
    ]
)


y_val_pred_raw = (
    y_scaler.inverse_transform(
        y_val_pred_scaled
    )
)


if not np.isfinite(
    y_val_pred_raw
).all():

    raise RuntimeError(
        "Validation raw prediction içinde NaN/Inf var."
    )


validation_raw_metrics_df = compute_validation_raw_metrics(

    y_true_raw=y_val_raw,

    y_pred_raw=y_val_pred_raw
)


validation_score_obj = compute_validation_score(

    metrics_df=
        validation_raw_metrics_df,

    denominators=
        denominators
)


print("\n" + "=" * 80)
print("VALIDATION RAW METRİKLER — SMOKE TEST")
print("=" * 80)

print(

    validation_raw_metrics_df.to_string(
        index=False
    )
)


print("\n" + "=" * 80)
print("BASELINE-NORMALIZE VALIDATION SCORE — SMOKE TEST")
print("=" * 80)

print(

    json.dumps(

        validation_score_obj,

        ensure_ascii=False,

        indent=2
    )
)


# ==========================================================
# 15. DOSYALARI KAYDET
# ==========================================================

history_df = pd.DataFrame(
    history
)


history_path = os.path.join(

    RESULTS_DIR,

    "small_model_training_history_v4.csv"
)


metrics_path = os.path.join(

    RESULTS_DIR,

    "small_model_validation_metrics_v4.csv"
)


score_path = os.path.join(

    RESULTS_DIR,

    "small_model_validation_score_v4.json"
)


summary_path = os.path.join(

    RESULTS_DIR,

    "small_model_test_summary_v4.json"
)


model_path = os.path.join(

    MODEL_DIR,

    "small_model_test_fullsharing_baseline_lb10_v4.pt"
)


history_df.to_csv(

    history_path,

    index=False
)


validation_raw_metrics_df.to_csv(

    metrics_path,

    index=False
)


with open(

    score_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        validation_score_obj,

        f,

        ensure_ascii=False,

        indent=2
    )


torch.save(

    {

        "model_state_dict":
            model.state_dict(),

        "config": {

            "project_version":
                "v4_repro",

            "feature_set":
                FEATURE_SET,

            "lookback":
                LOOKBACK,

            "architecture":
                "FullSharingMTL",

            "size":
                "small",

            "d_model":
                D_MODEL,

            "n_head":
                N_HEAD,

            "n_layers":
                N_LAYERS,

            "d_ff":
                D_FF,

            "dropout":
                DROPOUT,

            "loss_strategy":
                LOSS_STRATEGY,

            "lambda_return":
                LAMBDA_RETURN,

            "tau":
                TAU,

            "epochs":
                EPOCHS,

            "seed":
                SEED
        },

        "target_names":
            TARGET_NAMES,

        "asset_order":
            ASSET_ORDER,

        "trainable_parameter_count":
            parameter_count,

        "purpose":
            "smoke_test_only"
    },

    model_path
)


summary = {

    "created_at":
        datetime.now().isoformat(),

    "script":
        "04_small_model_test_v4.py",

    "project_version":
        "v4_repro",

    "status":
        "success",

    "purpose":
        (
            "Smoke test only: verifies "
            "data -> model -> loss -> backprop -> "
            "validation -> inverse scaling -> "
            "ValidationScore pipeline."
        ),

    "device":
        str(device),

    "seed":
        SEED,

    "feature_set":
        FEATURE_SET,

    "lookback":
        LOOKBACK,

    "architecture":
        "FullSharingMTL",

    "size":
        "small",

    "loss_strategy":
        LOSS_STRATEGY,

    "epochs":
        EPOCHS,

    "trainable_parameter_count":
        parameter_count,

    "validation_rows":
        int(
            len(
                X_val
            )
        ),

    "validation_anchor_start":
        str(
            pd.to_datetime(
                anchor_dates_val
            ).min().date()
        ),

    "validation_anchor_end":
        str(
            pd.to_datetime(
                anchor_dates_val
            ).max().date()
        ),

    "validation_target_realization_start":
        str(
            val_target_dt
            .min()
            .date()
        ),

    "validation_target_realization_end":
        str(
            val_target_dt
            .max()
            .date()
        ),

    "max_inverse_scaling_diff":
        max_inverse_diff,

    "validation_score":
        validation_score_obj,

    "test_arrays_loaded":
        False,

    "test_metrics_computed":
        False,

    "final_model_selection_performed":
        False,

    "files_created": {

        "training_history":
            history_path,

        "validation_metrics":
            metrics_path,

        "validation_score":
            score_path,

        "model_checkpoint":
            model_path
    },

    "important_notes": [

        "Bu dosyada test dizileri yüklenmemiştir.",

        "Bu dosyada test metriği hesaplanmamıştır.",

        "Bu dosyada final model seçimi yapılmamıştır.",

        "Bu dosya yalnızca smoke testtir.",

        "ValidationScore yalnızca validation denominator'ları ile hesaplanmıştır."
    ]
}


with open(

    summary_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        summary,

        f,

        ensure_ascii=False,

        indent=2
    )


# ==========================================================
# 16. SON KONTROLLER VE ÇIKTI
# ==========================================================

if len(
    validation_raw_metrics_df
) != 8:

    raise RuntimeError(
        "Beklenen 8 validation metric satırı üretilemedi."
    )


if not np.isfinite(
    validation_score_obj[
        "validation_score"
    ]
):

    raise RuntimeError(
        "ValidationScore finite değil."
    )


print("\n" + "=" * 80)
print("04_small_model_test_v4.py BAŞARIYLA TAMAMLANDI")
print("=" * 80)


print("\nÜretilen dosyalar:")

print(
    " -",
    history_path
)

print(
    " -",
    metrics_path
)

print(
    " -",
    score_path
)

print(
    " -",
    summary_path
)

print(
    " -",
    model_path
)


print("\nKURAL KONTROLÜ:")

print(
    "✅ Train ve validation dizileri yüklendi."
)

print(
    "✅ Test dizileri yüklenmedi."
)

print(
    "✅ 5 epoch smoke test tamamlandı."
)

print(
    "✅ Forward pass tamamlandı."
)

print(
    "✅ Return MSE hesaplandı."
)

print(
    "✅ Volatility PinballLoss(tau=0.5) hesaplandı."
)

print(
    "✅ Backpropagation tamamlandı."
)

print(
    "✅ Validation prediction inverse-scale edildi."
)

print(
    "✅ ValidationScore hesaplandı."
)

print(
    "✅ Test metriği hesaplanmadı."
)

print(
    "✅ Final model seçimi yapılmadı."
)

print("=" * 80)
'''


# ==========================================================
# SCRIPT'İ KAYDET
# ==========================================================

with open(
    script_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        script_code
    )


print(
    "04_small_model_test_v4.py oluşturuldu:"
)

print(
    script_path
)


# ==========================================================
# ÖNCE SYNTAX KONTROLÜ
# ==========================================================

with open(
    script_path,
    "r",
    encoding="utf-8"
) as f:

    code_to_run = f.read()


compile(
    code_to_run,
    script_path,
    "exec"
)


print(
    "\n✅ Syntax kontrolü geçti."
)


# ==========================================================
# SCRIPT'İ ÇALIŞTIR
# ==========================================================

print(
    "\nScript çalıştırılıyor...\n"
)


exec(
    compile(
        code_to_run,
        script_path,
        "exec"
    )
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ==========================================================
# 05a_mini_grid_v4.py OLUŞTUR + SYNTAX KONTROLÜ + ÇALIŞTIR
# ==========================================================

import os

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

script_path = os.path.join(
    BASE_DIR,
    "scripts",
    "05a_mini_grid_v4.py"
)

script_code = r'''
import os
import json
import pickle
import random
import copy
import itertools
from datetime import datetime

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader
)


# ==========================================================
# 1. YOLLAR
# ==========================================================

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

if not os.path.exists(BASE_DIR):

    raise FileNotFoundError(
        f"Proje klasörü yok:\n{BASE_DIR}\n"
        "Drive mount edildi mi?"
    )


CONFIG_DIR = os.path.join(
    BASE_DIR,
    "config"
)

PROCESSED_DIR = os.path.join(
    BASE_DIR,
    "data",
    "processed"
)

SEQUENCE_DIR = os.path.join(
    BASE_DIR,
    "data",
    "sequences"
)

RESULTS_DIR = os.path.join(
    BASE_DIR,
    "results",
    "mini_grid"
)

MODEL_DIR = os.path.join(
    BASE_DIR,
    "models",
    "mini_grid"
)


os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)


# ==========================================================
# 2. MINI-GRID SABİTLERİ
# ==========================================================

SEED = 42

EPOCHS = 5

BATCH_SIZE = 64

LR = 1e-3

WEIGHT_DECAY = 1e-4

GRAD_CLIP = 1.0

TAU = 0.5

LAMBDA_RETURN = 0.5


FEATURE_SET = "baseline"


LOOKBACKS = [
    10,
    30
]


ARCHITECTURES = [
    "FullSharingMTL",
    "NoSharing"
]


LOSS_STRATEGIES = [
    "FixedLambda_0.5",
    "PCGrad"
]


# medium model

D_MODEL = 64

N_HEAD = 4

N_LAYERS = 2

D_FF = 256

DROPOUT = 0.10


TOTAL_CONFIGS = (
    len(LOOKBACKS)
    *
    len(ARCHITECTURES)
    *
    len(LOSS_STRATEGIES)
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 80)

print(
    "05a — v4 MINI-GRID AUDIT"
)

print("=" * 80)

print(
    "[DEVICE]",
    DEVICE
)


if torch.cuda.is_available():

    print(
        "[GPU]",
        torch.cuda.get_device_name(0)
    )

else:

    print(
        "⚠️ GPU aktif değil; CPU kullanılacak."
    )


print(
    "[TOTAL CONFIGS]",
    TOTAL_CONFIGS
)

print(
    "[PURPOSE] Audit only — final model selection değildir."
)


# ==========================================================
# 3. ŞEMA VE DENOMINATOR
# ==========================================================

schema_path = os.path.join(
    CONFIG_DIR,
    "schema_v4.json"
)


denom_path = os.path.join(
    PROCESSED_DIR,
    "selection_baseline_denominators_v4.json"
)


for path in [
    schema_path,
    denom_path
]:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Gerekli dosya yok:\n{path}"
        )


with open(
    schema_path,
    "r",
    encoding="utf-8"
) as f:

    schema = json.load(f)


with open(
    denom_path,
    "r",
    encoding="utf-8"
) as f:

    denominators = json.load(f)


ASSETS = schema[
    "data"
][
    "assets"
]


TARGETS = schema[
    "targets"
][
    "definition"
]


if ASSETS != [
    "BIST100",
    "USDTRY",
    "EURTRY",
    "GOLD"
]:

    raise RuntimeError(
        f"Asset sırası beklenenden farklı: {ASSETS}"
    )


if len(TARGETS) != 8:

    raise RuntimeError(
        f"Target sayısı 8 değil: {len(TARGETS)}"
    )


# ==========================================================
# 4. YARDIMCI FONKSİYONLAR
# ==========================================================

def set_seed(
    seed=42
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


    if hasattr(
        torch.backends,
        "cudnn"
    ):

        torch.backends.cudnn.deterministic = True

        torch.backends.cudnn.benchmark = False



def clone_state_to_cpu(
    model
):

    return {

        key:
            value
            .detach()
            .cpu()
            .clone()

        for (
            key,
            value
        )

        in model.state_dict().items()
    }



def pinball_torch(
    pred,
    true,
    tau=0.5
):

    diff = (
        true
        -
        pred
    )


    return torch.maximum(

        tau
        *
        diff,

        (
            tau
            -
            1.0
        )
        *
        diff

    ).mean()



def pinball_np(
    true,
    pred,
    tau=0.5
):

    diff = (
        true
        -
        pred
    )


    return float(

        np.maximum(

            tau
            *
            diff,

            (
                tau
                -
                1.0
            )
            *
            diff

        ).mean()
    )



def score_validation(
    y_true_raw,
    y_pred_raw
):

    rows = []

    ret_ratios = {}

    vol_ratios = {}


    # ------------------------------------------------------
    # RETURN
    # ------------------------------------------------------

    for (
        i,
        asset
    ) in enumerate(
        ASSETS
    ):

        mae = float(

            np.mean(

                np.abs(

                    y_true_raw[
                        :,
                        i
                    ]

                    -

                    y_pred_raw[
                        :,
                        i
                    ]
                )
            )
        )


        denom = float(

            denominators[

                "return_denominator"

            ][

                asset

            ][

                "value"

            ]
        )


        if denom <= 0:

            raise RuntimeError(

                f"{asset} return denominator pozitif değil."
            )


        ratio = (
            mae
            /
            denom
        )


        ret_ratios[
            asset
        ] = float(
            ratio
        )


        rows.append(

            {

                "task":
                    "return",

                "asset":
                    asset,

                "MAE":
                    mae,

                "PinballLoss_tau_0.5":
                    np.nan,

                "baseline_ratio":
                    ratio
            }
        )


    # ------------------------------------------------------
    # VOLATILITY
    # ------------------------------------------------------

    for (
        i,
        asset
    ) in enumerate(
        ASSETS
    ):

        col = (
            4
            +
            i
        )


        mae = float(

            np.mean(

                np.abs(

                    y_true_raw[
                        :,
                        col
                    ]

                    -

                    y_pred_raw[
                        :,
                        col
                    ]
                )
            )
        )


        pb = pinball_np(

            y_true_raw[
                :,
                col
            ],

            y_pred_raw[
                :,
                col
            ],

            TAU
        )


        denom = float(

            denominators[

                "volatility_denominator"

            ][

                asset

            ][

                "value"

            ]
        )


        if denom <= 0:

            raise RuntimeError(

                f"{asset} volatility denominator pozitif değil."
            )


        ratio = (
            pb
            /
            denom
        )


        vol_ratios[
            asset
        ] = float(
            ratio
        )


        rows.append(

            {

                "task":
                    "volatility",

                "asset":
                    asset,

                "MAE":
                    mae,

                "PinballLoss_tau_0.5":
                    pb,

                "baseline_ratio":
                    ratio
            }
        )


    avg_ret = float(

        np.mean(

            list(
                ret_ratios.values()
            )
        )
    )


    avg_vol = float(

        np.mean(

            list(
                vol_ratios.values()
            )
        )
    )


    score = float(

        0.5
        *
        avg_ret

        +

        0.5
        *
        avg_vol
    )


    return (

        pd.DataFrame(
            rows
        ),

        {

            "return_ratios":
                ret_ratios,

            "vol_ratios":
                vol_ratios,

            "avg_return_ratio":
                avg_ret,

            "avg_vol_ratio":
                avg_vol,

            "validation_score":
                score,

            "lower_is_better":
                True
        }
    )


# ==========================================================
# 5. ORTAK ENCODER / HEAD ÜRETİCİLERİ
# ==========================================================

def make_encoder(

    d_model,

    n_head,

    n_layers,

    d_ff,

    dropout
):

    layer = nn.TransformerEncoderLayer(

        d_model=d_model,

        nhead=n_head,

        dim_feedforward=d_ff,

        dropout=dropout,

        activation="gelu",

        batch_first=True
    )


    return nn.TransformerEncoder(

        layer,

        num_layers=n_layers
    )



def make_head(

    d_model,

    dropout
):

    return nn.Sequential(

        nn.Linear(
            d_model,
            d_model
        ),

        nn.GELU(),

        nn.Dropout(
            dropout
        ),

        nn.Linear(
            d_model,
            4
        )
    )


# ==========================================================
# 6. FULL SHARING
# ==========================================================

class FullSharingMTL(
    nn.Module
):

    def __init__(

        self,

        n_features,

        lookback
    ):

        super().__init__()


        self.input_projection = nn.Linear(

            n_features,

            D_MODEL
        )


        self.positional_embedding = nn.Parameter(

            torch.zeros(

                1,

                lookback,

                D_MODEL
            )
        )


        self.encoder = make_encoder(

            D_MODEL,

            N_HEAD,

            N_LAYERS,

            D_FF,

            DROPOUT
        )


        self.norm = nn.LayerNorm(

            D_MODEL
        )


        self.return_head = make_head(

            D_MODEL,

            DROPOUT
        )


        self.vol_head = make_head(

            D_MODEL,

            DROPOUT
        )


    def forward(
        self,
        x
    ):

        h = self.input_projection(
            x
        )


        h = (

            h

            +

            self.positional_embedding[

                :,

                :h.size(1),

                :
            ]
        )


        h = self.encoder(
            h
        )


        h = self.norm(

            h[
                :,
                -1,
                :
            ]
        )


        return torch.cat(

            [

                self.return_head(
                    h
                ),

                self.vol_head(
                    h
                )
            ],

            dim=1
        )


    def pcgrad_groups(
        self
    ):

        shared = (

            list(
                self.input_projection.parameters()
            )

            +

            [
                self.positional_embedding
            ]

            +

            list(
                self.encoder.parameters()
            )

            +

            list(
                self.norm.parameters()
            )
        )


        return {

            "shared":
                shared,

            "return_specific":
                list(
                    self.return_head.parameters()
                ),

            "vol_specific":
                list(
                    self.vol_head.parameters()
                )
        }


# ==========================================================
# 7. NO SHARING
# ==========================================================

class NoSharing(
    nn.Module
):

    def __init__(

        self,

        n_features,

        lookback
    ):

        super().__init__()


        # RETURN BRANCH

        self.ret_projection = nn.Linear(

            n_features,

            D_MODEL
        )


        self.ret_positional = nn.Parameter(

            torch.zeros(

                1,

                lookback,

                D_MODEL
            )
        )


        self.ret_encoder = make_encoder(

            D_MODEL,

            N_HEAD,

            N_LAYERS,

            D_FF,

            DROPOUT
        )


        self.ret_norm = nn.LayerNorm(

            D_MODEL
        )


        self.return_head = make_head(

            D_MODEL,

            DROPOUT
        )


        # VOLATILITY BRANCH

        self.vol_projection = nn.Linear(

            n_features,

            D_MODEL
        )


        self.vol_positional = nn.Parameter(

            torch.zeros(

                1,

                lookback,

                D_MODEL
            )
        )


        self.vol_encoder = make_encoder(

            D_MODEL,

            N_HEAD,

            N_LAYERS,

            D_FF,

            DROPOUT
        )


        self.vol_norm = nn.LayerNorm(

            D_MODEL
        )


        self.vol_head = make_head(

            D_MODEL,

            DROPOUT
        )


    def forward(
        self,
        x
    ):

        hr = self.ret_projection(
            x
        )


        hr = (

            hr

            +

            self.ret_positional[

                :,

                :hr.size(1),

                :
            ]
        )


        hr = self.ret_encoder(
            hr
        )


        hr = self.ret_norm(

            hr[
                :,
                -1,
                :
            ]
        )


        hv = self.vol_projection(
            x
        )


        hv = (

            hv

            +

            self.vol_positional[

                :,

                :hv.size(1),

                :
            ]
        )


        hv = self.vol_encoder(
            hv
        )


        hv = self.vol_norm(

            hv[
                :,
                -1,
                :
            ]
        )


        return torch.cat(

            [

                self.return_head(
                    hr
                ),

                self.vol_head(
                    hv
                )
            ],

            dim=1
        )


    def pcgrad_groups(
        self
    ):

        ret = (

            list(
                self.ret_projection.parameters()
            )

            +

            [
                self.ret_positional
            ]

            +

            list(
                self.ret_encoder.parameters()
            )

            +

            list(
                self.ret_norm.parameters()
            )

            +

            list(
                self.return_head.parameters()
            )
        )


        vol = (

            list(
                self.vol_projection.parameters()
            )

            +

            [
                self.vol_positional
            ]

            +

            list(
                self.vol_encoder.parameters()
            )

            +

            list(
                self.vol_norm.parameters()
            )

            +

            list(
                self.vol_head.parameters()
            )
        )


        return {

            "shared":
                [],

            "return_specific":
                ret,

            "vol_specific":
                vol
        }


# ==========================================================
# 8. MODEL FACTORY
# ==========================================================

def build_model(

    architecture,

    n_features,

    lookback
):

    if architecture == "FullSharingMTL":

        return FullSharingMTL(

            n_features,

            lookback
        )


    if architecture == "NoSharing":

        return NoSharing(

            n_features,

            lookback
        )


    raise ValueError(

        f"Bilinmeyen mimari: {architecture}"
    )


# ==========================================================
# 9. LOSS
# ==========================================================

def task_losses(

    pred,

    true
):

    ret_loss = nn.functional.mse_loss(

        pred[
            :,
            :4
        ],

        true[
            :,
            :4
        ]
    )


    vol_loss = pinball_torch(

        pred[
            :,
            4:
        ],

        true[
            :,
            4:
        ],

        TAU
    )


    total = (

        LAMBDA_RETURN
        *
        ret_loss

        +

        (
            1.0
            -
            LAMBDA_RETURN
        )
        *
        vol_loss
    )


    return (

        total,

        ret_loss,

        vol_loss
    )


# ==========================================================
# 10. PCGRAD — SHARED PARAMETER ONLY
# ==========================================================

def grads_or_zeros(

    loss,

    params,

    retain_graph
):

    if not params:

        return []


    grads = torch.autograd.grad(

        loss,

        params,

        retain_graph=retain_graph,

        allow_unused=True
    )


    return [

        torch.zeros_like(
            parameter
        )

        if gradient is None

        else gradient

        for (
            parameter,
            gradient
        )

        in zip(
            params,
            grads
        )
    ]



def flatten_grads(
    grads
):

    if not grads:

        return None


    return torch.cat(

        [

            gradient.reshape(
                -1
            )

            for gradient in grads
        ]
    )



def assign_flat_grad(

    params,

    flat_grad
):

    offset = 0


    for parameter in params:

        n = parameter.numel()


        parameter.grad = (

            flat_grad[

                offset
                :
                offset + n

            ]

            .view_as(
                parameter
            )

            .detach()

            .clone()
        )


        offset += n


    if offset != flat_grad.numel():

        raise RuntimeError(

            "Flat gradient ile parameter boyutları uyuşmuyor."
        )



def pcgrad_backward_equal_weight(

    model,

    ret_loss,

    vol_loss
):

    groups = model.pcgrad_groups()


    shared = groups[
        "shared"
    ]


    ret_params = groups[
        "return_specific"
    ]


    vol_params = groups[
        "vol_specific"
    ]


    model.zero_grad(
        set_to_none=True
    )


    conflict = False

    shared_dot = None


    # ------------------------------------------------------
    # SHARED PARAMETRELER
    # ------------------------------------------------------

    if shared:

        g_ret = flatten_grads(

            grads_or_zeros(

                ret_loss,

                shared,

                retain_graph=True
            )
        )


        g_vol = flatten_grads(

            grads_or_zeros(

                vol_loss,

                shared,

                retain_graph=True
            )
        )


        dot = torch.dot(

            g_ret,

            g_vol
        )


        shared_dot = float(

            dot
            .detach()
            .cpu()
            .item()
        )


        ret_norm_sq = torch.dot(

            g_ret,

            g_ret
        )


        vol_norm_sq = torch.dot(

            g_vol,

            g_vol
        )


        eps = torch.finfo(

            g_ret.dtype

        ).eps


        if dot < 0:

            conflict = True


            g_ret_proj = (

                g_ret

                -

                dot
                /
                (
                    vol_norm_sq
                    +
                    eps
                )

                *
                g_vol
            )


            g_vol_proj = (

                g_vol

                -

                dot
                /
                (
                    ret_norm_sq
                    +
                    eps
                )

                *
                g_ret
            )


        else:

            g_ret_proj = g_ret

            g_vol_proj = g_vol


        assign_flat_grad(

            shared,

            0.5
            *
            (
                g_ret_proj
                +
                g_vol_proj
            )
        )


    # ------------------------------------------------------
    # TASK-SPECIFIC PARAMETRELER
    # ------------------------------------------------------

    ret_grads = grads_or_zeros(

        ret_loss,

        ret_params,

        retain_graph=True
    )


    vol_grads = grads_or_zeros(

        vol_loss,

        vol_params,

        retain_graph=False
    )


    for (
        parameter,
        gradient
    ) in zip(

        ret_params,

        ret_grads
    ):

        parameter.grad = (

            0.5
            *
            gradient

        ).detach().clone()


    for (
        parameter,
        gradient
    ) in zip(

        vol_params,

        vol_grads
    ):

        parameter.grad = (

            0.5
            *
            gradient

        ).detach().clone()


    return {

        "conflict":
            conflict,

        "shared_dot":
            shared_dot,

        "shared_param_count":
            int(

                sum(

                    parameter.numel()

                    for parameter in shared
                )
            )
    }


# ==========================================================
# 11. DATALOADER
# ==========================================================

def make_loaders(

    X_train,

    y_train,

    X_val,

    y_val
):

    train_ds = TensorDataset(

        torch.tensor(

            X_train,

            dtype=torch.float32
        ),

        torch.tensor(

            y_train,

            dtype=torch.float32
        )
    )


    val_ds = TensorDataset(

        torch.tensor(

            X_val,

            dtype=torch.float32
        ),

        torch.tensor(

            y_val,

            dtype=torch.float32
        )
    )


    generator = torch.Generator()


    generator.manual_seed(

        SEED
    )


    train_loader = DataLoader(

        train_ds,

        batch_size=BATCH_SIZE,

        shuffle=True,

        drop_last=False,

        generator=generator
    )


    val_loader = DataLoader(

        val_ds,

        batch_size=BATCH_SIZE,

        shuffle=False,

        drop_last=False
    )


    return (

        train_loader,

        val_loader
    )


# ==========================================================
# 12. TRAIN ONE EPOCH
# ==========================================================

def train_one_epoch(

    model,

    loader,

    optimizer,

    loss_strategy
):

    model.train()


    sums = {

        "total":
            0.0,

        "ret":
            0.0,

        "vol":
            0.0
    }


    n_obs = 0

    conflicts = 0

    pc_batches = 0

    dots = []


    for (
        X_batch,
        y_batch
    ) in loader:


        X_batch = X_batch.to(

            DEVICE
        )


        y_batch = y_batch.to(

            DEVICE
        )


        optimizer.zero_grad(

            set_to_none=True
        )


        pred = model(

            X_batch
        )


        (
            total,

            ret_loss,

            vol_loss

        ) = task_losses(

            pred,

            y_batch
        )


        if not torch.isfinite(

            total
        ):

            raise RuntimeError(

                "Training loss finite değil."
            )


        if loss_strategy == "FixedLambda_0.5":

            total.backward()


        elif loss_strategy == "PCGrad":

            info = pcgrad_backward_equal_weight(

                model,

                ret_loss,

                vol_loss
            )


            pc_batches += 1


            conflicts += int(

                info[
                    "conflict"
                ]
            )


            if info[
                "shared_dot"
            ] is not None:

                dots.append(

                    info[
                        "shared_dot"
                    ]
                )


        else:

            raise ValueError(

                f"Bilinmeyen loss strategy: {loss_strategy}"
            )


        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            GRAD_CLIP
        )


        optimizer.step()


        batch_size = X_batch.size(
            0
        )


        sums[
            "total"
        ] += (

            total.item()

            *
            batch_size
        )


        sums[
            "ret"
        ] += (

            ret_loss.item()

            *
            batch_size
        )


        sums[
            "vol"
        ] += (

            vol_loss.item()

            *
            batch_size
        )


        n_obs += batch_size


    return {

        "loss":
            sums[
                "total"
            ]
            /
            n_obs,

        "return_loss":
            sums[
                "ret"
            ]
            /
            n_obs,

        "vol_loss":
            sums[
                "vol"
            ]
            /
            n_obs,

        "pcgrad_conflict_batches":
            conflicts,

        "pcgrad_total_batches":
            pc_batches,

        "pcgrad_conflict_rate":
            (

                conflicts
                /
                pc_batches

                if pc_batches

                else 0.0
            ),

        "pcgrad_mean_shared_dot":
            (

                float(
                    np.mean(
                        dots
                    )
                )

                if dots

                else None
            )
    }


# ==========================================================
# 13. VALIDATION PREDICTION
# ==========================================================

@torch.no_grad()

def predict_scaled(

    model,

    loader
):

    model.eval()


    preds = []

    trues = []


    for (
        X_batch,
        y_batch
    ) in loader:


        pred = model(

            X_batch.to(
                DEVICE
            )
        )


        preds.append(

            pred
            .detach()
            .cpu()
            .numpy()
        )


        trues.append(

            y_batch.numpy()
        )


    return (

        np.vstack(
            preds
        ),

        np.vstack(
            trues
        )
    )


# ==========================================================
# 14. LOOKBACK VERİSİNİ YÜKLE
# TEST DİZİLERİ YÜKLENMEZ.
# ==========================================================

def load_data(

    lookback
):

    lb_dir = os.path.join(

        SEQUENCE_DIR,

        FEATURE_SET,

        f"lb{lookback}"
    )


    scaler_path = os.path.join(

        SEQUENCE_DIR,

        FEATURE_SET,

        "scalers.pkl"
    )


    files = {

        "X_train":
            os.path.join(
                lb_dir,
                "X_train.npy"
            ),

        "y_train":
            os.path.join(
                lb_dir,
                "y_train.npy"
            ),

        "X_val":
            os.path.join(
                lb_dir,
                "X_val.npy"
            ),

        "y_val":
            os.path.join(
                lb_dir,
                "y_val.npy"
            ),

        "y_val_raw":
            os.path.join(
                lb_dir,
                "y_val_raw.npy"
            )
    }


    for path in (

        list(
            files.values()
        )

        +

        [
            scaler_path
        ]
    ):

        if not os.path.exists(
            path
        ):

            raise FileNotFoundError(

                f"Gerekli dosya yok:\n{path}"
            )


    arrays = {

        key:
            np.load(
                path
            )

        for (
            key,
            path
        ) in files.items()
    }


    with open(

        scaler_path,

        "rb"

    ) as f:

        y_scaler = pickle.load(
            f
        )[
            "y_scaler"
        ]


    inverse_check = (

        y_scaler.inverse_transform(

            arrays[
                "y_val"
            ]
        )
    )


    max_diff = float(

        np.max(

            np.abs(

                inverse_check

                -

                arrays[
                    "y_val_raw"
                ]
            )
        )
    )


    if max_diff > 1e-5:

        raise RuntimeError(

            f"Inverse-scale kontrolü geçmedi: {max_diff}"
        )


    return (

        arrays[
            "X_train"
        ],

        arrays[
            "y_train"
        ],

        arrays[
            "X_val"
        ],

        arrays[
            "y_val"
        ],

        arrays[
            "y_val_raw"
        ],

        y_scaler,

        max_diff
    )


# ==========================================================
# 15. ÇIKTI DOSYALARI
# ==========================================================

results_path = os.path.join(

    RESULTS_DIR,

    "mini_grid_results_v4.csv"
)


ranked_path = os.path.join(

    RESULTS_DIR,

    "mini_grid_results_ranked_v4.csv"
)


history_path = os.path.join(

    RESULTS_DIR,

    "mini_grid_history_v4.csv"
)


pcgrad_audit_path = os.path.join(

    RESULTS_DIR,

    "mini_grid_pcgrad_audit_v4.csv"
)


summary_path = os.path.join(

    RESULTS_DIR,

    "mini_grid_summary_v4.json"
)


# ==========================================================
# 16. RESUME
# ==========================================================

if os.path.exists(
    results_path
):

    existing = pd.read_csv(

        results_path
    )

else:

    existing = pd.DataFrame()


success_keys = set()


if not existing.empty:

    done = existing[

        existing[
            "status"
        ]

        ==

        "success"
    ]


    success_keys = {

        (

            str(
                row[
                    "architecture"
                ]
            ),

            str(
                row[
                    "loss_strategy"
                ]
            ),

            int(
                row[
                    "lookback"
                ]
            )
        )

        for (
            _,
            row
        ) in done.iterrows()
    }


# ==========================================================
# 17. 8 KONFİGÜRASYON
# ==========================================================

configs = []


for (

    config_id,

    (
        architecture,
        loss_strategy,
        lookback
    )

) in enumerate(

    itertools.product(

        ARCHITECTURES,

        LOSS_STRATEGIES,

        LOOKBACKS
    ),

    start=1
):

    configs.append(

        {

            "config_id":
                config_id,

            "architecture":
                architecture,

            "loss_strategy":
                loss_strategy,

            "lookback":
                lookback
        }
    )


new_results = []

new_history = []


# ==========================================================
# 18. MINI-GRID
# ==========================================================

for config in configs:


    config_id = config[
        "config_id"
    ]


    architecture = config[
        "architecture"
    ]


    loss_strategy = config[
        "loss_strategy"
    ]


    lookback = config[
        "lookback"
    ]


    key = (

        architecture,

        loss_strategy,

        lookback
    )


    print(
        "\n"
        +
        "=" * 80
    )


    print(

        f"CONFIG {config_id}/{TOTAL_CONFIGS} | "

        f"{architecture} | "

        f"{loss_strategy} | "

        f"lb={lookback}"
    )


    print(
        "=" * 80
    )


    if key in success_keys:

        print(

            "[SKIP] Daha önce success."
        )

        continue


    try:

        set_seed(
            SEED
        )


        (

            X_train,

            y_train,

            X_val,

            y_val,

            y_val_raw,

            y_scaler,

            max_diff

        ) = load_data(

            lookback
        )


        (

            train_loader,

            val_loader

        ) = make_loaders(

            X_train,

            y_train,

            X_val,

            y_val
        )


        model = build_model(

            architecture,

            X_train.shape[
                2
            ],

            lookback

        ).to(
            DEVICE
        )


        parameter_count = int(

            sum(

                parameter.numel()

                for parameter in model.parameters()

                if parameter.requires_grad
            )
        )


        shared_count = int(

            sum(

                parameter.numel()

                for parameter in (

                    model
                    .pcgrad_groups()
                    [
                        "shared"
                    ]
                )
            )
        )


        optimizer = torch.optim.AdamW(

            model.parameters(),

            lr=LR,

            weight_decay=WEIGHT_DECAY
        )


        best_score = float(
            "inf"
        )


        best_epoch = None

        best_score_obj = None

        best_metrics = None

        best_state = None


        total_conflicts = 0

        total_pc_batches = 0


        # --------------------------------------------------
        # 5 EPOCH
        # --------------------------------------------------

        for epoch in range(

            1,

            EPOCHS
            +
            1
        ):


            train_info = train_one_epoch(

                model,

                train_loader,

                optimizer,

                loss_strategy
            )


            (

                pred_scaled,

                true_scaled

            ) = predict_scaled(

                model,

                val_loader
            )


            pred_raw = (

                y_scaler.inverse_transform(

                    pred_scaled
                )
            )


            true_raw_check = (

                y_scaler.inverse_transform(

                    true_scaled
                )
            )


            true_diff = float(

                np.max(

                    np.abs(

                        true_raw_check

                        -

                        y_val_raw
                    )
                )
            )


            if true_diff > 1e-5:

                raise RuntimeError(

                    "Validation true inverse-scale uyuşmuyor: "
                    f"{true_diff}"
                )


            (

                metrics_df,

                score_obj

            ) = score_validation(

                y_val_raw,

                pred_raw
            )


            score = score_obj[

                "validation_score"
            ]


            total_conflicts += (

                train_info[

                    "pcgrad_conflict_batches"
                ]
            )


            total_pc_batches += (

                train_info[

                    "pcgrad_total_batches"
                ]
            )


            new_history.append(

                {

                    "config_id":
                        config_id,

                    "architecture":
                        architecture,

                    "loss_strategy":
                        loss_strategy,

                    "lookback":
                        lookback,

                    "epoch":
                        epoch,

                    "train_loss":
                        train_info[
                            "loss"
                        ],

                    "train_return_loss":
                        train_info[
                            "return_loss"
                        ],

                    "train_vol_loss":
                        train_info[
                            "vol_loss"
                        ],

                    "validation_score":
                        score,

                    "avg_return_ratio":
                        score_obj[
                            "avg_return_ratio"
                        ],

                    "avg_vol_ratio":
                        score_obj[
                            "avg_vol_ratio"
                        ],

                    "pcgrad_conflict_batches":
                        train_info[
                            "pcgrad_conflict_batches"
                        ],

                    "pcgrad_total_batches":
                        train_info[
                            "pcgrad_total_batches"
                        ],

                    "pcgrad_conflict_rate":
                        train_info[
                            "pcgrad_conflict_rate"
                        ],

                    "pcgrad_mean_shared_dot":
                        train_info[
                            "pcgrad_mean_shared_dot"
                        ]
                }
            )


            print(

                f"Epoch {epoch:02d} | "

                f"train="
                f"{train_info['loss']:.6f} | "

                f"score="
                f"{score:.6f} | "

                f"ret="
                f"{score_obj['avg_return_ratio']:.6f} | "

                f"vol="
                f"{score_obj['avg_vol_ratio']:.6f}"
            )


            if score < best_score:


                best_score = float(
                    score
                )


                best_epoch = int(
                    epoch
                )


                best_score_obj = copy.deepcopy(

                    score_obj
                )


                best_metrics = metrics_df.copy()


                best_state = clone_state_to_cpu(

                    model
                )


        if best_state is None:

            raise RuntimeError(

                "Best checkpoint oluşmadı."
            )


        # --------------------------------------------------
        # CHECKPOINT
        # --------------------------------------------------

        model_path = os.path.join(

            MODEL_DIR,

            (
                f"mini_cfg{config_id:02d}_"
                f"{architecture}_"
                f"{loss_strategy}_"
                f"lb{lookback}_v4.pt"
            )
        )


        torch.save(

            {

                "model_state_dict":
                    best_state,

                "config": {

                    "config_id":
                        config_id,

                    "architecture":
                        architecture,

                    "loss_strategy":
                        loss_strategy,

                    "lookback":
                        lookback,

                    "feature_set":
                        FEATURE_SET,

                    "size":
                        "medium",

                    "d_model":
                        D_MODEL,

                    "n_head":
                        N_HEAD,

                    "n_layers":
                        N_LAYERS,

                    "d_ff":
                        D_FF,

                    "dropout":
                        DROPOUT,

                    "epochs":
                        EPOCHS,

                    "seed":
                        SEED,

                    "purpose":
                        "mini_grid_audit_only"
                },

                "best_epoch":
                    best_epoch,

                "best_validation_score":
                    best_score,

                "parameter_count":
                    parameter_count,

                "shared_param_count":
                    shared_count,

                "test_arrays_loaded":
                    False,

                "test_metrics_computed":
                    False
            },

            model_path
        )


        # --------------------------------------------------
        # BEST METRICS
        # --------------------------------------------------

        metrics_path = os.path.join(

            RESULTS_DIR,

            f"mini_cfg{config_id:02d}_best_metrics_v4.csv"
        )


        best_metrics.to_csv(

            metrics_path,

            index=False
        )


        # --------------------------------------------------
        # RESULT ROW
        # --------------------------------------------------

        row = {

            "config_id":
                config_id,

            "status":
                "success",

            "architecture":
                architecture,

            "loss_strategy":
                loss_strategy,

            "lookback":
                lookback,

            "feature_set":
                FEATURE_SET,

            "size":
                "medium",

            "seed":
                SEED,

            "epochs":
                EPOCHS,

            "best_epoch":
                best_epoch,

            "validation_score":
                best_score_obj[
                    "validation_score"
                ],

            "avg_return_ratio":
                best_score_obj[
                    "avg_return_ratio"
                ],

            "avg_vol_ratio":
                best_score_obj[
                    "avg_vol_ratio"
                ],

            "parameter_count":
                parameter_count,

            "shared_param_count":
                shared_count,

            "pcgrad_conflict_batches_total":
                total_conflicts,

            "pcgrad_batches_total":
                total_pc_batches,

            "pcgrad_conflict_rate_total":
                (

                    total_conflicts
                    /
                    total_pc_batches

                    if total_pc_batches

                    else 0.0
                ),

            "max_inverse_diff":
                max_diff,

            "model_checkpoint":
                model_path,

            "metrics_file":
                metrics_path,

            "test_arrays_loaded":
                False,

            "test_metrics_computed":
                False
        }


        for asset in ASSETS:


            row[

                f"return_ratio_{asset}"

            ] = best_score_obj[

                "return_ratios"

            ][

                asset

            ]


            row[

                f"vol_ratio_{asset}"

            ] = best_score_obj[

                "vol_ratios"

            ][

                asset

            ]


        new_results.append(
            row
        )


        pd.concat(

            [

                existing,

                pd.DataFrame(
                    new_results
                )
            ],

            ignore_index=True

        ).to_csv(

            results_path,

            index=False
        )


        print(

            f"[SUCCESS] best_epoch={best_epoch} | "
            f"best_score={best_score:.6f}"
        )


        del model

        del optimizer


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    except Exception as error:


        new_results.append(

            {

                "config_id":
                    config_id,

                "status":
                    "error",

                "architecture":
                    architecture,

                "loss_strategy":
                    loss_strategy,

                "lookback":
                    lookback,

                "feature_set":
                    FEATURE_SET,

                "size":
                    "medium",

                "seed":
                    SEED,

                "epochs":
                    EPOCHS,

                "error":
                    repr(
                        error
                    ),

                "test_arrays_loaded":
                    False,

                "test_metrics_computed":
                    False
            }
        )


        pd.concat(

            [

                existing,

                pd.DataFrame(
                    new_results
                )
            ],

            ignore_index=True

        ).to_csv(

            results_path,

            index=False
        )


        print(

            "[ERROR]",

            repr(
                error
            )
        )


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


        raise


# ==========================================================
# 19. HISTORY KAYDET
# ==========================================================

if new_history:


    new_history_df = pd.DataFrame(

        new_history
    )


    if os.path.exists(
        history_path
    ):

        old_history = pd.read_csv(

            history_path
        )


        history_df = pd.concat(

            [

                old_history,

                new_history_df
            ],

            ignore_index=True
        )


    else:

        history_df = new_history_df


    history_df.to_csv(

        history_path,

        index=False
    )


# ==========================================================
# 20. RANKING
# ==========================================================

final_results = pd.read_csv(

    results_path
)


success = final_results[

    final_results[
        "status"
    ]

    ==

    "success"

].copy()


success = (

    success

    .sort_values(

        [

            "validation_score",

            "config_id"
        ],

        ascending=[

            True,

            True
        ]
    )

    .reset_index(
        drop=True
    )
)


success[
    "rank"
] = np.arange(

    1,

    len(success)
    +
    1
)


success.to_csv(

    ranked_path,

    index=False
)


# ==========================================================
# 21. PCGRAD EŞDEĞERLİK AUDIT
# ==========================================================

audit_rows = []


for lookback in LOOKBACKS:


    rows = success[

        (

            success[
                "architecture"
            ]

            ==

            "NoSharing"
        )

        &

        (

            success[
                "lookback"
            ]

            ==

            lookback
        )
    ]


    fixed_row = rows[

        rows[
            "loss_strategy"
        ]

        ==

        "FixedLambda_0.5"
    ]


    pcgrad_row = rows[

        rows[
            "loss_strategy"
        ]

        ==

        "PCGrad"
    ]


    if (

        len(
            fixed_row
        )
        ==
        1

        and

        len(
            pcgrad_row
        )
        ==
        1
    ):


        fixed_score = float(

            fixed_row
            .iloc[0]
            [
                "validation_score"
            ]
        )


        pcgrad_score = float(

            pcgrad_row
            .iloc[0]
            [
                "validation_score"
            ]
        )


        difference = abs(

            fixed_score

            -

            pcgrad_score
        )


        audit_rows.append(

            {

                "architecture":
                    "NoSharing",

                "lookback":
                    lookback,

                "fixedlambda_score":
                    fixed_score,

                "pcgrad_score":
                    pcgrad_score,

                "absolute_score_difference":
                    difference,

                "numerically_equal_tol_1e_10":
                    bool(

                        difference
                        <=
                        1e-10
                    ),

                "interpretation":
                    (
                        "No shared parameters: "
                        "equal-weight PCGrad should reduce "
                        "to FixedLambda_0.5."
                    )
            }
        )


audit_df = pd.DataFrame(

    audit_rows
)


audit_df.to_csv(

    pcgrad_audit_path,

    index=False
)


# ==========================================================
# 22. SUMMARY
# ==========================================================

summary = {

    "project_version":
        "v4_repro",

    "created_at":
        datetime.now().isoformat(),

    "script":
        "05a_mini_grid_v4.py",

    "purpose":
        (
            "Audit only: FullSharingMTL/NoSharing, "
            "FixedLambda_0.5/PCGrad, lookback 10/30, "
            "checkpointing, resume, validation score ve "
            "NoSharing eşdeğerlik davranışını doğrular."
        ),

    "feature_set":
        FEATURE_SET,

    "size":
        "medium",

    "seed":
        SEED,

    "epochs":
        EPOCHS,

    "total_configs_expected":
        TOTAL_CONFIGS,

    "success_configs":
        int(
            len(
                success
            )
        ),

    "all_success":
        bool(

            len(
                success
            )

            ==

            TOTAL_CONFIGS
        ),

    "test_arrays_loaded":
        False,

    "test_metrics_computed":
        False,

    "pcgrad_method":
        (
            "Equal-weight shared-parameter PCGrad: "
            "projeksiyon yalnızca shared parametrelerde, "
            "iki görev gradienti çatıştığında uygulanır. "
            "Task-specific parametreler kendi görev gradientini "
            "0.5 katsayısıyla alır. NoSharing durumunda PCGrad, "
            "FixedLambda_0.5'e indirgenir."
        )
}


with open(

    summary_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        summary,

        f,

        ensure_ascii=False,

        indent=2
    )


# ==========================================================
# 23. SON ÇIKTI
# ==========================================================

print(
    "\n"
    +
    "=" * 80
)


print(
    "MINI-GRID RANKING"
)


print(
    "=" * 80
)


display_columns = [

    "rank",

    "config_id",

    "architecture",

    "loss_strategy",

    "lookback",

    "best_epoch",

    "validation_score",

    "avg_return_ratio",

    "avg_vol_ratio",

    "parameter_count",

    "shared_param_count",

    "pcgrad_conflict_rate_total"
]


print(

    success[

        display_columns

    ].to_string(

        index=False
    )
)


print(
    "\n"
    +
    "=" * 80
)


print(
    "PCGRAD AUDIT — NoSharing vs FixedLambda_0.5"
)


print(
    "=" * 80
)


if len(
    audit_df
) > 0:

    print(

        audit_df.to_string(

            index=False
        )
    )


else:

    print(

        "Karşılaştırılabilir çift yok."
    )


if len(
    success
) != TOTAL_CONFIGS:

    raise RuntimeError(

        f"Mini-grid tamamlanmadı: "
        f"{len(success)}/{TOTAL_CONFIGS} success"
    )


print(
    "\n"
    +
    "=" * 80
)


print(
    "05a_mini_grid_v4.py BAŞARIYLA TAMAMLANDI"
)


print(
    "=" * 80
)


print(
    "✅ 8/8 config success."
)

print(
    "✅ FullSharingMTL çalıştı."
)

print(
    "✅ NoSharing çalıştı."
)

print(
    "✅ FixedLambda_0.5 çalıştı."
)

print(
    "✅ Shared-parameter PCGrad çalıştı."
)

print(
    "✅ Best checkpoint gerçek CPU clone ile saklandı."
)

print(
    "✅ Resume yalnızca success configleri atlıyor."
)

print(
    "✅ ValidationScore yalnızca validation ile hesaplandı."
)

print(
    "✅ Test dizileri yüklenmedi."
)

print(
    "✅ Test metriği hesaplanmadı."
)

print(
    "✅ NoSharing PCGrad eşdeğerliği ayrıca audit edildi."
)

print(
    "=" * 80
)
'''


# ==========================================================
# SCRIPT'İ KAYDET
# ==========================================================

with open(
    script_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        script_code
    )


print(
    "05a_mini_grid_v4.py oluşturuldu:"
)

print(
    script_path
)


# ==========================================================
# SYNTAX KONTROLÜ
# ==========================================================

with open(
    script_path,
    "r",
    encoding="utf-8"
) as f:

    code_to_run = f.read()


compile(
    code_to_run,
    script_path,
    "exec"
)


print(
    "\n✅ Syntax kontrolü geçti."
)


# ==========================================================
# ÇALIŞTIR
# ==========================================================

print(
    "\nMini-grid çalıştırılıyor...\n"
)


exec(
    compile(
        code_to_run,
        script_path,
        "exec"
    )
)

In [ ]:
# ==========================================================
# 05_grid_search_v4.py — SYNTAX KONTROLÜ + ÇALIŞTIR
# ==========================================================

SCRIPT_PATH = (
    "/content/drive/MyDrive/"
    "tez_transformer_v4_repro/"
    "scripts/"
    "05_grid_search_v4.py"
)

# Script'i oku
with open(
    SCRIPT_PATH,
    "r",
    encoding="utf-8"
) as f:
    code_to_run = f.read()

# Önce syntax kontrolü
compile(
    code_to_run,
    SCRIPT_PATH,
    "exec"
)

print("✅ Syntax kontrolü geçti.")
print("\n05_grid_search_v4.py çalıştırılıyor...\n")

# Çalıştır
exec(
    compile(
        code_to_run,
        SCRIPT_PATH,
        "exec"
    )
)

In [ ]:
import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

ranked_path = os.path.join(
    BASE_DIR,
    "results",
    "grid_search",
    "grid_results_ranked_v4.csv"
)

df = pd.read_csv(ranked_path)

# En iyi validation konfigürasyonu
best = df.iloc[0]

print("=" * 90)
print("EN İYİ V4 GRID MODELİ — VARLIK × GÖREV BAZINDA DETAY")
print("=" * 90)

print(f"Mimari        : {best['architecture']}")
print(f"Loss          : {best['loss_strategy']}")
print(f"Lookback      : {int(best['lookback'])}")
print(f"Boyut         : {best['size']}")
print(f"Feature set   : {best['feature_set']}")
print(f"Best epoch    : {int(best['best_epoch'])}")
print(f"ValidationScore: {best['validation_score']:.9f}")

assets = ["BIST100", "USDTRY", "EURTRY", "GOLD"]

rows = []

for asset in assets:
    return_ratio = float(best[f"{asset}_return_ratio"])
    vol_ratio = float(best[f"{asset}_vol_ratio"])

    rows.append({
        "Varlık": asset,
        "ReturnRatio": return_ratio,
        "Return yorumu": (
            "Model daha iyi"
            if return_ratio < 1
            else "Baseline daha iyi"
        ),
        "VolRatio": vol_ratio,
        "Vol yorumu": (
            "Model daha iyi"
            if vol_ratio < 1
            else "Baseline daha iyi"
        )
    })

detail_df = pd.DataFrame(rows)

print("\n" + "=" * 90)
print("8 AYRI PERFORMANS ORANI")
print("=" * 90)

print(detail_df.to_string(index=False))

# Manuel ortalama kontrolü
avg_return_manual = detail_df["ReturnRatio"].mean()
avg_vol_manual = detail_df["VolRatio"].mean()

validation_score_manual = (
    0.5 * avg_return_manual
    +
    0.5 * avg_vol_manual
)

max_ratio = max(
    detail_df["ReturnRatio"].max(),
    detail_df["VolRatio"].max()
)

print("\n" + "=" * 90)
print("MANUEL FORMÜL KONTROLÜ")
print("=" * 90)

print(f"AvgReturnRatio manuel       : {avg_return_manual:.9f}")
print(f"Dosyadaki AvgReturnRatio    : {best['avg_return_ratio']:.9f}")

print(f"\nAvgVolRatio manuel          : {avg_vol_manual:.9f}")
print(f"Dosyadaki AvgVolRatio       : {best['avg_vol_ratio']:.9f}")

print(f"\nValidationScore manuel      : {validation_score_manual:.9f}")
print(f"Dosyadaki ValidationScore   : {best['validation_score']:.9f}")

print(f"\nEn kötü tekil oran          : {max_ratio:.9f}")
print(f"Catastrophic max ratio      : {best['catastrophic_max_ratio']:.9f}")

# Tutarlılık kontrolleri
assert abs(
    avg_return_manual - best["avg_return_ratio"]
) < 1e-10

assert abs(
    avg_vol_manual - best["avg_vol_ratio"]
) < 1e-10

assert abs(
    validation_score_manual - best["validation_score"]
) < 1e-10

assert abs(
    max_ratio - best["catastrophic_max_ratio"]
) < 1e-10

print("\n✅ Dört varlığın 8 ayrı oranı açıldı.")
print("✅ AvgReturnRatio yeniden hesaplandı ve doğrulandı.")
print("✅ AvgVolRatio yeniden hesaplandı ve doğrulandı.")
print("✅ ValidationScore yeniden hesaplandı ve doğrulandı.")
print("✅ Catastrophic max ratio yeniden hesaplandı ve doğrulandı.")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# ==========================================================
# 05_FINAL_AUDIT_v4
# Salt-okuma audit'i:
# - Hiçbir dosya değiştirmez
# - Test dizilerini yüklemez
# - Test metriği hesaplamaz
# - Yalnızca mevcut 480-grid sonuçlarını inceler
# ==========================================================

import os
import numpy as np
import pandas as pd

# ----------------------------------------------------------
# 1. YOLLAR
# ----------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

GRID_DIR = os.path.join(
    BASE_DIR,
    "results",
    "grid_search"
)

RANKED_PATH = os.path.join(
    GRID_DIR,
    "grid_results_ranked_v4.csv"
)

TOP10_PATH = os.path.join(
    GRID_DIR,
    "grid_top10_v4.csv"
)

# ----------------------------------------------------------
# 2. TEMEL KONTROLLER
# ----------------------------------------------------------

print("=" * 110)
print("05_FINAL_AUDIT_v4 — SALT OKUMA")
print("=" * 110)

if not os.path.exists(RANKED_PATH):
    raise FileNotFoundError(
        f"Ranked sonuç dosyası bulunamadı:\n{RANKED_PATH}"
    )

if not os.path.exists(TOP10_PATH):
    raise FileNotFoundError(
        f"Top-10 sonuç dosyası bulunamadı:\n{TOP10_PATH}"
    )

ranked = pd.read_csv(RANKED_PATH)
saved_top10 = pd.read_csv(TOP10_PATH)

print(f"[RANKED ROWS] {len(ranked)}")
print(f"[TOP10 ROWS]  {len(saved_top10)}")


def as_bool(value):
    """
    CSV'den bool veya string gelebilecek değerleri
    güvenli biçimde bool'a çevirir.
    """
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    text = str(value).strip().lower()

    if text in {"true", "1", "yes"}:
        return True

    if text in {"false", "0", "no", "nan", "none", ""}:
        return False

    raise ValueError(
        f"Bool'a çevrilemeyen değer: {value!r}"
    )


# ----------------------------------------------------------
# 3. 480-GRID BÜTÜNLÜK AUDIT'I
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("A. GRID BÜTÜNLÜK VE TEST KÖRLÜĞÜ")
print("=" * 110)

required_columns = [
    "rank",
    "config_id",
    "architecture",
    "loss_strategy",
    "lookback",
    "size",
    "feature_set",
    "seed",
    "best_epoch",
    "epochs_ran",
    "validation_score",
    "avg_return_ratio",
    "avg_vol_ratio",
    "catastrophic_max_ratio",
    "parameter_count",
    "loss_parameter_count",
    "total_trainable_parameter_count",
    "shared_param_count",
    "return_specific_param_count",
    "vol_specific_param_count",
    "status",
    "history_file",
    "metrics_file",
    "test_arrays_loaded",
    "test_metrics_computed",
]

missing = [
    col for col in required_columns
    if col not in ranked.columns
]

if missing:
    raise RuntimeError(
        f"Eksik zorunlu kolonlar: {missing}"
    )

unique_config_count = ranked["config_id"].nunique()
success_count = int(
    (ranked["status"] == "success").sum()
)

test_arrays_any = ranked[
    "test_arrays_loaded"
].map(as_bool).any()

test_metrics_any = ranked[
    "test_metrics_computed"
].map(as_bool).any()

print(f"Toplam satır              : {len(ranked)}")
print(f"Unique config             : {unique_config_count}")
print(f"Success config            : {success_count}")
print(f"Seed değerleri            : {sorted(ranked['seed'].unique().tolist())}")
print(f"Test arrays loaded?       : {test_arrays_any}")
print(f"Test metrics computed?    : {test_metrics_any}")

assert len(ranked) == 480, (
    f"Beklenen 480 satır, bulunan {len(ranked)}"
)

assert unique_config_count == 480, (
    f"Unique config sayısı 480 değil: {unique_config_count}"
)

assert success_count == 480, (
    f"Success sayısı 480 değil: {success_count}"
)

assert not test_arrays_any, (
    "Test array erişimi tespit edildi."
)

assert not test_metrics_any, (
    "Test metriği tespit edildi."
)

print("\n✅ 480/480 unique success.")
print("✅ Test dizileri yüklenmemiş.")
print("✅ Test metriği hesaplanmamış.")


# ----------------------------------------------------------
# 4. TOP-10 EXACT CONFIGLER
# ----------------------------------------------------------

top10 = ranked.head(10).copy()

top10["gap_to_best"] = (
    top10["validation_score"]
    - top10.iloc[0]["validation_score"]
)

top10["gap_from_previous"] = (
    top10["validation_score"].diff()
)

ratio_cols = []

for asset in [
    "BIST100",
    "USDTRY",
    "EURTRY",
    "GOLD",
]:
    ratio_cols.extend([
        f"{asset}_return_ratio",
        f"{asset}_vol_ratio",
    ])

for col in ratio_cols:
    if col not in ranked.columns:
        raise RuntimeError(
            f"Eksik ratio kolonu: {col}"
        )


def get_worst_ratio_label(row):
    values = row[ratio_cols].astype(float)
    return values.idxmax()


top10["worst_asset_task"] = top10.apply(
    get_worst_ratio_label,
    axis=1
)

top10["worst_ratio"] = top10[
    ratio_cols
].max(axis=1)

show_cols = [
    "rank",
    "config_id",
    "architecture",
    "loss_strategy",
    "lookback",
    "size",
    "feature_set",
    "best_epoch",
    "epochs_ran",
    "validation_score",
    "gap_to_best",
    "gap_from_previous",
    "avg_return_ratio",
    "avg_vol_ratio",
    "catastrophic_max_ratio",
    "worst_asset_task",
    "worst_ratio",
]

print("\n" + "=" * 110)
print("B. RESMÎ TOP-10 — EXACT CONFIGLER")
print("=" * 110)

display(
    top10[show_cols].reset_index(drop=True)
)


# ----------------------------------------------------------
# 5. KAYITLI TOP10 DOSYASI İLE UYUMLULUK
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("C. RANKED HEAD(10) ↔ grid_top10_v4.csv UYUMLULUK")
print("=" * 110)

compare_cols = [
    "config_id",
    "architecture",
    "loss_strategy",
    "lookback",
    "size",
    "feature_set",
    "best_epoch",
    "validation_score",
]

top10_match = (
    top10[compare_cols]
    .reset_index(drop=True)
    .equals(
        saved_top10[compare_cols]
        .reset_index(drop=True)
    )
)

print(
    "Ranked ilk 10 ile kayıtlı grid_top10_v4.csv aynı mı? :",
    top10_match
)

assert top10_match, (
    "Ranked Top-10 ile grid_top10_v4.csv uyuşmuyor."
)

print("✅ Top-10 dosya tutarlılığı doğrulandı.")


# ----------------------------------------------------------
# 6. TOP-10 BEST_EPOCH DAĞILIMI
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("D. TOP-10 BEST_EPOCH DAĞILIMI")
print("=" * 110)

epoch_summary = pd.DataFrame({
    "metric": [
        "minimum_best_epoch",
        "median_best_epoch",
        "maximum_best_epoch",
        "best_epoch_50_count",
        "best_epoch_45_or_more_count",
        "epochs_ran_50_count",
    ],
    "value": [
        int(top10["best_epoch"].min()),
        float(top10["best_epoch"].median()),
        int(top10["best_epoch"].max()),
        int((top10["best_epoch"] == 50).sum()),
        int((top10["best_epoch"] >= 45).sum()),
        int((top10["epochs_ran"] == 50).sum()),
    ]
})

display(epoch_summary)

display(
    top10[
        [
            "rank",
            "config_id",
            "architecture",
            "loss_strategy",
            "lookback",
            "size",
            "feature_set",
            "best_epoch",
            "epochs_ran",
            "validation_score",
        ]
    ].reset_index(drop=True)
)

print(
    "\nNOT: MIN_EPOCHS kararı bu dağılım görülmeden "
    "v3'ten otomatik taşınmamalıdır."
)


# ----------------------------------------------------------
# 7. TOP-10 KATEGORİ DAĞILIMLARI
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("E. TOP-10 KATEGORİ DAĞILIMLARI")
print("=" * 110)

distribution_columns = [
    "architecture",
    "loss_strategy",
    "lookback",
    "size",
    "feature_set",
]

for col in distribution_columns:
    print(f"\n--- {col} ---")

    counts = (
        top10[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="count")
    )

    display(counts)


# ----------------------------------------------------------
# 8. HER MİMARİNİN EN İYİ KONFİGÜRASYONU
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("F. HER MİMARİNİN EN İYİ KONFİGÜRASYONU")
print("=" * 110)

best_per_arch = (
    ranked
    .sort_values(
        ["validation_score", "config_id"],
        ascending=[True, True]
    )
    .groupby(
        "architecture",
        as_index=False,
        sort=False
    )
    .first()
)

best_per_arch = best_per_arch.sort_values(
    "validation_score"
).reset_index(drop=True)

display(
    best_per_arch[
        [
            "architecture",
            "config_id",
            "loss_strategy",
            "lookback",
            "size",
            "feature_set",
            "best_epoch",
            "validation_score",
            "avg_return_ratio",
            "avg_vol_ratio",
            "catastrophic_max_ratio",
            "parameter_count",
            "total_trainable_parameter_count",
        ]
    ]
)


# ----------------------------------------------------------
# 9. PARAMETRE / KAPASİTE TABLOSU
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("G. TOP-10 PARAMETRE / KAPASİTE AUDIT'I")
print("=" * 110)

capacity_cols = [
    "rank",
    "config_id",
    "architecture",
    "loss_strategy",
    "lookback",
    "size",
    "feature_set",
    "parameter_count",
    "loss_parameter_count",
    "total_trainable_parameter_count",
    "shared_param_count",
    "return_specific_param_count",
    "vol_specific_param_count",
    "validation_score",
]

display(
    top10[capacity_cols].reset_index(drop=True)
)

print(
    "\nNOT: Aynı 'small/medium/large' etiketi, "
    "farklı mimariler arasında eşit parametre sayısı "
    "anlamına gelmez."
)


# ----------------------------------------------------------
# 10. CATASTROPHIC / UNDERPERFORMANCE DAĞILIMI
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("H. BASELINE-ALTI PERFORMANS TANISAL DAĞILIMI")
print("=" * 110)


def diagnostic_summary(df, name):
    return {
        "scope": name,
        "n_configs": len(df),
        "catastrophic_min": float(
            df["catastrophic_max_ratio"].min()
        ),
        "catastrophic_median": float(
            df["catastrophic_max_ratio"].median()
        ),
        "catastrophic_max": float(
            df["catastrophic_max_ratio"].max()
        ),
        "count_gt_1_30": int(
            (
                df["catastrophic_max_ratio"]
                > 1.30
            ).sum()
        ),
        "count_gt_1_50": int(
            (
                df["catastrophic_max_ratio"]
                > 1.50
            ).sum()
        ),
    }


diagnostic_df = pd.DataFrame([
    diagnostic_summary(top10, "Top-10"),
    diagnostic_summary(ranked, "All-480"),
])

display(diagnostic_df)

print(
    "\nNOT: 1.30 ve 1.50 karar eşikleri değildir; "
    "yalnızca önceden tanımlanmış tanısal raporlama "
    "işaretleyicileridir."
)


# ----------------------------------------------------------
# 11. NoSharing FL0.5 ↔ PCGrad EŞDEĞERLİK AUDIT'I
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("I. NoSharing: FixedLambda_0.5 ↔ PCGrad EŞDEĞERLİK AUDIT'I")
print("=" * 110)

pair_numeric_cols = [
    "validation_score",
    "avg_return_ratio",
    "avg_vol_ratio",
    "catastrophic_max_ratio",
] + ratio_cols


def exact_float_array_equal(a, b):
    """
    CSV'den okunan saklanmış float değerlerin
    exact eşitliğini kontrol eder.
    """
    return np.array_equal(
        np.asarray(a, dtype=np.float64),
        np.asarray(b, dtype=np.float64),
        equal_nan=True
    )


pair_rows = []

no_sharing = ranked[
    ranked["architecture"] == "NoSharing"
].copy()

pair_keys = [
    "lookback",
    "size",
    "feature_set",
]

for key, group in no_sharing.groupby(
    pair_keys,
    dropna=False
):
    fixed = group[
        group["loss_strategy"]
        == "FixedLambda_0.5"
    ]

    pcgrad = group[
        group["loss_strategy"]
        == "PCGrad"
    ]

    if len(fixed) != 1 or len(pcgrad) != 1:
        continue

    fixed_row = fixed.iloc[0]
    pcgrad_row = pcgrad.iloc[0]

    fixed_vals = fixed_row[
        pair_numeric_cols
    ].to_numpy(dtype=np.float64)

    pcgrad_vals = pcgrad_row[
        pair_numeric_cols
    ].to_numpy(dtype=np.float64)

    stored_outputs_exact = exact_float_array_equal(
        fixed_vals,
        pcgrad_vals
    )

    max_abs_output_diff = float(
        np.nanmax(
            np.abs(
                fixed_vals
                - pcgrad_vals
            )
        )
    )

    best_epoch_equal = (
        int(fixed_row["best_epoch"])
        == int(pcgrad_row["best_epoch"])
    )

    epochs_ran_equal = (
        int(fixed_row["epochs_ran"])
        == int(pcgrad_row["epochs_ran"])
    )

    # --------------------------------------
    # History trajectory karşılaştırması
    # --------------------------------------

    history_exact = None
    history_max_abs_diff = np.nan

    fixed_history_path = str(
        fixed_row["history_file"]
    )

    pcgrad_history_path = str(
        pcgrad_row["history_file"]
    )

    if (
        os.path.exists(fixed_history_path)
        and os.path.exists(pcgrad_history_path)
    ):
        fixed_hist = pd.read_csv(
            fixed_history_path
        )

        pcgrad_hist = pd.read_csv(
            pcgrad_history_path
        )

        history_core_cols = [
            "epoch",
            "train_loss",
            "train_return_loss",
            "train_vol_loss",
            "val_equal_weight_proxy_loss",
            "val_return_loss",
            "val_vol_loss",
            "validation_score",
            "avg_return_ratio",
            "avg_vol_ratio",
        ]

        same_shape = (
            fixed_hist[history_core_cols].shape
            == pcgrad_hist[history_core_cols].shape
        )

        if same_shape:
            fixed_hist_vals = fixed_hist[
                history_core_cols
            ].to_numpy(dtype=np.float64)

            pcgrad_hist_vals = pcgrad_hist[
                history_core_cols
            ].to_numpy(dtype=np.float64)

            history_exact = exact_float_array_equal(
                fixed_hist_vals,
                pcgrad_hist_vals
            )

            history_max_abs_diff = float(
                np.nanmax(
                    np.abs(
                        fixed_hist_vals
                        - pcgrad_hist_vals
                    )
                )
            )

        else:
            history_exact = False

    # --------------------------------------
    # Final metrics karşılaştırması
    # --------------------------------------

    metrics_exact = None
    metrics_max_abs_diff = np.nan

    fixed_metrics_path = str(
        fixed_row["metrics_file"]
    )

    pcgrad_metrics_path = str(
        pcgrad_row["metrics_file"]
    )

    if (
        os.path.exists(fixed_metrics_path)
        and os.path.exists(pcgrad_metrics_path)
    ):
        fixed_metrics = pd.read_csv(
            fixed_metrics_path
        )

        pcgrad_metrics = pd.read_csv(
            pcgrad_metrics_path
        )

        metrics_numeric_cols = [
            "MAE",
            "RMSE",
            "R2",
            "PinballLoss_tau_0.5",
        ]

        labels_equal = (
            fixed_metrics[
                ["task", "asset"]
            ]
            .reset_index(drop=True)
            .equals(
                pcgrad_metrics[
                    ["task", "asset"]
                ]
                .reset_index(drop=True)
            )
        )

        same_shape = (
            fixed_metrics[
                metrics_numeric_cols
            ].shape
            == pcgrad_metrics[
                metrics_numeric_cols
            ].shape
        )

        if labels_equal and same_shape:
            fixed_metric_vals = fixed_metrics[
                metrics_numeric_cols
            ].to_numpy(dtype=np.float64)

            pcgrad_metric_vals = pcgrad_metrics[
                metrics_numeric_cols
            ].to_numpy(dtype=np.float64)

            metrics_exact = exact_float_array_equal(
                fixed_metric_vals,
                pcgrad_metric_vals
            )

            metrics_max_abs_diff = float(
                np.nanmax(
                    np.abs(
                        fixed_metric_vals
                        - pcgrad_metric_vals
                    )
                )
            )

        else:
            metrics_exact = False

    pair_rows.append({
        "lookback": key[0],
        "size": key[1],
        "feature_set": key[2],

        "fixed_rank": int(
            fixed_row["rank"]
        ),
        "pcgrad_rank": int(
            pcgrad_row["rank"]
        ),

        "fixed_score": float(
            fixed_row["validation_score"]
        ),
        "pcgrad_score": float(
            pcgrad_row["validation_score"]
        ),

        "best_epoch_equal":
            best_epoch_equal,

        "epochs_ran_equal":
            epochs_ran_equal,

        "all_stored_outputs_exact":
            stored_outputs_exact,

        "max_abs_output_diff":
            max_abs_output_diff,

        "history_core_exact":
            history_exact,

        "history_max_abs_diff":
            history_max_abs_diff,

        "final_metrics_exact":
            metrics_exact,

        "metrics_max_abs_diff":
            metrics_max_abs_diff,

        "fixed_in_top10":
            int(fixed_row["rank"]) <= 10,

        "pcgrad_in_top10":
            int(pcgrad_row["rank"]) <= 10,
    })


pair_audit = pd.DataFrame(pair_rows).sort_values(
    [
        "lookback",
        "size",
        "feature_set",
    ]
).reset_index(drop=True)

display(pair_audit)

if len(pair_audit) > 0:
    print(
        "\nExact stored-output eşit çift sayısı:",
        int(
            pair_audit[
                "all_stored_outputs_exact"
            ].sum()
        ),
        "/",
        len(pair_audit)
    )

    print(
        "Exact history-core eşit çift sayısı:",
        int(
            (
                pair_audit[
                    "history_core_exact"
                ] == True
            ).sum()
        ),
        "/",
        len(pair_audit)
    )

    print(
        "Exact final-metrics eşit çift sayısı:",
        int(
            (
                pair_audit[
                    "final_metrics_exact"
                ] == True
            ).sum()
        ),
        "/",
        len(pair_audit)
    )

print(
    "\nÖNEMLİ NOT:"
    "\nBuradaki 'exact' kontrol, CSV'lerde saklanmış sayısal "
    "çıktıların exact eşitliğidir."
    "\nBu tek başına model checkpoint state_dict'lerinin "
    "bit-düzeyinde aynı olduğu iddiası değildir."
)


# ----------------------------------------------------------
# 12. RAW TOP-10 İÇİNDE EŞDEĞER DUPLICATE TESPİTİ
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("J. RAW TOP-10 İÇİNDE EŞDEĞER DUPLICATE TESPİTİ")
print("=" * 110)


def canonical_equivalence_key(row):
    """
    Yalnızca teorik ve audit ile doğrulanan
    NoSharing FL0.5 ↔ PCGrad eşdeğerliğini
    aynı canonical key altında toplar.

    Diğer hiçbir config otomatik birleştirilmez.
    """

    architecture = row["architecture"]
    loss = row["loss_strategy"]

    if (
        architecture == "NoSharing"
        and loss in {
            "FixedLambda_0.5",
            "PCGrad",
        }
    ):
        canonical_loss = (
            "NoSharing_FL0.5_EQ_PCGrad"
        )
    else:
        canonical_loss = loss

    return (
        architecture,
        canonical_loss,
        int(row["lookback"]),
        row["size"],
        row["feature_set"],
    )


top10_keys = []

for _, row in top10.iterrows():
    top10_keys.append(
        canonical_equivalence_key(row)
    )

top10 = top10.copy()
top10["canonical_key"] = [
    str(key)
    for key in top10_keys
]

duplicate_groups = (
    top10
    .groupby("canonical_key")
    .filter(lambda x: len(x) > 1)
    .sort_values(
        [
            "canonical_key",
            "rank",
        ]
    )
)

if len(duplicate_groups) == 0:
    print(
        "Top-10 içinde tanımlı equivalence kuralına "
        "göre duplicate yok."
    )

else:
    print(
        f"Top-10 içinde duplicate satır sayısı: "
        f"{len(duplicate_groups)}"
    )

    print(
        f"Duplicate equivalence grup sayısı: "
        f"{duplicate_groups['canonical_key'].nunique()}"
    )

    display(
        duplicate_groups[
            [
                "rank",
                "config_id",
                "architecture",
                "loss_strategy",
                "lookback",
                "size",
                "feature_set",
                "best_epoch",
                "validation_score",
                "canonical_key",
            ]
        ].reset_index(drop=True)
    )


# ----------------------------------------------------------
# 13. DIAGNOSTIC TOP-10 DISTINCT
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("K. DIAGNOSTIC TOP-10 DISTINCT — KARAR DEĞİL")
print("=" * 110)

seen_keys = set()
distinct_rows = []

for _, row in ranked.iterrows():
    key = canonical_equivalence_key(row)

    if key in seen_keys:
        continue

    seen_keys.add(key)
    distinct_rows.append(row.copy())

    if len(distinct_rows) == 10:
        break

top10_distinct = pd.DataFrame(
    distinct_rows
).reset_index(drop=True)

top10_distinct["distinct_position"] = (
    np.arange(
        1,
        len(top10_distinct) + 1
    )
)

display(
    top10_distinct[
        [
            "distinct_position",
            "rank",
            "config_id",
            "architecture",
            "loss_strategy",
            "lookback",
            "size",
            "feature_set",
            "best_epoch",
            "validation_score",
            "avg_return_ratio",
            "avg_vol_ratio",
            "catastrophic_max_ratio",
        ]
    ]
)

entered_from_below = top10_distinct[
    top10_distinct["rank"] > 10
].copy()

print("\nRaw Top-10 dışından distinct listeye girenler:")

if len(entered_from_below) == 0:
    print("Yok.")

else:
    display(
        entered_from_below[
            [
                "distinct_position",
                "rank",
                "config_id",
                "architecture",
                "loss_strategy",
                "lookback",
                "size",
                "feature_set",
                "validation_score",
            ]
        ]
    )

print(
    "\nUYARI: Bu tablo yalnızca audit amaçlıdır."
    "\nHenüz 'Raw Top-10' veya 'Top-10 distinct' "
    "politikası seçilmemiştir."
)


# ----------------------------------------------------------
# 14. 06 PROTOKOL KARAR DESTEĞİ — OTOMATİK ÖZET
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("L. 06 PROTOKOL KARAR DESTEĞİ — ÖZET")
print("=" * 110)

n_best_epoch_50 = int(
    (top10["best_epoch"] == 50).sum()
)

n_best_epoch_45_plus = int(
    (top10["best_epoch"] >= 45).sum()
)

n_duplicate_rows = int(
    len(duplicate_groups)
)

n_duplicate_groups = int(
    duplicate_groups[
        "canonical_key"
    ].nunique()
) if len(duplicate_groups) > 0 else 0

n_distinct_entries_from_below = int(
    len(entered_from_below)
)

print(
    f"Top-10 içinde best_epoch = 50       : "
    f"{n_best_epoch_50}"
)

print(
    f"Top-10 içinde best_epoch >= 45      : "
    f"{n_best_epoch_45_plus}"
)

print(
    f"Top-10 duplicate satır sayısı       : "
    f"{n_duplicate_rows}"
)

print(
    f"Top-10 duplicate equivalence grubu  : "
    f"{n_duplicate_groups}"
)

print(
    f"Distinct listede rank > 10'dan giriş: "
    f"{n_distinct_entries_from_below}"
)

print("\nAÇIK KALAN KARARLAR:")
print(
    "1. MIN_EPOCHS — Top-10 best_epoch dağılımına göre."
)
print(
    "2. DUPLICATE POLICY — Raw Top-10 mı, Top-10 distinct mi?"
)
print(
    "3. PATIENCE ve MAX_EPOCHS — mevcut v4 dağılımına göre gerekçelendirilecek."
)
print(
    "4. SEEDS — üzerinde mutabık kalınan revizyon: [123, 777, 2026];"
)
print(
    "   MASTER'a yazılınca resmen kilitlenecek."
)

print("\n" + "=" * 110)
print("05_FINAL_AUDIT_v4 TAMAMLANDI")
print("=" * 110)

print("✅ Hiçbir dosya değiştirilmedi.")
print("✅ Hiçbir sonuç dosyasına yazılmadı.")
print("✅ Test dizileri yüklenmedi.")
print("✅ Test metriği hesaplanmadı.")
print("✅ 480-grid bütünlüğü kontrol edildi.")
print("✅ Top-10 exact configler açıldı.")
print("✅ Best-epoch dağılımı incelendi.")
print("✅ Mimari/loss/lookback/size/feature dağılımı açıldı.")
print("✅ Parametre sayıları incelendi.")
print("✅ NoSharing FL0.5 ↔ PCGrad eşdeğerliği programatik kontrol edildi.")
print("✅ Raw Top-10 duplicate'ları tespit edildi.")
print("✅ Diagnostic Top-10 distinct listesi oluşturuldu.")
print("=" * 110)

In [ ]:
# ==========================================================
# CODE MANIFEST v4
# Resmî v4 pipeline scriptlerinin SHA-256 kaydı
#
# - Scriptleri değiştirmez
# - Sonuç dosyalarını değiştirmez
# - Test verisine erişmez
# - Yalnızca mevcut script sürümlerini hash ile tanımlar
# ==========================================================

import os
import hashlib
import pandas as pd
from datetime import datetime, timezone

# ----------------------------------------------------------
# 1. YOLLAR
# ----------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

SCRIPTS_DIR = os.path.join(
    BASE_DIR,
    "scripts"
)

CONFIG_DIR = os.path.join(
    BASE_DIR,
    "config"
)

MANIFEST_PATH = os.path.join(
    CONFIG_DIR,
    "code_manifest_v4.csv"
)

os.makedirs(
    CONFIG_DIR,
    exist_ok=True
)

# ----------------------------------------------------------
# 2. RESMÎ v4 SCRIPT LİSTESİ
# ----------------------------------------------------------

SCRIPT_NAMES = [
    "00_setup_v4.py",
    "01_rebuild_from_frozen_raw_v4.py",
    "02_preprocessing_v4.py",
    "03_baseline_sanity_v4.py",
    "04_small_model_test_v4.py",
    "05a_mini_grid_v4.py",
    "05_grid_search_v4.py",
]

# ----------------------------------------------------------
# 3. SHA-256 FONKSİYONU
# ----------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024
):
    """
    Dosyanın SHA-256 hash'ini hesaplar.
    Dosyayı değiştirmez.
    """

    sha = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha.update(chunk)

    return sha.hexdigest()

# ----------------------------------------------------------
# 4. MANIFEST OLUŞTUR
# ----------------------------------------------------------

rows = []

manifest_created_utc = datetime.now(
    timezone.utc
).isoformat()

for script_name in SCRIPT_NAMES:

    script_path = os.path.join(
        SCRIPTS_DIR,
        script_name
    )

    if not os.path.exists(script_path):
        raise FileNotFoundError(
            f"Script bulunamadı:\n{script_path}"
        )

    file_size_bytes = os.path.getsize(
        script_path
    )

    modified_timestamp = os.path.getmtime(
        script_path
    )

    modified_utc = datetime.fromtimestamp(
        modified_timestamp,
        tz=timezone.utc
    ).isoformat()

    sha256 = sha256_file(
        script_path
    )

    rows.append({
        "script_name": script_name,
        "relative_path": os.path.relpath(
            script_path,
            BASE_DIR
        ),
        "sha256": sha256,
        "size_bytes": file_size_bytes,
        "modified_utc": modified_utc,
        "manifest_created_utc": manifest_created_utc,
    })

manifest_df = pd.DataFrame(rows)

# ----------------------------------------------------------
# 5. CSV'YE KAYDET
# ----------------------------------------------------------

manifest_df.to_csv(
    MANIFEST_PATH,
    index=False,
    encoding="utf-8"
)

# ----------------------------------------------------------
# 6. KAYIT SONRASI DOĞRULAMA
# ----------------------------------------------------------

saved_manifest = pd.read_csv(
    MANIFEST_PATH
)

assert len(saved_manifest) == len(SCRIPT_NAMES)

assert (
    saved_manifest["script_name"].tolist()
    == SCRIPT_NAMES
)

assert saved_manifest["sha256"].str.len().eq(
    64
).all()

# Her script için hash'i tekrar hesaplayıp
# kaydedilen manifest ile karşılaştır.
verification_rows = []

for _, row in saved_manifest.iterrows():

    script_path = os.path.join(
        BASE_DIR,
        row["relative_path"]
    )

    current_sha256 = sha256_file(
        script_path
    )

    verification_rows.append(
        current_sha256 == row["sha256"]
    )

saved_manifest[
    "hash_verification_passed"
] = verification_rows

assert saved_manifest[
    "hash_verification_passed"
].all()

# ----------------------------------------------------------
# 7. SONUÇ
# ----------------------------------------------------------

print("=" * 110)
print("CODE MANIFEST v4 — TAMAMLANDI")
print("=" * 110)

print(f"\nManifest yolu:\n{MANIFEST_PATH}")

print(
    f"\nKaydedilen resmî script sayısı: "
    f"{len(saved_manifest)}"
)

print("\nResmî v4 script hash'leri:\n")

display(
    saved_manifest[
        [
            "script_name",
            "sha256",
            "size_bytes",
            "hash_verification_passed",
        ]
    ]
)

print("\n" + "=" * 110)
print("DOĞRULAMA")
print("=" * 110)

print("✅ 7/7 resmî v4 script bulundu.")
print("✅ Her script için SHA-256 hesaplandı.")
print("✅ code_manifest_v4.csv oluşturuldu.")
print("✅ Kayıt sonrası tüm hash'ler yeniden doğrulandı.")
print("✅ Hiçbir script değiştirilmedi.")
print("✅ Hiçbir model sonucu değiştirilmedi.")
print("✅ Test verisine erişilmedi.")
print("=" * 110)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# ==========================================================
# 06 PRE-RUN HASH + COMPILE + MANIFEST KONTROLÜ
#
# Amaç:
# - 06 scripti gerçekten doğru dosya mı?
# - Python syntax geçiyor mu?
# - Hazırlanan resmî 06 sürümüyle SHA-256 birebir aynı mı?
# - code_manifest_v4.csv'ye güvenli biçimde eklendi mi?
#
# Bu hücre:
# - 06 scriptini DEĞİŞTİRMEZ
# - Model eğitimi BAŞLATMAZ
# - Test verisine ERİŞMEZ
# - Yalnızca doğrulama yapar ve manifesti günceller
# ==========================================================

import os
import hashlib
import py_compile
from datetime import datetime, timezone

import pandas as pd


# ----------------------------------------------------------
# 1. YOLLAR
# ----------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

SCRIPT_PATH = os.path.join(
    BASE_DIR,
    "scripts",
    "06_best_model_multiseed_v4.py"
)

MANIFEST_PATH = os.path.join(
    BASE_DIR,
    "config",
    "code_manifest_v4.csv"
)


# ----------------------------------------------------------
# 2. BEKLENEN RESMÎ 06 HASH'İ
# ----------------------------------------------------------

EXPECTED_SHA256 = (
    "35de2ee398699003dfef6be36b70c112f"
    "b2c0d1b1e9577cbf64bef58877e16d8"
)


# ----------------------------------------------------------
# 3. SHA-256 FONKSİYONU
# ----------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha.update(chunk)

    return sha.hexdigest()


# ----------------------------------------------------------
# 4. DOSYA VARLIK KONTROLÜ
# ----------------------------------------------------------

if not os.path.exists(SCRIPT_PATH):

    raise FileNotFoundError(
        f"06 script bulunamadı:\n{SCRIPT_PATH}"
    )

if not os.path.exists(MANIFEST_PATH):

    raise FileNotFoundError(
        f"Manifest bulunamadı:\n{MANIFEST_PATH}"
    )


print("=" * 110)
print("06 PRE-RUN KONTROLÜ")
print("=" * 110)

print(f"\nScript:\n{SCRIPT_PATH}")

print(f"\nManifest:\n{MANIFEST_PATH}")


# ----------------------------------------------------------
# 5. PYTHON COMPILE KONTROLÜ
# ----------------------------------------------------------

try:

    py_compile.compile(
        SCRIPT_PATH,
        doraise=True
    )

    compile_passed = True

except py_compile.PyCompileError as error:

    compile_passed = False

    raise RuntimeError(
        "06 script Python syntax kontrolünden geçemedi."
    ) from error


print("\n✅ Python compile kontrolü geçti.")


# ----------------------------------------------------------
# 6. SHA-256 KONTROLÜ
# ----------------------------------------------------------

actual_sha256 = sha256_file(
    SCRIPT_PATH
)

print("\nSHA-256")

print(f"Beklenen : {EXPECTED_SHA256}")
print(f"Gerçek   : {actual_sha256}")


if actual_sha256 != EXPECTED_SHA256:

    raise RuntimeError(
        "\n06 script hash'i beklenen resmî sürümle eşleşmiyor.\n"
        "ÇALIŞTIRMA DURDURULDU.\n\n"
        f"Beklenen: {EXPECTED_SHA256}\n"
        f"Gerçek  : {actual_sha256}"
    )


print("\n✅ 06 script SHA-256 birebir eşleşti.")


# ----------------------------------------------------------
# 7. MEVCUT MANIFESTİ OKU
# ----------------------------------------------------------

manifest_df = pd.read_csv(
    MANIFEST_PATH
)

required_columns = [
    "script_name",
    "relative_path",
    "sha256",
    "size_bytes",
    "modified_utc",
    "manifest_created_utc",
]

missing_columns = [
    col
    for col in required_columns
    if col not in manifest_df.columns
]

if missing_columns:

    raise RuntimeError(
        "Manifest schema eksik.\n"
        f"Eksik kolonlar: {missing_columns}"
    )


print(
    f"\nMevcut manifest satır sayısı: "
    f"{len(manifest_df)}"
)


# ----------------------------------------------------------
# 8. 06 KAYDINI HAZIRLA
# ----------------------------------------------------------

script_name = os.path.basename(
    SCRIPT_PATH
)

relative_path = os.path.relpath(
    SCRIPT_PATH,
    BASE_DIR
)

size_bytes = os.path.getsize(
    SCRIPT_PATH
)

modified_utc = datetime.fromtimestamp(
    os.path.getmtime(SCRIPT_PATH),
    tz=timezone.utc
).isoformat()

manifest_created_utc = datetime.now(
    timezone.utc
).isoformat()


new_row = {
    "script_name": script_name,
    "relative_path": relative_path,
    "sha256": actual_sha256,
    "size_bytes": size_bytes,
    "modified_utc": modified_utc,
    "manifest_created_utc": manifest_created_utc,
}


# ----------------------------------------------------------
# 9. DUPLICATE / VERSİYON KONTROLÜ
# ----------------------------------------------------------

same_exact_row = (
    (manifest_df["script_name"] == script_name)
    &
    (manifest_df["sha256"] == actual_sha256)
)


if same_exact_row.any():

    print(
        "\nℹ️ 06 script aynı SHA-256 ile zaten manifestte kayıtlı."
    )

    manifest_updated = False

else:

    manifest_df = pd.concat(
        [
            manifest_df,
            pd.DataFrame([new_row])
        ],
        ignore_index=True
    )

    manifest_df.to_csv(
        MANIFEST_PATH,
        index=False,
        encoding="utf-8"
    )

    manifest_updated = True

    print(
        "\n✅ 06 script code_manifest_v4.csv dosyasına eklendi."
    )


# ----------------------------------------------------------
# 10. KAYIT SONRASI YENİDEN DOĞRULAMA
# ----------------------------------------------------------

saved_manifest = pd.read_csv(
    MANIFEST_PATH
)

matching_rows = saved_manifest[
    (
        saved_manifest["script_name"] == script_name
    )
    &
    (
        saved_manifest["sha256"] == actual_sha256
    )
].copy()


if len(matching_rows) < 1:

    raise RuntimeError(
        "06 script manifest kaydı doğrulanamadı."
    )


current_sha256_check = sha256_file(
    SCRIPT_PATH
)

if current_sha256_check != actual_sha256:

    raise RuntimeError(
        "06 script hash'i kontrol sırasında değişti."
    )


# ----------------------------------------------------------
# 11. SONUÇ
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("06 PRE-RUN KONTROLÜ — TAMAMLANDI")
print("=" * 110)

print(f"\nScript adı       : {script_name}")
print(f"Boyut            : {size_bytes} bytes")
print(f"SHA-256          : {actual_sha256}")
print(f"Compile passed   : {compile_passed}")
print(f"Manifest updated : {manifest_updated}")

print("\n✅ 06 script bulundu.")
print("✅ Python syntax kontrolünden geçti.")
print("✅ Beklenen resmî SHA-256 ile birebir eşleşti.")
print("✅ Manifest kaydı doğrulandı.")
print("✅ 06 script değiştirilmedi.")
print("✅ Model eğitimi başlatılmadı.")
print("✅ Test verisine erişilmedi.")

print("=" * 110)

In [ ]:
!python "/content/drive/MyDrive/tez_transformer_v4_repro/scripts/06_best_model_multiseed_v4.py"

In [ ]:
# ==========================================================
# 06_FINAL_AUDIT_v4 — SALT OKUNUR NİHAİ MULTISEED AUDİTİ
#
# AMAÇ:
# 1. Top-10 distinct aday kümesini doğrula
# 2. 30/30 unique success run doğrula
# 3. Her config için exact seeds [123, 777, 2026] doğrula
# 4. MIN=45 / PATIENCE=15 / MAX=100 protokolünü doğrula
# 5. Test erişiminin olmadığını doğrula
# 6. Her run'ın metrics/history/checkpoint dosyasını doğrula
# 7. Metrics dosyalarından ValidationScore'u bağımsız yeniden hesapla
# 8. History minimum score = kayıtlı best score kontrolü yap
# 9. 10 aday için mean/std değerlerini bağımsız yeniden hesapla
# 10. Multiseed ranking'i bağımsız yeniden üret
# 11. Winner JSON ve üç winner checkpoint'ini doğrula
# 12. 06 script SHA-256 manifest kaydını doğrula
#
# BU HÜCRE:
# - HİÇBİR DOSYA YAZMAZ
# - HİÇBİR DOSYA DEĞİŞTİRMEZ
# - MODEL EĞİTMEZ
# - TEST VERİSİNE ERİŞMEZ
# ==========================================================

import os
import json
import hashlib
import ast

import numpy as np
import pandas as pd
import torch


# ==========================================================
# 1. YOLLAR
# ==========================================================

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

CONFIG_DIR = os.path.join(BASE_DIR, "config")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
SCRIPTS_DIR = os.path.join(BASE_DIR, "scripts")

RESULTS_DIR = os.path.join(BASE_DIR, "results", "multiseed")

CANDIDATES_CSV = os.path.join(
    RESULTS_DIR,
    "multiseed_candidates_distinct_v4.csv"
)

RUNS_CSV = os.path.join(
    RESULTS_DIR,
    "multiseed_runs_v4.csv"
)

SUMMARY_CSV = os.path.join(
    RESULTS_DIR,
    "multiseed_summary_v4.csv"
)

RANKED_SUMMARY_CSV = os.path.join(
    RESULTS_DIR,
    "multiseed_summary_ranked_v4.csv"
)

SUMMARY_JSON = os.path.join(
    RESULTS_DIR,
    "multiseed_summary_v4.json"
)

WINNER_JSON = os.path.join(
    RESULTS_DIR,
    "multiseed_winner_config_v4.json"
)

DENOMINATOR_PATH = os.path.join(
    PROCESSED_DIR,
    "selection_baseline_denominators_v4.json"
)

CODE_MANIFEST_PATH = os.path.join(
    CONFIG_DIR,
    "code_manifest_v4.csv"
)

SCRIPT_06_PATH = os.path.join(
    SCRIPTS_DIR,
    "06_best_model_multiseed_v4.py"
)


# ==========================================================
# 2. KİLİTLİ PROTOKOL
# ==========================================================

EXPECTED_SEEDS = [123, 777, 2026]

EXPECTED_SOURCE_RANKS = [
    1, 2, 4, 5, 6, 7, 8, 10, 11, 12
]

EXPECTED_CANDIDATES = 10
EXPECTED_RUNS = 30

EXPECTED_MIN_EPOCHS = 45
EXPECTED_PATIENCE = 15
EXPECTED_MAX_EPOCHS = 100

EXPECTED_06_SHA256 = (
    "35de2ee398699003dfef6be36b70c112f"
    "b2c0d1b1e9577cbf64bef58877e16d8"
)

ASSETS = [
    "BIST100",
    "USDTRY",
    "EURTRY",
    "GOLD"
]

TAU = 0.5

FLOAT_TOL = 1e-10


# ==========================================================
# 3. YARDIMCI FONKSİYONLAR
# ==========================================================

def sha256_file(path, chunk_size=1024 * 1024):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha.update(chunk)

    return sha.hexdigest()


def normalize_bool(value):

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if pd.isna(value):
        return False

    text = str(value).strip().lower()

    if text in {"true", "1", "yes"}:
        return True

    if text in {"false", "0", "no", ""}:
        return False

    raise ValueError(
        f"Boolean değere çevrilemeyen kayıt: {value!r}"
    )


def series_all_false(series):

    return all(
        not normalize_bool(value)
        for value in series
    )


def assert_close(actual, expected, name, tol=FLOAT_TOL):

    if not np.isclose(
        float(actual),
        float(expected),
        rtol=0.0,
        atol=tol
    ):

        raise RuntimeError(
            f"{name} uyuşmuyor.\n"
            f"Beklenen: {expected}\n"
            f"Gerçek   : {actual}\n"
            f"Fark     : {abs(float(actual) - float(expected))}"
        )


def pinball_np(y_true, y_pred, tau=0.5):

    diff = y_true - y_pred

    return float(
        np.mean(
            np.maximum(
                tau * diff,
                (tau - 1.0) * diff
            )
        )
    )


def load_checkpoint_cpu(path):

    try:

        return torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

    except TypeError:

        return torch.load(
            path,
            map_location="cpu"
        )


# ==========================================================
# 4. GEREKLİ DOSYALAR
# ==========================================================

required_paths = [
    CANDIDATES_CSV,
    RUNS_CSV,
    SUMMARY_CSV,
    RANKED_SUMMARY_CSV,
    SUMMARY_JSON,
    WINNER_JSON,
    DENOMINATOR_PATH,
    CODE_MANIFEST_PATH,
    SCRIPT_06_PATH
]

missing_paths = [
    path
    for path in required_paths
    if not os.path.exists(path)
]

if missing_paths:

    raise FileNotFoundError(
        "Gerekli dosyalar eksik:\n"
        + "\n".join(missing_paths)
    )


print("=" * 110)
print("06_FINAL_AUDIT_v4 — BAŞLADI")
print("=" * 110)

print("\n✅ Gerekli tüm ana dosyalar bulundu.")
print("✅ Audit salt okunur modda çalışıyor.")
print("✅ Test verisi yüklenmeyecek.")


# ==========================================================
# 5. 06 SCRIPT SHA-256 + MANIFEST
# ==========================================================

actual_06_sha = sha256_file(
    SCRIPT_06_PATH
)

if actual_06_sha != EXPECTED_06_SHA256:

    raise RuntimeError(
        "06 script SHA-256 beklenen resmî sürümle eşleşmiyor.\n"
        f"Beklenen: {EXPECTED_06_SHA256}\n"
        f"Gerçek  : {actual_06_sha}"
    )

manifest_df = pd.read_csv(
    CODE_MANIFEST_PATH
)

manifest_match = manifest_df[
    (
        manifest_df["script_name"]
        == "06_best_model_multiseed_v4.py"
    )
    &
    (
        manifest_df["sha256"]
        == actual_06_sha
    )
]

if len(manifest_match) < 1:

    raise RuntimeError(
        "06 script mevcut hash ile code manifest içinde bulunamadı."
    )


print("\n" + "-" * 110)
print("A. CODE PROVENANCE")
print("-" * 110)

print("✅ 06 script SHA-256 beklenen resmî hash ile eşleşiyor.")
print("✅ 06 script mevcut hash ile code_manifest_v4.csv içinde kayıtlı.")
print("SHA-256:", actual_06_sha)


# ==========================================================
# 6. DOSYALARI OKU
# ==========================================================

candidates = pd.read_csv(
    CANDIDATES_CSV
)

runs_all = pd.read_csv(
    RUNS_CSV
)

stored_summary = pd.read_csv(
    SUMMARY_CSV
)

stored_ranked = pd.read_csv(
    RANKED_SUMMARY_CSV
)

with open(
    SUMMARY_JSON,
    "r",
    encoding="utf-8"
) as f:

    summary_json = json.load(f)

with open(
    WINNER_JSON,
    "r",
    encoding="utf-8"
) as f:

    winner_json = json.load(f)

with open(
    DENOMINATOR_PATH,
    "r",
    encoding="utf-8"
) as f:

    denominators = json.load(f)


# ==========================================================
# 7. TOP-10 DISTINCT ADAY KÜMESİ
# ==========================================================

if len(candidates) != EXPECTED_CANDIDATES:

    raise RuntimeError(
        f"Candidate sayısı 10 değil: {len(candidates)}"
    )

if candidates["config_id"].nunique() != EXPECTED_CANDIDATES:

    raise RuntimeError(
        "Candidate config_id değerleri unique değil."
    )

actual_positions = sorted(
    candidates["candidate_position"]
    .astype(int)
    .tolist()
)

if actual_positions != list(
    range(1, EXPECTED_CANDIDATES + 1)
):

    raise RuntimeError(
        f"Candidate position yanlış: {actual_positions}"
    )

actual_source_ranks = (
    candidates
    .sort_values("candidate_position")
    ["source_rank"]
    .astype(int)
    .tolist()
)

if actual_source_ranks != EXPECTED_SOURCE_RANKS:

    raise RuntimeError(
        "Top-10 distinct source rank listesi yanlış.\n"
        f"Beklenen: {EXPECTED_SOURCE_RANKS}\n"
        f"Gerçek  : {actual_source_ranks}"
    )


print("\n" + "-" * 110)
print("B. TOP-10 DISTINCT ADAY KÜMESİ")
print("-" * 110)

print("✅ Candidate count = 10")
print("✅ 10/10 config_id unique")
print("✅ Candidate positions = 1..10")
print(
    "✅ Source ranks =",
    actual_source_ranks
)


# ==========================================================
# 8. RUN BÜTÜNLÜĞÜ
# ==========================================================

success_raw = runs_all[
    runs_all["status"] == "success"
].copy()

duplicate_success_pairs = (
    success_raw[
        ["config_id", "seed"]
    ]
    .duplicated()
    .sum()
)

if duplicate_success_pairs != 0:

    raise RuntimeError(
        "Duplicate successful (config_id, seed) pair bulundu: "
        f"{duplicate_success_pairs}"
    )

success = success_raw.copy()

candidate_ids = set(
    candidates["config_id"].astype(str)
)

success = success[
    success["config_id"]
    .astype(str)
    .isin(candidate_ids)
].copy()

if len(success) != EXPECTED_RUNS:

    raise RuntimeError(
        f"Successful unique run sayısı 30 değil: {len(success)}"
    )

if (
    success[
        ["config_id", "seed"]
    ]
    .drop_duplicates()
    .shape[0]
    != EXPECTED_RUNS
):

    raise RuntimeError(
        "30 run içinde duplicate pair bulundu."
    )

if success["config_id"].nunique() != EXPECTED_CANDIDATES:

    raise RuntimeError(
        "Success runlarda 10 distinct config yok."
    )


for config_id, group in success.groupby(
    "config_id"
):

    actual_seeds = sorted(
        group["seed"]
        .astype(int)
        .tolist()
    )

    if actual_seeds != EXPECTED_SEEDS:

        raise RuntimeError(
            f"Seed set yanlış:\n{config_id}\n"
            f"Beklenen: {EXPECTED_SEEDS}\n"
            f"Gerçek  : {actual_seeds}"
        )


print("\n" + "-" * 110)
print("C. 30-RUN BÜTÜNLÜĞÜ")
print("-" * 110)

print(
    f"Toplam raw kayıt        : {len(runs_all)}"
)

print(
    f"Success raw kayıt       : {len(success_raw)}"
)

print(
    f"Unique successful run   : {len(success)} / 30"
)

print(
    f"Distinct candidate      : {success['config_id'].nunique()} / 10"
)

print("✅ Duplicate successful pair yok.")
print("✅ Her candidate exact 3 seed içeriyor.")
print("✅ Her candidate seed seti = [123, 777, 2026]")


# ==========================================================
# 9. PROTOKOL SABİTLERİ
# ==========================================================

if not (
    success["max_epochs"]
    .astype(int)
    == EXPECTED_MAX_EPOCHS
).all():

    raise RuntimeError(
        "MAX_EPOCHS tüm runlarda 100 değil."
    )

if not (
    success["min_epochs_before_stop"]
    .astype(int)
    == EXPECTED_MIN_EPOCHS
).all():

    raise RuntimeError(
        "MIN_EPOCHS tüm runlarda 45 değil."
    )

if not (
    success["patience"]
    .astype(int)
    == EXPECTED_PATIENCE
).all():

    raise RuntimeError(
        "PATIENCE tüm runlarda 15 değil."
    )

if not (
    success["epochs_ran"]
    .astype(int)
    >= EXPECTED_MIN_EPOCHS
).all():

    raise RuntimeError(
        "45 epoch'tan önce biten run bulundu."
    )

if not (
    success["epochs_ran"]
    .astype(int)
    <= EXPECTED_MAX_EPOCHS
).all():

    raise RuntimeError(
        "100 epoch'u geçen run bulundu."
    )

if not (
    success["best_epoch"]
    .astype(int)
    <= success["epochs_ran"]
    .astype(int)
).all():

    raise RuntimeError(
        "best_epoch > epochs_ran olan run bulundu."
    )


early_stopped = success[
    success["epochs_ran"].astype(int)
    < EXPECTED_MAX_EPOCHS
].copy()

if len(early_stopped) > 0:

    bad_early_stop = early_stopped[
        (
            early_stopped["epochs_ran"].astype(int)
            - early_stopped["best_epoch"].astype(int)
        )
        < EXPECTED_PATIENCE
    ]

    if len(bad_early_stop) > 0:

        raise RuntimeError(
            "Early-stopped run içinde PATIENCE koşulunu "
            "sağlamayan kayıt bulundu."
        )


print("\n" + "-" * 110)
print("D. PROTOKOL KONTROLÜ")
print("-" * 110)

print("✅ MIN_EPOCHS = 45")
print("✅ PATIENCE = 15")
print("✅ MAX_EPOCHS = 100")
print("✅ Hiçbir run 45 epoch'tan önce bitmedi.")
print("✅ best_epoch <= epochs_ran tüm runlarda sağlandı.")
print("✅ Early-stop edilen runlar patience koşuluyla uyumlu.")


# ==========================================================
# 10. TEST KÖRLÜĞÜ
# ==========================================================

if not series_all_false(
    runs_all["test_arrays_loaded"]
):

    raise RuntimeError(
        "Run kayıtlarında test_arrays_loaded=True bulundu."
    )

if not series_all_false(
    runs_all["test_metrics_computed"]
):

    raise RuntimeError(
        "Run kayıtlarında test_metrics_computed=True bulundu."
    )

if normalize_bool(
    summary_json["test_arrays_loaded"]
):

    raise RuntimeError(
        "Summary JSON test_arrays_loaded=True."
    )

if normalize_bool(
    summary_json["test_metrics_computed"]
):

    raise RuntimeError(
        "Summary JSON test_metrics_computed=True."
    )

if normalize_bool(
    winner_json["test_arrays_loaded"]
):

    raise RuntimeError(
        "Winner JSON test_arrays_loaded=True."
    )

if normalize_bool(
    winner_json["test_metrics_computed"]
):

    raise RuntimeError(
        "Winner JSON test_metrics_computed=True."
    )


print("\n" + "-" * 110)
print("E. TEST KÖRLÜĞÜ")
print("-" * 110)

print("✅ Tüm run kayıtlarında test_arrays_loaded = False")
print("✅ Tüm run kayıtlarında test_metrics_computed = False")
print("✅ Summary JSON test flag'leri False")
print("✅ Winner JSON test flag'leri False")
print("✅ 06 audit sırasında test dosyası yüklenmedi.")


# ==========================================================
# 11. HER RUN İÇİN DOSYA BÜTÜNLÜĞÜ
# ==========================================================

missing_metrics = []
missing_history = []
missing_checkpoint = []

for _, row in success.iterrows():

    if not os.path.exists(
        str(row["metrics_file"])
    ):

        missing_metrics.append(
            row["config_id"]
        )

    if not os.path.exists(
        str(row["history_file"])
    ):

        missing_history.append(
            row["config_id"]
        )

    if not os.path.exists(
        str(row["checkpoint_file"])
    ):

        missing_checkpoint.append(
            row["config_id"]
        )


if missing_metrics:

    raise FileNotFoundError(
        f"Eksik metrics dosyası sayısı: {len(missing_metrics)}"
    )

if missing_history:

    raise FileNotFoundError(
        f"Eksik history dosyası sayısı: {len(missing_history)}"
    )

if missing_checkpoint:

    raise FileNotFoundError(
        f"Eksik checkpoint sayısı: {len(missing_checkpoint)}"
    )


print("\n" + "-" * 110)
print("F. RUN DOSYA BÜTÜNLÜĞÜ")
print("-" * 110)

print("✅ 30/30 metrics dosyası mevcut.")
print("✅ 30/30 history dosyası mevcut.")
print("✅ 30/30 checkpoint dosyası mevcut.")


# ==========================================================
# 12. METRICS DOSYALARINDAN SCORE'U BAĞIMSIZ YENİDEN HESAPLA
# ==========================================================

max_score_diff = 0.0
max_ratio_diff = 0.0

for _, row in success.iterrows():

    metrics = pd.read_csv(
        str(row["metrics_file"])
    )

    if len(metrics) != 8:

        raise RuntimeError(
            "Metrics dosyası 8 asset-task satırı içermiyor:\n"
            f"{row['metrics_file']}"
        )

    return_ratios = []
    vol_ratios = []

    for asset in ASSETS:

        ret_mae = float(
            metrics.loc[
                (
                    metrics["task"] == "return"
                )
                &
                (
                    metrics["asset"] == asset
                ),
                "MAE"
            ].iloc[0]
        )

        ret_denom = float(
            denominators[
                "return_denominator"
            ][asset]["value"]
        )

        vol_pb = float(
            metrics.loc[
                (
                    metrics["task"] == "volatility"
                )
                &
                (
                    metrics["asset"] == asset
                ),
                "PinballLoss_tau_0.5"
            ].iloc[0]
        )

        vol_denom = float(
            denominators[
                "volatility_denominator"
            ][asset]["value"]
        )

        ret_ratio = (
            ret_mae / ret_denom
        )

        vol_ratio = (
            vol_pb / vol_denom
        )

        return_ratios.append(
            ret_ratio
        )

        vol_ratios.append(
            vol_ratio
        )

        stored_ret_ratio = float(
            row[
                f"{asset}_return_ratio"
            ]
        )

        stored_vol_ratio = float(
            row[
                f"{asset}_vol_ratio"
            ]
        )

        max_ratio_diff = max(
            max_ratio_diff,
            abs(
                ret_ratio
                - stored_ret_ratio
            ),
            abs(
                vol_ratio
                - stored_vol_ratio
            )
        )

        assert_close(
            ret_ratio,
            stored_ret_ratio,
            (
                f"{asset} return ratio "
                f"{row['config_id']} "
                f"seed={row['seed']}"
            )
        )

        assert_close(
            vol_ratio,
            stored_vol_ratio,
            (
                f"{asset} vol ratio "
                f"{row['config_id']} "
                f"seed={row['seed']}"
            )
        )

    avg_return_ratio = float(
        np.mean(return_ratios)
    )

    avg_vol_ratio = float(
        np.mean(vol_ratios)
    )

    recomputed_score = float(
        0.5 * avg_return_ratio
        + 0.5 * avg_vol_ratio
    )

    score_diff = abs(
        recomputed_score
        - float(row["validation_score"])
    )

    max_score_diff = max(
        max_score_diff,
        score_diff
    )

    assert_close(
        avg_return_ratio,
        row["avg_return_ratio"],
        "avg_return_ratio"
    )

    assert_close(
        avg_vol_ratio,
        row["avg_vol_ratio"],
        "avg_vol_ratio"
    )

    assert_close(
        recomputed_score,
        row["validation_score"],
        (
            f"ValidationScore "
            f"{row['config_id']} "
            f"seed={row['seed']}"
        )
    )


print("\n" + "-" * 110)
print("G. BAĞIMSIZ METRİK → VALIDATIONSCORE YENİDEN HESABI")
print("-" * 110)

print("✅ 30/30 metrics dosyası bağımsız yeniden hesaplandı.")
print("✅ 8 asset-task oranı her run için yeniden doğrulandı.")
print("✅ AvgReturnRatio her run için doğrulandı.")
print("✅ AvgVolRatio her run için doğrulandı.")
print("✅ ValidationScore her run için bağımsız yeniden doğrulandı.")

print(
    "Maksimum ratio farkı :",
    f"{max_ratio_diff:.16e}"
)

print(
    "Maksimum score farkı :",
    f"{max_score_diff:.16e}"
)


# ==========================================================
# 13. HISTORY ↔ RUN SCORE KONTROLÜ
# ==========================================================

max_history_score_diff = 0.0

for _, row in success.iterrows():

    history = pd.read_csv(
        str(row["history_file"])
    )

    if len(history) != int(
        row["epochs_ran"]
    ):

        raise RuntimeError(
            "History satır sayısı epochs_ran ile uyuşmuyor:\n"
            f"{row['config_id']} seed={row['seed']}"
        )

    history_min_score = float(
        history[
            "validation_score"
        ].min()
    )

    run_score = float(
        row["validation_score"]
    )

    diff = abs(
        history_min_score
        - run_score
    )

    max_history_score_diff = max(
        max_history_score_diff,
        diff
    )

    assert_close(
        history_min_score,
        run_score,
        (
            f"History min score "
            f"{row['config_id']} "
            f"seed={row['seed']}"
        )
    )

    best_epoch_from_history = int(
        history.loc[
            history[
                "validation_score"
            ].idxmin(),
            "epoch"
        ]
    )

    if best_epoch_from_history != int(
        row["best_epoch"]
    ):

        raise RuntimeError(
            "History best_epoch ile run best_epoch uyuşmuyor:\n"
            f"{row['config_id']} seed={row['seed']}\n"
            f"History: {best_epoch_from_history}\n"
            f"Run    : {int(row['best_epoch'])}"
        )


print("\n" + "-" * 110)
print("H. HISTORY ↔ RUN KONTROLÜ")
print("-" * 110)

print("✅ 30/30 history satır sayısı epochs_ran ile eşleşiyor.")
print("✅ 30/30 history minimum ValidationScore run kaydıyla eşleşiyor.")
print("✅ 30/30 best_epoch history'den bağımsız yeniden doğrulandı.")

print(
    "Maksimum history-score farkı:",
    f"{max_history_score_diff:.16e}"
)


# ==========================================================
# 14. CONFIG-LEVEL SUMMARY'Yİ BAĞIMSIZ YENİDEN HESAPLA
# ==========================================================

recomputed_rows = []

for _, candidate in candidates.iterrows():

    config_id = str(
        candidate["config_id"]
    )

    group = success[
        success["config_id"]
        .astype(str)
        == config_id
    ].copy()

    if len(group) != 3:

        raise RuntimeError(
            f"Config 3 seed içermiyor: {config_id}"
        )

    recomputed_rows.append({
        "candidate_position": int(
            candidate["candidate_position"]
        ),
        "source_rank": int(
            candidate["source_rank"]
        ),
        "config_id": config_id,
        "architecture": str(
            candidate["architecture"]
        ),
        "loss_strategy": str(
            candidate["loss_strategy"]
        ),
        "lookback": int(
            candidate["lookback"]
        ),
        "size": str(
            candidate["size"]
        ),
        "feature_set": str(
            candidate["feature_set"]
        ),
        "n_seeds": 3,
        "mean_validation_score": float(
            group[
                "validation_score"
            ].mean()
        ),
        "std_validation_score_sample": float(
            group[
                "validation_score"
            ].std(ddof=1)
        ),
        "mean_avg_return_ratio": float(
            group[
                "avg_return_ratio"
            ].mean()
        ),
        "mean_avg_vol_ratio": float(
            group[
                "avg_vol_ratio"
            ].mean()
        ),
        "mean_catastrophic_max_ratio": float(
            group[
                "catastrophic_max_ratio"
            ].mean()
        ),
        "min_best_epoch": int(
            group[
                "best_epoch"
            ].min()
        ),
        "median_best_epoch": float(
            group[
                "best_epoch"
            ].median()
        ),
        "max_best_epoch": int(
            group[
                "best_epoch"
            ].max()
        ),
    })


recomputed_summary = pd.DataFrame(
    recomputed_rows
)


# Stored summary ile karşılaştır
stored_summary_indexed = (
    stored_summary
    .set_index("config_id")
)

recomputed_indexed = (
    recomputed_summary
    .set_index("config_id")
)

if set(
    stored_summary_indexed.index
) != set(
    recomputed_indexed.index
):

    raise RuntimeError(
        "Stored summary ve yeniden hesaplanan config setleri uyuşmuyor."
    )


summary_numeric_columns = [
    "mean_validation_score",
    "std_validation_score_sample",
    "mean_avg_return_ratio",
    "mean_avg_vol_ratio",
    "mean_catastrophic_max_ratio",
    "min_best_epoch",
    "median_best_epoch",
    "max_best_epoch",
]


max_summary_diff = 0.0

for config_id in recomputed_indexed.index:

    for col in summary_numeric_columns:

        actual = float(
            recomputed_indexed.loc[
                config_id,
                col
            ]
        )

        expected = float(
            stored_summary_indexed.loc[
                config_id,
                col
            ]
        )

        diff = abs(
            actual - expected
        )

        max_summary_diff = max(
            max_summary_diff,
            diff
        )

        assert_close(
            actual,
            expected,
            (
                f"Summary {col} "
                f"{config_id}"
            )
        )


print("\n" + "-" * 110)
print("I. CONFIG-LEVEL MEAN / STD YENİDEN HESABI")
print("-" * 110)

print("✅ 10/10 aday için mean ValidationScore yeniden hesaplandı.")
print("✅ 10/10 aday için sample std yeniden hesaplandı.")
print("✅ Mean return ratio yeniden hesaplandı.")
print("✅ Mean volatility ratio yeniden hesaplandı.")
print("✅ Stored summary ile bağımsız yeniden hesap aynı.")

print(
    "Maksimum summary farkı:",
    f"{max_summary_diff:.16e}"
)


# ==========================================================
# 15. RANKING'İ BAĞIMSIZ YENİDEN ÜRET
# ==========================================================

recomputed_ranked = (
    recomputed_summary
    .sort_values(
        [
            "mean_validation_score",
            "std_validation_score_sample",
            "candidate_position"
        ],
        ascending=[
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

recomputed_ranked.insert(
    0,
    "multiseed_rank",
    np.arange(
        1,
        len(recomputed_ranked) + 1
    )
)

stored_order = (
    stored_ranked
    .sort_values("multiseed_rank")
    ["config_id"]
    .astype(str)
    .tolist()
)

recomputed_order = (
    recomputed_ranked
    ["config_id"]
    .astype(str)
    .tolist()
)

if stored_order != recomputed_order:

    raise RuntimeError(
        "Stored ranking ile bağımsız yeniden hesaplanan ranking farklı.\n"
        f"Stored    : {stored_order}\n"
        f"Recomputed: {recomputed_order}"
    )


print("\n" + "-" * 110)
print("J. MULTISEED RANKING YENİDEN ÜRETİMİ")
print("-" * 110)

print("✅ Ranking bağımsız olarak yeniden üretildi.")
print("✅ Stored ranking ile 10/10 sıra birebir eşleşti.")

print(
    recomputed_ranked[
        [
            "multiseed_rank",
            "source_rank",
            "architecture",
            "loss_strategy",
            "lookback",
            "size",
            "feature_set",
            "mean_validation_score",
            "std_validation_score_sample",
        ]
    ].to_string(
        index=False
    )
)


# ==========================================================
# 16. WINNER JSON KONTROLÜ
# ==========================================================

derived_winner = (
    recomputed_ranked.iloc[0]
)

derived_winner_id = str(
    derived_winner["config_id"]
)

stored_winner = winner_json[
    "winner"
]

stored_winner_id = str(
    stored_winner["config_id"]
)

if derived_winner_id != stored_winner_id:

    raise RuntimeError(
        "Bağımsız winner ile Winner JSON uyuşmuyor.\n"
        f"Derived: {derived_winner_id}\n"
        f"Stored : {stored_winner_id}"
    )

assert_close(
    derived_winner[
        "mean_validation_score"
    ],
    stored_winner[
        "mean_validation_score"
    ],
    "Winner mean ValidationScore"
)

assert_close(
    derived_winner[
        "std_validation_score_sample"
    ],
    stored_winner[
        "std_validation_score_sample"
    ],
    "Winner sample std"
)


print("\n" + "-" * 110)
print("K. WINNER JSON KONTROLÜ")
print("-" * 110)

print("✅ Winner bağımsız ranking'den yeniden üretildi.")
print("✅ Winner JSON config_id ile eşleşti.")
print("✅ Winner mean ValidationScore eşleşti.")
print("✅ Winner sample std eşleşti.")


# ==========================================================
# 17. WINNER'IN 3 CHECKPOINT'İNİ DERİN DOĞRULA
# ==========================================================

winner_runs = (
    success[
        success["config_id"]
        .astype(str)
        == derived_winner_id
    ]
    .sort_values("seed")
    .copy()
)

if len(winner_runs) != 3:

    raise RuntimeError(
        "Winner için 3 seed checkpoint kaydı yok."
    )


checkpoint_rows = []

for _, row in winner_runs.iterrows():

    checkpoint_path = str(
        row["checkpoint_file"]
    )

    checkpoint = load_checkpoint_cpu(
        checkpoint_path
    )

    checkpoint_seed = int(
        checkpoint["seed"]
    )

    checkpoint_epoch = int(
        checkpoint["epoch"]
    )

    checkpoint_score = float(
        checkpoint["validation_score"]
    )

    checkpoint_config = checkpoint[
        "config"
    ]

    checkpoint_config_id = str(
        checkpoint_config["config_id"]
    )

    if checkpoint_config_id != derived_winner_id:

        raise RuntimeError(
            "Winner checkpoint config_id yanlış:\n"
            f"{checkpoint_path}"
        )

    if checkpoint_seed != int(
        row["seed"]
    ):

        raise RuntimeError(
            "Winner checkpoint seed yanlış:\n"
            f"{checkpoint_path}"
        )

    if checkpoint_epoch != int(
        row["best_epoch"]
    ):

        raise RuntimeError(
            "Winner checkpoint epoch yanlış:\n"
            f"{checkpoint_path}"
        )

    assert_close(
        checkpoint_score,
        row["validation_score"],
        "Winner checkpoint score"
    )

    protocol = checkpoint[
        "protocol"
    ]

    if sorted(
        protocol["seeds"]
    ) != EXPECTED_SEEDS:

        raise RuntimeError(
            "Checkpoint protocol seed set yanlış."
        )

    if int(
        protocol["min_epochs_before_stop"]
    ) != EXPECTED_MIN_EPOCHS:

        raise RuntimeError(
            "Checkpoint MIN_EPOCHS yanlış."
        )

    if int(
        protocol["patience"]
    ) != EXPECTED_PATIENCE:

        raise RuntimeError(
            "Checkpoint PATIENCE yanlış."
        )

    if int(
        protocol["max_epochs"]
    ) != EXPECTED_MAX_EPOCHS:

        raise RuntimeError(
            "Checkpoint MAX_EPOCHS yanlış."
        )

    if normalize_bool(
        checkpoint[
            "test_arrays_loaded"
        ]
    ):

        raise RuntimeError(
            "Winner checkpoint test_arrays_loaded=True."
        )

    if normalize_bool(
        checkpoint[
            "test_metrics_computed"
        ]
    ):

        raise RuntimeError(
            "Winner checkpoint test_metrics_computed=True."
        )

    checkpoint_rows.append({
        "seed": checkpoint_seed,
        "best_epoch": checkpoint_epoch,
        "validation_score": checkpoint_score,
        "checkpoint_file": checkpoint_path,
    })


checkpoint_audit_df = pd.DataFrame(
    checkpoint_rows
).sort_values("seed")


print("\n" + "-" * 110)
print("L. WINNER CHECKPOINT DERİN AUDİTİ")
print("-" * 110)

print("✅ Winner için 3/3 checkpoint mevcut.")
print("✅ Her checkpoint config_id doğru.")
print("✅ Her checkpoint seed doğru.")
print("✅ Her checkpoint best_epoch doğru.")
print("✅ Her checkpoint ValidationScore doğru.")
print("✅ Her checkpoint protokolü doğru.")
print("✅ Her checkpoint test flag'leri False.")

print(
    checkpoint_audit_df.to_string(
        index=False
    )
)


# ==========================================================
# 18. SUMMARY JSON KONTROLÜ
# ==========================================================

if int(
    summary_json["candidate_count"]
) != EXPECTED_CANDIDATES:

    raise RuntimeError(
        "Summary JSON candidate_count yanlış."
    )

if int(
    summary_json["expected_total_runs"]
) != EXPECTED_RUNS:

    raise RuntimeError(
        "Summary JSON expected_total_runs yanlış."
    )

if int(
    summary_json["successful_unique_runs"]
) != EXPECTED_RUNS:

    raise RuntimeError(
        "Summary JSON successful_unique_runs yanlış."
    )

if sorted(
    summary_json["seeds"]
) != EXPECTED_SEEDS:

    raise RuntimeError(
        "Summary JSON seed set yanlış."
    )

if summary_json[
    "source_ranks"
] != EXPECTED_SOURCE_RANKS:

    raise RuntimeError(
        "Summary JSON source rank listesi yanlış."
    )


print("\n" + "-" * 110)
print("M. SUMMARY JSON KONTROLÜ")
print("-" * 110)

print("✅ candidate_count = 10")
print("✅ expected_total_runs = 30")
print("✅ successful_unique_runs = 30")
print("✅ seeds = [123, 777, 2026]")
print("✅ source_ranks doğru")
print("✅ test flag'leri False")


# ==========================================================
# 19. SON RESMÎ AUDIT ÖZETİ
# ==========================================================

winner_mean = float(
    derived_winner[
        "mean_validation_score"
    ]
)

winner_std = float(
    derived_winner[
        "std_validation_score_sample"
    ]
)

winner_group = success[
    success["config_id"]
    .astype(str)
    == derived_winner_id
].copy()

winner_return_mean = float(
    winner_group[
        "avg_return_ratio"
    ].mean()
)

winner_vol_mean = float(
    winner_group[
        "avg_vol_ratio"
    ].mean()
)


print("\n" + "=" * 110)
print("06_FINAL_AUDIT_v4 — TÜM KONTROLLER GEÇTİ")
print("=" * 110)

print("\nAUDIT SONUCU:")
print("✅ 10/10 Top-10 distinct aday doğru.")
print("✅ Source ranks exact:", EXPECTED_SOURCE_RANKS)
print("✅ 30/30 unique successful run doğrulandı.")
print("✅ Her config exact seeds [123, 777, 2026] içeriyor.")
print("✅ MIN_EPOCHS=45 doğrulandı.")
print("✅ PATIENCE=15 doğrulandı.")
print("✅ MAX_EPOCHS=100 doğrulandı.")
print("✅ 30/30 metrics dosyası doğrulandı.")
print("✅ 30/30 history dosyası doğrulandı.")
print("✅ 30/30 checkpoint dosyası mevcut.")
print("✅ 30/30 ValidationScore bağımsız yeniden hesaplandı.")
print("✅ 10/10 mean/std summary bağımsız yeniden üretildi.")
print("✅ 10/10 ranking birebir yeniden üretildi.")
print("✅ Winner JSON doğrulandı.")
print("✅ Winner'ın 3/3 checkpoint'i derin doğrulandı.")
print("✅ 06 script SHA-256 manifest ile doğrulandı.")
print("✅ Test arrays loaded = False.")
print("✅ Test metrics computed = False.")
print("✅ Audit test verisine erişmedi.")
print("✅ Hiçbir dosya değiştirilmedi.")

print("\nRESMÎ MULTISEED WINNER:")
print("Architecture :", derived_winner["architecture"])
print("Loss         :", derived_winner["loss_strategy"])
print("Lookback     :", int(derived_winner["lookback"]))
print("Size         :", derived_winner["size"])
print("Feature set  :", derived_winner["feature_set"])
print("Source rank  :", int(derived_winner["source_rank"]))

print(
    "Mean ValidationScore :",
    f"{winner_mean:.16f}"
)

print(
    "Sample std           :",
    f"{winner_std:.16f}"
)

print(
    "Mean AvgReturnRatio  :",
    f"{winner_return_mean:.16f}"
)

print(
    "Mean AvgVolRatio     :",
    f"{winner_vol_mean:.16f}"
)

print("\nWINNER CONFIG ID:")
print(derived_winner_id)

print("\n" + "=" * 110)
print("SON HÜKÜM: 06 AŞAMASI AUDIT EDİLDİ VE 07 ÖNCESİ KAPATILMAYA HAZIR.")
print("=" * 110)

In [ ]:
# ==========================================================
# 07 PRE-RUN HASH + COMPILE + MANIFEST KONTROLÜ
#
# Amaç:
# - 07 script gerçekten doğru dosya mı?
# - Python syntax geçiyor mu?
# - Hazırlanan resmî 07 sürümüyle SHA-256 birebir aynı mı?
# - code_manifest_v4.csv'ye güvenli biçimde eklendi mi?
#
# Bu hücre:
# - 07 scriptini DEĞİŞTİRMEZ
# - Test setini AÇMAZ
# - Model değerlendirmesi BAŞLATMAZ
# - Yalnızca doğrulama yapar ve manifesti günceller
# ==========================================================

import os
import hashlib
import py_compile
from datetime import datetime, timezone

import pandas as pd


# ----------------------------------------------------------
# 1. YOLLAR
# ----------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

SCRIPT_PATH = os.path.join(
    BASE_DIR,
    "scripts",
    "07_final_test_evaluation_v4.py"
)

MANIFEST_PATH = os.path.join(
    BASE_DIR,
    "config",
    "code_manifest_v4.csv"
)


# ----------------------------------------------------------
# 2. BEKLENEN RESMÎ 07 HASH'İ
# ----------------------------------------------------------

EXPECTED_SHA256 = (
    "8b0e3cf2edb9508b4fddd402ddcdbf8c"
    "4d2acd6080ffe6fe1876ad818306cd74"
)


# ----------------------------------------------------------
# 3. SHA-256 FONKSİYONU
# ----------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha.update(chunk)

    return sha.hexdigest()


# ----------------------------------------------------------
# 4. DOSYA VARLIK KONTROLÜ
# ----------------------------------------------------------

if not os.path.exists(SCRIPT_PATH):

    raise FileNotFoundError(
        f"07 script bulunamadı:\n{SCRIPT_PATH}"
    )

if not os.path.exists(MANIFEST_PATH):

    raise FileNotFoundError(
        f"Manifest bulunamadı:\n{MANIFEST_PATH}"
    )


print("=" * 110)
print("07 PRE-RUN KONTROLÜ")
print("=" * 110)

print(f"\nScript:\n{SCRIPT_PATH}")

print(f"\nManifest:\n{MANIFEST_PATH}")


# ----------------------------------------------------------
# 5. PYTHON COMPILE KONTROLÜ
# ----------------------------------------------------------

try:

    py_compile.compile(
        SCRIPT_PATH,
        doraise=True
    )

    compile_passed = True

except py_compile.PyCompileError as error:

    compile_passed = False

    raise RuntimeError(
        "07 script Python syntax kontrolünden geçemedi."
    ) from error


print("\n✅ Python compile kontrolü geçti.")


# ----------------------------------------------------------
# 6. SHA-256 KONTROLÜ
# ----------------------------------------------------------

actual_sha256 = sha256_file(
    SCRIPT_PATH
)

print("\nSHA-256")

print(f"Beklenen : {EXPECTED_SHA256}")
print(f"Gerçek   : {actual_sha256}")


if actual_sha256 != EXPECTED_SHA256:

    raise RuntimeError(
        "\n07 script hash'i beklenen resmî sürümle eşleşmiyor.\n"
        "ÇALIŞTIRMA DURDURULDU.\n\n"
        f"Beklenen: {EXPECTED_SHA256}\n"
        f"Gerçek  : {actual_sha256}"
    )


print("\n✅ 07 script SHA-256 birebir eşleşti.")


# ----------------------------------------------------------
# 7. MEVCUT MANIFESTİ OKU
# ----------------------------------------------------------

manifest_df = pd.read_csv(
    MANIFEST_PATH
)

required_columns = [
    "script_name",
    "relative_path",
    "sha256",
    "size_bytes",
    "modified_utc",
    "manifest_created_utc",
]

missing_columns = [
    col
    for col in required_columns
    if col not in manifest_df.columns
]

if missing_columns:

    raise RuntimeError(
        "Manifest schema eksik.\n"
        f"Eksik kolonlar: {missing_columns}"
    )


print(
    f"\nMevcut manifest satır sayısı: "
    f"{len(manifest_df)}"
)


# ----------------------------------------------------------
# 8. 07 KAYDINI HAZIRLA
# ----------------------------------------------------------

script_name = os.path.basename(
    SCRIPT_PATH
)

relative_path = os.path.relpath(
    SCRIPT_PATH,
    BASE_DIR
)

size_bytes = os.path.getsize(
    SCRIPT_PATH
)

modified_utc = datetime.fromtimestamp(
    os.path.getmtime(SCRIPT_PATH),
    tz=timezone.utc
).isoformat()

manifest_created_utc = datetime.now(
    timezone.utc
).isoformat()


new_row = {
    "script_name": script_name,
    "relative_path": relative_path,
    "sha256": actual_sha256,
    "size_bytes": size_bytes,
    "modified_utc": modified_utc,
    "manifest_created_utc": manifest_created_utc,
}


# ----------------------------------------------------------
# 9. DUPLICATE / VERSİYON KONTROLÜ
# ----------------------------------------------------------

same_exact_row = (
    (manifest_df["script_name"] == script_name)
    &
    (manifest_df["sha256"] == actual_sha256)
)


if same_exact_row.any():

    print(
        "\nℹ️ 07 script aynı SHA-256 ile zaten manifestte kayıtlı."
    )

    manifest_updated = False

else:

    manifest_df = pd.concat(
        [
            manifest_df,
            pd.DataFrame([new_row])
        ],
        ignore_index=True
    )

    manifest_df.to_csv(
        MANIFEST_PATH,
        index=False,
        encoding="utf-8"
    )

    manifest_updated = True

    print(
        "\n✅ 07 script code_manifest_v4.csv dosyasına eklendi."
    )


# ----------------------------------------------------------
# 10. KAYIT SONRASI YENİDEN DOĞRULAMA
# ----------------------------------------------------------

saved_manifest = pd.read_csv(
    MANIFEST_PATH
)

matching_rows = saved_manifest[
    (
        saved_manifest["script_name"] == script_name
    )
    &
    (
        saved_manifest["sha256"] == actual_sha256
    )
].copy()


if len(matching_rows) < 1:

    raise RuntimeError(
        "07 script manifest kaydı doğrulanamadı."
    )


current_sha256_check = sha256_file(
    SCRIPT_PATH
)

if current_sha256_check != actual_sha256:

    raise RuntimeError(
        "07 script hash'i kontrol sırasında değişti."
    )


# ----------------------------------------------------------
# 11. SONUÇ
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("07 PRE-RUN KONTROLÜ — TAMAMLANDI")
print("=" * 110)

print(f"\nScript adı       : {script_name}")
print(f"Boyut            : {size_bytes} bytes")
print(f"SHA-256          : {actual_sha256}")
print(f"Compile passed   : {compile_passed}")
print(f"Manifest updated : {manifest_updated}")

print("\n✅ 07 script bulundu.")
print("✅ Python syntax kontrolünden geçti.")
print("✅ Beklenen resmî SHA-256 ile birebir eşleşti.")
print("✅ Manifest kaydı doğrulandı.")
print("✅ 07 script değiştirilmedi.")
print("✅ Test seti açılmadı.")
print("✅ Model değerlendirmesi başlatılmadı.")

print("=" * 110)

In [ ]:
!python "/content/drive/MyDrive/tez_transformer_v4_repro/scripts/07_final_test_evaluation_v4.py"

In [ ]:
# ==========================================================
# 07_FINAL_AUDIT_v4 — SALT OKUNUR NİHAİ TEST AUDİTİ
#
# AMAÇ:
# 1. 07 script SHA-256 + manifest kaydını doğrula
# 2. 06'dan gelen kilitli winner config'i doğrula
# 3. Exact 3 winner checkpoint'ini doğrula
# 4. Test seti / raw output boyutlarını doğrula
# 5. Seed tahminlerini doğrula
# 6. Ensemble'ın tam olarak 3 seed tahmininin
#    ham ölçekte aritmetik ortalaması olduğunu doğrula
# 7. 32 test metric satırını raw prediction'lardan
#    bağımsız yeniden hesapla
# 8. 4 satırlık final summary'yi bağımsız yeniden üret
# 9. CSV ↔ JSON tutarlılığını doğrula
# 10. Primary prediction'ın yalnızca
#     FinalWinner_3SeedEnsemble olduğunu doğrula
#
# BU HÜCRE:
# - HİÇBİR DOSYA YAZMAZ
# - HİÇBİR DOSYA DEĞİŞTİRMEZ
# - MODEL EĞİTMEZ
# - MODEL SEÇİMİ YAPMAZ
# - TEST SONUCUNA GÖRE KARAR DEĞİŞTİRMEZ
# ==========================================================

import os
import json
import hashlib

import numpy as np
import pandas as pd
import torch


# ==========================================================
# 1. YOLLAR
# ==========================================================

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

CONFIG_DIR = os.path.join(
    BASE_DIR,
    "config"
)

SCRIPTS_DIR = os.path.join(
    BASE_DIR,
    "scripts"
)

MULTISEED_RESULTS_DIR = os.path.join(
    BASE_DIR,
    "results",
    "multiseed"
)

FINAL_TEST_DIR = os.path.join(
    BASE_DIR,
    "results",
    "final_test"
)


SCRIPT_07_PATH = os.path.join(
    SCRIPTS_DIR,
    "07_final_test_evaluation_v4.py"
)

CODE_MANIFEST_PATH = os.path.join(
    CONFIG_DIR,
    "code_manifest_v4.csv"
)

WINNER_JSON = os.path.join(
    MULTISEED_RESULTS_DIR,
    "multiseed_winner_config_v4.json"
)

METRICS_LONG_CSV = os.path.join(
    FINAL_TEST_DIR,
    "final_test_metrics_long_v4.csv"
)

SUMMARY_CSV = os.path.join(
    FINAL_TEST_DIR,
    "final_test_summary_v4.csv"
)

SUMMARY_JSON = os.path.join(
    FINAL_TEST_DIR,
    "final_test_summary_v4.json"
)

Y_TRUE_RAW_PATH = os.path.join(
    FINAL_TEST_DIR,
    "final_test_y_true_raw_v4.npy"
)

ENSEMBLE_PRED_PATH = os.path.join(
    FINAL_TEST_DIR,
    "pred_final_ensemble_raw_v4.npy"
)


# ==========================================================
# 2. KİLİTLİ BEKLENEN DEĞERLER
# ==========================================================

EXPECTED_07_SHA256 = (
    "8b0e3cf2edb9508b4fddd402ddcdbf8c"
    "4d2acd6080ffe6fe1876ad818306cd74"
)

EXPECTED_WINNER_CONFIG_ID = (
    "arch=NoSharing"
    "__loss=FixedLambda_0.7"
    "__lb=10"
    "__size=small"
    "__feat=baseline"
)

EXPECTED_WINNER = {
    "architecture": "NoSharing",
    "loss_strategy": "FixedLambda_0.7",
    "lookback": 10,
    "size": "small",
    "feature_set": "baseline",
}

EXPECTED_SEEDS = [
    123,
    777,
    2026
]

EXPECTED_TEST_SAMPLES = 584

EXPECTED_ASSETS = [
    "BIST100",
    "USDTRY",
    "EURTRY",
    "GOLD"
]

EXPECTED_MODEL_LABELS = [
    "FinalWinner_Seed123",
    "FinalWinner_Seed777",
    "FinalWinner_Seed2026",
    "FinalWinner_3SeedEnsemble",
]

EXPECTED_PRIMARY_LABEL = (
    "FinalWinner_3SeedEnsemble"
)

EXPECTED_PRIMARY_POLICY = (
    "arithmetic_mean_of_three_locked_winner_seed_predictions_in_raw_scale"
)

TAU = 0.5

FLOAT_TOL = 1e-12


# ==========================================================
# 3. YARDIMCI FONKSİYONLAR
# ==========================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024
):

    sha = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            sha.update(
                chunk
            )

    return sha.hexdigest()


def normalize_bool(
    value
):

    if isinstance(
        value,
        (bool, np.bool_)
    ):
        return bool(
            value
        )

    if value is None:
        return False

    if (
        isinstance(
            value,
            float
        )
        and np.isnan(
            value
        )
    ):
        return False

    text = str(
        value
    ).strip().lower()

    if text in {
        "true",
        "1",
        "yes"
    }:
        return True

    if text in {
        "false",
        "0",
        "no",
        ""
    }:
        return False

    raise ValueError(
        f"Boolean değere çevrilemeyen kayıt: {value!r}"
    )


def assert_close(
    actual,
    expected,
    name,
    tol=FLOAT_TOL
):

    actual = float(
        actual
    )

    expected = float(
        expected
    )

    if not np.isclose(
        actual,
        expected,
        rtol=0.0,
        atol=tol
    ):

        raise RuntimeError(
            f"{name} uyuşmuyor.\n"
            f"Beklenen: {expected}\n"
            f"Gerçek   : {actual}\n"
            f"Fark     : {abs(actual - expected)}"
        )


def mae_np(
    y_true,
    y_pred
):

    return float(
        np.mean(
            np.abs(
                y_true
                - y_pred
            )
        )
    )


def rmse_np(
    y_true,
    y_pred
):

    return float(
        np.sqrt(
            np.mean(
                (
                    y_true
                    - y_pred
                ) ** 2
            )
        )
    )


def r2_np(
    y_true,
    y_pred
):

    ss_res = np.sum(
        (
            y_true
            - y_pred
        ) ** 2
    )

    ss_tot = np.sum(
        (
            y_true
            - np.mean(
                y_true
            )
        ) ** 2
    )

    if ss_tot == 0:

        return float(
            "nan"
        )

    return float(
        1.0
        - ss_res
        / ss_tot
    )


def pinball_np(
    y_true,
    y_pred,
    tau=0.5
):

    diff = (
        y_true
        - y_pred
    )

    loss = np.maximum(
        tau * diff,
        (
            tau
            - 1.0
        ) * diff
    )

    return float(
        np.mean(
            loss
        )
    )


def load_checkpoint_cpu(
    path
):

    try:

        return torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

    except TypeError:

        return torch.load(
            path,
            map_location="cpu"
        )


def compute_raw_metrics(
    y_true_raw,
    y_pred_raw,
    model_label,
    seed_value,
    primary_prediction
):

    rows = []

    for i, asset in enumerate(
        EXPECTED_ASSETS
    ):

        true = y_true_raw[
            :,
            i
        ]

        pred = y_pred_raw[
            :,
            i
        ]

        rows.append({
            "model_label":
                model_label,

            "seed":
                seed_value,

            "primary_prediction":
                primary_prediction,

            "task":
                "return",

            "asset":
                asset,

            "MAE":
                mae_np(
                    true,
                    pred
                ),

            "RMSE":
                rmse_np(
                    true,
                    pred
                ),

            "R2":
                r2_np(
                    true,
                    pred
                ),

            "PinballLoss_tau_0.5":
                np.nan,
        })


    for i, asset in enumerate(
        EXPECTED_ASSETS
    ):

        col = (
            4
            + i
        )

        true = y_true_raw[
            :,
            col
        ]

        pred = y_pred_raw[
            :,
            col
        ]

        rows.append({
            "model_label":
                model_label,

            "seed":
                seed_value,

            "primary_prediction":
                primary_prediction,

            "task":
                "volatility",

            "asset":
                asset,

            "MAE":
                mae_np(
                    true,
                    pred
                ),

            "RMSE":
                rmse_np(
                    true,
                    pred
                ),

            "R2":
                r2_np(
                    true,
                    pred
                ),

            "PinballLoss_tau_0.5":
                pinball_np(
                    true,
                    pred,
                    tau=TAU
                ),
        })


    return pd.DataFrame(
        rows
    )


def summarize_metrics(
    metrics_df,
    model_label,
    seed_value,
    primary_prediction
):

    ret = metrics_df[
        metrics_df[
            "task"
        ]
        == "return"
    ]

    vol = metrics_df[
        metrics_df[
            "task"
        ]
        == "volatility"
    ]

    return {
        "model_label":
            model_label,

        "seed":
            seed_value,

        "primary_prediction":
            primary_prediction,

        "avg_return_mae":
            float(
                ret[
                    "MAE"
                ].mean()
            ),

        "avg_return_rmse":
            float(
                ret[
                    "RMSE"
                ].mean()
            ),

        "avg_return_r2":
            float(
                ret[
                    "R2"
                ].mean()
            ),

        "avg_vol_mae":
            float(
                vol[
                    "MAE"
                ].mean()
            ),

        "avg_vol_rmse":
            float(
                vol[
                    "RMSE"
                ].mean()
            ),

        "avg_vol_r2":
            float(
                vol[
                    "R2"
                ].mean()
            ),

        "avg_vol_pinball_tau_0.5":
            float(
                vol[
                    "PinballLoss_tau_0.5"
                ].mean()
            ),
    }


# ==========================================================
# 4. GEREKLİ DOSYALAR
# ==========================================================

required_paths = [
    SCRIPT_07_PATH,
    CODE_MANIFEST_PATH,
    WINNER_JSON,
    METRICS_LONG_CSV,
    SUMMARY_CSV,
    SUMMARY_JSON,
    Y_TRUE_RAW_PATH,
    ENSEMBLE_PRED_PATH,
]

for seed in EXPECTED_SEEDS:

    required_paths.append(
        os.path.join(
            FINAL_TEST_DIR,
            f"pred_final_seed{seed}_raw_v4.npy"
        )
    )


missing_paths = [
    path
    for path in required_paths
    if not os.path.exists(
        path
    )
]


if missing_paths:

    raise FileNotFoundError(
        "Gerekli dosyalar eksik:\n"
        + "\n".join(
            missing_paths
        )
    )


print(
    "=" * 110
)

print(
    "07_FINAL_AUDIT_v4 — BAŞLADI"
)

print(
    "=" * 110
)

print(
    "\n✅ Gerekli tüm 07 çıktı dosyaları bulundu."
)

print(
    "✅ Audit salt okunur modda çalışıyor."
)

print(
    "✅ Model eğitimi yapılmayacak."
)

print(
    "✅ Model seçimi yapılmayacak."
)


# ==========================================================
# 5. 07 SCRIPT SHA-256 + MANIFEST
# ==========================================================

actual_07_sha = sha256_file(
    SCRIPT_07_PATH
)

if (
    actual_07_sha
    != EXPECTED_07_SHA256
):

    raise RuntimeError(
        "07 script SHA-256 beklenen resmî sürümle eşleşmiyor.\n"
        f"Beklenen: {EXPECTED_07_SHA256}\n"
        f"Gerçek  : {actual_07_sha}"
    )


manifest_df = pd.read_csv(
    CODE_MANIFEST_PATH
)


manifest_match = manifest_df[
    (
        manifest_df[
            "script_name"
        ]
        == "07_final_test_evaluation_v4.py"
    )
    &
    (
        manifest_df[
            "sha256"
        ]
        == actual_07_sha
    )
]


if len(
    manifest_match
) < 1:

    raise RuntimeError(
        "07 script mevcut SHA-256 ile "
        "code_manifest_v4.csv içinde bulunamadı."
    )


print(
    "\n"
    + "-" * 110
)

print(
    "A. CODE PROVENANCE"
)

print(
    "-" * 110
)

print(
    "✅ 07 script SHA-256 beklenen resmî hash ile eşleşiyor."
)

print(
    "✅ 07 script mevcut hash ile manifest içinde kayıtlı."
)

print(
    "SHA-256:",
    actual_07_sha
)


# ==========================================================
# 6. 06 WINNER JSON + KİLİTLİ CONFIG
# ==========================================================

with open(
    WINNER_JSON,
    "r",
    encoding="utf-8"
) as f:

    winner_payload = json.load(
        f
    )


winner = winner_payload[
    "winner"
]


if str(
    winner[
        "config_id"
    ]
) != EXPECTED_WINNER_CONFIG_ID:

    raise RuntimeError(
        "Winner config_id kilitli değerle uyuşmuyor.\n"
        f"Beklenen: {EXPECTED_WINNER_CONFIG_ID}\n"
        f"Gerçek  : {winner['config_id']}"
    )


for key, expected in EXPECTED_WINNER.items():

    actual = winner[
        key
    ]

    if str(
        actual
    ) != str(
        expected
    ):

        raise RuntimeError(
            f"Winner {key} uyuşmuyor.\n"
            f"Beklenen: {expected}\n"
            f"Gerçek  : {actual}"
        )


winner_seed_checkpoints = list(
    winner[
        "seed_checkpoints"
    ]
)


actual_winner_seeds = sorted(
    int(
        item[
            "seed"
        ]
    )
    for item in winner_seed_checkpoints
)


if (
    actual_winner_seeds
    != EXPECTED_SEEDS
):

    raise RuntimeError(
        "Winner seed set yanlış.\n"
        f"Beklenen: {EXPECTED_SEEDS}\n"
        f"Gerçek  : {actual_winner_seeds}"
    )


print(
    "\n"
    + "-" * 110
)

print(
    "B. KİLİTLİ WINNER KONTROLÜ"
)

print(
    "-" * 110
)

print(
    "✅ Winner config_id doğru."
)

print(
    "✅ Architecture = NoSharing"
)

print(
    "✅ Loss = FixedLambda_0.7"
)

print(
    "✅ Lookback = 10"
)

print(
    "✅ Size = small"
)

print(
    "✅ Feature set = baseline"
)

print(
    "✅ Winner seeds = [123, 777, 2026]"
)


# ==========================================================
# 7. 3/3 WINNER CHECKPOINT DERİN KONTROLÜ
# ==========================================================

checkpoint_audit_rows = []


for item in sorted(
    winner_seed_checkpoints,
    key=lambda x: int(
        x[
            "seed"
        ]
    )
):

    seed = int(
        item[
            "seed"
        ]
    )

    checkpoint_path = str(
        item[
            "checkpoint_file"
        ]
    )


    if not os.path.exists(
        checkpoint_path
    ):

        raise FileNotFoundError(
            f"Winner checkpoint eksik:\n"
            f"{checkpoint_path}"
        )


    checkpoint = load_checkpoint_cpu(
        checkpoint_path
    )


    if str(
        checkpoint[
            "config"
        ][
            "config_id"
        ]
    ) != EXPECTED_WINNER_CONFIG_ID:

        raise RuntimeError(
            f"Checkpoint config_id yanlış: seed={seed}"
        )


    if int(
        checkpoint[
            "seed"
        ]
    ) != seed:

        raise RuntimeError(
            f"Checkpoint seed yanlış: seed={seed}"
        )


    if int(
        checkpoint[
            "epoch"
        ]
    ) != int(
        item[
            "best_epoch"
        ]
    ):

        raise RuntimeError(
            f"Checkpoint best_epoch yanlış: seed={seed}"
        )


    assert_close(
        checkpoint[
            "validation_score"
        ],
        item[
            "validation_score"
        ],
        f"Checkpoint ValidationScore seed={seed}"
    )


    checkpoint_audit_rows.append({
        "seed":
            seed,

        "best_epoch":
            int(
                checkpoint[
                    "epoch"
                ]
            ),

        "validation_score":
            float(
                checkpoint[
                    "validation_score"
                ]
            ),

        "checkpoint_sha256":
            sha256_file(
                checkpoint_path
            ),

        "checkpoint_file":
            checkpoint_path,
    })


checkpoint_audit_df = pd.DataFrame(
    checkpoint_audit_rows
)


print(
    "\n"
    + "-" * 110
)

print(
    "C. WINNER CHECKPOINT DERİN AUDİTİ"
)

print(
    "-" * 110
)

print(
    "✅ 3/3 checkpoint mevcut."
)

print(
    "✅ 3/3 config_id doğru."
)

print(
    "✅ 3/3 seed metadata doğru."
)

print(
    "✅ 3/3 best_epoch doğru."
)

print(
    "✅ 3/3 ValidationScore doğru."
)

print(
    checkpoint_audit_df[
        [
            "seed",
            "best_epoch",
            "validation_score"
        ]
    ].to_string(
        index=False
    )
)


# ==========================================================
# 8. 07 ÇIKTILARINI YÜKLE
# ==========================================================

metrics_stored = pd.read_csv(
    METRICS_LONG_CSV
)

summary_stored = pd.read_csv(
    SUMMARY_CSV
)


with open(
    SUMMARY_JSON,
    "r",
    encoding="utf-8"
) as f:

    summary_json = json.load(
        f
    )


y_true_raw = np.load(
    Y_TRUE_RAW_PATH
)


ensemble_pred_raw = np.load(
    ENSEMBLE_PRED_PATH
)


seed_predictions = {}


for seed in EXPECTED_SEEDS:

    seed_path = os.path.join(
        FINAL_TEST_DIR,
        f"pred_final_seed{seed}_raw_v4.npy"
    )

    seed_predictions[
        seed
    ] = np.load(
        seed_path
    )


# ==========================================================
# 9. DİZİ BOYUTU VE SONLULUK KONTROLÜ
# ==========================================================

expected_shape = (
    EXPECTED_TEST_SAMPLES,
    8
)


if (
    y_true_raw.shape
    != expected_shape
):

    raise RuntimeError(
        "y_true_raw shape yanlış.\n"
        f"Beklenen: {expected_shape}\n"
        f"Gerçek  : {y_true_raw.shape}"
    )


if (
    ensemble_pred_raw.shape
    != expected_shape
):

    raise RuntimeError(
        "Ensemble prediction shape yanlış.\n"
        f"Beklenen: {expected_shape}\n"
        f"Gerçek  : {ensemble_pred_raw.shape}"
    )


if not np.isfinite(
    y_true_raw
).all():

    raise RuntimeError(
        "y_true_raw içinde NaN/Inf var."
    )


if not np.isfinite(
    ensemble_pred_raw
).all():

    raise RuntimeError(
        "Ensemble prediction içinde NaN/Inf var."
    )


for seed in EXPECTED_SEEDS:

    arr = seed_predictions[
        seed
    ]

    if (
        arr.shape
        != expected_shape
    ):

        raise RuntimeError(
            f"Seed {seed} prediction shape yanlış: {arr.shape}"
        )

    if not np.isfinite(
        arr
    ).all():

        raise RuntimeError(
            f"Seed {seed} prediction içinde NaN/Inf var."
        )


print(
    "\n"
    + "-" * 110
)

print(
    "D. TEST OUTPUT DİZİ BÜTÜNLÜĞÜ"
)

print(
    "-" * 110
)

print(
    "✅ y_true_raw shape = (584, 8)"
)

print(
    "✅ Ensemble prediction shape = (584, 8)"
)

print(
    "✅ Seed 123 prediction shape = (584, 8)"
)

print(
    "✅ Seed 777 prediction shape = (584, 8)"
)

print(
    "✅ Seed 2026 prediction shape = (584, 8)"
)

print(
    "✅ Tüm target ve prediction değerleri finite."
)


# ==========================================================
# 10. ENSEMBLE'I BAĞIMSIZ YENİDEN ÜRET
# ==========================================================

ensemble_recomputed = np.mean(
    np.stack(
        [
            seed_predictions[
                seed
            ]
            for seed in EXPECTED_SEEDS
        ],
        axis=0
    ),
    axis=0
)


max_ensemble_diff = float(
    np.max(
        np.abs(
            ensemble_recomputed
            - ensemble_pred_raw
        )
    )
)


if (
    max_ensemble_diff
    > FLOAT_TOL
):

    raise RuntimeError(
        "Stored ensemble, 3 seed aritmetik ortalamasıyla uyuşmuyor.\n"
        f"Max fark: {max_ensemble_diff}"
    )


print(
    "\n"
    + "-" * 110
)

print(
    "E. 3-SEED ENSEMBLE YENİDEN ÜRETİMİ"
)

print(
    "-" * 110
)

print(
    "✅ Ensemble, exact 3 winner-seed prediction'dan yeniden üretildi."
)

print(
    "✅ Averaging raw target scale üzerinde doğrulandı."
)

print(
    "✅ Stored ensemble ile bağımsız yeniden üretim eşleşti."
)

print(
    "Maksimum ensemble farkı:",
    f"{max_ensemble_diff:.16e}"
)


# ==========================================================
# 11. 32 METRİK SATIRINI BAĞIMSIZ YENİDEN HESAPLA
# ==========================================================

recomputed_metric_frames = []


for seed in EXPECTED_SEEDS:

    model_label = (
        f"FinalWinner_Seed{seed}"
    )

    recomputed_metric_frames.append(
        compute_raw_metrics(
            y_true_raw=y_true_raw,
            y_pred_raw=seed_predictions[
                seed
            ],
            model_label=model_label,
            seed_value=seed,
            primary_prediction=False
        )
    )


recomputed_metric_frames.append(
    compute_raw_metrics(
        y_true_raw=y_true_raw,
        y_pred_raw=ensemble_pred_raw,
        model_label=EXPECTED_PRIMARY_LABEL,
        seed_value=np.nan,
        primary_prediction=True
    )
)


metrics_recomputed = pd.concat(
    recomputed_metric_frames,
    ignore_index=True
)


if len(
    metrics_stored
) != 32:

    raise RuntimeError(
        f"Stored metrics satır sayısı 32 değil: {len(metrics_stored)}"
    )


if len(
    metrics_recomputed
) != 32:

    raise RuntimeError(
        "Recomputed metrics satır sayısı 32 değil."
    )


stored_labels = sorted(
    metrics_stored[
        "model_label"
    ].unique().tolist()
)


if stored_labels != sorted(
    EXPECTED_MODEL_LABELS
):

    raise RuntimeError(
        "Stored model_label seti yanlış.\n"
        f"Beklenen: {sorted(EXPECTED_MODEL_LABELS)}\n"
        f"Gerçek  : {stored_labels}"
    )


key_cols = [
    "model_label",
    "task",
    "asset"
]


stored_indexed = (
    metrics_stored
    .set_index(
        key_cols
    )
    .sort_index()
)


recomputed_indexed = (
    metrics_recomputed
    .set_index(
        key_cols
    )
    .sort_index()
)


if list(
    stored_indexed.index
) != list(
    recomputed_indexed.index
):

    raise RuntimeError(
        "Stored ve recomputed metrics indexleri uyuşmuyor."
    )


numeric_metric_cols = [
    "MAE",
    "RMSE",
    "R2",
    "PinballLoss_tau_0.5"
]


max_metric_diff = 0.0


for idx in recomputed_indexed.index:

    for col in numeric_metric_cols:

        actual = recomputed_indexed.loc[
            idx,
            col
        ]

        expected = stored_indexed.loc[
            idx,
            col
        ]


        if (
            pd.isna(
                actual
            )
            and pd.isna(
                expected
            )
        ):
            continue


        diff = abs(
            float(
                actual
            )
            - float(
                expected
            )
        )


        max_metric_diff = max(
            max_metric_diff,
            diff
        )


        assert_close(
            actual,
            expected,
            (
                f"{col} | "
                f"{idx}"
            )
        )


print(
    "\n"
    + "-" * 110
)

print(
    "F. RAW PREDICTION → TEST METRİK YENİDEN HESABI"
)

print(
    "-" * 110
)

print(
    "✅ 32/32 metrics satırı raw prediction'lardan yeniden hesaplandı."
)

print(
    "✅ Return MAE yeniden doğrulandı."
)

print(
    "✅ Return RMSE yeniden doğrulandı."
)

print(
    "✅ Return R² yeniden doğrulandı."
)

print(
    "✅ Volatility MAE yeniden doğrulandı."
)

print(
    "✅ Volatility RMSE yeniden doğrulandı."
)

print(
    "✅ Volatility R² yeniden doğrulandı."
)

print(
    "✅ Volatility PinballLoss tau=0.5 yeniden doğrulandı."
)

print(
    "Maksimum metric farkı:",
    f"{max_metric_diff:.16e}"
)


# ==========================================================
# 12. 4 SATIRLIK SUMMARY'Yİ BAĞIMSIZ YENİDEN ÜRET
# ==========================================================

summary_rows_recomputed = []


for seed in EXPECTED_SEEDS:

    label = (
        f"FinalWinner_Seed{seed}"
    )

    df = metrics_recomputed[
        metrics_recomputed[
            "model_label"
        ]
        == label
    ].copy()

    summary_rows_recomputed.append(
        summarize_metrics(
            metrics_df=df,
            model_label=label,
            seed_value=seed,
            primary_prediction=False
        )
    )


ensemble_metrics_recomputed = (
    metrics_recomputed[
        metrics_recomputed[
            "model_label"
        ]
        == EXPECTED_PRIMARY_LABEL
    ].copy()
)


summary_rows_recomputed.append(
    summarize_metrics(
        metrics_df=
            ensemble_metrics_recomputed,

        model_label=
            EXPECTED_PRIMARY_LABEL,

        seed_value=
            np.nan,

        primary_prediction=
            True
    )
)


summary_recomputed = pd.DataFrame(
    summary_rows_recomputed
)


if len(
    summary_stored
) != 4:

    raise RuntimeError(
        f"Stored summary satır sayısı 4 değil: {len(summary_stored)}"
    )


if sorted(
    summary_stored[
        "model_label"
    ].tolist()
) != sorted(
    EXPECTED_MODEL_LABELS
):

    raise RuntimeError(
        "Stored summary model_label seti yanlış."
    )


stored_primary_count = sum(
    normalize_bool(
        value
    )
    for value in summary_stored[
        "primary_prediction"
    ]
)


if (
    stored_primary_count
    != 1
):

    raise RuntimeError(
        "Stored summary içinde exact 1 primary prediction yok."
    )


stored_primary_label = str(
    summary_stored.loc[
        summary_stored[
            "primary_prediction"
        ].apply(
            normalize_bool
        ),
        "model_label"
    ].iloc[
        0
    ]
)


if (
    stored_primary_label
    != EXPECTED_PRIMARY_LABEL
):

    raise RuntimeError(
        "Primary prediction label ensemble değil.\n"
        f"Gerçek: {stored_primary_label}"
    )


summary_stored_indexed = (
    summary_stored
    .set_index(
        "model_label"
    )
    .sort_index()
)


summary_recomputed_indexed = (
    summary_recomputed
    .set_index(
        "model_label"
    )
    .sort_index()
)


summary_numeric_cols = [
    "avg_return_mae",
    "avg_return_rmse",
    "avg_return_r2",
    "avg_vol_mae",
    "avg_vol_rmse",
    "avg_vol_r2",
    "avg_vol_pinball_tau_0.5"
]


max_summary_diff = 0.0


for label in EXPECTED_MODEL_LABELS:

    for col in summary_numeric_cols:

        actual = (
            summary_recomputed_indexed.loc[
                label,
                col
            ]
        )

        expected = (
            summary_stored_indexed.loc[
                label,
                col
            ]
        )


        diff = abs(
            float(
                actual
            )
            - float(
                expected
            )
        )


        max_summary_diff = max(
            max_summary_diff,
            diff
        )


        assert_close(
            actual,
            expected,
            (
                f"Summary {col} | {label}"
            )
        )


print(
    "\n"
    + "-" * 110
)

print(
    "G. FINAL SUMMARY YENİDEN ÜRETİMİ"
)

print(
    "-" * 110
)

print(
    "✅ 4/4 prediction set summary bağımsız yeniden üretildi."
)

print(
    "✅ Exact 1 primary prediction doğrulandı."
)

print(
    "✅ Primary prediction = FinalWinner_3SeedEnsemble"
)

print(
    "✅ Stored summary ile recomputed summary eşleşti."
)

print(
    "Maksimum summary farkı:",
    f"{max_summary_diff:.16e}"
)


# ==========================================================
# 13. SUMMARY JSON AUDİTİ
# ==========================================================

if normalize_bool(
    summary_json[
        "model_selection_inside_07"
    ]
):

    raise RuntimeError(
        "JSON model_selection_inside_07=True."
    )


if normalize_bool(
    summary_json[
        "hyperparameter_change_inside_07"
    ]
):

    raise RuntimeError(
        "JSON hyperparameter_change_inside_07=True."
    )


if normalize_bool(
    summary_json[
        "retraining_inside_07"
    ]
):

    raise RuntimeError(
        "JSON retraining_inside_07=True."
    )


if normalize_bool(
    summary_json[
        "baseline_comparison_inside_07"
    ]
):

    raise RuntimeError(
        "JSON baseline_comparison_inside_07=True."
    )


if normalize_bool(
    summary_json[
        "statistical_model_comparison_inside_07"
    ]
):

    raise RuntimeError(
        "JSON statistical_model_comparison_inside_07=True."
    )


if not normalize_bool(
    summary_json[
        "test_access_started"
    ]
):

    raise RuntimeError(
        "JSON test_access_started=False."
    )


if not normalize_bool(
    summary_json[
        "test_metrics_computed"
    ]
):

    raise RuntimeError(
        "JSON test_metrics_computed=False."
    )


if str(
    summary_json[
        "winner_config_id"
    ]
) != EXPECTED_WINNER_CONFIG_ID:

    raise RuntimeError(
        "JSON winner_config_id yanlış."
    )


if sorted(
    summary_json[
        "expected_seeds"
    ]
) != EXPECTED_SEEDS:

    raise RuntimeError(
        "JSON expected seeds yanlış."
    )


if int(
    summary_json[
        "test_sample_count"
    ]
) != EXPECTED_TEST_SAMPLES:

    raise RuntimeError(
        "JSON test_sample_count 584 değil."
    )


if str(
    summary_json[
        "primary_prediction"
    ]
) != EXPECTED_PRIMARY_LABEL:

    raise RuntimeError(
        "JSON primary_prediction yanlış."
    )


if str(
    summary_json[
        "primary_test_policy"
    ]
) != EXPECTED_PRIMARY_POLICY:

    raise RuntimeError(
        "JSON primary_test_policy yanlış."
    )


if not normalize_bool(
    summary_json[
        "06_script_sha256_verified"
    ]
):

    raise RuntimeError(
        "JSON 06_script_sha256_verified=False."
    )


print(
    "\n"
    + "-" * 110
)

print(
    "H. SUMMARY JSON + METODOLOJİK KURAL AUDİTİ"
)

print(
    "-" * 110
)

print(
    "✅ model_selection_inside_07 = False"
)

print(
    "✅ hyperparameter_change_inside_07 = False"
)

print(
    "✅ retraining_inside_07 = False"
)

print(
    "✅ baseline_comparison_inside_07 = False"
)

print(
    "✅ statistical_model_comparison_inside_07 = False"
)

print(
    "✅ test_access_started = True"
)

print(
    "✅ test_metrics_computed = True"
)

print(
    "✅ winner_config_id doğru"
)

print(
    "✅ expected_seeds = [123, 777, 2026]"
)

print(
    "✅ test_sample_count = 584"
)

print(
    "✅ primary_prediction = FinalWinner_3SeedEnsemble"
)

print(
    "✅ primary_test_policy doğru"
)


# ==========================================================
# 14. ANA FINAL TEST METRİKLERİNİ YENİDEN ÇIKAR
# ==========================================================

primary_metrics = (
    metrics_recomputed[
        metrics_recomputed[
            "model_label"
        ]
        == EXPECTED_PRIMARY_LABEL
    ]
    .copy()
)


primary_summary = (
    summary_recomputed[
        summary_recomputed[
            "model_label"
        ]
        == EXPECTED_PRIMARY_LABEL
    ]
    .iloc[
        0
    ]
)


# ==========================================================
# 15. SON RESMÎ AUDIT ÖZETİ
# ==========================================================

print(
    "\n"
    + "=" * 110
)

print(
    "07_FINAL_AUDIT_v4 — TÜM KONTROLLER GEÇTİ"
)

print(
    "=" * 110
)


print(
    "\nAUDIT SONUCU:"
)

print(
    "✅ 07 script SHA-256 manifest ile doğrulandı."
)

print(
    "✅ 06 kilitli winner config doğrulandı."
)

print(
    "✅ Exact seeds [123, 777, 2026] doğrulandı."
)

print(
    "✅ Winner'ın 3/3 checkpoint'i derin doğrulandı."
)

print(
    "✅ y_true_raw shape = (584, 8)."
)

print(
    "✅ 3/3 seed prediction shape = (584, 8)."
)

print(
    "✅ Ensemble prediction shape = (584, 8)."
)

print(
    "✅ Ensemble, raw ölçekte exact 3-seed aritmetik ortalama olarak yeniden üretildi."
)

print(
    "✅ 32/32 test metric satırı raw prediction'lardan bağımsız yeniden hesaplandı."
)

print(
    "✅ 4/4 final summary satırı bağımsız yeniden üretildi."
)

print(
    "✅ Exact 1 primary prediction doğrulandı."
)

print(
    "✅ Primary prediction = FinalWinner_3SeedEnsemble."
)

print(
    "✅ 07 içinde model seçimi yapılmadığı doğrulandı."
)

print(
    "✅ 07 içinde hyperparameter değişmediği doğrulandı."
)

print(
    "✅ 07 içinde yeniden eğitim yapılmadığı doğrulandı."
)

print(
    "✅ Test access started = True."
)

print(
    "✅ Test metrics computed = True."
)

print(
    "✅ Audit hiçbir dosyayı değiştirmedi."
)


print(
    "\nNUMERİK YENİDEN HESAP FARKLARI:"
)

print(
    "Max ensemble diff :",
    f"{max_ensemble_diff:.16e}"
)

print(
    "Max metric diff   :",
    f"{max_metric_diff:.16e}"
)

print(
    "Max summary diff  :",
    f"{max_summary_diff:.16e}"
)


print(
    "\nRESMÎ FINAL MODEL:"
)

print(
    EXPECTED_WINNER_CONFIG_ID
)


print(
    "\nPRIMARY TEST POLICY:"
)

print(
    EXPECTED_PRIMARY_POLICY
)


print(
    "\nPRIMARY FINAL TEST SUMMARY:"
)

print(
    f"Avg Return MAE           : "
    f"{float(primary_summary['avg_return_mae']):.16f}"
)

print(
    f"Avg Return RMSE          : "
    f"{float(primary_summary['avg_return_rmse']):.16f}"
)

print(
    f"Avg Return R²            : "
    f"{float(primary_summary['avg_return_r2']):.16f}"
)

print(
    f"Avg Volatility MAE       : "
    f"{float(primary_summary['avg_vol_mae']):.16f}"
)

print(
    f"Avg Volatility RMSE      : "
    f"{float(primary_summary['avg_vol_rmse']):.16f}"
)

print(
    f"Avg Volatility R²        : "
    f"{float(primary_summary['avg_vol_r2']):.16f}"
)

print(
    f"Avg Vol Pinball tau=0.5  : "
    f"{float(primary_summary['avg_vol_pinball_tau_0.5']):.16f}"
)


print(
    "\nPRIMARY ASSET-TASK METRİKLER:"
)

print(
    primary_metrics[
        [
            "task",
            "asset",
            "MAE",
            "RMSE",
            "R2",
            "PinballLoss_tau_0.5"
        ]
    ].to_string(
        index=False
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "SON HÜKÜM: 07 AŞAMASI AUDIT EDİLDİ VE 08 ÖNCESİ KAPATILMAYA HAZIR."
)

print(
    "=" * 110
)

In [ ]:
# ==========================================================
# 08A PRE-RUN HASH + COMPILE + MANIFEST KONTROLÜ
#
# Amaç:
# - 08A script doğru dosya mı?
# - Python syntax geçiyor mu?
# - Hazırlanan resmî 08A sürümüyle SHA-256 birebir aynı mı?
# - code_manifest_v4.csv'ye güvenli biçimde eklendi mi?
#
# Bu hücre:
# - 08A scriptini değiştirmez
# - Baseline hesabı başlatmaz
# - Sonuç üretmez
# - Sadece provenance doğrular ve manifesti günceller
# ==========================================================

import os
import hashlib
import py_compile
from datetime import datetime, timezone

import pandas as pd


# ----------------------------------------------------------
# 1. YOLLAR
# ----------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

SCRIPT_PATH = os.path.join(
    BASE_DIR,
    "scripts",
    "08A_naive_baselines_test_v4.py"
)

MANIFEST_PATH = os.path.join(
    BASE_DIR,
    "config",
    "code_manifest_v4.csv"
)


# ----------------------------------------------------------
# 2. BEKLENEN RESMÎ 08A HASH'İ
# ----------------------------------------------------------

EXPECTED_SHA256 = (
    "95a9658e97f57eaa1a9bb63ec29d8159"
    "432f06438bd57ff54fc8ab43013487e8"
)


# ----------------------------------------------------------
# 3. SHA-256 FONKSİYONU
# ----------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha.update(chunk)

    return sha.hexdigest()


# ----------------------------------------------------------
# 4. DOSYA VARLIK KONTROLÜ
# ----------------------------------------------------------

if not os.path.exists(SCRIPT_PATH):

    raise FileNotFoundError(
        f"08A script bulunamadı:\n{SCRIPT_PATH}"
    )

if not os.path.exists(MANIFEST_PATH):

    raise FileNotFoundError(
        f"Manifest bulunamadı:\n{MANIFEST_PATH}"
    )


print("=" * 110)
print("08A PRE-RUN KONTROLÜ")
print("=" * 110)

print(f"\nScript:\n{SCRIPT_PATH}")
print(f"\nManifest:\n{MANIFEST_PATH}")


# ----------------------------------------------------------
# 5. PYTHON COMPILE KONTROLÜ
# ----------------------------------------------------------

try:

    py_compile.compile(
        SCRIPT_PATH,
        doraise=True
    )

    compile_passed = True

except py_compile.PyCompileError as error:

    raise RuntimeError(
        "08A script Python syntax kontrolünden geçemedi."
    ) from error


print("\n✅ Python compile kontrolü geçti.")


# ----------------------------------------------------------
# 6. SHA-256 KONTROLÜ
# ----------------------------------------------------------

actual_sha256 = sha256_file(
    SCRIPT_PATH
)

print("\nSHA-256")
print(f"Beklenen : {EXPECTED_SHA256}")
print(f"Gerçek   : {actual_sha256}")


if actual_sha256 != EXPECTED_SHA256:

    raise RuntimeError(
        "\n08A script hash'i beklenen resmî sürümle eşleşmiyor.\n"
        "ÇALIŞTIRMA DURDURULDU.\n\n"
        f"Beklenen: {EXPECTED_SHA256}\n"
        f"Gerçek  : {actual_sha256}"
    )


print("\n✅ 08A script SHA-256 birebir eşleşti.")


# ----------------------------------------------------------
# 7. MEVCUT MANIFESTİ OKU
# ----------------------------------------------------------

manifest_df = pd.read_csv(
    MANIFEST_PATH
)

required_columns = [
    "script_name",
    "relative_path",
    "sha256",
    "size_bytes",
    "modified_utc",
    "manifest_created_utc",
]

missing_columns = [
    col
    for col in required_columns
    if col not in manifest_df.columns
]

if missing_columns:

    raise RuntimeError(
        "Manifest schema eksik.\n"
        f"Eksik kolonlar: {missing_columns}"
    )


print(
    f"\nMevcut manifest satır sayısı: {len(manifest_df)}"
)


# ----------------------------------------------------------
# 8. 08A KAYDINI HAZIRLA
# ----------------------------------------------------------

script_name = os.path.basename(
    SCRIPT_PATH
)

relative_path = os.path.relpath(
    SCRIPT_PATH,
    BASE_DIR
)

size_bytes = os.path.getsize(
    SCRIPT_PATH
)

modified_utc = datetime.fromtimestamp(
    os.path.getmtime(SCRIPT_PATH),
    tz=timezone.utc
).isoformat()

manifest_created_utc = datetime.now(
    timezone.utc
).isoformat()


new_row = {
    "script_name": script_name,
    "relative_path": relative_path,
    "sha256": actual_sha256,
    "size_bytes": size_bytes,
    "modified_utc": modified_utc,
    "manifest_created_utc": manifest_created_utc,
}


# ----------------------------------------------------------
# 9. DUPLICATE / VERSİYON KONTROLÜ
# ----------------------------------------------------------

same_exact_row = (
    (manifest_df["script_name"] == script_name)
    &
    (manifest_df["sha256"] == actual_sha256)
)


if same_exact_row.any():

    print(
        "\nℹ️ 08A script aynı SHA-256 ile zaten manifestte kayıtlı."
    )

    manifest_updated = False

else:

    manifest_df = pd.concat(
        [
            manifest_df,
            pd.DataFrame([new_row])
        ],
        ignore_index=True
    )

    manifest_df.to_csv(
        MANIFEST_PATH,
        index=False,
        encoding="utf-8"
    )

    manifest_updated = True

    print(
        "\n✅ 08A script code_manifest_v4.csv dosyasına eklendi."
    )


# ----------------------------------------------------------
# 10. KAYIT SONRASI YENİDEN DOĞRULAMA
# ----------------------------------------------------------

saved_manifest = pd.read_csv(
    MANIFEST_PATH
)

matching_rows = saved_manifest[
    (
        saved_manifest["script_name"] == script_name
    )
    &
    (
        saved_manifest["sha256"] == actual_sha256
    )
].copy()


if len(matching_rows) < 1:

    raise RuntimeError(
        "08A script manifest kaydı doğrulanamadı."
    )


current_sha256_check = sha256_file(
    SCRIPT_PATH
)

if current_sha256_check != actual_sha256:

    raise RuntimeError(
        "08A script hash'i kontrol sırasında değişti."
    )


# ----------------------------------------------------------
# 11. SONUÇ
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("08A PRE-RUN KONTROLÜ — TAMAMLANDI")
print("=" * 110)

print(f"\nScript adı       : {script_name}")
print(f"Boyut            : {size_bytes} bytes")
print(f"SHA-256          : {actual_sha256}")
print(f"Compile passed   : {compile_passed}")
print(f"Manifest updated : {manifest_updated}")

print("\n✅ 08A script bulundu.")
print("✅ Python syntax kontrolünden geçti.")
print("✅ Beklenen resmî SHA-256 ile birebir eşleşti.")
print("✅ Manifest kaydı doğrulandı.")
print("✅ 08A script değiştirilmedi.")
print("✅ Baseline hesabı henüz başlatılmadı.")

print("=" * 110)

In [ ]:
!python "/content/drive/MyDrive/tez_transformer_v4_repro/scripts/08A_naive_baselines_test_v4.py"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

print("Klasör var mı?:", os.path.exists(BASE_DIR))
print("İçerik:", os.listdir(BASE_DIR)[:20] if os.path.exists(BASE_DIR) else "Bulunamadı")

In [ ]:
!python "/content/drive/MyDrive/tez_transformer_v4_repro/scripts/08A_naive_baselines_test_v4.py"

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/tez_transformer_v4_repro")

SEARCH_DIRS = [
    BASE_DIR / "data" / "processed",
    BASE_DIR / "data" / "sequences",
    BASE_DIR / "config",
    BASE_DIR / "results",
]

KEYWORDS = [
    "date",
    "dates",
    "anchor",
    "target",
    "realization",
    "split",
    "metadata",
    "index",
    "timestamp",
]

matches = []

for search_dir in SEARCH_DIRS:
    if not search_dir.exists():
        continue

    for path in search_dir.rglob("*"):
        if not path.is_file():
            continue

        name_lower = path.name.lower()

        if any(keyword in name_lower for keyword in KEYWORDS):
            matches.append(path)

print("=" * 100)
print("V4 TARİH / SPLIT / METADATA DOSYA ARAMASI")
print("=" * 100)

if not matches:
    print("\n❌ Dosya adında tarih/split/metadata anahtar kelimesi taşıyan dosya bulunamadı.")
else:
    print(f"\nBulunan dosya sayısı: {len(matches)}\n")

    for i, path in enumerate(sorted(matches), start=1):
        relative = path.relative_to(BASE_DIR)
        print(f"{i:03d}. {relative}")

print("\n" + "=" * 100)
print("ARAMA TAMAMLANDI")
print("=" * 100)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/tez_transformer_v4_repro")

SEQ_DIR = (
    BASE_DIR
    / "data"
    / "sequences"
    / "baseline"
    / "lb10"
)

PATHS = {
    "anchor_dates_val":
        SEQ_DIR / "anchor_dates_val.npy",

    "anchor_dates_test":
        SEQ_DIR / "anchor_dates_test.npy",

    "target_dates_val":
        SEQ_DIR / "target_realization_dates_val.npy",

    "target_dates_test":
        SEQ_DIR / "target_realization_dates_test.npy",

    "split_meta":
        BASE_DIR / "data" / "processed" / "split_meta_v4.json",
}


print("=" * 110)
print("08A TARİH-SEMANTİĞİ ÖN KONTROLÜ")
print("=" * 110)


# ----------------------------------------------------------
# 1. DOSYA VARLIK KONTROLÜ
# ----------------------------------------------------------

for name, path in PATHS.items():

    print(
        f"{name:25s}: "
        f"{'VAR ✅' if path.exists() else 'YOK ❌'}"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Eksik dosya:\n{path}"
        )


# ----------------------------------------------------------
# 2. TARİH DİZİLERİNİ YÜKLE
# ----------------------------------------------------------

anchor_dates_val = np.load(
    PATHS["anchor_dates_val"],
    allow_pickle=True
)

anchor_dates_test = np.load(
    PATHS["anchor_dates_test"],
    allow_pickle=True
)

target_dates_val = np.load(
    PATHS["target_dates_val"],
    allow_pickle=True
)

target_dates_test = np.load(
    PATHS["target_dates_test"],
    allow_pickle=True
)


# Pandas datetime'e normalize et

anchor_dates_val = pd.to_datetime(
    anchor_dates_val
)

anchor_dates_test = pd.to_datetime(
    anchor_dates_test
)

target_dates_val = pd.to_datetime(
    target_dates_val
)

target_dates_test = pd.to_datetime(
    target_dates_test
)


# ----------------------------------------------------------
# 3. SHAPE / LENGTH
# ----------------------------------------------------------

print("\n" + "-" * 110)
print("A. TARİH DİZİ BOYUTLARI")
print("-" * 110)

print(
    "anchor_dates_val       :",
    len(anchor_dates_val)
)

print(
    "target_dates_val       :",
    len(target_dates_val)
)

print(
    "anchor_dates_test      :",
    len(anchor_dates_test)
)

print(
    "target_dates_test      :",
    len(target_dates_test)
)


# ----------------------------------------------------------
# 4. KRİTİK SINIR TARİHLERİ
# ----------------------------------------------------------

print("\n" + "-" * 110)
print("B. KRİTİK VAL → TEST SINIRI")
print("-" * 110)

print(
    "Son validation anchor              :",
    anchor_dates_val[-1]
)

print(
    "Son validation target realization  :",
    target_dates_val[-1]
)

print(
    "İlk test anchor                    :",
    anchor_dates_test[0]
)

print(
    "İlk test target realization        :",
    target_dates_test[0]
)


# ----------------------------------------------------------
# 5. EXACT SINIR EŞİTLİĞİ
# ----------------------------------------------------------

boundary_equal = (
    target_dates_val[-1]
    == anchor_dates_test[0]
)

print("\n" + "-" * 110)
print("C. FIRST TEST PERSISTENCE BOUNDARY")
print("-" * 110)

print(
    "target_dates_val[-1] == anchor_dates_test[0] :",
    boundary_equal
)


if not boundary_equal:

    raise RuntimeError(
        "Validation son target realization tarihi "
        "ilk test anchor tarihiyle eşleşmiyor."
    )


# ----------------------------------------------------------
# 6. TEST İÇİ 1-ADIM TARİH HİZALAMASI
# ----------------------------------------------------------

internal_alignment = (
    anchor_dates_test[1:]
    == target_dates_test[:-1]
)

all_internal_aligned = bool(
    np.all(
        internal_alignment
    )
)

print("\n" + "-" * 110)
print("D. TEST İÇİ 1-ADIM PERSISTENCE HİZALAMASI")
print("-" * 110)

print(
    "anchor_dates_test[1:] "
    "== target_dates_test[:-1] :",
    all_internal_aligned
)

print(
    "Doğru eşleşen satır sayısı:",
    int(
        np.sum(
            internal_alignment
        )
    ),
    "/",
    len(
        internal_alignment
    )
)


if not all_internal_aligned:

    bad_idx = np.where(
        ~internal_alignment
    )[0][:10]

    raise RuntimeError(
        "Test içi tarih hizalamasında uyuşmazlık var.\n"
        f"İlk hatalı indeksler: {bad_idx.tolist()}"
    )


# ----------------------------------------------------------
# 7. HER TEST TARGET'I ANCHOR'DAN SONRA MI?
# ----------------------------------------------------------

target_after_anchor = (
    target_dates_test
    > anchor_dates_test
)

all_targets_after_anchor = bool(
    np.all(
        target_after_anchor
    )
)

print("\n" + "-" * 110)
print("E. FORWARD TARGET TEMPORAL ORDER")
print("-" * 110)

print(
    "Tüm target realization tarihleri "
    "ilgili anchor tarihinden sonra mı?:",
    all_targets_after_anchor
)

print(
    "Doğru satır sayısı:",
    int(
        np.sum(
            target_after_anchor
        )
    ),
    "/",
    len(
        target_after_anchor
    )
)


if not all_targets_after_anchor:

    bad_idx = np.where(
        ~target_after_anchor
    )[0][:10]

    raise RuntimeError(
        "Target realization tarihi anchor'dan sonra olmayan "
        "satır bulundu.\n"
        f"İlk hatalı indeksler: {bad_idx.tolist()}"
    )


# ----------------------------------------------------------
# 8. MONOTONİKLİK
# ----------------------------------------------------------

anchor_test_increasing = bool(
    pd.Index(
        anchor_dates_test
    ).is_monotonic_increasing
)

target_test_increasing = bool(
    pd.Index(
        target_dates_test
    ).is_monotonic_increasing
)

print("\n" + "-" * 110)
print("F. KRONOLOJİK SIRALAMA")
print("-" * 110)

print(
    "anchor_dates_test monotonik artan :",
    anchor_test_increasing
)

print(
    "target_dates_test monotonik artan :",
    target_test_increasing
)


if not anchor_test_increasing:
    raise RuntimeError(
        "Test anchor tarihleri kronolojik değil."
    )

if not target_test_increasing:
    raise RuntimeError(
        "Test target realization tarihleri kronolojik değil."
    )


# ----------------------------------------------------------
# 9. SPLIT META'YI OKU
# ----------------------------------------------------------

with open(
    PATHS["split_meta"],
    "r",
    encoding="utf-8"
) as f:

    split_meta = json.load(
        f
    )


print("\n" + "-" * 110)
print("G. SPLIT META")
print("-" * 110)

print(
    json.dumps(
        split_meta,
        ensure_ascii=False,
        indent=2
    )
)


# ----------------------------------------------------------
# 10. SON HÜKÜM
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("08A TARİH-SEMANTİĞİ ÖN KONTROLÜ — TÜM KONTROLLER GEÇTİ")
print("=" * 110)

print(
    "✅ Son validation target realization "
    "= ilk test anchor."
)

print(
    "✅ İlk test persistence state'i "
    "geçmişte gerçekleşmiş bilgiye dayanıyor."
)

print(
    "✅ Test içindeki tüm persistence satırları "
    "bir önceki target realization ile hizalı."
)

print(
    "✅ Tüm target realization tarihleri "
    "ilgili anchor tarihinden sonra."
)

print(
    "✅ Test tarih dizileri kronolojik."
)

print(
    "✅ Off-by-one / look-ahead tarih semantiği "
    "seviyesinde bulunmadı."
)

print("=" * 110)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/tez_transformer_v4_repro")

print("Proje klasörü var mı?:", BASE_DIR.exists())
print("Yol:", BASE_DIR)

In [ ]:
# ==========================================================
# 08A_FINAL_AUDIT_v4 — GÜÇLENDİRİLMİŞ SALT OKUNUR AUDİT
#
# Bu audit bağımsız olarak şunları doğrular:
#
# A) CODE PROVENANCE
#    - 08A script SHA-256
#    - code_manifest_v4.csv kaydı
#
# B) KİLİTLİ FINAL MODEL
#    - NoSharing + FL0.7 + lb10 + small + baseline
#    - primary = 3-seed raw-scale ensemble
#    - seeds = [123, 777, 2026]
#
# C) TARİH-SEMANTİĞİ
#    - Son validation target realization
#      == ilk test anchor
#    - İlk test target realization > ilk test anchor
#    - Test içi 583/583:
#      anchor_test[i] == target_test[i-1]
#    - 584/584:
#      target_test[i] > anchor_test[i]
#    - Tarihler kronolojik
#    - split_meta ile tarih dizileri çapraz eşleşiyor
#
# D) PERSISTENCE DİZİLERİ
#    previous_observed_raw =
#      vstack([y_val_raw[-1], y_test_raw[:-1]])
#
#    - ReturnZero yeniden üret
#    - ReturnPersistence yeniden üret
#    - VolPersistence yeniden üret
#    - stored NPY dosyalarıyla birebir karşılaştır
#
# E) METRİKLER
#    - 20/20 08A metric satırını ham prediction'lardan
#      bağımsız yeniden hesapla
#
# F) 07 ↔ AUDIT ÇAPRAZ KONTROLÜ
#    - Final modelin 8 asset-task metriğini yeniden hesapla
#    - 07 resmî final metrics CSV ile doğrudan karşılaştır
#
# G) 12 BASELINE KARŞILAŞTIRMASI
#    - 4 ReturnZero
#    - 4 ReturnPersistence
#    - 4 VolPersistence
#
# H) 3 comparison summary satırı
#
# I) Strong-naive test oranları
#
# J) 09 için 20 loss-series
#
# K) JSON metodolojik kuralları
#
# BU HÜCRE:
# - HİÇBİR DOSYA YAZMAZ
# - HİÇBİR DOSYA DEĞİŞTİRMEZ
# - MODEL EĞİTMEZ
# - MODEL SEÇİMİ YAPMAZ
# - HYPERPARAMETER DEĞİŞTİRMEZ
# ==========================================================


import os
import json
import hashlib

import numpy as np
import pandas as pd


# ==========================================================
# 1. YOLLAR
# ==========================================================

BASE_DIR = "/content/drive/MyDrive/tez_transformer_v4_repro"

CONFIG_DIR = os.path.join(
    BASE_DIR,
    "config"
)

SCRIPTS_DIR = os.path.join(
    BASE_DIR,
    "scripts"
)

SEQUENCE_DIR = os.path.join(
    BASE_DIR,
    "data",
    "sequences",
    "baseline",
    "lb10"
)

PROCESSED_DIR = os.path.join(
    BASE_DIR,
    "data",
    "processed"
)

FINAL_TEST_DIR = os.path.join(
    BASE_DIR,
    "results",
    "final_test"
)

MULTISEED_DIR = os.path.join(
    BASE_DIR,
    "results",
    "multiseed"
)

NAIVE_DIR = os.path.join(
    BASE_DIR,
    "results",
    "baselines",
    "naive"
)


# ----------------------------------------------------------
# SCRIPT / MANIFEST
# ----------------------------------------------------------

SCRIPT_08A_PATH = os.path.join(
    SCRIPTS_DIR,
    "08A_naive_baselines_test_v4.py"
)

CODE_MANIFEST_PATH = os.path.join(
    CONFIG_DIR,
    "code_manifest_v4.csv"
)


# ----------------------------------------------------------
# 06 / 07 FINAL
# ----------------------------------------------------------

WINNER_JSON = os.path.join(
    MULTISEED_DIR,
    "multiseed_winner_config_v4.json"
)

FINAL_SUMMARY_JSON = os.path.join(
    FINAL_TEST_DIR,
    "final_test_summary_v4.json"
)

FINAL_METRICS_CSV = os.path.join(
    FINAL_TEST_DIR,
    "final_test_metrics_long_v4.csv"
)

FINAL_Y_TRUE_RAW = os.path.join(
    FINAL_TEST_DIR,
    "final_test_y_true_raw_v4.npy"
)

FINAL_ENSEMBLE_PRED_RAW = os.path.join(
    FINAL_TEST_DIR,
    "pred_final_ensemble_raw_v4.npy"
)


# ----------------------------------------------------------
# SEQUENCE RAW TARGETS
# ----------------------------------------------------------

Y_VAL_RAW_PATH = os.path.join(
    SEQUENCE_DIR,
    "y_val_raw.npy"
)

Y_TEST_RAW_SEQUENCE_PATH = os.path.join(
    SEQUENCE_DIR,
    "y_test_raw.npy"
)


# ----------------------------------------------------------
# TARİH DİZİLERİ
# ----------------------------------------------------------

ANCHOR_DATES_VAL_PATH = os.path.join(
    SEQUENCE_DIR,
    "anchor_dates_val.npy"
)

ANCHOR_DATES_TEST_PATH = os.path.join(
    SEQUENCE_DIR,
    "anchor_dates_test.npy"
)

TARGET_DATES_VAL_PATH = os.path.join(
    SEQUENCE_DIR,
    "target_realization_dates_val.npy"
)

TARGET_DATES_TEST_PATH = os.path.join(
    SEQUENCE_DIR,
    "target_realization_dates_test.npy"
)

SPLIT_META_PATH = os.path.join(
    PROCESSED_DIR,
    "split_meta_v4.json"
)


# ----------------------------------------------------------
# 08A OUTPUTS
# ----------------------------------------------------------

METRICS_LONG_CSV = os.path.join(
    NAIVE_DIR,
    "naive_baseline_metrics_long_v4.csv"
)

COMPARISON_CSV = os.path.join(
    NAIVE_DIR,
    "naive_baseline_comparison_v4.csv"
)

COMPARISON_SUMMARY_CSV = os.path.join(
    NAIVE_DIR,
    "naive_baseline_comparison_summary_v4.csv"
)

SUMMARY_JSON = os.path.join(
    NAIVE_DIR,
    "naive_baseline_summary_v4.json"
)

RETURN_ZERO_PRED_PATH = os.path.join(
    NAIVE_DIR,
    "pred_return_zero_raw_v4.npy"
)

RETURN_PERSISTENCE_PRED_PATH = os.path.join(
    NAIVE_DIR,
    "pred_return_persistence_raw_v4.npy"
)

VOL_PERSISTENCE_PRED_PATH = os.path.join(
    NAIVE_DIR,
    "pred_vol_persistence_raw_v4.npy"
)

LOSS_SERIES_PATH = os.path.join(
    NAIVE_DIR,
    "naive_baseline_loss_series_v4.npz"
)


# ==========================================================
# 2. KİLİTLİ BEKLENEN DEĞERLER
# ==========================================================

EXPECTED_08A_SHA256 = (
    "95a9658e97f57eaa1a9bb63ec29d8159"
    "432f06438bd57ff54fc8ab43013487e8"
)

EXPECTED_WINNER_CONFIG_ID = (
    "arch=NoSharing"
    "__loss=FixedLambda_0.7"
    "__lb=10"
    "__size=small"
    "__feat=baseline"
)

EXPECTED_PRIMARY_LABEL = (
    "FinalWinner_3SeedEnsemble"
)

EXPECTED_PRIMARY_POLICY = (
    "arithmetic_mean_of_three_locked_winner_seed_predictions_in_raw_scale"
)

EXPECTED_SEEDS = [
    123,
    777,
    2026
]

EXPECTED_ASSETS = [
    "BIST100",
    "USDTRY",
    "EURTRY",
    "GOLD"
]

EXPECTED_TEST_SAMPLES = 584

EXPECTED_TEST_SHAPE = (
    584,
    8
)

EXPECTED_DATE_BOUNDARY = pd.Timestamp(
    "2022-10-05"
)

EXPECTED_FIRST_TEST_TARGET_DATE = pd.Timestamp(
    "2022-10-06"
)

TAU = 0.5

WARNING_THRESHOLD = 1.30

SEVERE_THRESHOLD = 1.50

FLOAT_TOL = 1e-12


# ==========================================================
# 3. YARDIMCI FONKSİYONLAR
# ==========================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024
):
    sha = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            sha.update(
                chunk
            )

    return sha.hexdigest()


def normalize_bool(
    value
):
    if isinstance(
        value,
        (bool, np.bool_)
    ):
        return bool(
            value
        )

    if value is None:
        return False

    if (
        isinstance(
            value,
            float
        )
        and np.isnan(
            value
        )
    ):
        return False

    text = str(
        value
    ).strip().lower()

    if text in {
        "true",
        "1",
        "yes"
    }:
        return True

    if text in {
        "false",
        "0",
        "no",
        ""
    }:
        return False

    raise ValueError(
        f"Boolean değere çevrilemeyen kayıt: {value!r}"
    )


def assert_close(
    actual,
    expected,
    name,
    tol=FLOAT_TOL
):
    actual = float(
        actual
    )

    expected = float(
        expected
    )

    if not np.isclose(
        actual,
        expected,
        rtol=0.0,
        atol=tol
    ):

        raise RuntimeError(
            f"{name} uyuşmuyor.\n"
            f"Beklenen: {expected}\n"
            f"Gerçek   : {actual}\n"
            f"Fark     : {abs(actual - expected)}"
        )


def mae_np(
    y_true,
    y_pred
):
    return float(
        np.mean(
            np.abs(
                y_true
                - y_pred
            )
        )
    )


def rmse_np(
    y_true,
    y_pred
):
    return float(
        np.sqrt(
            np.mean(
                (
                    y_true
                    - y_pred
                ) ** 2
            )
        )
    )


def r2_np(
    y_true,
    y_pred
):
    ss_res = np.sum(
        (
            y_true
            - y_pred
        ) ** 2
    )

    ss_tot = np.sum(
        (
            y_true
            - np.mean(
                y_true
            )
        ) ** 2
    )

    if ss_tot == 0:

        return float(
            "nan"
        )

    return float(
        1.0
        - ss_res
        / ss_tot
    )


def pinball_loss_series_np(
    y_true,
    y_pred,
    tau=0.5
):
    diff = (
        y_true
        - y_pred
    )

    return np.maximum(
        tau * diff,
        (
            tau
            - 1.0
        ) * diff
    )


def pinball_np(
    y_true,
    y_pred,
    tau=0.5
):
    return float(
        np.mean(
            pinball_loss_series_np(
                y_true,
                y_pred,
                tau=tau
            )
        )
    )


def build_return_metric_rows(
    model_label,
    y_true_return,
    y_pred_return
):
    rows = []

    for i, asset in enumerate(
        EXPECTED_ASSETS
    ):

        true = y_true_return[
            :,
            i
        ]

        pred = y_pred_return[
            :,
            i
        ]

        rows.append({
            "model_label":
                model_label,

            "task":
                "return",

            "asset":
                asset,

            "MAE":
                mae_np(
                    true,
                    pred
                ),

            "RMSE":
                rmse_np(
                    true,
                    pred
                ),

            "R2":
                r2_np(
                    true,
                    pred
                ),

            "PinballLoss_tau_0.5":
                np.nan,
        })

    return rows


def build_vol_metric_rows(
    model_label,
    y_true_vol,
    y_pred_vol
):
    rows = []

    for i, asset in enumerate(
        EXPECTED_ASSETS
    ):

        true = y_true_vol[
            :,
            i
        ]

        pred = y_pred_vol[
            :,
            i
        ]

        rows.append({
            "model_label":
                model_label,

            "task":
                "volatility",

            "asset":
                asset,

            "MAE":
                mae_np(
                    true,
                    pred
                ),

            "RMSE":
                rmse_np(
                    true,
                    pred
                ),

            "R2":
                r2_np(
                    true,
                    pred
                ),

            "PinballLoss_tau_0.5":
                pinball_np(
                    true,
                    pred,
                    tau=TAU
                ),
        })

    return rows


# ==========================================================
# 4. GEREKLİ DOSYALAR
# ==========================================================

required_paths = [
    SCRIPT_08A_PATH,
    CODE_MANIFEST_PATH,
    WINNER_JSON,
    FINAL_SUMMARY_JSON,
    FINAL_METRICS_CSV,
    FINAL_Y_TRUE_RAW,
    FINAL_ENSEMBLE_PRED_RAW,
    Y_VAL_RAW_PATH,
    Y_TEST_RAW_SEQUENCE_PATH,
    ANCHOR_DATES_VAL_PATH,
    ANCHOR_DATES_TEST_PATH,
    TARGET_DATES_VAL_PATH,
    TARGET_DATES_TEST_PATH,
    SPLIT_META_PATH,
    METRICS_LONG_CSV,
    COMPARISON_CSV,
    COMPARISON_SUMMARY_CSV,
    SUMMARY_JSON,
    RETURN_ZERO_PRED_PATH,
    RETURN_PERSISTENCE_PRED_PATH,
    VOL_PERSISTENCE_PRED_PATH,
    LOSS_SERIES_PATH,
]


missing_paths = [
    path
    for path in required_paths
    if not os.path.exists(
        path
    )
]


if missing_paths:

    raise FileNotFoundError(
        "Gerekli dosyalar eksik:\n"
        + "\n".join(
            missing_paths
        )
    )


print(
    "=" * 110
)

print(
    "08A_FINAL_AUDIT_v4 — GÜÇLENDİRİLMİŞ AUDİT BAŞLADI"
)

print(
    "=" * 110
)

print(
    "\n✅ Gerekli tüm 08A ve tarih dosyaları bulundu."
)

print(
    "✅ Audit salt okunur modda çalışıyor."
)

print(
    "✅ Hiçbir model eğitilmeyecek."
)

print(
    "✅ Hiçbir model seçilmeyecek."
)

print(
    "✅ Hiçbir dosya değiştirilmeyecek."
)


# ==========================================================
# 5. A — CODE PROVENANCE
# ==========================================================

actual_08a_sha = sha256_file(
    SCRIPT_08A_PATH
)


if (
    actual_08a_sha
    != EXPECTED_08A_SHA256
):

    raise RuntimeError(
        "08A script SHA-256 beklenen resmî sürümle eşleşmiyor.\n"
        f"Beklenen: {EXPECTED_08A_SHA256}\n"
        f"Gerçek  : {actual_08a_sha}"
    )


manifest_df = pd.read_csv(
    CODE_MANIFEST_PATH
)


manifest_match = manifest_df[
    (
        manifest_df[
            "script_name"
        ]
        == "08A_naive_baselines_test_v4.py"
    )
    &
    (
        manifest_df[
            "sha256"
        ]
        == actual_08a_sha
    )
]


if len(
    manifest_match
) < 1:

    raise RuntimeError(
        "08A script mevcut SHA-256 ile manifest içinde bulunamadı."
    )


print(
    "\n"
    + "-" * 110
)

print(
    "A. CODE PROVENANCE"
)

print(
    "-" * 110
)

print(
    "✅ 08A script SHA-256 beklenen resmî hash ile eşleşiyor."
)

print(
    "✅ 08A script mevcut hash ile manifest içinde kayıtlı."
)

print(
    "SHA-256:",
    actual_08a_sha
)


# ==========================================================
# 6. B — KİLİTLİ FINAL MODEL
# ==========================================================

with open(
    WINNER_JSON,
    "r",
    encoding="utf-8"
) as f:

    winner_payload = json.load(
        f
    )


winner = winner_payload[
    "winner"
]


if str(
    winner[
        "config_id"
    ]
) != EXPECTED_WINNER_CONFIG_ID:

    raise RuntimeError(
        "Kilitli winner config değişmiş."
    )


with open(
    FINAL_SUMMARY_JSON,
    "r",
    encoding="utf-8"
) as f:

    final_summary_json = json.load(
        f
    )


if str(
    final_summary_json[
        "winner_config_id"
    ]
) != EXPECTED_WINNER_CONFIG_ID:

    raise RuntimeError(
        "07 final winner config uyuşmuyor."
    )


if str(
    final_summary_json[
        "primary_prediction"
    ]
) != EXPECTED_PRIMARY_LABEL:

    raise RuntimeError(
        "07 primary prediction ensemble değil."
    )


if str(
    final_summary_json[
        "primary_test_policy"
    ]
) != EXPECTED_PRIMARY_POLICY:

    raise RuntimeError(
        "07 primary test policy değişmiş."
    )


if sorted(
    final_summary_json[
        "expected_seeds"
    ]
) != EXPECTED_SEEDS:

    raise RuntimeError(
        "07 expected seeds yanlış."
    )


print(
    "\n"
    + "-" * 110
)

print(
    "B. KİLİTLİ FINAL MODEL KONTROLÜ"
)

print(
    "-" * 110
)

print(
    "✅ Final winner config değişmedi."
)

print(
    "✅ Primary prediction = FinalWinner_3SeedEnsemble"
)

print(
    "✅ Primary test policy değişmedi."
)

print(
    "✅ Winner seeds = [123, 777, 2026]"
)


# ==========================================================
# 7. C — TARİH DİZİLERİNİ YÜKLE VE NORMALİZE ET
# ==========================================================

anchor_dates_val = pd.to_datetime(
    np.load(
        ANCHOR_DATES_VAL_PATH,
        allow_pickle=True
    )
)

anchor_dates_test = pd.to_datetime(
    np.load(
        ANCHOR_DATES_TEST_PATH,
        allow_pickle=True
    )
)

target_dates_val = pd.to_datetime(
    np.load(
        TARGET_DATES_VAL_PATH,
        allow_pickle=True
    )
)

target_dates_test = pd.to_datetime(
    np.load(
        TARGET_DATES_TEST_PATH,
        allow_pickle=True
    )
)


if len(
    anchor_dates_val
) != 584:

    raise RuntimeError(
        "anchor_dates_val length 584 değil."
    )


if len(
    target_dates_val
) != 584:

    raise RuntimeError(
        "target_dates_val length 584 değil."
    )


if len(
    anchor_dates_test
) != 584:

    raise RuntimeError(
        "anchor_dates_test length 584 değil."
    )


if len(
    target_dates_test
) != 584:

    raise RuntimeError(
        "target_dates_test length 584 değil."
    )


# ----------------------------------------------------------
# Tarih sınırı
# ----------------------------------------------------------

boundary_equal = bool(
    target_dates_val[
        -1
    ]
    == anchor_dates_test[
        0
    ]
)


if not boundary_equal:

    raise RuntimeError(
        "Son validation target realization "
        "ilk test anchor ile eşleşmiyor."
    )


# ----------------------------------------------------------
# 583 test-içi hizalama
# ----------------------------------------------------------

internal_alignment = (
    anchor_dates_test[
        1:
    ]
    == target_dates_test[
        :-1
    ]
)


internal_alignment_count = int(
    np.sum(
        internal_alignment
    )
)


if internal_alignment_count != 583:

    raise RuntimeError(
        "Test-içi persistence tarih hizalaması 583/583 değil.\n"
        f"Doğru eşleşme: {internal_alignment_count}/583"
    )


# ----------------------------------------------------------
# 584/584 target > anchor
# ----------------------------------------------------------

target_after_anchor = (
    target_dates_test
    > anchor_dates_test
)


target_after_anchor_count = int(
    np.sum(
        target_after_anchor
    )
)


if target_after_anchor_count != 584:

    raise RuntimeError(
        "Tüm test target realization tarihleri "
        "ilgili anchor tarihinden sonra değil.\n"
        f"Doğru eşleşme: {target_after_anchor_count}/584"
    )


# ----------------------------------------------------------
# Kronolojik sıralama
# ----------------------------------------------------------

if not pd.Index(
    anchor_dates_test
).is_monotonic_increasing:

    raise RuntimeError(
        "Test anchor tarihleri kronolojik değil."
    )


if not pd.Index(
    target_dates_test
).is_monotonic_increasing:

    raise RuntimeError(
        "Test target realization tarihleri kronolojik değil."
    )


# ----------------------------------------------------------
# Beklenen kritik tarihler
# ----------------------------------------------------------

if (
    target_dates_val[
        -1
    ]
    != EXPECTED_DATE_BOUNDARY
):

    raise RuntimeError(
        "Beklenen validation target boundary tarihi farklı.\n"
        f"Beklenen: {EXPECTED_DATE_BOUNDARY}\n"
        f"Gerçek  : {target_dates_val[-1]}"
    )


if (
    anchor_dates_test[
        0
    ]
    != EXPECTED_DATE_BOUNDARY
):

    raise RuntimeError(
        "Beklenen first test anchor tarihi farklı."
    )


if (
    target_dates_test[
        0
    ]
    != EXPECTED_FIRST_TEST_TARGET_DATE
):

    raise RuntimeError(
        "Beklenen first test target realization tarihi farklı."
    )


print(
    "\n"
    + "-" * 110
)

print(
    "C. TARİH-SEMANTİĞİ PERSISTENCE AUDİTİ"
)

print(
    "-" * 110
)

print(
    "✅ Son validation target realization "
    "= ilk test anchor."
)

print(
    "   Tarih:",
    target_dates_val[-1]
)

print(
    "✅ İlk test target realization "
    "> ilk test anchor."
)

print(
    "   Anchor:",
    anchor_dates_test[0]
)

print(
    "   Target:",
    target_dates_test[0]
)

print(
    "✅ Test içi 583/583 persistence tarih hizalaması doğru."
)

print(
    "✅ Toplam 584 test gözleminin tamamı tarih düzeyinde kapsandı."
)

print(
    "✅ 584/584 target realization tarihi ilgili anchor'dan sonra."
)

print(
    "✅ Test tarih dizileri kronolojik."
)

print(
    "✅ Off-by-one / look-ahead tarih semantiği bulgusu yok."
)


# ==========================================================
# 8. SPLIT META ÇAPRAZ KONTROLÜ
# ==========================================================

with open(
    SPLIT_META_PATH,
    "r",
    encoding="utf-8"
) as f:

    split_meta = json.load(
        f
    )


meta_val_target_end = pd.Timestamp(
    split_meta[
        "validation"
    ][
        "target_realization_end"
    ]
)

meta_test_anchor_start = pd.Timestamp(
    split_meta[
        "test"
    ][
        "anchor_start"
    ]
)

meta_test_target_start = pd.Timestamp(
    split_meta[
        "test"
    ][
        "target_realization_start"
    ]
)


if (
    meta_val_target_end
    != target_dates_val[-1]
):

    raise RuntimeError(
        "split_meta validation target end "
        "tarih dizisiyle uyuşmuyor."
    )


if (
    meta_test_anchor_start
    != anchor_dates_test[0]
):

    raise RuntimeError(
        "split_meta test anchor start "
        "tarih dizisiyle uyuşmuyor."
    )


if (
    meta_test_target_start
    != target_dates_test[0]
):

    raise RuntimeError(
        "split_meta test target start "
        "tarih dizisiyle uyuşmuyor."
    )


if not normalize_bool(
    split_meta[
        "target_sets_disjoint"
    ]
):

    raise RuntimeError(
        "split_meta target_sets_disjoint=False."
    )


print(
    "\n"
    + "-" * 110
)

print(
    "D. SPLIT META ÇAPRAZ KONTROLÜ"
)

print(
    "-" * 110
)

print(
    "✅ split_meta validation target end eşleşti."
)

print(
    "✅ split_meta test anchor start eşleşti."
)

print(
    "✅ split_meta test target realization start eşleşti."
)

print(
    "✅ target_sets_disjoint = True"
)


# ==========================================================
# 9. HAM DİZİLERİ YÜKLE
# ==========================================================

y_true_raw = np.load(
    FINAL_Y_TRUE_RAW
)

final_pred_raw = np.load(
    FINAL_ENSEMBLE_PRED_RAW
)

y_val_raw = np.load(
    Y_VAL_RAW_PATH
)

y_test_raw_sequence = np.load(
    Y_TEST_RAW_SEQUENCE_PATH
)

return_zero_stored = np.load(
    RETURN_ZERO_PRED_PATH
)

return_persistence_stored = np.load(
    RETURN_PERSISTENCE_PRED_PATH
)

vol_persistence_stored = np.load(
    VOL_PERSISTENCE_PRED_PATH
)


if (
    y_true_raw.shape
    != EXPECTED_TEST_SHAPE
):

    raise RuntimeError(
        f"y_true_raw shape yanlış: {y_true_raw.shape}"
    )


if (
    final_pred_raw.shape
    != EXPECTED_TEST_SHAPE
):

    raise RuntimeError(
        f"final_pred_raw shape yanlış: {final_pred_raw.shape}"
    )


if (
    y_test_raw_sequence.shape
    != EXPECTED_TEST_SHAPE
):

    raise RuntimeError(
        f"y_test_raw_sequence shape yanlış: "
        f"{y_test_raw_sequence.shape}"
    )


if (
    y_val_raw.ndim != 2
    or y_val_raw.shape[1] != 8
):

    raise RuntimeError(
        f"y_val_raw shape geçersiz: {y_val_raw.shape}"
    )


all_arrays = {
    "y_true_raw":
        y_true_raw,

    "final_pred_raw":
        final_pred_raw,

    "y_val_raw":
        y_val_raw,

    "y_test_raw_sequence":
        y_test_raw_sequence,

    "return_zero_stored":
        return_zero_stored,

    "return_persistence_stored":
        return_persistence_stored,

    "vol_persistence_stored":
        vol_persistence_stored,
}


for name, arr in all_arrays.items():

    if not np.isfinite(
        arr
    ).all():

        raise RuntimeError(
            f"{name} içinde NaN/Inf var."
        )


max_truth_diff = float(
    np.max(
        np.abs(
            y_true_raw
            - y_test_raw_sequence
        )
    )
)


if (
    max_truth_diff
    > FLOAT_TOL
):

    raise RuntimeError(
        "07 final truth ile sequence y_test_raw uyuşmuyor."
    )


print(
    "\n"
    + "-" * 110
)

print(
    "E. HAM DİZİ BÜTÜNLÜĞÜ"
)

print(
    "-" * 110
)

print(
    "✅ y_true_raw shape = (584, 8)"
)

print(
    "✅ final_pred_raw shape = (584, 8)"
)

print(
    "✅ sequence y_test_raw shape = (584, 8)"
)

print(
    "✅ 07 final truth ile sequence truth birebir eşleşiyor."
)

print(
    "Max truth diff:",
    f"{max_truth_diff:.16e}"
)


# ==========================================================
# 10. PERSISTENCE'I BAĞIMSIZ YENİDEN KUR
# ==========================================================

previous_observed_recomputed = np.vstack(
    [
        y_val_raw[
            -1,
            :
        ],

        y_true_raw[
            :-1,
            :
        ]
    ]
)


if (
    previous_observed_recomputed.shape
    != EXPECTED_TEST_SHAPE
):

    raise RuntimeError(
        "previous_observed_recomputed shape yanlış."
    )


# Boundary numeric alignment

max_first_boundary_numeric_diff = float(
    np.max(
        np.abs(
            previous_observed_recomputed[
                0,
                :
            ]
            - y_val_raw[
                -1,
                :
            ]
        )
    )
)


# Test-içi numeric alignment

max_internal_numeric_shift_diff = float(
    np.max(
        np.abs(
            previous_observed_recomputed[
                1:,
                :
            ]
            - y_true_raw[
                :-1,
                :
            ]
        )
    )
)


if (
    max_first_boundary_numeric_diff
    > FLOAT_TOL
):

    raise RuntimeError(
        "İlk persistence boundary numeric hizalaması yanlış."
    )


if (
    max_internal_numeric_shift_diff
    > FLOAT_TOL
):

    raise RuntimeError(
        "Test-içi persistence numeric shift yanlış."
    )


y_true_return = y_true_raw[
    :,
    :4
]

y_true_vol = y_true_raw[
    :,
    4:
]

final_return_pred = final_pred_raw[
    :,
    :4
]

final_vol_pred = final_pred_raw[
    :,
    4:
]


return_zero_recomputed = np.zeros_like(
    y_true_return
)

return_persistence_recomputed = (
    previous_observed_recomputed[
        :,
        :4
    ].copy()
)

vol_persistence_recomputed = (
    previous_observed_recomputed[
        :,
        4:
    ].copy()
)


max_return_zero_diff = float(
    np.max(
        np.abs(
            return_zero_recomputed
            - return_zero_stored
        )
    )
)

max_return_persistence_diff = float(
    np.max(
        np.abs(
            return_persistence_recomputed
            - return_persistence_stored
        )
    )
)

max_vol_persistence_diff = float(
    np.max(
        np.abs(
            vol_persistence_recomputed
            - vol_persistence_stored
        )
    )
)


if (
    max_return_zero_diff
    > FLOAT_TOL
):

    raise RuntimeError(
        "Stored ReturnZero prediction yanlış."
    )


if (
    max_return_persistence_diff
    > FLOAT_TOL
):

    raise RuntimeError(
        "Stored ReturnPersistence prediction yanlış."
    )


if (
    max_vol_persistence_diff
    > FLOAT_TOL
):

    raise RuntimeError(
        "Stored VolPersistence prediction yanlış."
    )


print(
    "\n"
    + "-" * 110
)

print(
    "F. PERSISTENCE VE NAİF TAHMİN YENİDEN ÜRETİMİ"
)

print(
    "-" * 110
)

print(
    "✅ Numeric persistence boundary hizalaması doğru."
)

print(
    "✅ Numeric test-içi 1-adım shift hizalaması doğru."
)

print(
    "✅ ReturnZero bağımsız yeniden üretildi."
)

print(
    "✅ ReturnPersistence bağımsız yeniden üretildi."
)

print(
    "✅ VolPersistence bağımsız yeniden üretildi."
)

print(
    "✅ Stored NPY dosyaları birebir eşleşti."
)


# ==========================================================
# 11. 20 METRİK SATIRINI BAĞIMSIZ YENİDEN HESAPLA
# ==========================================================

metric_rows = []


metric_rows.extend(
    build_return_metric_rows(
        model_label=
            "ReturnZero",

        y_true_return=
            y_true_return,

        y_pred_return=
            return_zero_recomputed
    )
)


metric_rows.extend(
    build_return_metric_rows(
        model_label=
            "ReturnPersistence",

        y_true_return=
            y_true_return,

        y_pred_return=
            return_persistence_recomputed
    )
)


metric_rows.extend(
    build_return_metric_rows(
        model_label=
            EXPECTED_PRIMARY_LABEL,

        y_true_return=
            y_true_return,

        y_pred_return=
            final_return_pred
    )
)


metric_rows.extend(
    build_vol_metric_rows(
        model_label=
            "VolPersistence",

        y_true_vol=
            y_true_vol,

        y_pred_vol=
            vol_persistence_recomputed
    )
)


metric_rows.extend(
    build_vol_metric_rows(
        model_label=
            EXPECTED_PRIMARY_LABEL,

        y_true_vol=
            y_true_vol,

        y_pred_vol=
            final_vol_pred
    )
)


metrics_recomputed = pd.DataFrame(
    metric_rows
)

metrics_stored = pd.read_csv(
    METRICS_LONG_CSV
)


if len(
    metrics_recomputed
) != 20:

    raise RuntimeError(
        "Recomputed metrics satır sayısı 20 değil."
    )


if len(
    metrics_stored
) != 20:

    raise RuntimeError(
        "Stored metrics satır sayısı 20 değil."
    )


key_cols = [
    "model_label",
    "task",
    "asset"
]


stored_indexed = (
    metrics_stored
    .set_index(
        key_cols
    )
    .sort_index()
)


recomputed_indexed = (
    metrics_recomputed
    .set_index(
        key_cols
    )
    .sort_index()
)


if list(
    stored_indexed.index
) != list(
    recomputed_indexed.index
):

    raise RuntimeError(
        "Stored ve recomputed metric indexleri uyuşmuyor."
    )


metric_cols = [
    "MAE",
    "RMSE",
    "R2",
    "PinballLoss_tau_0.5"
]


max_metric_diff = 0.0


for idx in recomputed_indexed.index:

    for col in metric_cols:

        actual = recomputed_indexed.loc[
            idx,
            col
        ]

        expected = stored_indexed.loc[
            idx,
            col
        ]


        if (
            pd.isna(
                actual
            )
            and pd.isna(
                expected
            )
        ):
            continue


        diff = abs(
            float(
                actual
            )
            - float(
                expected
            )
        )


        max_metric_diff = max(
            max_metric_diff,
            diff
        )


        assert_close(
            actual,
            expected,
            f"08A metric {idx} {col}"
        )


print(
    "\n"
    + "-" * 110
)

print(
    "G. HAM TAHMİN → 20/20 METRİK YENİDEN HESABI"
)

print(
    "-" * 110
)

print(
    "✅ 20/20 metric satırı bağımsız yeniden hesaplandı."
)

print(
    "✅ Return MAE / RMSE / R² doğrulandı."
)

print(
    "✅ Volatility MAE / RMSE / R² doğrulandı."
)

print(
    "✅ Volatility PinballLoss tau=0.5 doğrulandı."
)

print(
    "Max 08A metric diff:",
    f"{max_metric_diff:.16e}"
)


# ==========================================================
# 12. H — 07 ↔ AUDIT FINAL METRİK ÇAPRAZ KONTROLÜ
# ==========================================================

final_metrics_07 = pd.read_csv(
    FINAL_METRICS_CSV
)


final_primary_07 = final_metrics_07[
    final_metrics_07[
        "model_label"
    ]
    == EXPECTED_PRIMARY_LABEL
].copy()


if len(
    final_primary_07
) != 8:

    raise RuntimeError(
        "07 final primary metric satır sayısı 8 değil."
    )


final_primary_audit = metrics_recomputed[
    metrics_recomputed[
        "model_label"
    ]
    == EXPECTED_PRIMARY_LABEL
].copy()


if len(
    final_primary_audit
) != 8:

    raise RuntimeError(
        "Audit final primary metric satır sayısı 8 değil."
    )


cross_key = [
    "task",
    "asset"
]


m07 = (
    final_primary_07
    .set_index(
        cross_key
    )
    .sort_index()
)


maudit = (
    final_primary_audit
    .set_index(
        cross_key
    )
    .sort_index()
)


if list(
    m07.index
) != list(
    maudit.index
):

    raise RuntimeError(
        "07 ve audit final metric indexleri uyuşmuyor."
    )


max_07_final_crosscheck_diff = 0.0


for idx in maudit.index:

    for col in metric_cols:

        value_07 = m07.loc[
            idx,
            col
        ]

        value_audit = maudit.loc[
            idx,
            col
        ]


        if (
            pd.isna(
                value_07
            )
            and pd.isna(
                value_audit
            )
        ):
            continue


        diff = abs(
            float(
                value_07
            )
            - float(
                value_audit
            )
        )


        max_07_final_crosscheck_diff = max(
            max_07_final_crosscheck_diff,
            diff
        )


        assert_close(
            value_audit,
            value_07,
            f"07↔Audit final metric {idx} {col}"
        )


print(
    "\n"
    + "-" * 110
)

print(
    "H. 07 ↔ AUDIT FINAL METRİK ÇAPRAZ KONTROLÜ"
)

print(
    "-" * 110
)

print(
    "✅ Final modelin 8/8 asset-task metriği raw prediction'lardan yeniden üretildi."
)

print(
    "✅ 07 resmî final metrics CSV ile doğrudan karşılaştırıldı."
)

print(
    "✅ 07 ↔ audit final metric zinciri eşleşti."
)

print(
    "Max 07-final crosscheck diff:",
    f"{max_07_final_crosscheck_diff:.16e}"
)


# ==========================================================
# 13. 12 KARŞILAŞTIRMAYI YENİDEN HESAPLA
# ==========================================================

comparison_rows = []


for asset in EXPECTED_ASSETS:

    final_row = metrics_recomputed[
        (
            metrics_recomputed[
                "model_label"
            ]
            == EXPECTED_PRIMARY_LABEL
        )
        &
        (
            metrics_recomputed[
                "task"
            ]
            == "return"
        )
        &
        (
            metrics_recomputed[
                "asset"
            ]
            == asset
        )
    ].iloc[
        0
    ]


    for baseline_model in [
        "ReturnZero",
        "ReturnPersistence",
    ]:

        baseline_row = metrics_recomputed[
            (
                metrics_recomputed[
                    "model_label"
                ]
                == baseline_model
            )
            &
            (
                metrics_recomputed[
                    "task"
                ]
                == "return"
            )
            &
            (
                metrics_recomputed[
                    "asset"
                ]
                == asset
            )
        ].iloc[
            0
        ]


        final_error = float(
            final_row[
                "MAE"
            ]
        )

        baseline_error = float(
            baseline_row[
                "MAE"
            ]
        )

        ratio = (
            final_error
            / baseline_error
        )


        comparison_rows.append({
            "task":
                "return",

            "asset":
                asset,

            "primary_metric":
                "MAE",

            "final_model":
                EXPECTED_PRIMARY_LABEL,

            "baseline_model":
                baseline_model,

            "final_error":
                final_error,

            "baseline_error":
                baseline_error,

            "final_minus_baseline":
                final_error
                - baseline_error,

            "final_to_baseline_ratio":
                ratio,

            "final_beats_baseline":
                bool(
                    ratio < 1.0
                ),

            "warning_gt_1_30":
                bool(
                    ratio
                    > WARNING_THRESHOLD
                ),

            "severe_gt_1_50":
                bool(
                    ratio
                    > SEVERE_THRESHOLD
                ),
        })


for asset in EXPECTED_ASSETS:

    final_row = metrics_recomputed[
        (
            metrics_recomputed[
                "model_label"
            ]
            == EXPECTED_PRIMARY_LABEL
        )
        &
        (
            metrics_recomputed[
                "task"
            ]
            == "volatility"
        )
        &
        (
            metrics_recomputed[
                "asset"
            ]
            == asset
        )
    ].iloc[
        0
    ]


    baseline_row = metrics_recomputed[
        (
            metrics_recomputed[
                "model_label"
            ]
            == "VolPersistence"
        )
        &
        (
            metrics_recomputed[
                "task"
            ]
            == "volatility"
        )
        &
        (
            metrics_recomputed[
                "asset"
            ]
            == asset
        )
    ].iloc[
        0
    ]


    final_error = float(
        final_row[
            "PinballLoss_tau_0.5"
        ]
    )

    baseline_error = float(
        baseline_row[
            "PinballLoss_tau_0.5"
        ]
    )

    ratio = (
        final_error
        / baseline_error
    )


    comparison_rows.append({
        "task":
            "volatility",

        "asset":
            asset,

        "primary_metric":
            "PinballLoss_tau_0.5",

        "final_model":
            EXPECTED_PRIMARY_LABEL,

        "baseline_model":
            "VolPersistence",

        "final_error":
            final_error,

        "baseline_error":
            baseline_error,

        "final_minus_baseline":
            final_error
            - baseline_error,

        "final_to_baseline_ratio":
            ratio,

        "final_beats_baseline":
            bool(
                ratio < 1.0
            ),

        "warning_gt_1_30":
            bool(
                ratio
                > WARNING_THRESHOLD
            ),

        "severe_gt_1_50":
            bool(
                ratio
                > SEVERE_THRESHOLD
            ),
    })


comparison_recomputed = pd.DataFrame(
    comparison_rows
)

comparison_stored = pd.read_csv(
    COMPARISON_CSV
)


if len(
    comparison_recomputed
) != 12:

    raise RuntimeError(
        "Recomputed comparison satır sayısı 12 değil."
    )


if len(
    comparison_stored
) != 12:

    raise RuntimeError(
        "Stored comparison satır sayısı 12 değil."
    )


comparison_key = [
    "task",
    "asset",
    "baseline_model"
]


stored_comp_idx = (
    comparison_stored
    .set_index(
        comparison_key
    )
    .sort_index()
)


recomputed_comp_idx = (
    comparison_recomputed
    .set_index(
        comparison_key
    )
    .sort_index()
)


if list(
    stored_comp_idx.index
) != list(
    recomputed_comp_idx.index
):

    raise RuntimeError(
        "Stored ve recomputed comparison indexleri uyuşmuyor."
    )


comparison_numeric_cols = [
    "final_error",
    "baseline_error",
    "final_minus_baseline",
    "final_to_baseline_ratio"
]


max_comparison_diff = 0.0


for idx in recomputed_comp_idx.index:

    for col in comparison_numeric_cols:

        actual = float(
            recomputed_comp_idx.loc[
                idx,
                col
            ]
        )

        expected = float(
            stored_comp_idx.loc[
                idx,
                col
            ]
        )


        diff = abs(
            actual
            - expected
        )


        max_comparison_diff = max(
            max_comparison_diff,
            diff
        )


        assert_close(
            actual,
            expected,
            f"Comparison {idx} {col}"
        )


    for col in [
        "final_beats_baseline",
        "warning_gt_1_30",
        "severe_gt_1_50"
    ]:

        actual_bool = normalize_bool(
            recomputed_comp_idx.loc[
                idx,
                col
            ]
        )

        expected_bool = normalize_bool(
            stored_comp_idx.loc[
                idx,
                col
            ]
        )


        if actual_bool != expected_bool:

            raise RuntimeError(
                f"Comparison boolean uyuşmuyor: {idx} {col}"
            )


print(
    "\n"
    + "-" * 110
)

print(
    "I. 12/12 FINAL-vs-BASELINE KARŞILAŞTIRMA AUDİTİ"
)

print(
    "-" * 110
)

print(
    "✅ 4/4 ReturnZero karşılaştırması yeniden üretildi."
)

print(
    "✅ 4/4 ReturnPersistence karşılaştırması yeniden üretildi."
)

print(
    "✅ 4/4 VolPersistence karşılaştırması yeniden üretildi."
)

print(
    "✅ 12/12 ratio yeniden doğrulandı."
)

print(
    "✅ 12/12 final_beats_baseline flag doğrulandı."
)

print(
    "✅ 1.30 diagnostic warning flagleri doğrulandı."
)

print(
    "✅ 1.50 diagnostic severe flagleri doğrulandı."
)

print(
    "Max comparison diff:",
    f"{max_comparison_diff:.16e}"
)


# ==========================================================
# 14. 3 SATIRLIK COMPARISON SUMMARY
# ==========================================================

summary_rows = []


for (
    task,
    baseline_model
), group in comparison_recomputed.groupby(
    [
        "task",
        "baseline_model"
    ],
    sort=False
):

    ratios = group[
        "final_to_baseline_ratio"
    ].astype(
        float
    )


    summary_rows.append({
        "task":
            task,

        "baseline_model":
            baseline_model,

        "n_assets":
            int(
                len(
                    group
                )
            ),

        "mean_final_to_baseline_ratio":
            float(
                ratios.mean()
            ),

        "median_final_to_baseline_ratio":
            float(
                ratios.median()
            ),

        "min_final_to_baseline_ratio":
            float(
                ratios.min()
            ),

        "max_final_to_baseline_ratio":
            float(
                ratios.max()
            ),

        "n_final_beats_baseline":
            int(
                group[
                    "final_beats_baseline"
                ].sum()
            ),

        "n_warning_gt_1_30":
            int(
                group[
                    "warning_gt_1_30"
                ].sum()
            ),

        "n_severe_gt_1_50":
            int(
                group[
                    "severe_gt_1_50"
                ].sum()
            ),
    })


comparison_summary_recomputed = pd.DataFrame(
    summary_rows
)

comparison_summary_stored = pd.read_csv(
    COMPARISON_SUMMARY_CSV
)


if len(
    comparison_summary_recomputed
) != 3:

    raise RuntimeError(
        "Recomputed comparison summary 3 satır değil."
    )


if len(
    comparison_summary_stored
) != 3:

    raise RuntimeError(
        "Stored comparison summary 3 satır değil."
    )


summary_key = [
    "task",
    "baseline_model"
]


stored_sum_idx = (
    comparison_summary_stored
    .set_index(
        summary_key
    )
    .sort_index()
)


recomputed_sum_idx = (
    comparison_summary_recomputed
    .set_index(
        summary_key
    )
    .sort_index()
)


if list(
    stored_sum_idx.index
) != list(
    recomputed_sum_idx.index
):

    raise RuntimeError(
        "Stored ve recomputed summary indexleri uyuşmuyor."
    )


summary_numeric_cols = [
    "mean_final_to_baseline_ratio",
    "median_final_to_baseline_ratio",
    "min_final_to_baseline_ratio",
    "max_final_to_baseline_ratio",
]


summary_integer_cols = [
    "n_assets",
    "n_final_beats_baseline",
    "n_warning_gt_1_30",
    "n_severe_gt_1_50",
]


max_summary_diff = 0.0


for idx in recomputed_sum_idx.index:

    for col in summary_numeric_cols:

        actual = float(
            recomputed_sum_idx.loc[
                idx,
                col
            ]
        )

        expected = float(
            stored_sum_idx.loc[
                idx,
                col
            ]
        )


        diff = abs(
            actual
            - expected
        )


        max_summary_diff = max(
            max_summary_diff,
            diff
        )


        assert_close(
            actual,
            expected,
            f"Summary {idx} {col}"
        )


    for col in summary_integer_cols:

        actual = int(
            recomputed_sum_idx.loc[
                idx,
                col
            ]
        )

        expected = int(
            stored_sum_idx.loc[
                idx,
                col
            ]
        )


        if actual != expected:

            raise RuntimeError(
                f"Summary integer uyuşmuyor: {idx} {col}"
            )


print(
    "\n"
    + "-" * 110
)

print(
    "J. COMPARISON SUMMARY YENİDEN ÜRETİMİ"
)

print(
    "-" * 110
)

print(
    "✅ 3/3 comparison summary satırı yeniden üretildi."
)

print(
    "✅ Mean / median / min / max ratio değerleri doğrulandı."
)

print(
    "✅ Baseline kazanım sayıları doğrulandı."
)

print(
    "✅ Diagnostic flag sayıları doğrulandı."
)

print(
    "Max summary diff:",
    f"{max_summary_diff:.16e}"
)


# ==========================================================
# 15. STRONG-NAIVE TEST ORANLARI
# ==========================================================

return_vs_zero = comparison_recomputed[
    (
        comparison_recomputed[
            "task"
        ]
        == "return"
    )
    &
    (
        comparison_recomputed[
            "baseline_model"
        ]
        == "ReturnZero"
    )
].copy()


return_vs_persistence = comparison_recomputed[
    (
        comparison_recomputed[
            "task"
        ]
        == "return"
    )
    &
    (
        comparison_recomputed[
            "baseline_model"
        ]
        == "ReturnPersistence"
    )
].copy()


vol_vs_persistence = comparison_recomputed[
    (
        comparison_recomputed[
            "task"
        ]
        == "volatility"
    )
    &
    (
        comparison_recomputed[
            "baseline_model"
        ]
        == "VolPersistence"
    )
].copy()


avg_return_ratio_vs_zero = float(
    return_vs_zero[
        "final_to_baseline_ratio"
    ].mean()
)


avg_return_ratio_vs_persistence = float(
    return_vs_persistence[
        "final_to_baseline_ratio"
    ].mean()
)


avg_vol_ratio_vs_persistence = float(
    vol_vs_persistence[
        "final_to_baseline_ratio"
    ].mean()
)


diagnostic_score = float(
    0.5
    * avg_return_ratio_vs_zero
    + 0.5
    * avg_vol_ratio_vs_persistence
)


with open(
    SUMMARY_JSON,
    "r",
    encoding="utf-8"
) as f:

    summary_json = json.load(
        f
    )


stored_ratios = summary_json[
    "strong_naive_test_ratios"
]


assert_close(
    avg_return_ratio_vs_zero,
    stored_ratios[
        "avg_return_ratio_vs_ReturnZero"
    ],
    "Avg Return Ratio vs ReturnZero"
)


assert_close(
    avg_return_ratio_vs_persistence,
    stored_ratios[
        "avg_return_ratio_vs_ReturnPersistence"
    ],
    "Avg Return Ratio vs ReturnPersistence"
)


assert_close(
    avg_vol_ratio_vs_persistence,
    stored_ratios[
        "avg_vol_ratio_vs_VolPersistence"
    ],
    "Avg Vol Ratio vs VolPersistence"
)


assert_close(
    diagnostic_score,
    stored_ratios[
        "test_strong_naive_diagnostic_score"
    ],
    "Test Strong-Naive Diagnostic Score"
)


print(
    "\n"
    + "-" * 110
)

print(
    "K. STRONG-NAIVE TEST ORANLARI AUDİTİ"
)

print(
    "-" * 110
)

print(
    "✅ Avg Return Ratio vs ReturnZero yeniden doğrulandı."
)

print(
    "✅ Avg Return Ratio vs ReturnPersistence yeniden doğrulandı."
)

print(
    "✅ Avg Vol Ratio vs VolPersistence yeniden doğrulandı."
)

print(
    "✅ Test Strong-Naive Diagnostic Score yeniden doğrulandı."
)

print(
    f"Avg Return Ratio vs ReturnZero        : "
    f"{avg_return_ratio_vs_zero:.16f}"
)

print(
    f"Avg Return Ratio vs ReturnPersistence : "
    f"{avg_return_ratio_vs_persistence:.16f}"
)

print(
    f"Avg Vol Ratio vs VolPersistence       : "
    f"{avg_vol_ratio_vs_persistence:.16f}"
)

print(
    f"Test Strong-Naive Diagnostic Score    : "
    f"{diagnostic_score:.16f}"
)


# ==========================================================
# 16. 09 İÇİN 20 LOSS-SERIES AUDİTİ
# ==========================================================

loss_stored = np.load(
    LOSS_SERIES_PATH
)


loss_expected = {}


for i, asset in enumerate(
    EXPECTED_ASSETS
):

    loss_expected[
        f"final_return_abs__{asset}"
    ] = np.abs(
        y_true_return[
            :,
            i
        ]
        - final_return_pred[
            :,
            i
        ]
    )


    loss_expected[
        f"return_zero_abs__{asset}"
    ] = np.abs(
        y_true_return[
            :,
            i
        ]
        - return_zero_recomputed[
            :,
            i
        ]
    )


    loss_expected[
        f"return_persistence_abs__{asset}"
    ] = np.abs(
        y_true_return[
            :,
            i
        ]
        - return_persistence_recomputed[
            :,
            i
        ]
    )


    loss_expected[
        f"final_vol_pinball__{asset}"
    ] = pinball_loss_series_np(
        y_true_vol[
            :,
            i
        ],
        final_vol_pred[
            :,
            i
        ],
        tau=TAU
    )


    loss_expected[
        f"vol_persistence_pinball__{asset}"
    ] = pinball_loss_series_np(
        y_true_vol[
            :,
            i
        ],
        vol_persistence_recomputed[
            :,
            i
        ],
        tau=TAU
    )


stored_loss_keys = sorted(
    loss_stored.files
)

expected_loss_keys = sorted(
    loss_expected.keys()
)


if (
    stored_loss_keys
    != expected_loss_keys
):

    raise RuntimeError(
        "Loss-series key seti yanlış."
    )


if len(
    stored_loss_keys
) != 20:

    raise RuntimeError(
        f"Loss-series sayısı 20 değil: "
        f"{len(stored_loss_keys)}"
    )


max_loss_series_diff = 0.0


for key in expected_loss_keys:

    stored_arr = loss_stored[
        key
    ]

    expected_arr = loss_expected[
        key
    ]


    if stored_arr.shape != (
        EXPECTED_TEST_SAMPLES,
    ):

        raise RuntimeError(
            f"Loss-series shape yanlış: "
            f"{key} {stored_arr.shape}"
        )


    if not np.isfinite(
        stored_arr
    ).all():

        raise RuntimeError(
            f"Loss-series NaN/Inf içeriyor: {key}"
        )


    diff = float(
        np.max(
            np.abs(
                stored_arr
                - expected_arr
            )
        )
    )


    max_loss_series_diff = max(
        max_loss_series_diff,
        diff
    )


    if (
        diff
        > FLOAT_TOL
    ):

        raise RuntimeError(
            f"Loss-series uyuşmuyor: {key}\n"
            f"Max fark: {diff}"
        )


print(
    "\n"
    + "-" * 110
)

print(
    "L. 09 İÇİN LOSS-SERIES AUDİTİ"
)

print(
    "-" * 110
)

print(
    "✅ Exact 20 loss-series anahtarı doğrulandı."
)

print(
    "✅ Her loss-series exact 584 gözlem içeriyor."
)

print(
    "✅ Return absolute-loss serileri yeniden üretildi."
)

print(
    "✅ Volatility pinball-loss serileri yeniden üretildi."
)

print(
    "✅ Stored NPZ ile bağımsız yeniden üretim eşleşti."
)

print(
    "Max loss-series diff:",
    f"{max_loss_series_diff:.16e}"
)


# ==========================================================
# 17. JSON METODOLOJİK KURAL AUDİTİ
# ==========================================================

if normalize_bool(
    summary_json[
        "model_selection_inside_08A"
    ]
):

    raise RuntimeError(
        "JSON model_selection_inside_08A=True."
    )


if normalize_bool(
    summary_json[
        "hyperparameter_change_inside_08A"
    ]
):

    raise RuntimeError(
        "JSON hyperparameter_change_inside_08A=True."
    )


if normalize_bool(
    summary_json[
        "retraining_inside_08A"
    ]
):

    raise RuntimeError(
        "JSON retraining_inside_08A=True."
    )


if normalize_bool(
    summary_json[
        "final_model_changed"
    ]
):

    raise RuntimeError(
        "JSON final_model_changed=True."
    )


if str(
    summary_json[
        "winner_config_id"
    ]
) != EXPECTED_WINNER_CONFIG_ID:

    raise RuntimeError(
        "08A JSON winner_config_id yanlış."
    )


if str(
    summary_json[
        "primary_final_prediction"
    ]
) != EXPECTED_PRIMARY_LABEL:

    raise RuntimeError(
        "08A JSON primary prediction yanlış."
    )


if str(
    summary_json[
        "primary_test_policy"
    ]
) != EXPECTED_PRIMARY_POLICY:

    raise RuntimeError(
        "08A JSON primary policy yanlış."
    )


if int(
    summary_json[
        "test_sample_count"
    ]
) != EXPECTED_TEST_SAMPLES:

    raise RuntimeError(
        "08A JSON test_sample_count 584 değil."
    )


thresholds = summary_json[
    "diagnostic_thresholds"
]


assert_close(
    thresholds[
        "warning_gt"
    ],
    WARNING_THRESHOLD,
    "Diagnostic warning threshold"
)


assert_close(
    thresholds[
        "severe_gt"
    ],
    SEVERE_THRESHOLD,
    "Diagnostic severe threshold"
)


print(
    "\n"
    + "-" * 110
)

print(
    "M. JSON + METODOLOJİK KURAL AUDİTİ"
)

print(
    "-" * 110
)

print(
    "✅ model_selection_inside_08A = False"
)

print(
    "✅ hyperparameter_change_inside_08A = False"
)

print(
    "✅ retraining_inside_08A = False"
)

print(
    "✅ final_model_changed = False"
)

print(
    "✅ winner_config_id değişmedi."
)

print(
    "✅ primary final prediction değişmedi."
)

print(
    "✅ test_sample_count = 584"
)

print(
    "✅ Diagnostic warning threshold = 1.30"
)

print(
    "✅ Diagnostic severe threshold = 1.50"
)

print(
    "✅ Eşikler yalnızca tanısal raporlama amacıyla korunuyor."
)


# ==========================================================
# 18. ANA BULGULAR
# ==========================================================

return_zero_wins = int(
    return_vs_zero[
        "final_beats_baseline"
    ].sum()
)


return_persistence_wins = int(
    return_vs_persistence[
        "final_beats_baseline"
    ].sum()
)


vol_persistence_wins = int(
    vol_vs_persistence[
        "final_beats_baseline"
    ].sum()
)


# ==========================================================
# 19. SON RESMÎ AUDIT ÖZETİ
# ==========================================================

print(
    "\n"
    + "=" * 110
)

print(
    "08A_FINAL_AUDIT_v4 — TÜM KONTROLLER GEÇTİ"
)

print(
    "=" * 110
)


print(
    "\nAUDIT SONUCU:"
)

print(
    "✅ 08A script SHA-256 manifest ile doğrulandı."
)

print(
    "✅ Kilitli final model değişmedi."
)

print(
    "✅ 07 primary 3-seed ensemble politikası değişmedi."
)

print(
    "✅ 584 test gözlemi doğrulandı."
)

print(
    "✅ Tarih dizileri common datetime tipe normalize edildi."
)

print(
    "✅ Son validation target realization = ilk test anchor."
)

print(
    "✅ İlk test target realization > ilk test anchor."
)

print(
    "✅ 583/583 test-içi persistence tarih hizalaması doğrulandı."
)

print(
    "✅ 584/584 target realization > ilgili anchor doğrulandı."
)

print(
    "✅ split_meta ile tarih dizileri çapraz doğrulandı."
)

print(
    "✅ Persistence numeric boundary hizalaması doğrulandı."
)

print(
    "✅ Persistence test-içi numeric shift doğrulandı."
)

print(
    "✅ Look-ahead / off-by-one bulgusuna rastlanmadı."
)

print(
    "✅ ReturnZero stored prediction yeniden üretildi."
)

print(
    "✅ ReturnPersistence stored prediction yeniden üretildi."
)

print(
    "✅ VolPersistence stored prediction yeniden üretildi."
)

print(
    "✅ 20/20 metric satırı bağımsız yeniden hesaplandı."
)

print(
    "✅ 07 final modelin 8/8 asset-task metriği audit içinde yeniden üretildi."
)

print(
    "✅ 07 resmî final metrics CSV ile doğrudan çapraz eşleşme doğrulandı."
)

print(
    "✅ 12/12 final-vs-baseline karşılaştırması yeniden üretildi."
)

print(
    "✅ 3/3 comparison summary satırı yeniden üretildi."
)

print(
    "✅ Strong-naive test oranları yeniden hesaplandı."
)

print(
    "✅ 20/20 loss-series bağımsız yeniden üretildi."
)

print(
    "✅ 08A içinde model seçimi yapılmadığı doğrulandı."
)

print(
    "✅ 08A içinde hyperparameter değişmediği doğrulandı."
)

print(
    "✅ 08A içinde yeniden eğitim yapılmadığı doğrulandı."
)

print(
    "✅ Final modelin değiştirilmediği doğrulandı."
)

print(
    "✅ Audit hiçbir dosyayı değiştirmedi."
)


print(
    "\nNUMERİK YENİDEN HESAP FARKLARI:"
)

print(
    "Max truth diff                   :",
    f"{max_truth_diff:.16e}"
)

print(
    "Max first-boundary numeric diff  :",
    f"{max_first_boundary_numeric_diff:.16e}"
)

print(
    "Max internal numeric shift diff  :",
    f"{max_internal_numeric_shift_diff:.16e}"
)

print(
    "Max ReturnZero prediction diff   :",
    f"{max_return_zero_diff:.16e}"
)

print(
    "Max ReturnPersistence pred diff  :",
    f"{max_return_persistence_diff:.16e}"
)

print(
    "Max VolPersistence pred diff     :",
    f"{max_vol_persistence_diff:.16e}"
)

print(
    "Max 08A metric diff              :",
    f"{max_metric_diff:.16e}"
)

print(
    "Max 07-final crosscheck diff     :",
    f"{max_07_final_crosscheck_diff:.16e}"
)

print(
    "Max comparison diff              :",
    f"{max_comparison_diff:.16e}"
)

print(
    "Max summary diff                 :",
    f"{max_summary_diff:.16e}"
)

print(
    "Max loss-series diff             :",
    f"{max_loss_series_diff:.16e}"
)


print(
    "\nRESMÎ 08A BULGULARI:"
)

print(
    f"Final model vs ReturnZero         : "
    f"{return_zero_wins}/4 varlıkta "
    f"final model daha düşük MAE"
)

print(
    f"Final model vs ReturnPersistence  : "
    f"{return_persistence_wins}/4 varlıkta "
    f"final model daha düşük MAE"
)

print(
    f"Final model vs VolPersistence     : "
    f"{vol_persistence_wins}/4 varlıkta "
    f"final model daha düşük PinballLoss"
)


print(
    "\nSTRONG-NAIVE TEST RATIOS:"
)

print(
    f"Avg Return Ratio vs ReturnZero        : "
    f"{avg_return_ratio_vs_zero:.16f}"
)

print(
    f"Avg Return Ratio vs ReturnPersistence : "
    f"{avg_return_ratio_vs_persistence:.16f}"
)

print(
    f"Avg Vol Ratio vs VolPersistence       : "
    f"{avg_vol_ratio_vs_persistence:.16f}"
)

print(
    f"Test Strong-Naive Diagnostic Score    : "
    f"{diagnostic_score:.16f}"
)


print(
    "\n"
    + "=" * 110
)

print(
    "SON HÜKÜM: 08A AŞAMASI GÜÇLENDİRİLMİŞ AUDIT İLE "
    "DOĞRULANDI VE 08B ÖNCESİ KAPATILMAYA HAZIR."
)

print(
    "=" * 110
)

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

In [ ]:
from pathlib import Path
import json
import os

BASE_DIR = Path("/content/drive/MyDrive/tez_transformer_v4_repro")

paths = {
    "BASE_DIR": BASE_DIR,
    "CONFIG_DIR": BASE_DIR / "config",
    "SCRIPTS_DIR": BASE_DIR / "scripts",
    "SEQUENCE_BASELINE_LB10": BASE_DIR / "data" / "sequences" / "baseline" / "lb10",
    "FINAL_TEST_DIR": BASE_DIR / "results" / "final_test",
    "NAIVE_BASELINE_DIR": BASE_DIR / "results" / "baselines" / "naive",
    "MULTISEED_DIR": BASE_DIR / "results" / "multiseed",
    "BASELINES_DIR": BASE_DIR / "results" / "baselines",
}

print("=" * 110)
print("08B LEARNED BASELINES — ÖN ENVANTER KONTROLÜ")
print("=" * 110)

print("\nA. ANA KLASÖRLER")
for name, path in paths.items():
    print(f"{name:28s}: {'VAR ✅' if path.exists() else 'YOK ❌'} | {path}")

seq_dir = paths["SEQUENCE_BASELINE_LB10"]

print("\n" + "-" * 110)
print("B. baseline/lb10 SEQUENCE DOSYALARI")
print("-" * 110)

if seq_dir.exists():
    seq_files = sorted([p.name for p in seq_dir.iterdir() if p.is_file()])
    for file in seq_files:
        print(file)
else:
    print("❌ Sequence klasörü yok.")

required_sequence_candidates = [
    "X_train.npy",
    "X_val.npy",
    "X_test.npy",
    "y_train.npy",
    "y_val.npy",
    "y_test.npy",
    "y_train_raw.npy",
    "y_val_raw.npy",
    "y_test_raw.npy",
    "anchor_dates_test.npy",
    "target_realization_dates_test.npy",
]

print("\n" + "-" * 110)
print("C. 08B İÇİN BEKLENEN TEMEL SEQUENCE DOSYALARI")
print("-" * 110)

for fname in required_sequence_candidates:
    path = seq_dir / fname
    print(f"{fname:36s}: {'VAR ✅' if path.exists() else 'YOK ❌'}")

print("\n" + "-" * 110)
print("D. 07 FINAL TEST ÇIKTILARI")
print("-" * 110)

final_required = [
    "final_test_metrics_long_v4.csv",
    "final_test_summary_v4.json",
    "final_test_y_true_raw_v4.npy",
    "pred_final_ensemble_raw_v4.npy",
    "pred_final_seed123_raw_v4.npy",
    "pred_final_seed777_raw_v4.npy",
    "pred_final_seed2026_raw_v4.npy",
]

for fname in final_required:
    path = paths["FINAL_TEST_DIR"] / fname
    print(f"{fname:42s}: {'VAR ✅' if path.exists() else 'YOK ❌'}")

print("\n" + "-" * 110)
print("E. 08A NAIVE BASELINE ÇIKTILARI")
print("-" * 110)

naive_required = [
    "naive_baseline_metrics_long_v4.csv",
    "naive_baseline_comparison_v4.csv",
    "naive_baseline_comparison_summary_v4.csv",
    "naive_baseline_summary_v4.json",
    "naive_baseline_loss_series_v4.npz",
]

for fname in naive_required:
    path = paths["NAIVE_BASELINE_DIR"] / fname
    print(f"{fname:48s}: {'VAR ✅' if path.exists() else 'YOK ❌'}")

print("\n" + "-" * 110)
print("F. SCRIPT / MANIFEST KONTROLÜ")
print("-" * 110)

script_manifest_required = [
    BASE_DIR / "config" / "code_manifest_v4.csv",
    BASE_DIR / "scripts" / "07_final_test_evaluation_v4.py",
    BASE_DIR / "scripts" / "08A_naive_baselines_test_v4.py",
]

for path in script_manifest_required:
    print(f"{path.name:40s}: {'VAR ✅' if path.exists() else 'YOK ❌'} | {path}")

print("\n" + "-" * 110)
print("G. 08B ÇIKTI KLASÖRÜ ÖNERİSİ")
print("-" * 110)

learned_dir = BASE_DIR / "results" / "baselines" / "learned"
print("08B learned baseline çıktı klasörü:")
print(learned_dir)
print("Şu anda var mı?:", learned_dir.exists())

print("\n" + "=" * 110)
print("08B ÖN ENVANTER KONTROLÜ TAMAMLANDI")
print("=" * 110)

In [ ]:
from pathlib import Path
import json
import numpy as np
import xgboost as xgb

BASE_DIR = Path("/content/drive/MyDrive/tez_transformer_v4_repro")
SEQ_DIR = BASE_DIR / "data" / "sequences" / "baseline" / "lb10"

print("=" * 110)
print("08B — SON TEKNİK PREFLIGHT")
print("=" * 110)

# ----------------------------------------------------------
# 1. XGBoost version
# ----------------------------------------------------------
print("\nA. XGBOOST")
print("-" * 110)
print("XGBoost version:", xgb.__version__)

# ----------------------------------------------------------
# 2. Sequence shapes / dtypes
# ----------------------------------------------------------
print("\nB. SEQUENCE SHAPES / DTYPES")
print("-" * 110)

for fname in [
    "X_train.npy", "X_val.npy", "X_test.npy",
    "y_train.npy", "y_val.npy", "y_test.npy",
    "y_train_raw.npy", "y_val_raw.npy", "y_test_raw.npy"
]:
    p = SEQ_DIR / fname
    arr = np.load(p)
    print(
        f"{fname:20s} | "
        f"shape={str(arr.shape):15s} | "
        f"dtype={arr.dtype}"
    )

# ----------------------------------------------------------
# 3. Sequence metadata
# ----------------------------------------------------------
print("\nC. SEQUENCE META")
print("-" * 110)

meta_path = SEQ_DIR / "sequence_meta.json"
if meta_path.exists():
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    print(json.dumps(meta, indent=2, ensure_ascii=False))
else:
    print("sequence_meta.json YOK ❌")

# ----------------------------------------------------------
# 4. Scaler inventory
# ----------------------------------------------------------
print("\nD. SCALER / PREPROCESSOR ENVANTERİ")
print("-" * 110)

patterns = [
    "*scaler*",
    "*Scaler*",
    "*standard*",
    "*preprocess*"
]

found = set()

for pattern in patterns:
    for p in BASE_DIR.rglob(pattern):
        if p.is_file():
            found.add(p)

if found:
    for p in sorted(found):
        print(p)
else:
    print("Scaler/preprocessor isimli dosya bulunamadı.")

print("\n" + "=" * 110)
print("08B SON TEKNİK PREFLIGHT TAMAMLANDI")
print("=" * 110)

In [ ]:
from pathlib import Path
import numpy as np
import xgboost as xgb
import re

BASE_DIR = Path("/content/drive/MyDrive/tez_transformer_v4_repro")

print("=" * 110)
print("08B — EK KRİTİK PREFLIGHT: XGBOOST OBJECTIVE + FINAL LOSS KOD TEYİDİ")
print("=" * 110)

# ==========================================================
# A. XGBOOST OBJECTIVE FİİLİ ÇALIŞMA TESTİ
# ==========================================================

print("\nA. XGBOOST OBJECTIVE FİİLİ ÇALIŞMA TESTİ")
print("-" * 110)

rng = np.random.default_rng(42)
X_dummy = rng.normal(size=(50, 5))
y_dummy = rng.normal(size=50)

quantile_ok = False
squarederror_ok = False

try:
    model_q = xgb.XGBRegressor(
        objective="reg:quantileerror",
        quantile_alpha=0.5,
        n_estimators=5,
        max_depth=2,
        learning_rate=0.1,
        random_state=42,
        n_jobs=1
    )
    model_q.fit(X_dummy, y_dummy)
    pred_q = model_q.predict(X_dummy)

    assert pred_q.shape == (50,)
    assert np.isfinite(pred_q).all()

    quantile_ok = True
    print("✅ reg:quantileerror fiilen çalışıyor.")
    print("   quantile_alpha = 0.5")
    print("   Volatility XGBoost için doğrudan quantile/pinball objective kullanılabilir.")

except Exception as e:
    print("❌ reg:quantileerror çalışmadı.")
    print("Hata:", repr(e))


try:
    model_mse = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=5,
        max_depth=2,
        learning_rate=0.1,
        random_state=42,
        n_jobs=1
    )
    model_mse.fit(X_dummy, y_dummy)
    pred_mse = model_mse.predict(X_dummy)

    assert pred_mse.shape == (50,)
    assert np.isfinite(pred_mse).all()

    squarederror_ok = True
    print("✅ reg:squarederror fiilen çalışıyor.")
    print("   Return XGBoost için MSE uyumlu objective kullanılabilir.")

except Exception as e:
    print("❌ reg:squarederror çalışmadı.")
    print("Hata:", repr(e))


# ==========================================================
# B. 05 / 06 SCRIPT KODUNDAN LOSS TEYİDİ
# ==========================================================

print("\n" + "-" * 110)
print("B. FINAL MODEL LOSS — KORUNMUŞ SCRIPT KODUNDAN TEYİT")
print("-" * 110)

scripts = [
    BASE_DIR / "scripts" / "05_grid_search_v4.py",
    BASE_DIR / "scripts" / "06_best_model_multiseed_v4.py",
]

keywords = [
    "MSELoss",
    "mse_loss",
    "return_loss",
    "ret_loss",
    "PinballLoss",
    "pinball",
    "vol_loss",
    "FixedLambda",
    "lambda_value",
]

all_matches = {}

for sp in scripts:
    print(f"\nDOSYA: {sp.name}")

    if not sp.exists():
        print("❌ DOSYA YOK")
        continue

    code = sp.read_text(encoding="utf-8", errors="ignore")
    lines = code.splitlines()

    matches = []

    for lineno, line in enumerate(lines, start=1):
        if any(re.search(re.escape(kw), line, flags=re.IGNORECASE) for kw in keywords):
            matches.append((lineno, line.rstrip()))

    all_matches[sp.name] = matches

    if not matches:
        print("⚠️ İlgili keyword bulunamadı.")
        continue

    for lineno, line in matches:
        print(f"{lineno:5d}: {line}")


# ==========================================================
# C. BAĞLAMLI KOD PARÇALARI
# ==========================================================

print("\n" + "-" * 110)
print("C. LOSS TANIMLARININ BAĞLAMLI KOD PARÇALARI")
print("-" * 110)

context_patterns = [
    r"MSELoss",
    r"mse_loss",
    r"return_loss",
    r"ret_loss",
    r"PinballLoss",
    r"pinball",
    r"vol_loss",
]

for sp in scripts:
    if not sp.exists():
        continue

    code = sp.read_text(encoding="utf-8", errors="ignore")
    lines = code.splitlines()

    printed_ranges = set()

    print(f"\n### {sp.name}")

    for i, line in enumerate(lines):
        if any(re.search(pattern, line, flags=re.IGNORECASE) for pattern in context_patterns):
            start = max(0, i - 3)
            end = min(len(lines), i + 4)

            key = (start, end)
            if key in printed_ranges:
                continue

            printed_ranges.add(key)

            print("\n" + "." * 80)
            for j in range(start, end):
                marker = ">>" if j == i else "  "
                print(f"{marker} {j+1:5d}: {lines[j]}")


# ==========================================================
# D. PREFLIGHT ÖZETİ
# ==========================================================

print("\n" + "=" * 110)
print("08B EK PREFLIGHT ÖZETİ")
print("=" * 110)

print("XGBoost reg:quantileerror çalışıyor :", quantile_ok)
print("XGBoost reg:squarederror çalışıyor  :", squarederror_ok)

print("\nSONRAKİ YORUM KURALI:")
print("- Return loss'un MSE olduğu yalnızca yukarıdaki korunmuş 05/06 kod satırlarıyla teyit edilecek.")
print("- Volatility loss'un PinballLoss(tau=0.5) olduğu yine korunmuş kod satırlarıyla teyit edilecek.")
print("- Sadece keyword bulunması yeterli kanıt sayılmayacak; gerçek loss hesaplama satırının bağlamına bakılacak.")
print("- XGBoost objective kararı dummy fit'in fiilen başarılı olup olmadığına göre verilecek.")

print("\n" + "=" * 110)
print("08B EK KRİTİK PREFLIGHT TAMAMLANDI")
print("=" * 110)

In [ ]:
from pathlib import Path
import re

BASE_DIR = Path("/content/drive/MyDrive/tez_transformer_v4_repro")
SCRIPTS_DIR = BASE_DIR / "scripts"

SCRIPTS = [
    SCRIPTS_DIR / "05_grid_search_v4.py",
    SCRIPTS_DIR / "06_best_model_multiseed_v4.py",
]

print("=" * 110)
print("08B — FINAL NOSHARING MİMARİ + EĞİTİM PROTOKOLÜ KOD ÇIKARIMI")
print("=" * 110)

# 08B için kritik anahtarlar
KEYWORDS = [
    # Sabitler / eğitim
    "SEED",
    "BATCH_SIZE",
    "MAX_EPOCHS",
    "MIN_EPOCHS",
    "MIN_EPOCH",
    "PATIENCE",
    "LEARNING_RATE",
    "LR",
    "WEIGHT_DECAY",
    "GRAD_CLIP",
    "DROPOUT",
    "TAU",

    # Mimari
    "SIZE_CONFIG",
    "small",
    "d_model",
    "nhead",
    "n_head",
    "num_layers",
    "n_layers",
    "dim_feedforward",
    "d_ff",

    # Model sınıfları
    "class NoSharing",
    "class Transformer",
    "class TaskHead",
    "class MLP",
    "PositionalEncoding",

    # Eğitim / optimizer / scheduler
    "AdamW",
    "ReduceLROnPlateau",
    "scheduler",
    "early stopping",
    "early_stopping",
    "clip_grad_norm",

    # Checkpoint / seçim
    "best_state",
    "best_epoch",
    "ValidationScore",
    "validation_score",

    # Loss
    "MSELoss",
    "pinball_loss_torch",
    "split_task_losses",
    "fixed_lambda_loss",
]

def print_context(lines, center_idx, before=4, after=8):
    start = max(0, center_idx - before)
    end = min(len(lines), center_idx + after + 1)

    for j in range(start, end):
        marker = ">>" if j == center_idx else "  "
        print(f"{marker} {j+1:5d}: {lines[j]}")

for script_path in SCRIPTS:
    print("\n" + "#" * 110)
    print(f"DOSYA: {script_path.name}")
    print("#" * 110)

    if not script_path.exists():
        print("❌ DOSYA BULUNAMADI")
        continue

    code = script_path.read_text(encoding="utf-8", errors="ignore")
    lines = code.splitlines()

    matched_centers = set()

    for keyword in KEYWORDS:
        hits = [
            i for i, line in enumerate(lines)
            if re.search(re.escape(keyword), line, flags=re.IGNORECASE)
        ]

        if not hits:
            continue

        print("\n" + "-" * 110)
        print(f"ANAHTAR: {keyword!r} | EŞLEŞME SAYISI: {len(hits)}")
        print("-" * 110)

        # Çok fazla tekrar varsa ilk 5 anlamlı bağlam yeterli
        for i in hits[:5]:
            # Aynı merkezi tekrar basma
            if i in matched_centers:
                continue
            matched_centers.add(i)

            print_context(
                lines=lines,
                center_idx=i,
                before=4,
                after=8
            )
            print("." * 80)

print("\n" + "=" * 110)
print("08B MİMARİ + EĞİTİM PROTOKOLÜ KOD ÇIKARIMI TAMAMLANDI")
print("=" * 110)

print("""
BU ÇIKTIDAN SONRA KİLİTLENECEKLER:
1. Final NoSharing-small branch mimarisi
2. Task-head yapısı
3. Dropout
4. Optimizer
5. Learning rate
6. Weight decay
7. Batch size
8. Gradient clipping
9. Early stopping / patience
10. Checkpoint-selection mantığı

NOT:
- 08B scriptinde bu bilgiler korunmuş 05/06 kodundan alınacak.
- Tahmin sonucu görülerek hiçbir mimari veya eğitim ayarı değiştirilmeyecek.
- Test seti model seçimi için kullanılmayacak.
""")

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

In [ ]:
from pathlib import Path

SCRIPT_PATH = Path(
    "/content/drive/MyDrive/tez_transformer_v4_repro/scripts/08B_learned_baselines_test_v4.py"
)

print("Var mı:", SCRIPT_PATH.exists())
print("Yol:", SCRIPT_PATH)

In [ ]:
from pathlib import Path
import ast
import hashlib
import py_compile
import pandas as pd
from datetime import datetime, timezone


# ==========================================================
# 08B — ROBUST PRE-RUN AUDIT
# AST + SHA-256 + ORDER + TEST ISOLATION + MANIFEST
# ==========================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/tez_transformer_v4_repro"
)

SCRIPT_PATH = (
    BASE_DIR
    / "scripts"
    / "08B_learned_baselines_test_v4.py"
)

MANIFEST_PATH = (
    BASE_DIR
    / "config"
    / "code_manifest_v4.csv"
)

EXPECTED_SHA256 = (
    "df4fa2b29a6b522fc410f693a30520e045c144a15f7c7368af5eb6a1f0f12566"
)


print("=" * 110)
print("08B — ROBUST PRE-RUN AUDIT")
print("AST + SHA-256 + ORDER + TEST ISOLATION + MANIFEST")
print("=" * 110)


# ==========================================================
# 1. FILE EXISTS
# ==========================================================

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(
        f"08B script bulunamadı:\n{SCRIPT_PATH}"
    )

print("[FILE] FOUND ✅")
print(f"[PATH]  {SCRIPT_PATH}")
print(f"[BYTES] {SCRIPT_PATH.stat().st_size}")


# ==========================================================
# 2. PY_COMPILE
# ==========================================================

py_compile.compile(
    str(SCRIPT_PATH),
    doraise=True,
)

print("[PY_COMPILE] PASS ✅")


# ==========================================================
# 3. SHA-256
# ==========================================================

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


actual_sha256 = sha256_file(SCRIPT_PATH)

print(f"[SHA256 ACTUAL]   {actual_sha256}")
print(f"[SHA256 EXPECTED] {EXPECTED_SHA256}")

if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        "08B script SHA-256 mismatch.\n"
        "İlk resmî koşu DURDURULDU."
    )

print("[SHA256] EXACT MATCH ✅")


# ==========================================================
# 4. READ + AST PARSE
# ==========================================================

code = SCRIPT_PATH.read_text(
    encoding="utf-8",
    errors="strict",
)

tree = ast.parse(
    code,
    filename=str(SCRIPT_PATH),
)

print("[AST PARSE] PASS ✅")


# ==========================================================
# 5. EXTRACT TOP-LEVEL CONSTANTS
# ==========================================================

constants = {}

for node in tree.body:

    target_name = None
    value_node = None

    if isinstance(node, ast.Assign):

        if (
            len(node.targets) == 1
            and isinstance(node.targets[0], ast.Name)
        ):
            target_name = node.targets[0].id
            value_node = node.value

    elif isinstance(node, ast.AnnAssign):

        if isinstance(node.target, ast.Name):
            target_name = node.target.id
            value_node = node.value

    if target_name is not None and value_node is not None:

        try:
            constants[target_name] = ast.literal_eval(
                value_node
            )
        except Exception:
            pass


expected_constants = {

    # Seeds
    "SEEDS": [123, 777, 2026],

    # Neural training budget
    "BATCH_SIZE": 64,
    "MAX_EPOCHS": 100,
    "MIN_EPOCHS": 45,
    "PATIENCE": 15,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-4,
    "GRAD_CLIP": 1.0,
    "DROPOUT": 0.10,
    "TAU": 0.5,

    # Transformer
    "D_MODEL": 32,
    "N_HEAD": 4,
    "N_LAYERS": 2,
    "D_FF": 128,

    # LSTM grid
    "LSTM_HIDDEN": [32, 64, 128],
    "LSTM_LAYERS": [1, 2],

    # XGBoost grid
    "XGB_DEPTHS": [3, 4, 6],
    "XGB_LRS": [0.03, 0.05],
    "XGB_N_ESTIMATORS": 1000,
    "XGB_EARLY_STOP": 30,
    "XGB_SUBSAMPLE": 0.8,
    "XGB_COLSAMPLE": 0.8,
    "XGB_TREE_METHOD": "hist",
}


constant_failures = []

for name, expected in expected_constants.items():

    actual = constants.get(name, "__MISSING__")

    if actual != expected:

        constant_failures.append(
            {
                "name": name,
                "expected": expected,
                "actual": actual,
            }
        )

        print(
            f"[CONSTANT] {name}: FAIL ❌ | "
            f"expected={expected!r}, actual={actual!r}"
        )

    else:

        print(
            f"[CONSTANT] {name}: PASS ✅ | "
            f"{actual!r}"
        )


if constant_failures:

    raise RuntimeError(
        "Locked constant mismatch bulundu:\n"
        + "\n".join(
            (
                f"- {x['name']}: "
                f"expected={x['expected']!r}, "
                f"actual={x['actual']!r}"
            )
            for x in constant_failures
        )
    )


# ==========================================================
# 6. CLASS + FUNCTION INVENTORY
# ==========================================================

class_names = {
    node.name
    for node in ast.walk(tree)
    if isinstance(node, ast.ClassDef)
}

function_nodes = {
    node.name: node
    for node in ast.walk(tree)
    if isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef)
    )
}

required_classes = {
    "TransformerBlock",
    "SingleTaskTransformer",
    "SingleTaskLSTM",
}

required_functions = {
    "make_head",
    "pinball_torch",
    "task_loss",
    "train_neural_run",
    "train_xgb_run",
    "make_xgb",
    "summarize_family_task",
}

for name in sorted(required_classes):

    if name not in class_names:
        raise RuntimeError(
            f"Gerekli class eksik: {name}"
        )

    print(f"[CLASS] {name}: PASS ✅")


for name in sorted(required_functions):

    if name not in function_nodes:
        raise RuntimeError(
            f"Gerekli function eksik: {name}"
        )

    print(f"[FUNCTION] {name}: PASS ✅")


# ==========================================================
# 7. MSE LOSS STRUCTURE
# ==========================================================

mse_ok = False

for node in tree.body:

    if not isinstance(node, ast.Assign):
        continue

    if len(node.targets) != 1:
        continue

    target = node.targets[0]

    if not (
        isinstance(target, ast.Name)
        and target.id == "MSE"
    ):
        continue

    value = node.value

    if not isinstance(value, ast.Call):
        continue

    func = value.func

    if (
        isinstance(func, ast.Attribute)
        and isinstance(func.value, ast.Name)
        and func.value.id == "nn"
        and func.attr == "MSELoss"
    ):
        mse_ok = True
        break


if not mse_ok:
    raise RuntimeError(
        "MSE = nn.MSELoss() AST ile doğrulanamadı."
    )

print("[LOSS] Return neural = nn.MSELoss() ✅")


# ==========================================================
# 8. PINBALL LOSS STRUCTURE
# ==========================================================

pinball_source = ast.get_source_segment(
    code,
    function_nodes["pinball_torch"],
)

required_pinball_tokens = [
    "true - pred",
    "torch.maximum",
    "(tau - 1.0)",
    ".mean()",
]

for token in required_pinball_tokens:

    if token not in pinball_source:
        raise RuntimeError(
            f"Pinball loss yapısında eksik token: {token}"
        )

print(
    "[LOSS] Volatility neural = "
    "PinballLoss(tau=0.5) ✅"
)


# ==========================================================
# 9. XGBOOST OBJECTIVES
# ==========================================================

required_xgb_tokens = [
    "reg:squarederror",
    "reg:quantileerror",
    "quantile_alpha=TAU",
]

for token in required_xgb_tokens:

    if token not in code:
        raise RuntimeError(
            f"XGBoost objective token eksik: {token}"
        )

    print(
        f"[XGBOOST OBJECTIVE] {token}: PASS ✅"
    )


# ==========================================================
# 10. PROTOCOL LOCK EXISTS
# ==========================================================

if "LOCKED_BEFORE_08B_RESULTS" not in code:
    raise RuntimeError(
        "Pre-result protocol lock status bulunamadı."
    )

print(
    "[PROTOCOL LOCK] "
    "LOCKED_BEFORE_08B_RESULTS found ✅"
)


# ==========================================================
# 11. LOCATE CRITICAL LINE NUMBERS VIA AST
# ==========================================================

def assigned_name(node) -> str | None:

    if isinstance(node, ast.Assign):

        for target in node.targets:

            if isinstance(target, ast.Name):
                return target.id

    elif isinstance(node, ast.AnnAssign):

        if isinstance(node.target, ast.Name):
            return node.target.id

    return None


protocol_dump_line = None
selection_lock_dump_line = None
x_test_load_line = None
training_call_lines = []


for node in ast.walk(tree):

    # X_test assignment
    if isinstance(
        node,
        (ast.Assign, ast.AnnAssign),
    ):

        name = assigned_name(node)

        if name == "X_test":
            x_test_load_line = node.lineno

    # Function calls
    if isinstance(node, ast.Call):

        if isinstance(node.func, ast.Name):

            func_name = node.func.id

            if func_name in {
                "train_neural_run",
                "train_xgb_run",
            }:
                training_call_lines.append(
                    node.lineno
                )

            if func_name == "dump_json":

                if len(node.args) >= 2:

                    arg0 = node.args[0]
                    arg1 = node.args[1]

                    if (
                        isinstance(arg0, ast.Name)
                        and isinstance(arg1, ast.Name)
                    ):

                        if (
                            arg0.id == "protocol"
                            and arg1.id == "PROTOCOL_JSON"
                        ):
                            protocol_dump_line = node.lineno

                        if (
                            arg0.id == "selection_lock"
                            and arg1.id == "SELECTION_LOCK_JSON"
                        ):
                            selection_lock_dump_line = (
                                node.lineno
                            )


if protocol_dump_line is None:
    raise RuntimeError(
        "Protocol-lock JSON write line bulunamadı."
    )

if selection_lock_dump_line is None:
    raise RuntimeError(
        "Selection-lock JSON write line bulunamadı."
    )

if x_test_load_line is None:
    raise RuntimeError(
        "X_test load line bulunamadı."
    )

if not training_call_lines:
    raise RuntimeError(
        "Training call line bulunamadı."
    )


first_training_call_line = min(
    training_call_lines
)


print(
    f"[ORDER] protocol lock write line : "
    f"{protocol_dump_line}"
)

print(
    f"[ORDER] first training call line : "
    f"{first_training_call_line}"
)

print(
    f"[ORDER] selection lock write line: "
    f"{selection_lock_dump_line}"
)

print(
    f"[ORDER] X_test load line         : "
    f"{x_test_load_line}"
)


# Protocol must be locked before training.
if not (
    protocol_dump_line
    < first_training_call_line
):
    raise RuntimeError(
        "Protocol lock training'den önce değil."
    )

print(
    "[ORDER] Protocol locked "
    "BEFORE training ✅"
)


# Selection must be locked before test load.
if not (
    selection_lock_dump_line
    < x_test_load_line
):
    raise RuntimeError(
        "KRİTİK HATA: "
        "X_test selection lock'tan önce yükleniyor."
    )

print(
    "[ORDER] Validation selection locked "
    "BEFORE X_test load ✅"
)


# ==========================================================
# 12. TEST ISOLATION INSIDE TRAINING FUNCTIONS
# ==========================================================

forbidden_names = {
    "X_test",
    "y_test",
    "y_test_raw",
    "anchor_test",
    "target_test",
    "anchor_dates_test",
    "target_dates_test",
}


for function_name in [
    "train_neural_run",
    "train_xgb_run",
]:

    fn_node = function_nodes[function_name]

    used_names = {
        node.id
        for node in ast.walk(fn_node)
        if isinstance(node, ast.Name)
    }

    found = sorted(
        forbidden_names & used_names
    )

    if found:
        raise RuntimeError(
            f"{function_name} içinde test değişkeni "
            f"kullanımı bulundu: {found}"
        )

    print(
        f"[TEST ISOLATION] "
        f"{function_name}: PASS ✅"
    )


# ==========================================================
# 13. XGBOOST EARLY STOPPING ON VALIDATION
# ==========================================================

xgb_train_source = ast.get_source_segment(
    code,
    function_nodes["train_xgb_run"],
)

required_xgb_validation_tokens = [
    "eval_set",
    "X_val_flat",
    "y_val",
]

for token in required_xgb_validation_tokens:

    if token not in xgb_train_source:
        raise RuntimeError(
            "XGBoost validation early-stopping "
            f"yapısında eksik token: {token}"
        )

if "X_test" in xgb_train_source:
    raise RuntimeError(
        "XGBoost training function içinde X_test bulundu."
    )

print(
    "[XGBOOST EARLY STOP] "
    "Validation-based and test-isolated ✅"
)


# ==========================================================
# 14. FROZEN SCALER / NO REFIT
# ==========================================================

forbidden_scaler_patterns = [
    "StandardScaler(",
    ".fit_transform(",
    "y_scaler.fit(",
    "target_scaler.fit(",
    "scaler.fit(",
]

found_scaler_refit_patterns = [
    token
    for token in forbidden_scaler_patterns
    if token in code
]

if found_scaler_refit_patterns:
    raise RuntimeError(
        "Olası scaler refit pattern bulundu: "
        f"{found_scaler_refit_patterns}"
    )

required_scaler_tokens = [
    "SCALERS_PATH",
    "pickle.load",
    "find_target_scaler",
    "Y_MEAN",
    "Y_SCALE",
]

for token in required_scaler_tokens:

    if token not in code:
        raise RuntimeError(
            f"Frozen scaler token eksik: {token}"
        )

print(
    "[SCALER] Frozen scaler present; "
    "no refit pattern detected ✅"
)


# ==========================================================
# 15. 3-SEED CONFIG SELECTION
# ==========================================================

selection_source = ast.get_source_segment(
    code,
    function_nodes["summarize_family_task"],
)

required_selection_tokens = [
    "found != SEEDS",
    "np.mean(s)",
    "np.std(s, ddof=1)",
    "mean_validation_task_score",
    "std_validation_task_score",
]

for token in required_selection_tokens:

    if token not in selection_source:
        raise RuntimeError(
            f"3-seed config-selection token eksik: {token}"
        )

print(
    "[CONFIG SELECTION] "
    "3-seed mean validation score + "
    "sample-std tie-break ✅"
)


# ==========================================================
# 16. RAW-SCALE 3-SEED ENSEMBLE
# ==========================================================

ensemble_required_tokens = [
    "np.mean(np.stack([seed_preds[s] for s in SEEDS]",
    (
        "exact = "
        "(seed_preds[123] + seed_preds[777] "
        "+ seed_preds[2026]) / 3.0"
    ),
]

for token in ensemble_required_tokens:

    if token not in code:
        raise RuntimeError(
            f"3-seed ensemble token eksik:\n{token}"
        )

print(
    "[ENSEMBLE] "
    "3-seed arithmetic mean + exact reconstruction check ✅"
)


# ==========================================================
# 17. MANIFEST PRE-RUN REGISTRATION
# ==========================================================

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Manifest bulunamadı:\n{MANIFEST_PATH}"
    )

manifest_df = pd.read_csv(MANIFEST_PATH)

print(
    "[MANIFEST COLUMNS]",
    manifest_df.columns.tolist(),
)


def first_existing_column(
    columns,
    candidates,
):

    for candidate in candidates:

        if candidate in columns:
            return candidate

    return None


name_col = first_existing_column(
    manifest_df.columns,
    [
        "script_name",
        "filename",
        "file_name",
        "script",
        "script_file",
    ],
)

hash_col = first_existing_column(
    manifest_df.columns,
    [
        "sha256",
        "sha256_hash",
        "hash",
    ],
)


if name_col is None:
    raise RuntimeError(
        "Manifest içinde script-name kolonu "
        "otomatik bulunamadı.\n"
        f"Columns={manifest_df.columns.tolist()}"
    )

if hash_col is None:
    raise RuntimeError(
        "Manifest içinde SHA-256 kolonu "
        "otomatik bulunamadı.\n"
        f"Columns={manifest_df.columns.tolist()}"
    )


existing_exact = bool(
    (
        manifest_df[name_col]
        .astype(str)
        .str.strip()
        == SCRIPT_PATH.name
    )
    &
    (
        manifest_df[hash_col]
        .astype(str)
        .str.strip()
        == actual_sha256
    )
).any()


if existing_exact:

    print(
        "[MANIFEST] Exact script+hash "
        "already registered; duplicate not added ✅"
    )

else:

    new_row = {
        col: ""
        for col in manifest_df.columns
    }

    new_row[name_col] = SCRIPT_PATH.name
    new_row[hash_col] = actual_sha256


    optional_mappings = {

        "path": str(SCRIPT_PATH),
        "script_path": str(SCRIPT_PATH),
        "filepath": str(SCRIPT_PATH),

        "bytes": SCRIPT_PATH.stat().st_size,
        "size_bytes": SCRIPT_PATH.stat().st_size,
        "file_size_bytes": SCRIPT_PATH.stat().st_size,

        "stage": "08B_learned_baselines",

        "registration_type": "pre_run",

        "registered_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),

        "created_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),

        "note": (
            "py_compile passed; exact SHA-256 matched; "
            "AST-based locked-protocol checks passed "
            "before first official 08B run"
        ),

        "notes": (
            "py_compile passed; exact SHA-256 matched; "
            "AST-based locked-protocol checks passed "
            "before first official 08B run"
        ),
    }


    for col, value in optional_mappings.items():

        if col in new_row:
            new_row[col] = value


    updated_manifest = pd.concat(
        [
            manifest_df,
            pd.DataFrame([new_row]),
        ],
        ignore_index=True,
    )

    updated_manifest.to_csv(
        MANIFEST_PATH,
        index=False,
    )

    print(
        "[MANIFEST] Pre-run registration added ✅"
    )


# ==========================================================
# 18. POST-WRITE MANIFEST CROSS-CHECK
# ==========================================================

manifest_check = pd.read_csv(
    MANIFEST_PATH
)

exact_rows = manifest_check[
    (
        manifest_check[name_col]
        .astype(str)
        .str.strip()
        == SCRIPT_PATH.name
    )
    &
    (
        manifest_check[hash_col]
        .astype(str)
        .str.strip()
        == actual_sha256
    )
]


if exact_rows.empty:
    raise RuntimeError(
        "Manifest post-check failed: "
        "exact script+hash row bulunamadı."
    )

print(
    "[MANIFEST POST-CHECK] "
    "Exact script+hash row found ✅"
)


# ==========================================================
# 19. FINAL VERDICT
# ==========================================================

print("\n" + "=" * 110)
print("08B ROBUST PRE-RUN SON HÜKÜM")
print("=" * 110)

print("File exists                              : PASS ✅")
print("py_compile                               : PASS ✅")
print("SHA-256 exact match                      : PASS ✅")
print("AST parse                                : PASS ✅")
print("Locked constants                         : PASS ✅")
print("Required classes/functions               : PASS ✅")
print("Neural return loss = MSE                 : PASS ✅")
print("Neural vol loss = Pinball tau=0.5        : PASS ✅")
print("XGBoost objectives                       : PASS ✅")
print("Protocol locked before training          : PASS ✅")
print("Selection locked before X_test load      : PASS ✅")
print("Neural training test isolation           : PASS ✅")
print("XGBoost training test isolation          : PASS ✅")
print("XGBoost early stopping on validation     : PASS ✅")
print("Frozen scaler / no refit                 : PASS ✅")
print("3-seed mean config selection             : PASS ✅")
print("3-seed raw-scale ensemble                : PASS ✅")
print("Manifest pre-run registration            : PASS ✅")
print("Manifest post-write exact cross-check    : PASS ✅")

print("\nEXPECTED SHA-256:")
print(EXPECTED_SHA256)

print("\nSONUÇ:")
print(
    "08B_learned_baselines_test_v4.py "
    "ilk resmî koşu öncesi robust pre-run "
    "kontrollerini geçti."
)

print("=" * 110)

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
from datetime import datetime, timezone


# ==========================================================
# 08B — MANIFEST PRE-RUN FIX + FINAL CROSS-CHECK
# ==========================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/tez_transformer_v4_repro"
)

SCRIPT_PATH = (
    BASE_DIR
    / "scripts"
    / "08B_learned_baselines_test_v4.py"
)

MANIFEST_PATH = (
    BASE_DIR
    / "config"
    / "code_manifest_v4.csv"
)

EXPECTED_SHA256 = (
    "df4fa2b29a6b522fc410f693a30520e045c144a15f7c7368af5eb6a1f0f12566"
)


print("=" * 110)
print("08B — MANIFEST PRE-RUN FIX + FINAL CROSS-CHECK")
print("=" * 110)


# ----------------------------------------------------------
# 1. FILE + HASH
# ----------------------------------------------------------

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(
        f"08B script bulunamadı:\n{SCRIPT_PATH}"
    )

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Manifest bulunamadı:\n{MANIFEST_PATH}"
    )


def sha256_file(path: Path) -> str:

    h = hashlib.sha256()

    with path.open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


actual_sha256 = sha256_file(SCRIPT_PATH)

print(f"[SHA256 ACTUAL]   {actual_sha256}")
print(f"[SHA256 EXPECTED] {EXPECTED_SHA256}")

if actual_sha256 != EXPECTED_SHA256:

    raise RuntimeError(
        "08B script SHA-256 mismatch. "
        "Manifest kaydı DURDURULDU."
    )

print("[SHA256] EXACT MATCH ✅")


# ----------------------------------------------------------
# 2. READ MANIFEST
# ----------------------------------------------------------

manifest_df = pd.read_csv(MANIFEST_PATH)

print(
    "[MANIFEST BEFORE] rows =",
    len(manifest_df)
)

print(
    "[MANIFEST COLUMNS]",
    manifest_df.columns.tolist()
)


required_columns = {
    "script_name",
    "relative_path",
    "sha256",
    "size_bytes",
    "modified_utc",
    "manifest_created_utc",
}

missing_columns = (
    required_columns
    - set(manifest_df.columns)
)

if missing_columns:

    raise RuntimeError(
        "Manifest kolonları beklenen şemayla uyuşmuyor:\n"
        + str(sorted(missing_columns))
    )

print("[MANIFEST SCHEMA] PASS ✅")


# ----------------------------------------------------------
# 3. EXACT DUPLICATE CHECK — FIXED
# ----------------------------------------------------------

exact_mask = (

    manifest_df["script_name"]
    .astype(str)
    .str.strip()
    .eq(SCRIPT_PATH.name)

    &

    manifest_df["sha256"]
    .astype(str)
    .str.strip()
    .eq(actual_sha256)
)


existing_exact = bool(
    exact_mask.any()
)


if existing_exact:

    print(
        "[MANIFEST] Exact script+hash already registered; "
        "duplicate row not added ✅"
    )

else:

    stat = SCRIPT_PATH.stat()

    now_utc = datetime.now(
        timezone.utc
    ).isoformat()

    modified_utc = datetime.fromtimestamp(
        stat.st_mtime,
        tz=timezone.utc,
    ).isoformat()

    new_row = {
        "script_name": SCRIPT_PATH.name,
        "relative_path": str(
            SCRIPT_PATH.relative_to(BASE_DIR)
        ),
        "sha256": actual_sha256,
        "size_bytes": int(stat.st_size),
        "modified_utc": modified_utc,
        "manifest_created_utc": now_utc,
    }

    updated_manifest = pd.concat(
        [
            manifest_df,
            pd.DataFrame([new_row]),
        ],
        ignore_index=True,
    )

    updated_manifest.to_csv(
        MANIFEST_PATH,
        index=False,
    )

    print(
        "[MANIFEST] 08B pre-run registration added ✅"
    )


# ----------------------------------------------------------
# 4. POST-WRITE EXACT CROSS-CHECK
# ----------------------------------------------------------

manifest_check = pd.read_csv(
    MANIFEST_PATH
)

post_mask = (

    manifest_check["script_name"]
    .astype(str)
    .str.strip()
    .eq(SCRIPT_PATH.name)

    &

    manifest_check["sha256"]
    .astype(str)
    .str.strip()
    .eq(actual_sha256)
)


exact_rows = manifest_check.loc[
    post_mask
]


if exact_rows.empty:

    raise RuntimeError(
        "Manifest post-check FAILED: "
        "exact 08B script+hash row bulunamadı."
    )


print(
    "[MANIFEST POST-CHECK] "
    "Exact script+hash row found ✅"
)

print(
    "[MANIFEST AFTER] rows =",
    len(manifest_check)
)


# ----------------------------------------------------------
# 5. EXACT ROW DISPLAY
# ----------------------------------------------------------

print("\nREGISTERED 08B ROW")
print("-" * 110)

print(
    exact_rows.to_string(
        index=False
    )
)


# ----------------------------------------------------------
# 6. FINAL VERDICT
# ----------------------------------------------------------

print("\n" + "=" * 110)
print("08B PRE-RUN SON HÜKÜM")
print("=" * 110)

print("Previous robust audit critical checks : PASS ✅")
print("py_compile                           : PASS ✅")
print("SHA-256 exact match                  : PASS ✅")
print("Manifest schema                      : PASS ✅")
print("Manifest exact registration          : PASS ✅")
print("Manifest post-write cross-check      : PASS ✅")

print("\nSHA-256:")
print(actual_sha256)

print("\nSONUÇ:")
print(
    "08B_learned_baselines_test_v4.py "
    "ilk resmî koşu öncesi tüm pre-run "
    "kontrollerini tamamladı."
)

print("=" * 110)

In [ ]:
SCRIPT_PATH = "/content/drive/MyDrive/tez_transformer_v4_repro/scripts/08B_learned_baselines_test_v4.py"

exec(
    compile(
        open(SCRIPT_PATH, "r", encoding="utf-8").read(),
        SCRIPT_PATH,
        "exec"
    )
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

AUDIT_PATH = Path(
    "/content/drive/MyDrive/tez_transformer_v4_repro/scripts/08B_FINAL_AUDIT_v4.py"
)

print("Dosya var mı:", AUDIT_PATH.exists())
print("Yol:", AUDIT_PATH)

In [ ]:
from pathlib import Path
import ast
import hashlib
import py_compile
import pandas as pd
from datetime import datetime, timezone


# ==========================================================
# 08B_FINAL_AUDIT — ROBUST PRE-RUN
#
# Kontroller:
# 1) File exists
# 2) Exact size
# 3) py_compile
# 4) Exact SHA-256
# 5) AST parse
# 6) Locked constants
# 7) Locked hashes + selections
# 8) Required audit functions/classes
# 9) No model training inside audit
# 10) Audit write-isolation
# 11) Independent reconstruction structure
# 12) Manifest pre-run registration
# 13) Manifest exact post-check
# ==========================================================


BASE_DIR = Path(
    "/content/drive/MyDrive/tez_transformer_v4_repro"
)

SCRIPT_PATH = (
    BASE_DIR
    / "scripts"
    / "08B_FINAL_AUDIT_v4.py"
)

MANIFEST_PATH = (
    BASE_DIR
    / "config"
    / "code_manifest_v4.csv"
)


EXPECTED_SIZE_BYTES = 66115

EXPECTED_SHA256 = (
    "b5f69e161ac5baa86f8df5aa7fa61867"
    "dfd300c2c9ccc212c2cd4d7be155ac63"
)


print("=" * 110)
print("08B_FINAL_AUDIT — ROBUST PRE-RUN")
print("COMPILE + SHA-256 + AST + SOURCE ISOLATION + MANIFEST")
print("=" * 110)


# ==========================================================
# 1. FILE EXISTS
# ==========================================================

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(
        f"Audit script bulunamadı:\n{SCRIPT_PATH}"
    )

actual_size = SCRIPT_PATH.stat().st_size

print("[FILE] FOUND ✅")
print(f"[PATH]  {SCRIPT_PATH}")
print(f"[BYTES] {actual_size}")


if actual_size != EXPECTED_SIZE_BYTES:
    raise RuntimeError(
        "08B_FINAL_AUDIT file-size mismatch.\n"
        f"Expected: {EXPECTED_SIZE_BYTES}\n"
        f"Actual  : {actual_size}"
    )

print("[FILE SIZE] EXACT MATCH ✅")


# ==========================================================
# 2. PY_COMPILE
# ==========================================================

py_compile.compile(
    str(SCRIPT_PATH),
    doraise=True,
)

print("[PY_COMPILE] PASS ✅")


# ==========================================================
# 3. SHA-256
# ==========================================================

def sha256_file(path: Path) -> str:

    h = hashlib.sha256()

    with path.open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


actual_sha256 = sha256_file(SCRIPT_PATH)


print(f"[SHA256 ACTUAL]   {actual_sha256}")
print(f"[SHA256 EXPECTED] {EXPECTED_SHA256}")


if actual_sha256 != EXPECTED_SHA256:

    raise RuntimeError(
        "08B_FINAL_AUDIT SHA-256 mismatch.\n"
        "Audit koşusu DURDURULDU."
    )

print("[SHA256] EXACT MATCH ✅")


# ==========================================================
# 4. READ + AST PARSE
# ==========================================================

code = SCRIPT_PATH.read_text(
    encoding="utf-8",
    errors="strict",
)

tree = ast.parse(
    code,
    filename=str(SCRIPT_PATH),
)

print("[AST PARSE] PASS ✅")


# ==========================================================
# 5. EXTRACT TOP-LEVEL LITERAL CONSTANTS
# ==========================================================

constants = {}


for node in tree.body:

    name = None
    value_node = None

    if isinstance(node, ast.Assign):

        if (
            len(node.targets) == 1
            and isinstance(node.targets[0], ast.Name)
        ):
            name = node.targets[0].id
            value_node = node.value

    elif isinstance(node, ast.AnnAssign):

        if isinstance(node.target, ast.Name):
            name = node.target.id
            value_node = node.value


    if name is not None and value_node is not None:

        try:
            constants[name] = ast.literal_eval(
                value_node
            )
        except Exception:
            pass


# ==========================================================
# 6. LOCKED CONSTANTS
# ==========================================================

expected_constants = {

    "PROJECT_VERSION": "v4_repro",

    "FEATURE_SET": "baseline",

    "LOOKBACK": 10,

    "SEEDS": [123, 777, 2026],

    "ASSETS": [
        "BIST100",
        "USDTRY",
        "EURTRY",
        "GOLD",
    ],

    "TARGET_ORDER": [
        "BIST100_NextRet",
        "USDTRY_NextRet",
        "EURTRY_NextRet",
        "GOLD_NextRet",
        "BIST100_NextVol",
        "USDTRY_NextVol",
        "EURTRY_NextVol",
        "GOLD_NextVol",
    ],

    "TAU": 0.5,

    "EXPECTED_TEST_N": 584,

    "D_MODEL": 32,

    "N_HEAD": 4,

    "N_LAYERS": 2,

    "D_FF": 128,

    "DROPOUT": 0.10,

    "BATCH_SIZE": 64,
}


constant_failures = []


for name, expected in expected_constants.items():

    actual = constants.get(
        name,
        "__MISSING__",
    )

    if actual != expected:

        constant_failures.append(
            (
                name,
                expected,
                actual,
            )
        )

        print(
            f"[CONSTANT] {name}: FAIL ❌ | "
            f"expected={expected!r}, actual={actual!r}"
        )

    else:

        print(
            f"[CONSTANT] {name}: PASS ✅ | "
            f"{actual!r}"
        )


if constant_failures:

    raise RuntimeError(
        "Locked constant mismatch:\n"
        + "\n".join(
            (
                f"- {name}: "
                f"expected={expected!r}, "
                f"actual={actual!r}"
            )
            for name, expected, actual
            in constant_failures
        )
    )


# ==========================================================
# 7. EXPECTED HASH CHAIN
# ==========================================================

expected_hashes = {

    "05_grid_search_v4.py":
        "5d250d9d727cef15e6411cd027aad6089"
        "bf62b2cbf4b2c13a8c0f28ff7191a78",

    "06_best_model_multiseed_v4.py":
        "35de2ee398699003dfef6be36b70c112f"
        "b2c0d1b1e9577cbf64bef58877e16d8",

    "07_final_test_evaluation_v4.py":
        "8b0e3cf2edb9508b4fddd402ddcdbf8c"
        "4d2acd6080ffe6fe1876ad818306cd74",

    "08A_naive_baselines_test_v4.py":
        "95a9658e97f57eaa1a9bb63ec29d8159"
        "432f06438bd57ff54fc8ab43013487e8",

    "08B_learned_baselines_test_v4.py":
        "df4fa2b29a6b522fc410f693a30520e0"
        "45c144a15f7c7368af5eb6a1f0f12566",
}


actual_hashes = constants.get(
    "EXPECTED_HASHES"
)


if actual_hashes != expected_hashes:

    raise RuntimeError(
        "Audit script içindeki EXPECTED_HASHES "
        "kilitli provenance zinciriyle uyuşmuyor."
    )

print("[HASH CHAIN] 05/06/07/08A/08B exact locks PASS ✅")


# ==========================================================
# 8. LOCKED EXPECTED SELECTIONS
# ==========================================================

expected_selections = {

    (
        "SingleTaskTransformer",
        "return",
    ):
        "transformer_fixed_branchmatched",

    (
        "SingleTaskTransformer",
        "volatility",
    ):
        "transformer_fixed_branchmatched",

    (
        "SingleTaskLSTM",
        "return",
    ):
        "lstm_h64_l2",

    (
        "SingleTaskLSTM",
        "volatility",
    ):
        "lstm_h128_l1",

    (
        "XGBoost",
        "return",
    ):
        "xgb_d3_lr0p03",

    (
        "XGBoost",
        "volatility",
    ):
        "xgb_d3_lr0p05",
}


actual_selections = constants.get(
    "EXPECTED_SELECTIONS"
)


if actual_selections != expected_selections:

    raise RuntimeError(
        "EXPECTED_SELECTIONS mismatch.\n"
        f"Expected={expected_selections}\n"
        f"Actual={actual_selections}"
    )

print("[SELECTION LOCKS] Exact expected selections PASS ✅")


# ==========================================================
# 9. LOCKED FINAL REFERENCE VALUES
#
# Bunlar audit sonucunu üretmek için değil,
# 07 prediction array'den yeniden hesaplanan sonuçların
# kilitli 07 değerlerle eşleşmesini kontrol etmek içindir.
# ==========================================================

expected_final_return = (
    0.0075620993156917
)

expected_final_vol = (
    0.0050393493147567
)


actual_final_return = constants.get(
    "EXPECTED_FINAL_AVG_RETURN_MAE"
)

actual_final_vol = constants.get(
    "EXPECTED_FINAL_AVG_VOL_PINBALL"
)


if actual_final_return != expected_final_return:

    raise RuntimeError(
        "Locked final Avg Return MAE mismatch."
    )


if actual_final_vol != expected_final_vol:

    raise RuntimeError(
        "Locked final Avg Vol Pinball mismatch."
    )


print("[07 FINAL REFERENCE] Locked values exact PASS ✅")


# ==========================================================
# 10. CLASS + FUNCTION INVENTORY
# ==========================================================

class_names = {

    node.name

    for node in ast.walk(tree)

    if isinstance(
        node,
        ast.ClassDef,
    )
}


function_nodes = {

    node.name: node

    for node in ast.walk(tree)

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    )
}


required_classes = {

    "TransformerBlock",

    "SingleTaskTransformer",

    "SingleTaskLSTM",
}


required_functions = {

    "sha256_file",

    "record_check",

    "assert_close",

    "max_abs_diff",

    "mae",

    "rmse",

    "r2",

    "pinball",

    "pinball_series",

    "find_target_scaler",

    "task_score",

    "make_xgb",

    "xgb_predict_best",

    "build_neural",

    "selection_record",

    "rebuild_neural_seed_prediction",

    "rebuild_xgb_seed_prediction",

    "metrics_for",

    "get_param_row",
}


for name in sorted(required_classes):

    if name not in class_names:

        raise RuntimeError(
            f"Gerekli audit class eksik: {name}"
        )

    print(
        f"[CLASS] {name}: PASS ✅"
    )


for name in sorted(required_functions):

    if name not in function_nodes:

        raise RuntimeError(
            f"Gerekli audit function eksik: {name}"
        )

    print(
        f"[FUNCTION] {name}: PASS ✅"
    )


# ==========================================================
# 11. AUDIT MUST NOT TRAIN ANY MODEL
# ==========================================================

forbidden_training_tokens = [

    ".fit(",

    ".backward(",

    "optimizer.step(",

    "optimizer.zero_grad(",

    "torch.optim.",

    "AdamW(",

    "model.train(",

    "torch.save(",

    ".save_model(",
]


found_training_tokens = [

    token

    for token in forbidden_training_tokens

    if token in code
]


if found_training_tokens:

    raise RuntimeError(
        "KRİTİK: Audit script içinde model training "
        "veya model-write token bulundu:\n"
        + str(found_training_tokens)
    )


print(
    "[NO RETRAINING] "
    "No fit/backward/optimizer/model-write primitive found ✅"
)


# ==========================================================
# 12. NO EXECUTION OF 08B SOURCE
#
# Audit 08B source codeunu AST ile okuyabilir,
# fakat onu exec/eval/compile ile çalıştırmamalıdır.
# ==========================================================

dangerous_call_names = []


for node in ast.walk(tree):

    if not isinstance(
        node,
        ast.Call,
    ):
        continue

    if isinstance(
        node.func,
        ast.Name,
    ):

        if node.func.id in {
            "exec",
            "eval",
            "compile",
        }:

            dangerous_call_names.append(
                (
                    node.func.id,
                    node.lineno,
                )
            )


if dangerous_call_names:

    raise RuntimeError(
        "Audit source execution primitive found:\n"
        + str(dangerous_call_names)
    )


print(
    "[SOURCE ISOLATION] "
    "No exec/eval/compile call found ✅"
)


# ==========================================================
# 13. REQUIRED INDEPENDENT AUDIT SOURCES
# ==========================================================

required_source_tokens = [

    "GRID_CSV",

    "GRID_SUMMARY_CSV",

    "SELECTION_CSV",

    "SELECTION_LOCK_JSON",

    "METRICS_CSV",

    "COMPARISON_CSV",

    "PARAM_CSV",

    "LOSS_NPZ",

    "PROTOCOL_JSON",

    "CKPT_DIR",

    "XGB_DIR",

    "HIST_DIR",

    "META_DIR",

    "PRED_DIR",

    "FINAL_TEST_DIR",

    "SEQ_DIR",

    "rebuild_neural_seed_prediction",

    "rebuild_xgb_seed_prediction",
]


missing_source_tokens = [

    token

    for token in required_source_tokens

    if token not in code
]


if missing_source_tokens:

    raise RuntimeError(
        "Independent audit structure token missing:\n"
        + str(missing_source_tokens)
    )


print(
    "[INDEPENDENT SOURCES] "
    "Grid/history/meta/checkpoint/model/prediction/"
    "07-final/date sources present ✅"
)


# ==========================================================
# 14. AUDIT WRITE ISOLATION
#
# Audit yalnız audit_08B klasöründeki:
# - checks CSV
# - task comparison CSV
# - result JSON
# dosyalarını yazmalıdır.
# ==========================================================

allowed_to_csv_targets = {

    "AUDIT_TASK_COMPARISON_CSV",

    "AUDIT_CHECKS_CSV",
}


found_to_csv_targets = []


for node in ast.walk(tree):

    if not isinstance(
        node,
        ast.Call,
    ):
        continue

    if not isinstance(
        node.func,
        ast.Attribute,
    ):
        continue

    if node.func.attr != "to_csv":
        continue

    if not node.args:

        raise RuntimeError(
            "to_csv call without positional target found."
        )

    target = node.args[0]

    if not isinstance(
        target,
        ast.Name,
    ):

        raise RuntimeError(
            "Audit içinde dinamik/unknown to_csv "
            f"target bulundu, line={node.lineno}"
        )

    found_to_csv_targets.append(
        target.id
    )


if set(found_to_csv_targets) != allowed_to_csv_targets:

    raise RuntimeError(
        "Audit CSV write targets mismatch.\n"
        f"Expected={sorted(allowed_to_csv_targets)}\n"
        f"Actual={sorted(found_to_csv_targets)}"
    )


dump_json_targets = []


for node in ast.walk(tree):

    if not isinstance(
        node,
        ast.Call,
    ):
        continue

    if not isinstance(
        node.func,
        ast.Name,
    ):
        continue

    if node.func.id != "dump_json":
        continue

    if len(node.args) < 2:
        raise RuntimeError(
            "dump_json call missing target."
        )

    target = node.args[1]

    if not isinstance(
        target,
        ast.Name,
    ):
        raise RuntimeError(
            "Dynamic dump_json target found."
        )

    dump_json_targets.append(
        target.id
    )


if dump_json_targets != [
    "AUDIT_RESULT_JSON"
]:

    raise RuntimeError(
        "Audit JSON write target mismatch.\n"
        f"Actual={dump_json_targets}"
    )


forbidden_write_tokens = [

    "np.save(",

    "np.savez(",

    "np.savez_compressed(",

    "torch.save(",

    ".save_model(",

    ".to_excel(",
]


found_forbidden_writes = [

    token

    for token in forbidden_write_tokens

    if token in code
]


if found_forbidden_writes:

    raise RuntimeError(
        "Unexpected output-write primitive found:\n"
        + str(found_forbidden_writes)
    )


print(
    "[WRITE ISOLATION] "
    "Audit writes only its own 2 CSV + 1 JSON outputs ✅"
)


# ==========================================================
# 15. MANIFEST
# ==========================================================

if not MANIFEST_PATH.exists():

    raise FileNotFoundError(
        f"Manifest bulunamadı:\n{MANIFEST_PATH}"
    )


manifest_df = pd.read_csv(
    MANIFEST_PATH
)


print(
    "[MANIFEST BEFORE] rows =",
    len(manifest_df),
)

print(
    "[MANIFEST COLUMNS]",
    manifest_df.columns.tolist(),
)


required_manifest_columns = {

    "script_name",

    "relative_path",

    "sha256",

    "size_bytes",

    "modified_utc",

    "manifest_created_utc",
}


missing_manifest_columns = (

    required_manifest_columns

    - set(manifest_df.columns)
)


if missing_manifest_columns:

    raise RuntimeError(
        "Manifest schema mismatch:\n"
        + str(
            sorted(
                missing_manifest_columns
            )
        )
    )


print("[MANIFEST SCHEMA] PASS ✅")


# ==========================================================
# 16. EXACT DUPLICATE CHECK
# ==========================================================

exact_mask = (

    manifest_df["script_name"]
    .astype(str)
    .str.strip()
    .eq(SCRIPT_PATH.name)

    &

    manifest_df["sha256"]
    .astype(str)
    .str.strip()
    .eq(actual_sha256)
)


existing_exact = bool(
    exact_mask.any()
)


if existing_exact:

    print(
        "[MANIFEST] Exact audit script+hash "
        "already registered; duplicate not added ✅"
    )


else:

    stat = SCRIPT_PATH.stat()

    now_utc = datetime.now(
        timezone.utc
    ).isoformat()

    modified_utc = datetime.fromtimestamp(
        stat.st_mtime,
        tz=timezone.utc,
    ).isoformat()


    new_row = {

        "script_name":
            SCRIPT_PATH.name,

        "relative_path":
            str(
                SCRIPT_PATH.relative_to(
                    BASE_DIR
                )
            ),

        "sha256":
            actual_sha256,

        "size_bytes":
            int(
                stat.st_size
            ),

        "modified_utc":
            modified_utc,

        "manifest_created_utc":
            now_utc,
    }


    updated_manifest = pd.concat(

        [
            manifest_df,

            pd.DataFrame(
                [new_row]
            ),
        ],

        ignore_index=True,
    )


    updated_manifest.to_csv(
        MANIFEST_PATH,
        index=False,
    )


    print(
        "[MANIFEST] "
        "08B_FINAL_AUDIT pre-run registration added ✅"
    )


# ==========================================================
# 17. POST-WRITE EXACT CROSS-CHECK
# ==========================================================

manifest_check = pd.read_csv(
    MANIFEST_PATH
)


post_mask = (

    manifest_check["script_name"]
    .astype(str)
    .str.strip()
    .eq(SCRIPT_PATH.name)

    &

    manifest_check["sha256"]
    .astype(str)
    .str.strip()
    .eq(actual_sha256)
)


exact_rows = manifest_check.loc[
    post_mask
]


if exact_rows.empty:

    raise RuntimeError(
        "Manifest post-check FAILED: "
        "exact audit script+hash row bulunamadı."
    )


print(
    "[MANIFEST POST-CHECK] "
    "Exact audit script+hash row found ✅"
)

print(
    "[MANIFEST AFTER] rows =",
    len(manifest_check),
)


# ==========================================================
# 18. DISPLAY REGISTERED ROW
# ==========================================================

print("\nREGISTERED 08B_FINAL_AUDIT ROW")
print("-" * 110)

print(
    exact_rows.to_string(
        index=False
    )
)


# ==========================================================
# 19. FINAL VERDICT
# ==========================================================

print("\n" + "=" * 110)
print("08B_FINAL_AUDIT PRE-RUN SON HÜKÜM")
print("=" * 110)

print("File exists                              : PASS ✅")
print("File size exact match                    : PASS ✅")
print("py_compile                               : PASS ✅")
print("SHA-256 exact match                      : PASS ✅")
print("AST parse                                : PASS ✅")
print("Locked constants                         : PASS ✅")
print("05/06/07/08A/08B hash chain              : PASS ✅")
print("Expected selections                      : PASS ✅")
print("07 locked final reference values         : PASS ✅")
print("Required classes/functions               : PASS ✅")
print("No model retraining inside audit         : PASS ✅")
print("No exec/eval/compile of 08B source       : PASS ✅")
print("Independent reconstruction sources       : PASS ✅")
print("Audit write isolation                    : PASS ✅")
print("Manifest schema                          : PASS ✅")
print("Manifest pre-run registration            : PASS ✅")
print("Manifest exact post-write cross-check    : PASS ✅")

print("\nSHA-256:")
print(actual_sha256)

print("\nSONUÇ:")
print(
    "08B_FINAL_AUDIT_v4.py "
    "ilk resmî audit koşusu öncesi "
    "tüm pre-run kontrollerini geçti."
)

print("=" * 110)

In [ ]:
AUDIT_PATH = "/content/drive/MyDrive/tez_transformer_v4_repro/scripts/08B_FINAL_AUDIT_v4.py"

exec(
    compile(
        open(AUDIT_PATH, "r", encoding="utf-8").read(),
        AUDIT_PATH,
        "exec"
    )
)

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/tez_transformer_v4_repro/scripts")

FILES = [
    BASE / "02_preprocessing_v4.py",
    BASE / "06_best_model_multiseed_v4.py",
]

KEYWORDS = [
    "rolling(20)",
    "Vol20",
    "NextVol",
    "shift(-1)",
    "LOOKBACK",
    "FEATURE_SET",
    "baseline",
    "X_train",
]

for path in FILES:
    print("\n" + "=" * 110)
    print(path.name)
    print("=" * 110)

    if not path.exists():
        print("DOSYA BULUNAMADI")
        continue

    lines = path.read_text(encoding="utf-8").splitlines()

    matched = set()

    for i, line in enumerate(lines):
        if any(k.lower() in line.lower() for k in KEYWORDS):
            start = max(0, i - 3)
            end = min(len(lines), i + 4)

            for j in range(start, end):
                matched.add(j)

    for j in sorted(matched):
        print(f"{j+1:5d}: {lines[j]}")

In [ ]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True
)

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/tez_transformer_v4_repro/scripts")

FILES = [
    BASE / "01_rebuild_from_frozen_raw_v4.py",
]

KEYWORDS = [
    "Vol20",
    "rolling",
    ".std(",
    "sqrt(252)",
    "NextVol",
    "shift(-1)",
    "targets_all",
    "features_baseline",
    "LogRet",
]

for path in FILES:
    print("\n" + "=" * 110)
    print(path.name)
    print("=" * 110)

    if not path.exists():
        print("DOSYA BULUNAMADI")
        continue

    lines = path.read_text(encoding="utf-8").splitlines()
    matched = set()

    for i, line in enumerate(lines):
        if any(k.lower() in line.lower() for k in KEYWORDS):
            start = max(0, i - 5)
            end = min(len(lines), i + 8)

            for j in range(start, end):
                matched.add(j)

    for j in sorted(matched):
        print(f"{j+1:5d}: {lines[j]}")

In [ ]:
!pip install arch

import inspect
import numpy as np
import pandas as pd
import scipy
import arch

from arch.univariate import StudentsT


dist = StudentsT(seed=2026)

print("=" * 90)
print("08C ENVIRONMENT + DISTRIBUTION API PREFLIGHT")
print("=" * 90)

print("arch version   :", arch.__version__)
print("pandas version :", pd.__version__)
print("numpy version  :", np.__version__)
print("scipy version  :", scipy.__version__)

print("\nDistribution:")
print("name            :", dist.name)
print("parameters      :", dist.parameter_names())
print("cdf signature   :", inspect.signature(dist.cdf))
print("ppf signature   :", inspect.signature(dist.ppf))
print("moment signature:", inspect.signature(dist.moment))

# Standartlaştırılmış Student-t kontrolü:
# nu=8 yalnızca API ve birim varyans kontrolü içindir;
# resmî model parametresi değildir.
nu_test = [8.0]

print("\nStandardized Student-t API check:")
print("median, nu=8   :", dist.ppf(0.5, nu_test))
print("second moment  :", dist.moment(2, nu_test))

print("\nEXPECTED:")
print("- median yaklaşık 0")
print("- second moment yaklaşık 1")
print("- hiçbir model eğitilmedi")
print("- hiçbir test sonucu okunmadı")
print("=" * 90)


In [ ]:
%pip install -q arch==8.0.0

In [ ]:
!pip install arch

import inspect
import numpy as np
import pandas as pd
import scipy
import arch

from arch.univariate import StudentsT


dist = StudentsT(seed=2026)

print("=" * 90)
print("08C ENVIRONMENT + DISTRIBUTION API PREFLIGHT")
print("=" * 90)

print("arch version   :", arch.__version__)
print("pandas version :", pd.__version__)
print("numpy version  :", np.__version__)
print("scipy version  :", scipy.__version__)

print("\nDistribution:")
print("name            :", dist.name)
print("parameters      :", dist.parameter_names())
print("cdf signature   :", inspect.signature(dist.cdf))
print("ppf signature   :", inspect.signature(dist.ppf))
print("moment signature:", inspect.signature(dist.moment))

# Standartlaştırılmış Student-t kontrolü:
# nu=8 yalnızca API ve birim varyans kontrolü içindir;
# resmî model parametresi değildir.
nu_test = [8.0]

print("\nStandardized Student-t API check:")
print("median, nu=8   :", dist.ppf(0.5, nu_test))
print("second moment  :", dist.moment(2, nu_test))

print("\nEXPECTED:")
print("- median yaklaşık 0")
print("- second moment yaklaşık 1")
print("- hiçbir model eğitilmedi")
print("- hiçbir test sonucu okunmadı")
print("=" * 90)


In [ ]:
import numpy as np
from scipy.optimize import brentq
from arch.univariate import StudentsT

dist = StudentsT()
nu = 8.0

# Sentetik koşullu standart sapma ve merkez — tez verisi değildir.
sigma = 0.012
c = 0.0015

def scalar_cdf_standardized_t(z, nu_value):
    value = dist.cdf(
        np.asarray([z], dtype=float),
        [nu_value]
    )
    return float(np.asarray(value).reshape(-1)[0])

def predictive_cdf(x):
    # Zero-mean predictive return:
    # R = sigma * Z, Z ~ standardized Student-t
    return scalar_cdf_standardized_t(x / sigma, nu)

def central_mass(q):
    return predictive_cdf(c + q) - predictive_cdf(c - q)

def root_function(q):
    return central_mass(q) - 0.5

# Deterministik üst sınır araması
upper = sigma

for _ in range(100):
    if root_function(upper) >= 0:
        break
    upper *= 2.0
else:
    raise RuntimeError("q için geçerli kök aralığı bulunamadı.")

q = brentq(
    root_function,
    0.0,
    upper,
    xtol=1e-14,
    rtol=1e-12,
    maxiter=200
)

mass = central_mass(q)

# c=0 özel durumunda exact symmetry kontrolü:
# P(|R| <= q0)=0.5 => q0=sigma*PPF(0.75)
q0_root = brentq(
    lambda x: (
        scalar_cdf_standardized_t(x / sigma, nu)
        - scalar_cdf_standardized_t(-x / sigma, nu)
        - 0.5
    ),
    0.0,
    upper
)

q0_ppf = sigma * float(
    np.asarray(
        dist.ppf(0.75, [nu])
    ).reshape(-1)[0]
)

print("=" * 90)
print("08C DETERMINISTIC CDF-ROOT PREFLIGHT")
print("=" * 90)
print("nu                       :", nu)
print("sigma                    :", sigma)
print("c                        :", c)
print("q                        :", q)
print("central probability mass :", mass)
print("mass error               :", abs(mass - 0.5))
print("c=0 root q               :", q0_root)
print("c=0 ppf q                :", q0_ppf)
print("symmetry check difference:", abs(q0_root - q0_ppf))

passed = (
    abs(mass - 0.5) < 1e-10
    and abs(q0_root - q0_ppf) < 1e-10
)

print("\nPREFLIGHT PASSED:", passed)
print("No model trained. No project/test data accessed.")
print("=" * 90)

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/tez_transformer_v4_repro")

folders = [
    BASE / "data" / "processed",
    BASE / "data" / "sequences" / "baseline" / "lb10",
]

for folder in folders:
    print("\n" + "=" * 100)
    print(folder)
    print("=" * 100)

    if not folder.exists():
        print("KLASÖR BULUNAMADI")
        continue

    for path in sorted(folder.iterdir()):
        if path.is_file():
            print(path.name)

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd


BASE = Path("/content/drive/MyDrive/tez_transformer_v4_repro")

PROCESSED = BASE / "data" / "processed"
SEQ = BASE / "data" / "sequences" / "baseline" / "lb10"


# ==========================================================
# 1. DOSYA YOLLARI
# ==========================================================

FILES = {
    "prices_clean":
        PROCESSED / "prices_clean.csv",

    "features_baseline":
        PROCESSED / "features_baseline.csv",

    "targets_all":
        PROCESSED / "targets_all.csv",

    "target_realization_dates":
        PROCESSED / "target_realization_dates.csv",

    "split_meta":
        PROCESSED / "split_meta_v4.json",

    "meta_v4":
        PROCESSED / "meta_v4.json",

    "sequence_meta":
        SEQ / "sequence_meta.json",

    "anchor_dates_test":
        SEQ / "anchor_dates_test.npy",

    "target_realization_dates_test":
        SEQ / "target_realization_dates_test.npy",

    "y_test_raw":
        SEQ / "y_test_raw.npy",
}


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


print("=" * 110)
print("08C EXACT INPUT + DATE + TARGET PREFLIGHT")
print("=" * 110)


# ==========================================================
# 2. DOSYA VARLIĞI VE SHA-256
# ==========================================================

print("\n[1] FILE EXISTENCE + SHA-256")

for name, path in FILES.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Eksik dosya: {name}\n{path}"
        )

    print(
        f"{name:32s} "
        f"{path.stat().st_size:12d} bytes  "
        f"{sha256_file(path)}"
    )


# ==========================================================
# 3. CSV VERİLERİNİ OKU
# ==========================================================

prices = pd.read_csv(
    FILES["prices_clean"],
    index_col=0,
    parse_dates=True,
)

features = pd.read_csv(
    FILES["features_baseline"],
    index_col=0,
    parse_dates=True,
)

targets = pd.read_csv(
    FILES["targets_all"],
    index_col=0,
    parse_dates=True,
)

target_dates_csv = pd.read_csv(
    FILES["target_realization_dates"],
    index_col=0,
    parse_dates=True,
)


print("\n[2] CSV STRUCTURE")

for name, df in [
    ("prices_clean", prices),
    ("features_baseline", features),
    ("targets_all", targets),
    ("target_realization_dates", target_dates_csv),
]:
    print(f"\n{name}")
    print("shape :", df.shape)
    print("start :", df.index.min())
    print("end   :", df.index.max())
    print("columns:")
    print(list(df.columns))


# ==========================================================
# 4. LOGRET KAYNAK EŞİTLİĞİ
# ==========================================================

print("\n[3] FEATURES LOGRET vs PRICES-RECOMPUTED LOGRET")

logret_columns = [
    col for col in features.columns
    if col.endswith("_LogRet")
]

if len(logret_columns) != 4:
    raise RuntimeError(
        f"Beklenen 4 LogRet sütunu bulunamadı: {logret_columns}"
    )

logret_diffs = {}

for feature_col in logret_columns:

    asset = feature_col.removesuffix("_LogRet")

    if asset not in prices.columns:
        raise KeyError(
            f"{asset} prices_clean sütunlarında yok."
        )

    recomputed = np.log(
        prices[asset] / prices[asset].shift(1)
    )

    common_index = (
        features.index
        .intersection(recomputed.dropna().index)
    )

    stored_values = (
        features.loc[common_index, feature_col]
        .to_numpy(dtype=float)
    )

    recomputed_values = (
        recomputed.loc[common_index]
        .to_numpy(dtype=float)
    )

    max_diff = float(
        np.max(
            np.abs(
                stored_values - recomputed_values
            )
        )
    )

    logret_diffs[asset] = max_diff

    print(
        f"{asset:10s} "
        f"n={len(common_index):4d}  "
        f"max_abs_diff={max_diff:.18e}"
    )


# ==========================================================
# 5. TEST TARİHLERİ VE HAM HEDEFLER
# ==========================================================

anchor_dates = pd.DatetimeIndex(
    pd.to_datetime(
        np.load(
            FILES["anchor_dates_test"],
            allow_pickle=False,
        ).astype(str)
    )
)

realization_dates_npy = pd.DatetimeIndex(
    pd.to_datetime(
        np.load(
            FILES["target_realization_dates_test"],
            allow_pickle=False,
        ).astype(str)
    )
)

y_test_raw = np.load(
    FILES["y_test_raw"],
    allow_pickle=False,
)


print("\n[4] TEST DATE INVENTORY")

print("anchor count       :", len(anchor_dates))
print("anchor unique      :", anchor_dates.nunique())
print("anchor monotonic   :", anchor_dates.is_monotonic_increasing)
print("anchor start       :", anchor_dates.min())
print("anchor end         :", anchor_dates.max())

print("realization count  :", len(realization_dates_npy))
print("realization unique :", realization_dates_npy.nunique())
print("realization start  :", realization_dates_npy.min())
print("realization end    :", realization_dates_npy.max())

print("y_test_raw shape   :", y_test_raw.shape)


if len(anchor_dates) != 584:
    raise RuntimeError(
        f"Test anchor sayısı 584 değil: {len(anchor_dates)}"
    )

if len(realization_dates_npy) != 584:
    raise RuntimeError(
        "Test realization date sayısı 584 değil."
    )

if y_test_raw.shape != (584, 8):
    raise RuntimeError(
        f"y_test_raw shape yanlış: {y_test_raw.shape}"
    )

if not anchor_dates.is_unique:
    raise RuntimeError(
        "Test anchor tarihleri unique değil."
    )

if not anchor_dates.is_monotonic_increasing:
    raise RuntimeError(
        "Test anchor tarihleri kronolojik değil."
    )

if not np.all(
    realization_dates_npy.values
    >
    anchor_dates.values
):
    raise RuntimeError(
        "Bazı realization tarihleri anchor sonrasında değil."
    )


# ==========================================================
# 6. y_test_raw == targets_all[anchor]
# ==========================================================

missing_target_dates = (
    anchor_dates.difference(targets.index)
)

if len(missing_target_dates) > 0:
    raise RuntimeError(
        "Bazı test anchor tarihleri targets_all içinde yok:\n"
        f"{missing_target_dates}"
    )

targets_from_csv = (
    targets.loc[anchor_dates]
    .to_numpy(dtype=float)
)

target_max_diff = float(
    np.max(
        np.abs(
            y_test_raw - targets_from_csv
        )
    )
)

print("\n[5] RAW TARGET RECONSTRUCTION")
print(
    "y_test_raw vs targets_all max abs diff:",
    f"{target_max_diff:.18e}"
)


# ==========================================================
# 7. REALIZATION DATE CSV == NPY
# ==========================================================

if target_dates_csv.shape[1] != 1:
    raise RuntimeError(
        "target_realization_dates.csv tek sütun değil."
    )

missing_realization_anchors = (
    anchor_dates.difference(
        target_dates_csv.index
    )
)

if len(missing_realization_anchors) > 0:
    raise RuntimeError(
        "Bazı anchor tarihleri realization CSV içinde yok."
    )

realization_dates_csv = pd.DatetimeIndex(
    pd.to_datetime(
        target_dates_csv
        .loc[anchor_dates]
        .iloc[:, 0]
        .to_numpy()
    )
)

date_equal = np.array_equal(
    realization_dates_csv.values,
    realization_dates_npy.values,
)

print("\n[6] REALIZATION DATE RECONSTRUCTION")
print("CSV dates == NPY dates:", date_equal)

if not date_equal:
    raise RuntimeError(
        "CSV ve NPY realization tarihleri birebir aynı değil."
    )


# ==========================================================
# 8. SPLIT META
# ==========================================================

split_meta = json.loads(
    FILES["split_meta"].read_text(
        encoding="utf-8"
    )
)

print("\n[7] SPLIT META")
print(
    json.dumps(
        split_meta,
        indent=2,
        ensure_ascii=False,
    )
)


# ==========================================================
# 9. SONUÇ
# ==========================================================

passed = (
    max(logret_diffs.values()) < 1e-12
    and target_max_diff < 1e-12
    and date_equal
    and len(anchor_dates) == 584
    and y_test_raw.shape == (584, 8)
)

print("\n" + "=" * 110)
print("PREFLIGHT PASSED:", passed)
print("No model trained.")
print("No file modified.")
print("No protocol locked.")
print("=" * 110)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


BASE = Path("/content/drive/MyDrive/tez_transformer_v4_repro")

TARGETS_PATH = (
    BASE / "data" / "processed" / "targets_all.csv"
)

ANCHOR_PATH = (
    BASE
    / "data"
    / "sequences"
    / "baseline"
    / "lb10"
    / "anchor_dates_test.npy"
)

Y_TEST_RAW_PATH = (
    BASE
    / "data"
    / "sequences"
    / "baseline"
    / "lb10"
    / "y_test_raw.npy"
)


targets = pd.read_csv(
    TARGETS_PATH,
    index_col=0,
    parse_dates=True,
)

anchor_dates = pd.DatetimeIndex(
    pd.to_datetime(
        np.load(
            ANCHOR_PATH,
            allow_pickle=False,
        ).astype(str)
    )
)

y_test_raw = np.load(
    Y_TEST_RAW_PATH,
    allow_pickle=False,
)

targets_csv_float64 = (
    targets.loc[anchor_dates]
    .to_numpy(dtype=np.float64)
)

# CSV değerlerini resmî NPY'nin saklama tipine dönüştür.
targets_cast_to_npy_dtype = (
    targets_csv_float64
    .astype(y_test_raw.dtype)
)


raw_float64_diff = float(
    np.max(
        np.abs(
            y_test_raw.astype(np.float64)
            - targets_csv_float64
        )
    )
)

same_dtype_diff = float(
    np.max(
        np.abs(
            y_test_raw
            - targets_cast_to_npy_dtype
        )
    )
)

exact_after_dtype_alignment = np.array_equal(
    y_test_raw,
    targets_cast_to_npy_dtype,
)

per_column_diff = np.max(
    np.abs(
        y_test_raw.astype(np.float64)
        - targets_csv_float64
    ),
    axis=0,
)


print("=" * 100)
print("08C TARGET DTYPE-AWARE RECONSTRUCTION PREFLIGHT")
print("=" * 100)

print("y_test_raw dtype                  :", y_test_raw.dtype)
print("targets CSV dtype                 :", targets_csv_float64.dtype)
print("y_test_raw shape                  :", y_test_raw.shape)

print("\nBefore dtype alignment:")
print("max abs diff                      :", raw_float64_diff)

print("\nAfter CSV -> NPY dtype alignment:")
print("max abs diff                      :", same_dtype_diff)
print("exact array equality              :", exact_after_dtype_alignment)

print("\nPer-column float64 comparison diff:")
for column, value in zip(targets.columns, per_column_diff):
    print(f"{column:24s}: {value:.18e}")

passed = (
    y_test_raw.shape == (584, 8)
    and exact_after_dtype_alignment
    and same_dtype_diff == 0.0
)

print("\nPREFLIGHT PASSED:", passed)
print("No model trained.")
print("No file modified.")
print("No protocol locked.")
print("=" * 100)

In [ ]:
%run /content/drive/MyDrive/tez_transformer_v4_repro/scripts/08C_create_garch_protocol_lock_v4.py

In [ ]:
%run /content/drive/MyDrive/tez_transformer_v4_repro/scripts/08C_validate_garch_protocol_lock_v4.py

In [ ]:
%run /content/drive/MyDrive/tez_transformer_v4_repro/scripts/08C_validate_garch_protocol_lock_v4.py